In [ ]:
# Solar Q experiments. One cell: all internal ablations, shared frozen predictions.
# Mount competition + original v1output + GitHub U-Net + v3output Blended + complete v1-4output.
# Optional: v15data for the saved A3 quality model, and the other G/Q output for frozen-cache reuse.
# Historical A3 CSV reproduction was cancelled by the user. No 1132-count/equality acceptance.
# Neural checkpoints stay frozen; no Kaggle submission is performed.
from pathlib import Path
import importlib.util
import json
import os
import subprocess
import sys

CONFIG = {'line': 'Q', 'input_root': '/kaggle/input', 'working_root': '/kaggle/working', 'cache_source': '', 'original_bundle_source': '', 'v1_source': '', 'legacy_root': '', 'github_checkpoint': '', 'blended_checkpoint': '', 'annotation_path': '', 'test_image_dir': '', 'install_missing': True, 'max_hours': 10.5, 'max_cache_gib': 24.0, 'min_free_gib': 1.0, 'seed': 15026, 'resume': True, 'reuse_v1_cache': True, 'historical_csv_comparison': False, 'tree_levels': 4096, 'tree_budget': 48, 'tree_min_persistence': 0.015, 'tree_min_level': 0.25, 'graph_roi_pixels': 1200000, 'graph_max_nodes': 12000, 'repair_budget': 8, 'repair_pairs': 4, 'repair_max_gap': 64.0, 'repair_min_support': 0.25, 'repair_max_detour': 2.5, 'repair_max_mean_turn': 0.4, 'repair_max_area_ratio': 1.8, 'repair_split_families': 4, 'graph_beam_width': 32, 'graph_weights': (0.0, 0.02, 0.05), 'stability_deltas': (0.025, 0.05, 0.1), 'stability_roi_margin': 16, 'taus': (0.16, 0.2, 0.24, 0.28, 0.32, 0.36), 'orphan_floors': (0.5, 0.6, 0.7), 'visualization_count': 8}
ORIGINAL_NOTEBOOK_SHA256 = '862b417f08d92ecdfca3c2faa7a5f73979de60cb8ca69ad390bad123ade89374'
MODULES = {'APACHE_REFERENCE_LICENSE': '\n                                 Apache License\n                           Version 2.0, January 2004\n                        http://www.apache.org/licenses/\n\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n   1. Definitions.\n\n      "License" shall mean the terms and conditions for use, reproduction,\n      and distribution as defined by Sections 1 through 9 of this document.\n\n      "Licensor" shall mean the copyright owner or entity authorized by\n      the copyright owner that is granting the License.\n\n      "Legal Entity" shall mean the union of the acting entity and all\n      other entities that control, are controlled by, or are under common\n      control with that entity. For the purposes of this definition,\n      "control" means (i) the power, direct or indirect, to cause the\n      direction or management of such entity, whether by contract or\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n      outstanding shares, or (iii) beneficial ownership of such entity.\n\n      "You" (or "Your") shall mean an individual or Legal Entity\n      exercising permissions granted by this License.\n\n      "Source" form shall mean the preferred form for making modifications,\n      including but not limited to software source code, documentation\n      source, and configuration files.\n\n      "Object" form shall mean any form resulting from mechanical\n      transformation or translation of a Source form, including but\n      not limited to compiled object code, generated documentation,\n      and conversions to other media types.\n\n      "Work" shall mean the work of authorship, whether in Source or\n      Object form, made available under the License, as indicated by a\n      copyright notice that is included in or attached to the work\n      (an example is provided in the Appendix below).\n\n      "Derivative Works" shall mean any work, whether in Source or Object\n      form, that is based on (or derived from) the Work and for which the\n      editorial revisions, annotations, elaborations, or other modifications\n      represent, as a whole, an original work of authorship. For the purposes\n      of this License, Derivative Works shall not include works that remain\n      separable from, or merely link (or bind by name) to the interfaces of,\n      the Work and Derivative Works thereof.\n\n      "Contribution" shall mean any work of authorship, including\n      the original version of the Work and any modifications or additions\n      to that Work or Derivative Works thereof, that is intentionally\n      submitted to Licensor for inclusion in the Work by the copyright owner\n      or by an individual or Legal Entity authorized to submit on behalf of\n      the copyright owner. For the purposes of this definition, "submitted"\n      means any form of electronic, verbal, or written communication sent\n      to the Licensor or its representatives, including but not limited to\n      communication on electronic mailing lists, source code control systems,\n      and issue tracking systems that are managed by, or on behalf of, the\n      Licensor for the purpose of discussing and improving the Work, but\n      excluding communication that is conspicuously marked or otherwise\n      designated in writing by the copyright owner as "Not a Contribution."\n\n      "Contributor" shall mean Licensor and any individual or Legal Entity\n      on behalf of whom a Contribution has been received by Licensor and\n      subsequently incorporated within the Work.\n\n   2. Grant of Copyright License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      copyright license to reproduce, prepare Derivative Works of,\n      publicly display, publicly perform, sublicense, and distribute the\n      Work and such Derivative Works in Source or Object form.\n\n   3. Grant of Patent License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      (except as stated in this section) patent license to make, have made,\n      use, offer to sell, sell, import, and otherwise transfer the Work,\n      where such license applies only to those patent claims licensable\n      by such Contributor that are necessarily infringed by their\n      Contribution(s) alone or by combination of their Contribution(s)\n      with the Work to which such Contribution(s) was submitted. If You\n      institute patent litigation against any entity (including a\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\n      or a Contribution incorporated within the Work constitutes direct\n      or contributory patent infringement, then any patent licenses\n      granted to You under this License for that Work shall terminate\n      as of the date such litigation is filed.\n\n   4. Redistribution. You may reproduce and distribute copies of the\n      Work or Derivative Works thereof in any medium, with or without\n      modifications, and in Source or Object form, provided that You\n      meet the following conditions:\n\n      (a) You must give any other recipients of the Work or\n          Derivative Works a copy of this License; and\n\n      (b) You must cause any modified files to carry prominent notices\n          stating that You changed the files; and\n\n      (c) You must retain, in the Source form of any Derivative Works\n          that You distribute, all copyright, patent, trademark, and\n          attribution notices from the Source form of the Work,\n          excluding those notices that do not pertain to any part of\n          the Derivative Works; and\n\n      (d) If the Work includes a "NOTICE" text file as part of its\n          distribution, then any Derivative Works that You distribute must\n          include a readable copy of the attribution notices contained\n          within such NOTICE file, excluding those notices that do not\n          pertain to any part of the Derivative Works, in at least one\n          of the following places: within a NOTICE text file distributed\n          as part of the Derivative Works; within the Source form or\n          documentation, if provided along with the Derivative Works; or,\n          within a display generated by the Derivative Works, if and\n          wherever such third-party notices normally appear. The contents\n          of the NOTICE file are for informational purposes only and\n          do not modify the License. You may add Your own attribution\n          notices within Derivative Works that You distribute, alongside\n          or as an addendum to the NOTICE text from the Work, provided\n          that such additional attribution notices cannot be construed\n          as modifying the License.\n\n      You may add Your own copyright statement to Your modifications and\n      may provide additional or different license terms and conditions\n      for use, reproduction, or distribution of Your modifications, or\n      for any such Derivative Works as a whole, provided Your use,\n      reproduction, and distribution of the Work otherwise complies with\n      the conditions stated in this License.\n\n   5. Submission of Contributions. Unless You explicitly state otherwise,\n      any Contribution intentionally submitted for inclusion in the Work\n      by You to the Licensor shall be under the terms and conditions of\n      this License, without any additional terms or conditions.\n      Notwithstanding the above, nothing herein shall supersede or modify\n      the terms of any separate license agreement you may have executed\n      with Licensor regarding such Contributions.\n\n   6. Trademarks. This License does not grant permission to use the trade\n      names, trademarks, service marks, or product names of the Licensor,\n      except as required for reasonable and customary use in describing the\n      origin of the Work and reproducing the content of the NOTICE file.\n\n   7. Disclaimer of Warranty. Unless required by applicable law or\n      agreed to in writing, Licensor provides the Work (and each\n      Contributor provides its Contributions) on an "AS IS" BASIS,\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n      implied, including, without limitation, any warranties or conditions\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n      PARTICULAR PURPOSE. You are solely responsible for determining the\n      appropriateness of using or redistributing the Work and assume any\n      risks associated with Your exercise of permissions under this License.\n\n   8. Limitation of Liability. In no event and under no legal theory,\n      whether in tort (including negligence), contract, or otherwise,\n      unless required by applicable law (such as deliberate and grossly\n      negligent acts) or agreed to in writing, shall any Contributor be\n      liable to You for damages, including any direct, indirect, special,\n      incidental, or consequential damages of any character arising as a\n      result of this License or out of the use or inability to use the\n      Work (including but not limited to damages for loss of goodwill,\n      work stoppage, computer failure or malfunction, or any and all\n      other commercial damages or losses), even if such Contributor\n      has been advised of the possibility of such damages.\n\n   9. Accepting Warranty or Additional Liability. While redistributing\n      the Work or Derivative Works thereof, You may choose to offer,\n      and charge a fee for, acceptance of support, warranty, indemnity,\n      or other liability obligations and/or rights consistent with this\n      License. However, in accepting such obligations, You may act only\n      on Your own behalf and on Your sole responsibility, not on behalf\n      of any other Contributor, and only if You agree to indemnify,\n      defend, and hold each Contributor harmless for any liability\n      incurred by, or claims asserted against, such Contributor by reason\n      of your accepting any such warranty or additional liability.\n\n   END OF TERMS AND CONDITIONS\n\n   APPENDIX: How to apply the Apache License to your work.\n\n      To apply the Apache License to your work, attach the following\n      boilerplate notice, with the fields enclosed by brackets "[]"\n      replaced with your own identifying information. (Don\'t include\n      the brackets!)  The text should be enclosed in the appropriate\n      comment syntax for the file format. We also recommend that a\n      file or class name and description of purpose be included on the\n      same "printed page" as the copyright notice for easier\n      identification within third-party archives.\n\n   Copyright [yyyy] [name of copyright owner]\n\n   Licensed under the Apache License, Version 2.0 (the "License");\n   you may not use this file except in compliance with the License.\n   You may obtain a copy of the License at\n\n       http://www.apache.org/licenses/LICENSE-2.0\n\n   Unless required by applicable law or agreed to in writing, software\n   distributed under the License is distributed on an "AS IS" BASIS,\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n   See the License for the specific language governing permissions and\n   limitations under the License.\n', 'GITHUB_REFERENCE_LICENSE': 'MIT License\n\nCopyright (c) 2026 Amey Pawar\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n\n', 'NOTICE.md': "# Source acknowledgments\n\nThe Mask R-CNN v2 architecture, anchors, ROI mask pooling, SGD/DDP/AMP training design, ResNet18 U-Net decoder and crop geometry are based on Nomannic19's Apache-2.0 Kaggle notebook, supplied locally as `cv-solar-filament-mask-r-cnn-u-net-refiner.ipynb` (35 cells). `reference_models.py` retains the original architecture definitions. V1 changes the data handling, preprocessing, refiner input, losses, checkpoint selection, postprocessing, calibration and orchestration.\n\nOriginal: https://www.kaggle.com/code/nomannic19/cv-solar-filament-mask-r-cnn-u-net-refiner\n\n`disk.py` is adapted from ameyypawar/filament-seg, commit 81a739bb7786a44b989f5d76afbb6f9d22b5d1dd, under the MIT license. The original MIT license is included as GITHUB_REFERENCE_LICENSE. The V1 radial profile uses edge-padded smoothing to avoid convolution boundary artifacts.\n\nReference: https://github.com/ameyypawar/filament-seg\n\nThe official-compatible scorer preserves the all-pairs, strict IoU > 0.5 and pooled annotator semantics of the host Self_Evaluation_Notebook, version 6. Tests compare its counts and PQ with functions extracted from that public source.\n\nOfficial: https://www.kaggle.com/code/azimahmadzadeh/self-evaluation-notebook\n\nBCE + Tversky and crop-level segmentation are informed by the locally supplied MAGFiLO notebook. Its heavy YOLO architecture is not used. Artem Shelamanov's public portfolio informs the ensemble/quality interface only; its unpublished code is not claimed to have been reproduced.\n\nhttps://www.shelamanov.com/portfolio/solar\n", 'blended_frozen_source.py': 'import math\n\nimport os\n\nimport cv2\n\nimport numpy as np\n\nimport timm\n\nimport torch\n\nimport torch.nn as nn\n\nimport torch.nn.functional as F\n\nfrom pycocotools import mask as coco_mask\n\nFULL = 2048                     # native image size / submission RLE size\n\nCROP = 1872                     # centred region that gets predicted\n\nOFFSET = (FULL - CROP) // 2     # 88\n\nTILE = 1056                     # 33*32, so the encoder needs no internal padding\n\nTILES = 2                       # tiles per axis -> 2x2 = 4 forward passes per image\n\nSTEP = (CROP - TILE) // (TILES - 1)   # 816px between neighbouring tile origins\n\nOVERLAP = TILE - STEP                 # 240px shared by each neighbouring pair\n\nBLEND_FLOOR = 0.05              # blend weight at a tile\'s outermost pixel; never 0\n\nTTA = 8                         # dihedral transforms averaged at inference:\n\nENCODER = "efficientvit_b2.r288_in1k"\n\nDECODER_CH = (192, 96, 48, 24, 12)\n\nBOTTLENECK_CH = 192     # width inside the encoder->decoder bottleneck (0 = no bottleneck)\n\nBOTTLENECK_DIL = (1, 2, 4)   # dilations of its 3x3 convs, widening context at /32\n\nTILE_BATCH = 4          # tiles per forward pass when assembling a mosaic (= all 4)\n\nPERCENTILES = (10, 25, 50, 75, 90)\n\nNORM_RADIUS = 900       # px; every annotated filament is within 903px of the centre\n\nDEVICE = "cuda"\n\nAMP_DTYPE = torch.float16\n\n_yy, _xx = np.ogrid[:FULL, :FULL]\n\nDISK_MASK = ((_yy - FULL // 2) ** 2 + (_xx - FULL // 2) ** 2) <= NORM_RADIUS ** 2\n\ndef norm_stats(img):\n    """(centre, scale) for one full frame: median and 16-84 spread inside the disk.\n\n    Exact for uint8 via a 256-bin histogram, which is a single O(n) pass -- np.percentile\n    would sort ~2.5M values per call, and this runs once per image on every val pass.\n    """\n    hist = np.bincount(img[DISK_MASK], minlength=256).astype(np.float64)\n    cdf = np.cumsum(hist) / max(hist.sum(), 1.0)\n    lo, mid, hi = np.searchsorted(cdf, (0.16, 0.50, 0.84))\n    return float(mid), float(max(hi - lo, 1.0))\n\ndef tile_starts():\n    """Start coordinate of each tile along one axis, in full-frame coords.\n\n    TILES positions spread evenly from the crop\'s near edge to its far edge, so the\n    first and last tiles sit flush against the crop and nothing is left uncovered.\n    Rounding shares any leftover pixel out between the gaps rather than piling it all\n    on the last one, so neighbouring overlaps differ by at most a pixel.\n    """\n    span = CROP - TILE\n    return [OFFSET + round(i * span / (TILES - 1)) for i in range(TILES)]\n\ndef tile_layout():\n    """One (tile_r, tile_c) per tile, full-frame coords. TILES^2 of them."""\n    starts = tile_starts()\n    return [(r, c) for r in starts for c in starts]\n\ndef blend_window():\n    """TILE x TILE weight map: flat 1 through the middle, tapering to BLEND_FLOOR at\n    the rim. This is what makes sense of a pixel that several tiles predicted.\n\n    A pixel on a tile\'s edge was predicted with context on one side only. The same\n    pixel sits well inside the neighbouring tile, with context all round it. So each\n    prediction is weighted by its distance from its OWN tile\'s edge and the total is\n    divided by the total weight: the tile that saw the pixel properly dominates, and\n    the two crossfade instead of meeting at a cut. A hard cut would leave a seam right\n    where connected components get labelled, and a filament crossing it could break in\n    two -- exactly the one-to-many fragmentation the submission is scored on.\n\n    The ramp is a raised cosine over OVERLAP px, the exact width two neighbours share,\n    so one tile fades out as the other fades in and their weights sum to ~1 across the\n    band. It stops at BLEND_FLOOR rather than 0 because the outermost ring of the crop\n    is covered by ONE tile: there the floor is the only weight present, cancels in the\n    division and leaves that tile\'s prediction untouched. A true zero would be 0/0.\n\n    The window is separable, so the floor at a corner is BLEND_FLOOR^2 -- still a\n    common factor between whatever tiles cover it, and still cancelling.\n    """\n    d = np.minimum(np.arange(TILE), np.arange(TILE)[::-1]).astype(np.float32)\n    ramp = 0.5 * (1.0 - np.cos(np.pi * np.clip(d / max(OVERLAP, 1), 0.0, 1.0)))\n    line = BLEND_FLOOR + (1.0 - BLEND_FLOOR) * ramp\n    return np.outer(line, line).astype(np.float32)\n\nBLEND = blend_window()          # TILE^2 floats, built once and reused for every image\n\n_BLEND_GPU = None               # its device copy, made on first use\n\ndef tile_at(arr, tile_r, tile_c):\n    """TILE x TILE slice of a full-frame array -- always fully inside the frame."""\n    return np.ascontiguousarray(arr[tile_r:tile_r + TILE, tile_c:tile_c + TILE])\n\ndef normalize(batch_u8, centre, scale):\n    """uint8 (B, H, W) numpy -> (B, 1, H, W) tensor on DEVICE, per-image normalized."""\n    x = torch.from_numpy(batch_u8).to(DEVICE).float().sub_(centre).div_(scale)\n    return x.unsqueeze(1).to(memory_format=torch.channels_last)\n\n@torch.no_grad()\ndef mosaic_predict(model, img):\n    """Reassemble a FULL x FULL probability map from OVERLAPPING tiles.\n\n    img is a uint8 FULL x FULL array. Each tile\'s prediction is accumulated into the\n    canvas under the blend window, and the window itself is accumulated alongside it;\n    dividing the two at the end turns the sum into a weighted average, so a pixel that\n    several tiles predicted gets a single reconciled probability rather than whichever\n    tile happened to be written last.\n\n    Pixels outside the centred crop are left at 0 -- no tile touches them, so both\n    accumulators stay 0 there and the clamped division returns 0. They are black sky\n    outside the solar disk and hold no ground truth.\n    """\n    global _BLEND_GPU\n    if _BLEND_GPU is None:\n        _BLEND_GPU = torch.from_numpy(BLEND).to(DEVICE)\n    win = _BLEND_GPU\n\n    def forward(x):\n        """Mean sigmoid over the first TTA elements of the dihedral group.\n\n        Training applies flips and RandomRotate90(p=1.0), so the model is already close\n        to D4-equivariant and the transforms mostly disagree on noise. Averaging cancels\n        that, which matters here specifically because the noise it cancels is what\n        breaks a faint filament into two components and costs one-to-many.\n\n        Each transform is undone on the OUTPUT before averaging, so this averages\n        predictions in image space, never images.\n        """\n        if TTA <= 1:\n            return torch.sigmoid(model(x))\n        outs = []\n        for flip in range(2 if TTA >= 8 else 1):\n            xf = torch.flip(x, dims=[-1]) if flip else x\n            for k in range(4 if TTA >= 4 else 1):\n                xr = torch.rot90(xf, k, dims=(-2, -1))\n                y = torch.sigmoid(model(xr.contiguous(memory_format=torch.channels_last)))\n                y = torch.rot90(y, -k, dims=(-2, -1))\n                outs.append(torch.flip(y, dims=[-1]) if flip else y)\n        return torch.stack(outs).mean(0)\n\n    centre, scale = norm_stats(img)\n    layout = tile_layout()\n    acc = torch.zeros((FULL, FULL), device=DEVICE, dtype=torch.float32)\n    wsum = torch.zeros((FULL, FULL), device=DEVICE, dtype=torch.float32)\n    for k in range(0, len(layout), TILE_BATCH):\n        chunk = layout[k:k + TILE_BATCH]\n        tiles = np.stack([tile_at(img, tr, tc) for tr, tc in chunk])\n        with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=torch.device(DEVICE).type == "cuda"):\n            probs = forward(normalize(tiles, centre, scale)).float()\n        for (tr, tc), p in zip(chunk, probs):\n            acc[tr:tr + TILE, tc:tc + TILE] += p[0] * win\n            wsum[tr:tr + TILE, tc:tc + TILE] += win\n    return acc / wsum.clamp(min=1e-6)\n\nclass ConvBlock(nn.Module):\n    def __init__(self, cin, cout):\n        super().__init__()\n        self.block = nn.Sequential(\n            nn.Conv2d(cin, cout, 3, padding=1, bias=False),\n            nn.BatchNorm2d(cout),\n            nn.ReLU(inplace=True),\n            nn.Conv2d(cout, cout, 3, padding=1, bias=False),\n            nn.BatchNorm2d(cout),\n            nn.ReLU(inplace=True),\n        )\n\n    def forward(self, x):\n        return self.block(x)\n\nclass Bottleneck(nn.Module):\n    """Residual squeeze-transform-expand block sitting between encoder and decoder.\n\n    The encoder hands over its deepest stage (/32) and the decoder starts upsampling\n    from it immediately, so nothing ever mixes information across that map at full\n    width. This block does: 1x1 down to BOTTLENECK_CH, a stack of 3x3 convs with\n    growing dilation (context out to several hundred input pixels at /32, which is the\n    scale a filament spine spans), 1x1 back up, added residually.\n\n    The final BatchNorm is zero-initialised, so at step 0 the block is exactly the\n    identity and training starts from the same place the plain U-Net would.\n    """\n\n    def __init__(self, ch, mid, dilations):\n        super().__init__()\n        self.reduce = nn.Sequential(\n            nn.Conv2d(ch, mid, 1, bias=False), nn.BatchNorm2d(mid), nn.ReLU(inplace=True))\n        self.convs = nn.Sequential(*[\n            nn.Sequential(nn.Conv2d(mid, mid, 3, padding=d, dilation=d, bias=False),\n                          nn.BatchNorm2d(mid), nn.ReLU(inplace=True))\n            for d in dilations])\n        self.expand = nn.Sequential(\n            nn.Conv2d(mid, ch, 1, bias=False), nn.BatchNorm2d(ch))\n        nn.init.zeros_(self.expand[1].weight)   # start as identity\n        self.act = nn.ReLU(inplace=True)\n\n    def forward(self, x):\n        return self.act(x + self.expand(self.convs(self.reduce(x))))\n\nclass UNet(nn.Module):\n    """timm features_only encoder + U-Net decoder that upsamples back to /1.\n\n    The decoder is driven by the encoder\'s stage reductions, so it works whether\n    the encoder exposes 5 stages starting at /2 (resnet34: 2 4 8 16 32) or 4\n    stages starting at /4 (efficientvit_b1: 4 8 16 32). Levels without a matching\n    encoder stage simply get no skip connection.\n    """\n\n    def __init__(self, pretrained=True):\n        super().__init__()\n        self.encoder = timm.create_model(\n            ENCODER, pretrained=pretrained, features_only=True, in_chans=1\n        )\n        enc_ch = self.encoder.feature_info.channels()\n        self.enc_red = self.encoder.feature_info.reduction()\n        red2ch = dict(zip(self.enc_red, enc_ch))\n\n        deepest = max(self.enc_red)\n        n_up = int(math.log2(deepest))          # /32 -> 5 upsampling steps to /1\n        cin = enc_ch[-1]\n        blocks, skip_reds = [], []\n        for i in range(n_up):\n            red = deepest // (2 ** (i + 1))     # resolution level after this upsample\n            skip_ch = red2ch.get(red, 0)\n            cout = DECODER_CH[min(i, len(DECODER_CH) - 1)]\n            blocks.append(ConvBlock(cin + skip_ch, cout))\n            skip_reds.append(red if skip_ch else None)\n            cin = cout\n        self.blocks = nn.ModuleList(blocks)\n        self.skip_reds = skip_reds\n        self.bottleneck = (Bottleneck(enc_ch[-1], BOTTLENECK_CH, BOTTLENECK_DIL)\n                           if BOTTLENECK_CH else nn.Identity())\n        self.head = nn.Conv2d(cin, 1, 1)\n\n    def forward(self, x):\n        # The decoder doubles resolution 5x, so the encoder stages only line up with\n        # their skips when the input is a multiple of 32. The mosaic geometry is solved\n        # so TILE already is one (1056 = 33*32), which is the whole point of the\n        # TILE % 32 assert up top -- this pad is a no-op at the configured geometry and\n        # only fires if someone retunes TILE off the grid, in which case it adds <32px\n        # of reflected border that the crop below takes straight back off.\n        h, w = x.shape[-2:]\n        pad_h, pad_w = (-h) % 32, (-w) % 32\n        if pad_h or pad_w:\n            x = F.pad(x, (0, pad_w, 0, pad_h), mode="reflect")\n\n        feats = self.encoder(x)\n        red2feat = dict(zip(self.enc_red, feats))\n        out = self.bottleneck(feats[-1])\n        for block, red in zip(self.blocks, self.skip_reds):\n            out = F.interpolate(out, scale_factor=2, mode="nearest")\n            if red is not None:\n                out = torch.cat([out, red2feat[red]], dim=1)\n            out = block(out)\n        out = self.head(out)\n        return out[..., :h, :w] if (pad_h or pad_w) else out\n', 'filament_seg/LICENSE': 'MIT License\n\nCopyright (c) 2026 Amey Pawar\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n\n', 'filament_seg/__init__.py': '"""Solar Filament Segmentation Challenge 2026 (IEEE Big Data Cup).\n\nPackage layout:\n\n* :mod:`filament_seg.config`      paths and dataset constants\n* :mod:`filament_seg.rle`         mask <-> COCO RLE <-> submission CSV\n* :mod:`filament_seg.metrics`     local Panoptic Quality + rubric diagnostics\n* :mod:`filament_seg.data`        MAGFiLO annotations and leak-free splits\n* :mod:`filament_seg.disk`        solar disk detection, limb-darkening removal\n* :mod:`filament_seg.postprocess` binary mask -> filament instances\n* :mod:`filament_seg.baseline`    model-free detector (pipeline validation)\n"""\n\n__version__ = "0.1.0"\n\n', 'filament_seg/baseline.py': '"""A model-free filament detector.\n\nPurpose: get a *valid submission on the leaderboard before any training starts*,\nso the RLE encoding, the CSV format and the local evaluator are all proven\nagainst the real scorer. It is also the honest floor that any learned model has\nto beat.\n\nMethod -- essentially the classical H-alpha filament recipe:\n\n1. locate the solar disk and mask off everything outside it\n2. divide out limb darkening so one threshold works at any radius\n3. call a pixel a filament when it is ``k`` standard deviations darker than the\n   quiet Sun around it\n4. morphological cleanup, gap-bridging and an area filter to form instances\n\nExpect a modest PQ. The value is in the plumbing, not the score.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\n\nfrom .disk import Disk, detect_disk, flatten, read_image\nfrom .postprocess import PostprocessParams, binary_to_instances\nfrom .rle import Rle, labels_to_rles\n\n\n@dataclass\nclass BaselineParams:\n    """Thresholding knobs. Tune ``k`` first -- it dominates the score."""\n\n    #: Darkness threshold in standard deviations below the quiet-Sun level.\n    #: 2.0 comes from a grid sweep over 40 validation images; PQ is fairly\n    #: flat between k=1.6 and k=2.4 but collapses above k=2.8.\n    k: float = 2.0\n    #: Shrink factor applied to the disk radius; the limb itself is noisy.\n    disk_shrink: float = 0.96\n    #: Gaussian smoothing (in pixels) applied before thresholding.\n    blur_sigma: float = 2.0\n    postprocess: PostprocessParams = field(default_factory=PostprocessParams)\n\n\ndef segment(\n    image: np.ndarray, params: BaselineParams | None = None\n) -> tuple[np.ndarray, Disk]:\n    """Return an integer instance-label map and the detected disk."""\n    params = params or BaselineParams()\n\n    disk = detect_disk(image)\n    disk_mask = disk.mask(image.shape, shrink=params.disk_shrink)\n\n    flat = flatten(image, disk)\n    if params.blur_sigma > 0:\n        flat = cv2.GaussianBlur(flat, (0, 0), sigmaX=params.blur_sigma)\n\n    inside = flat[disk_mask]\n    if inside.size == 0:\n        return np.zeros(image.shape, dtype=np.int32), disk\n\n    # Robust centre/spread: filaments are a small, dark minority of disk pixels,\n    # so the median and the MAD are barely moved by them.\n    median = float(np.median(inside))\n    mad = float(np.median(np.abs(inside - median)))\n    sigma = 1.4826 * mad if mad > 0 else float(inside.std())\n    threshold = median - params.k * max(sigma, 1e-6)\n\n    binary = (flat < threshold) & disk_mask\n    labels = binary_to_instances(binary, params.postprocess, restrict_to=disk_mask)\n    return labels, disk\n\n\ndef predict_rles(\n    image_path: str | Path, params: BaselineParams | None = None\n) -> list[Rle]:\n    """Predict filament instances for one image file, as RLEs."""\n    labels, _ = segment(read_image(image_path), params)\n    return labels_to_rles(labels)\n\n', 'filament_seg/config.py': '"""Project-wide paths and constants.\n\nEvery path can be overridden with an environment variable so the same code runs\nunchanged locally and inside a Kaggle notebook (where the data lives under\n``/kaggle/input``).\n"""\n\nfrom __future__ import annotations\n\nimport os\nfrom pathlib import Path\n\n# --- Layout -----------------------------------------------------------------\n\nREPO_ROOT = Path(__file__).resolve().parent.parent\n\nDATA_ROOT = Path(\n    os.environ.get("FILAMENT_DATA_ROOT", REPO_ROOT / "data" / "MAGFiLO_1.0_Kaggle_2026")\n)\nOUTPUT_ROOT = Path(os.environ.get("FILAMENT_OUTPUT_ROOT", REPO_ROOT / "outputs"))\n\nTRAIN_DIR = DATA_ROOT / "train"\nTRAIN_IMAGE_DIR = TRAIN_DIR / "train_images"\nTRAIN_ANNOTATIONS = TRAIN_DIR / "MAGFiLO_1.0_Annotations_kaggle2026_train.json"\nTEST_IMAGE_DIR = DATA_ROOT / "test" / "test_images"\n\nSPLIT_PATH = OUTPUT_ROOT / "splits.json"\n\n#: Preprocessing cache (flattened images, masks, disk geometry) that\n#: scripts/preprocess.py builds and every other script reads.\nCACHE_DIR = REPO_ROOT / "data" / "cache"\n\n# --- Dataset constants ------------------------------------------------------\n\n#: All GONG H-alpha observations in this competition are 2048x2048 grayscale.\n#: The submission format relies on this being fixed (RLE size is not submitted).\nIMAGE_HEIGHT = 2048\nIMAGE_WIDTH = 2048\nIMAGE_SIZE = (IMAGE_HEIGHT, IMAGE_WIDTH)\n\n#: Chirality classes exist in MAGFiLO but are irrelevant to this competition,\n#: which is scored on masks only.\nCATEGORY_NAMES = {1: "Left", 2: "Right", 3: "Unidentifiable", 4: "Ambiguous"}\n\n# --- Evaluation constants ---------------------------------------------------\n\n#: Panoptic Quality matches a prediction to a ground-truth segment when their\n#: IoU exceeds this value. Above 0.5 the matching is guaranteed unique.\nPQ_IOU_THRESHOLD = 0.5\n\n#: Minimum overlap fraction used when counting one-to-many / many-to-one\n#: relations for the fragmentation diagnostics (part of the judging rubric).\nFRAGMENT_OVERLAP_THRESHOLD = 0.10\n\n\ndef ensure_output_dir() -> Path:\n    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)\n    return OUTPUT_ROOT\n\n', 'filament_seg/data.py': '"""Loading MAGFiLO annotations and building leak-free train/validation splits.\n\nThe dataset has one structural trap that quietly inflates validation scores:\n\n* **The same observation is annotated by several annotators independently.** They\n  appear as distinct ``image["id"]`` values that differ only in the batch prefix\n  (``010101-2016...`` vs ``010102-2016...``) but share a ``file_name``. Splitting\n  on ``image_id`` puts the same pixels in both train and validation.\n\n* **Filaments live for days to weeks.** Two observations from the same day (or\n  the same week) are near-duplicates. Splitting randomly by file name still\n  leaks. ``group_by="date"`` is the default here; ``"month"`` is the stricter\n  option and gives a more honest estimate of generalisation at the cost of a\n  pessimistic bias relative to the leaderboard.\n"""\n\nfrom __future__ import annotations\n\nimport json\nimport random\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Iterable, Literal, Sequence\n\nfrom .config import IMAGE_HEIGHT, IMAGE_WIDTH, TRAIN_ANNOTATIONS\nfrom .rle import Rle, polygon_to_rle\n\nGroupBy = Literal["file", "date", "month"]\n\n\n@dataclass(frozen=True)\nclass ImageRecord:\n    """One annotated view of one observation."""\n\n    image_id: str  # e.g. "010401-20160920230134Lh"\n    file_name: str  # e.g. "20160920230134Lh.jpg"\n    width: int\n    height: int\n\n    @property\n    def stem(self) -> str:\n        """Observation name without extension, e.g. ``20160920230134Lh``."""\n        return Path(self.file_name).stem\n\n    @property\n    def annotator(self) -> str:\n        """Annotator batch prefix, or ``""`` when the id carries none."""\n        return self.image_id.split("-", 1)[0] if "-" in self.image_id else ""\n\n    @property\n    def date(self) -> str:\n        return self.stem[:8]\n\n    @property\n    def month(self) -> str:\n        return self.stem[:6]\n\n    @property\n    def observatory(self) -> str:\n        """Two-letter GONG site code, e.g. ``Bh`` (Big Bear), ``Lh`` (Learmonth)."""\n        return self.stem[14:16]\n\n    def group_key(self, group_by: GroupBy = "date") -> str:\n        return {"file": self.stem, "date": self.date, "month": self.month}[group_by]\n\n\n@dataclass\nclass Annotations:\n    """Parsed COCO file, indexed the ways this project actually needs."""\n\n    images: dict[str, ImageRecord]\n    annotations_by_image: dict[str, list[dict]]\n    categories: dict[int, str]\n\n    def __len__(self) -> int:\n        return len(self.images)\n\n    @property\n    def image_ids(self) -> list[str]:\n        return sorted(self.images)\n\n    def by_file_stem(self) -> dict[str, list[str]]:\n        """``{observation stem: [image_id per annotator]}``."""\n        grouped: dict[str, list[str]] = {}\n        for image_id, record in self.images.items():\n            grouped.setdefault(record.stem, []).append(image_id)\n        return {k: sorted(v) for k, v in sorted(grouped.items())}\n\n    def gt_rles(self, image_id: str) -> list[Rle]:\n        """Ground-truth instance RLEs for one annotated image."""\n        record = self.images[image_id]\n        out: list[Rle] = []\n        for ann in self.annotations_by_image.get(image_id, []):\n            segmentation = ann.get("segmentation")\n            if not segmentation:\n                continue\n            out.append(polygon_to_rle(segmentation, record.height, record.width))\n        return out\n\n    def gt_dict(self, image_ids: Iterable[str]) -> dict[str, list[Rle]]:\n        return {image_id: self.gt_rles(image_id) for image_id in image_ids}\n\n\ndef load_annotations(path: str | Path = TRAIN_ANNOTATIONS) -> Annotations:\n    with open(path, "r", encoding="utf-8") as handle:\n        raw = json.load(handle)\n\n    images = {}\n    for entry in raw.get("images", []):\n        images[str(entry["id"])] = ImageRecord(\n            image_id=str(entry["id"]),\n            file_name=entry["file_name"],\n            width=int(entry.get("width") or IMAGE_WIDTH),\n            height=int(entry.get("height") or IMAGE_HEIGHT),\n        )\n\n    annotations_by_image: dict[str, list[dict]] = {}\n    for ann in raw.get("annotations", []):\n        annotations_by_image.setdefault(str(ann["image_id"]), []).append(ann)\n\n    categories = {int(c["id"]): c["name"] for c in raw.get("categories", [])}\n    return Annotations(images, annotations_by_image, categories)\n\n\ndef make_split(\n    annotations: Annotations,\n    val_fraction: float = 0.2,\n    group_by: GroupBy = "date",\n    seed: int = 0,\n) -> dict[str, list[str]]:\n    """Split image ids into train/val so that no group spans both sides.\n\n    Returns ``{"train": [...], "val": [...], "group_by": ...}`` with image ids\n    (annotator-prefixed), which is what the loaders and the evaluator key on.\n    """\n    groups: dict[str, list[str]] = {}\n    for image_id, record in annotations.images.items():\n        groups.setdefault(record.group_key(group_by), []).append(image_id)\n\n    keys = sorted(groups)\n    rng = random.Random(seed)\n    rng.shuffle(keys)\n\n    n_val = max(1, round(len(keys) * val_fraction))\n    val_keys = set(keys[:n_val])\n\n    train_ids = sorted(i for k in keys if k not in val_keys for i in groups[k])\n    val_ids = sorted(i for k in keys if k in val_keys for i in groups[k])\n    return {\n        "group_by": group_by,\n        "seed": seed,\n        "val_fraction": val_fraction,\n        "n_groups": len(keys),\n        "train": train_ids,\n        "val": val_ids,\n    }\n\n\ndef deduplicate_by_file(\n    annotations: Annotations, image_ids: Sequence[str], seed: int = 0\n) -> list[str]:\n    """Keep one annotator per observation.\n\n    Used for training, where the duplicates are label noise rather than extra\n    signal. Validation should not use it: the official scorer matches the\n    predictions against every annotator\'s view separately and pools the\n    counts, so scoring against one random annotator optimises a different\n    objective. See :func:`records_for_stems`.\n    """\n    rng = random.Random(seed)\n    chosen: list[str] = []\n    grouped: dict[str, list[str]] = {}\n    for image_id in image_ids:\n        grouped.setdefault(annotations.images[image_id].stem, []).append(image_id)\n    for stem in sorted(grouped):\n        chosen.append(rng.choice(sorted(grouped[stem])))\n    return sorted(chosen)\n\n\ndef load_split(path: str | Path) -> dict:\n    """The train/val split written by scripts/make_splits.py."""\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef stems_of(annotations: Annotations, image_ids: Iterable[str]) -> list[str]:\n    """The observations (file stems) that ``image_ids`` are views of, sorted."""\n    return sorted({annotations.images[i].stem for i in image_ids})\n\n\ndef sample_stems(\n    annotations: Annotations, image_ids: Iterable[str], n: int | None = None, seed: int = 0\n) -> list[str]:\n    """A fixed random sample of ``n`` observations (all of them if ``n`` is falsy).\n\n    Sampling observations rather than taking the first ``n`` sorted view ids\n    matters: view ids start with the annotator batch, so a sorted prefix comes\n    entirely from one batch of annotators. Shuffling with a fixed seed gives\n    the same sample on every call, so per-epoch scores stay comparable.\n    """\n    stems = stems_of(annotations, image_ids)\n    random.Random(seed).shuffle(stems)\n    return sorted(stems[:n] if n else stems)\n\n\ndef records_for_stems(\n    annotations: Annotations, image_ids: Iterable[str], stems: Iterable[str]\n) -> list[str]:\n    """Every annotator view among ``image_ids`` of the given observations.\n\n    This is the unit the official scorer works in: one prediction set per\n    observation, matched against each annotator\'s view of it separately, with\n    TP/FP/FN pooled over all views.\n    """\n    wanted = set(stems)\n    return sorted(i for i in image_ids if annotations.images[i].stem in wanted)\n\n\ndef test_image_paths(directory: str | Path) -> list[Path]:\n    directory = Path(directory)\n    patterns = ("*.jpeg", "*.jpg", "*.JPEG", "*.JPG", "*.png")\n    paths: list[Path] = []\n    for pattern in patterns:\n        paths.extend(directory.glob(pattern))\n    return sorted(set(paths))\n\n', 'filament_seg/dataset.py': '"""Cropped-tile dataset for training the filament segmentation network.\n\nTwo things shape this dataset more than anything else:\n\n* **Filaments are rare.** They cover roughly 0.2% of disk pixels (median area\n  1228 px in a 2048x2048 frame), so a uniformly random crop is empty the\n  overwhelming majority of the time. ``filament_bias`` fixes this by centring\n  most crops on an actual ground-truth filament, jittered so the network\n  still has to localise rather than just find the middle of the crop.\n\n* **Native resolution only.** Filaments are a few pixels wide; anything that\n  resamples the image -- including the elastic/scale augmentation that is\n  standard for most segmentation tasks -- destroys the structures the metric\n  is graded on. Only flips, 90-degree rotations and brightness/contrast\n  jitter are used here.\n\nEverything expensive and parameter-independent (disk detection, limb\ndarkening) is expected to already live in ``scripts/preprocess.py``\'s cache;\nthis module only reads that cache and crops it.\n"""\n\nfrom __future__ import annotations\n\nimport json\nimport random\nfrom collections import OrderedDict\nfrom pathlib import Path\nfrom typing import Sequence\n\nimport albumentations as A\nimport cv2\nimport numpy as np\nimport torch\nfrom pycocotools import mask as mask_utils\nfrom torch.utils.data import Dataset\n\nfrom .data import Annotations, deduplicate_by_file\nfrom .disk import Disk\n\n#: Flips and 90-degree rotations preserve every pixel exactly (no\n#: interpolation), which is why they\'re safe for structures a few pixels\n#: wide. Brightness/contrast jitter only ever touches the "image" target\n#: below, never the "radius" one -- see the ``additional_targets`` note.\n_AUGMENT = A.Compose(\n    [\n        A.HorizontalFlip(p=0.5),\n        A.VerticalFlip(p=0.5),\n        A.RandomRotate90(p=1.0),\n        A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),\n    ],\n    # albumentations only applies colour transforms to "image"-typed targets;\n    # tagging the radius map as "mask" gets it the flip/rotation but keeps it\n    # immune to brightness/contrast, which would corrupt its meaning (it\'s a\n    # normalised coordinate, not an intensity).\n    additional_targets={"radius": "mask"},\n)\n\n\ndef load_disk_geometry(path: str | Path) -> dict[str, Disk]:\n    """Load ``scripts/preprocess.py``\'s ``disk.json`` as ``{stem: Disk}``."""\n    raw = json.loads(Path(path).read_text(encoding="utf-8"))\n    return {stem: Disk(**geometry) for stem, geometry in raw.items()}\n\n\ndef load_model_input(flat_dir: str | Path, stem: str, disk: Disk) -> np.ndarray | None:\n    """The 2-channel full-frame network input for one observation.\n\n    Channel 0 is the cached limb-darkening-flattened intensity in [0, 1],\n    channel 1 the normalised radius map. Returns None if the flat image is not\n    cached, so callers can decide whether that is fatal.\n    """\n    flat_u8 = cv2.imread(str(Path(flat_dir) / f"{stem}.png"), cv2.IMREAD_GRAYSCALE)\n    if flat_u8 is None:\n        return None\n    radius = disk.radius_map(flat_u8.shape).astype(np.float32)\n    return np.stack([flat_u8.astype(np.float32) / 255.0, radius], axis=0)\n\n\ndef _instance_centroids(annotations: Annotations, image_id: str) -> list[tuple[float, float]]:\n    """Bounding-box centre of each ground-truth filament in ``image_id``.\n\n    A bbox centre is not the exact centroid of a curved ribbon, but it is a\n    fine jitter target and, unlike a full decode, stays cheap even for the\n    handful of images with two dozen instances.\n    """\n    centroids = []\n    for rle in annotations.gt_rles(image_id):\n        x, y, w, h = mask_utils.toBbox(rle)\n        centroids.append((float(y + h / 2.0), float(x + w / 2.0)))\n    return centroids\n\n\nclass _BoundedCache:\n    """Tiny LRU cache of decoded flat images (4 MB each), bounded so memory\n    stays flat over an epoch when the images are not preloaded. The sampler\n    shuffles crops, so it rarely hits; decoding the PNG is the cost it saves.\n    """\n\n    def __init__(self, maxsize: int = 8) -> None:\n        self._maxsize = maxsize\n        self._data: "OrderedDict[str, np.ndarray]" = OrderedDict()\n\n    def get(self, key: str) -> np.ndarray | None:\n        entry = self._data.get(key)\n        if entry is not None:\n            self._data.move_to_end(key)\n        return entry\n\n    def put(self, key: str, entry: np.ndarray) -> None:\n        self._data[key] = entry\n        self._data.move_to_end(key)\n        while len(self._data) > self._maxsize:\n            self._data.popitem(last=False)\n\n\nclass FilamentCrops(Dataset):\n    """Random crops from cached flattened images, biased toward filaments.\n\n    ``target="annotator"`` (the default) trains against whichever single\n    annotator\'s view was selected -- the original behaviour. ``target=\n    "consensus"`` trains against ``scripts/preprocess.py``\'s per-stem soft\n    consensus mask instead: MAGFiLO\'s measured inter-annotator Panoptic\n    Quality is only 0.343, so one annotator\'s view carries a lot of\n    annotator-specific noise that a consensus average regresses out. The\n    consensus mask is keyed by observation stem, not by view, so under this\n    target ``image_ids`` is deduplicated to one view per stem -- the target\n    no longer depends on which view was picked, only the crop geometry does.\n    ``target="union"`` is the same mask cut at zero: a pixel is filament if\n    any annotator drew it.\n\n    ``preload=True`` decodes every flat image once, up front (about 4 MB\n    each), instead of once per crop. Decoding is most of the time a crop\n    takes, so this is what lets two training runs share a 4-CPU machine.\n    DataLoader workers forked after that share the arrays rather than copy\n    them; under the "spawn" start method (macOS) each worker would get its\n    own copy, so leave it off there.\n    """\n\n    def __init__(\n        self,\n        image_ids: Sequence[str],\n        annotations: Annotations,\n        cache_dir: str | Path,\n        crop_size: int = 512,\n        crops_per_image: int = 8,\n        filament_bias: float = 0.7,\n        augment: bool = True,\n        seed: int = 0,\n        target: str = "annotator",\n        preload: bool = False,\n    ) -> None:\n        if target not in ("annotator", "consensus", "union"):\n            raise ValueError(f"unknown target: {target!r}")\n\n        self.annotations = annotations\n        self.cache_dir = Path(cache_dir)\n        self.crop_size = crop_size\n        self.crops_per_image = crops_per_image\n        self.filament_bias = filament_bias\n        self.augment = augment\n        self.seed = seed\n        self.target = target\n\n        image_ids = list(image_ids)\n        if target in ("consensus", "union"):\n            image_ids = deduplicate_by_file(annotations, image_ids, seed=seed)\n        self.image_ids = image_ids\n\n        self._flat_dir = self.cache_dir / "flat"\n        self._mask_dir = self.cache_dir / "mask"\n        self._consensus_dir = self.cache_dir / "consensus"\n        self._disk = load_disk_geometry(self.cache_dir / "disk.json")\n        self._centroids: dict[str, list[tuple[float, float]]] = {\n            image_id: _instance_centroids(annotations, image_id) for image_id in self.image_ids\n        }\n\n        self._cache = _BoundedCache()\n        self._preloaded: dict[str, np.ndarray] = {}\n        if preload:\n            for stem in sorted({annotations.images[i].stem for i in self.image_ids}):\n                self._preloaded[stem] = self._read_flat(stem)\n        # One RNG per DataLoader worker process -- see _rng().\n        self._rngs: dict[int, random.Random] = {}\n\n    def __len__(self) -> int:\n        return len(self.image_ids) * self.crops_per_image\n\n    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:\n        image_id = self.image_ids[index // self.crops_per_image]\n        record = self.annotations.images[image_id]\n        stem = record.stem\n        shape = (record.height, record.width)\n        rng = self._rng()\n\n        flat_u8 = self._flat(stem)\n        y0, x0 = self._sample_crop_origin(rng, image_id, stem, shape)\n        size = self.crop_size\n\n        flat_crop = flat_u8[y0 : y0 + size, x0 : x0 + size].astype(np.float32) / 255.0\n        radius_crop = self._disk[stem].radius_window(y0, x0, *flat_crop.shape).astype(np.float32)\n        mask_crop = self._load_mask(image_id, stem, shape)[\n            y0 : y0 + size, x0 : x0 + size\n        ].astype(np.float32)\n\n        if self.augment:\n            out = _AUGMENT(image=flat_crop, mask=mask_crop, radius=radius_crop)\n            flat_crop, mask_crop, radius_crop = out["image"], out["mask"], out["radius"]\n\n        x = np.stack([flat_crop, radius_crop], axis=0).astype(np.float32)\n        y = mask_crop[None, :, :].astype(np.float32)\n        return torch.from_numpy(x), torch.from_numpy(y)\n\n    # --- internals -----------------------------------------------------\n\n    def _rng(self) -> random.Random:\n        """Per-worker RNG.\n\n        ``self`` -- and any ``random.Random`` already stored on it -- is\n        copied wholesale into each DataLoader worker process at fork time;\n        keying by worker id and creating the generator lazily means sibling\n        workers don\'t replay an identical crop sequence.\n        """\n        worker = torch.utils.data.get_worker_info()\n        key = 0 if worker is None else worker.id + 1\n        rng = self._rngs.get(key)\n        if rng is None:\n            rng = random.Random(self.seed + key)\n            self._rngs[key] = rng\n        return rng\n\n    def _flat(self, stem: str) -> np.ndarray:\n        flat_u8 = self._preloaded.get(stem)\n        if flat_u8 is None:\n            flat_u8 = self._cache.get(stem)\n        if flat_u8 is None:\n            flat_u8 = self._read_flat(stem)\n            self._cache.put(stem, flat_u8)\n        return flat_u8\n\n    def _read_flat(self, stem: str) -> np.ndarray:\n        flat_path = self._flat_dir / f"{stem}.png"\n        flat_u8 = cv2.imread(str(flat_path), cv2.IMREAD_GRAYSCALE)\n        if flat_u8 is None:\n            raise FileNotFoundError(\n                f"missing cached flat image: {flat_path} -- run scripts/preprocess.py first"\n            )\n        return flat_u8\n\n    def _load_mask(self, image_id: str, stem: str, shape: tuple[int, int]) -> np.ndarray:\n        if self.target in ("consensus", "union"):\n            consensus_path = self._consensus_dir / f"{stem}.png"\n            mask = cv2.imread(str(consensus_path), cv2.IMREAD_GRAYSCALE)\n            if mask is None:\n                raise FileNotFoundError(\n                    f"missing cached consensus mask: {consensus_path} -- "\n                    "run scripts/preprocess.py first"\n                )\n            if self.target == "union":\n                return (mask > 0).astype(np.uint8)\n            # Soft agreement fraction, not thresholded -- 255 means every\n            # annotator of this stem agreed, not just a majority.\n            return mask.astype(np.float32) / 255.0\n\n        mask_path = self._mask_dir / f"{image_id}.png"\n        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)\n        if mask is None:\n            # Every one of the 1154 training views has at least one annotated\n            # filament, so a missing mask is always a cache fault (a --limit\n            # preprocessing run, a wiped directory). Treating it as "no\n            # filaments" would silently teach the model to predict nothing.\n            raise FileNotFoundError(\n                f"missing cached mask: {mask_path} -- run scripts/preprocess.py first"\n            )\n        return (mask > 0).astype(np.uint8)\n\n    def _sample_crop_origin(\n        self, rng: random.Random, image_id: str, stem: str, shape: tuple[int, int]\n    ) -> tuple[int, int]:\n        """Pick the crop\'s top-left corner, in image coordinates."""\n        height, width = shape\n        size = self.crop_size\n        centroids = self._centroids[image_id]\n\n        if centroids and rng.random() < self.filament_bias:\n            cy, cx = rng.choice(centroids)\n            jitter = size / 4.0\n            cy += rng.uniform(-jitter, jitter)\n            cx += rng.uniform(-jitter, jitter)\n        else:\n            # Uniform over the disk\'s *area*, not its bounding box -- sampling\n            # (angle, radius) uniformly would crowd crops toward the centre.\n            disk = self._disk[stem]\n            angle = rng.uniform(0.0, 2.0 * np.pi)\n            r = disk.radius * (rng.random() ** 0.5)\n            cy = disk.cy + r * np.sin(angle)\n            cx = disk.cx + r * np.cos(angle)\n\n        y0 = int(np.clip(cy - size / 2.0, 0, max(height - size, 0)))\n        x0 = int(np.clip(cx - size / 2.0, 0, max(width - size, 0)))\n        return y0, x0\n\n', 'filament_seg/disk.py': '"""Solar disk geometry and limb-darkening correction.\n\nTwo preprocessing steps that matter more than the choice of model:\n\n1. **Disk masking.** Everything outside the solar limb is sky and detector\n   noise. It is uniformly dark, so any "filaments are dark" rule fires on it.\n   Masking it away removes a large class of false positives for free -- and\n   under Panoptic Quality every false positive costs 0.5 in the denominator.\n\n2. **Limb-darkening correction.** A full-disk H-alpha image is systematically\n   darker toward the limb, so a single global darkness threshold is far too\n   eager near the edge and far too shy at disk centre. Dividing by the median\n   radial intensity profile makes one threshold mean the same thing everywhere.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\n\n\n@dataclass(frozen=True)\nclass Disk:\n    cx: float\n    cy: float\n    radius: float\n\n    def mask(self, shape: tuple[int, int], shrink: float = 0.98) -> np.ndarray:\n        """Boolean mask of the disk, shrunk slightly to drop the noisy limb."""\n        height, width = shape\n        ys, xs = np.ogrid[:height, :width]\n        r2 = (xs - self.cx) ** 2 + (ys - self.cy) ** 2\n        return r2 <= (self.radius * shrink) ** 2\n\n    def radius_map(self, shape: tuple[int, int]) -> np.ndarray:\n        """Normalised radial distance (1.0 at the limb)."""\n        return self.radius_window(0, 0, *shape)\n\n    def radius_window(self, y0: int, x0: int, height: int, width: int) -> np.ndarray:\n        """``radius_map`` for rows ``y0:y0+height`` and columns ``x0:x0+width`` only."""\n        ys, xs = np.ogrid[y0 : y0 + height, x0 : x0 + width]\n        return np.sqrt((xs - self.cx) ** 2 + (ys - self.cy) ** 2) / max(self.radius, 1.0)\n\n\ndef read_image(path: str | Path) -> np.ndarray:\n    """Load a GONG H-alpha JPEG as a single-channel uint8 array.\n\n    The competition data is grayscale; reading it as BGR and averaging would\n    only add rounding noise.\n    """\n    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)\n    if image is None:\n        raise FileNotFoundError(f"could not read image: {path}")\n    return image\n\n\n#: GONG H-alpha frames are usually registered to a standard geometry: the\n#: disk centred and about 0.439 frame-widths in radius (899 px at 2048). Many\n#: frames are not, so this is only one of several starting guesses.\n_STANDARD_RADIUS_FRACTION = 0.439\n#: Rays cast from a candidate centre when locating the limb.\n_N_RAYS = 720\n#: How far inside each starting guess to look as well, in px at 2048. The halo\n#: can reach 150 px beyond the limb, so a guess locked onto it needs this.\n_INWARD_OFFSETS = (0.0, 50.0, 100.0, 150.0)\n#: A ray "agrees" with a circle if its sharpest edge lies within this many px\n#: (at 2048) of it. Fixed, not data-adaptive, so scattered edges score low.\n_AGREEMENT_TOLERANCE = 3.0\n#: A circle counts as well supported if this fraction of the best circle\'s\n#: agreeing rays also agree with it.\n_SUPPORT_RATIO = 0.8\n\n\ndef detect_disk(image: np.ndarray) -> Disk:\n    """Locate the solar disk by its limb.\n\n    GONG frames carry a smooth "halo" annulus outside the limb -- scattered\n    light, and the area the frame covered before it was registered -- that can\n    be nearly as bright as the disk itself, and its outer border is a crisp\n    circle too. Thresholding the image, which this function used to rely on\n    alone, locks onto that border: about 35 px beyond the limb on a typical\n    frame and up to 150 px on the worst.\n\n    Instead, several starting circles (an intensity threshold, a texture\n    threshold -- the disk is textured, the halo smooth -- and the standard GONG\n    geometry), each also shrunk by up to 150 px, are snapped to the sharpest\n    radial edge nearby and scored by how many rays agree with the result. The\n    halo always lies outside the limb, so of the well-supported circles the\n    smallest is the limb.\n    """\n    scale = min(image.shape) / 2048.0\n    blurred = cv2.GaussianBlur(image.astype(np.float32), (0, 0), max(1.5 * scale, 0.8))\n    seeds = [_threshold_disk(image), _texture_disk(image), _standard_disk(image)]\n\n    fits: list[tuple[Disk, float]] = []\n    for seed in seeds:\n        if seed is None:\n            continue\n        for offset in _INWARD_OFFSETS:\n            start = Disk(seed.cx, seed.cy, seed.radius - offset * scale)\n            if start.radius <= 0:\n                continue\n            disk, _ = _refine_limb(blurred, start, inside=60.0, outside=20.0)\n            # A second, narrow pass lets the centre settle once the radius is close.\n            fits.append(_refine_limb(blurred, disk, inside=10.0, outside=10.0))\n\n    best_support = max((support for _, support in fits), default=0.0)\n    # Fewer than half the rays agreeing means nothing found a clean limb; the\n    # plain threshold circle is then the least surprising answer.\n    if best_support < 0.5:\n        return seeds[0]\n    supported = [disk for disk, support in fits if support >= _SUPPORT_RATIO * best_support]\n    return min(supported, key=lambda disk: disk.radius)\n\n\ndef _fit_circle(xs: np.ndarray, ys: np.ndarray, iterations: int = 4) -> tuple[Disk, np.ndarray]:\n    """Least-squares (Kasa) circle fit, refitted without outliers.\n\n    Returns the circle and a boolean inlier mask over the input points.\n    """\n    keep = np.ones(xs.size, dtype=bool)\n    disk = Disk(0.0, 0.0, 1.0)\n    for _ in range(iterations):\n        if keep.sum() < 3:\n            break\n        design = np.column_stack([xs[keep], ys[keep], np.ones(int(keep.sum()))])\n        target = -(xs[keep] ** 2 + ys[keep] ** 2)\n        d, e, f = np.linalg.lstsq(design, target, rcond=None)[0]\n        cx, cy = -d / 2.0, -e / 2.0\n        disk = Disk(float(cx), float(cy), float(np.sqrt(max(cx * cx + cy * cy - f, 1.0))))\n        residual = np.hypot(xs - disk.cx, ys - disk.cy) - disk.radius\n        spread = 1.4826 * np.median(np.abs(residual[keep] - np.median(residual[keep])))\n        keep = np.abs(residual) <= max(3.0 * spread, 2.0)\n    return disk, keep\n\n\ndef _refine_limb(\n    blurred: np.ndarray, start: Disk, inside: float, outside: float\n) -> tuple[Disk, float]:\n    """Snap ``start`` to the sharpest radial edge within the search band.\n\n    ``blurred`` is the lightly smoothed float image. Band lengths are in px at\n    2048 and scale with the frame, so the same code works on small test\n    images. Returns the refined circle and the fraction of all rays whose\n    sharpest edge lies within ``_AGREEMENT_TOLERANCE`` of it.\n    """\n    height, width = blurred.shape\n    scale = min(height, width) / 2048.0\n    theta = np.linspace(0.0, 2.0 * np.pi, _N_RAYS, endpoint=False)[:, None]\n    radii = np.arange(start.radius - inside * scale, start.radius + outside * scale + 1.0, 1.0)\n    map_x = (start.cx + radii[None, :] * np.cos(theta)).astype(np.float32)\n    map_y = (start.cy + radii[None, :] * np.sin(theta)).astype(np.float32)\n    profiles = cv2.remap(blurred, map_x, map_y, cv2.INTER_LINEAR,\n                         borderMode=cv2.BORDER_CONSTANT, borderValue=0)\n    in_frame = (map_x >= 1) & (map_x <= width - 2) & (map_y >= 1) & (map_y <= height - 2)\n    gradient = np.abs(np.gradient(profiles, axis=1))\n    gradient[~in_frame] = -1.0\n    edge = np.argmax(gradient, axis=1)\n    valid = gradient[np.arange(_N_RAYS), edge] > 0\n    # A real edge is a peak inside the band. A maximum on the band\'s boundary\n    # is just the smooth limb-darkening slope rising toward it -- perfectly\n    # circular, so without this it would pass for a well-supported limb.\n    valid &= (edge > 1) & (edge < radii.size - 2)\n    if valid.sum() < 3:\n        return start, 0.0\n    # Sub-pixel peak position from a parabola through the peak and its two\n    # neighbours: the flattening downstream is sensitive to fractions of a\n    # pixel in the radius.\n    rows = np.flatnonzero(valid)\n    peak = edge[valid]\n    before, at, after = (gradient[rows, peak - 1], gradient[rows, peak],\n                         gradient[rows, peak + 1])\n    curvature = before - 2.0 * at + after\n    shift = np.divide(0.5 * (before - after), curvature,\n                      out=np.zeros_like(at), where=curvature < 0)\n    r = radii[peak] + np.clip(shift, -0.5, 0.5)\n    xs = start.cx + r * np.cos(theta[valid, 0])\n    ys = start.cy + r * np.sin(theta[valid, 0])\n    disk, _ = _fit_circle(xs, ys)\n    residual = np.hypot(xs - disk.cx, ys - disk.cy) - disk.radius\n    agreeing = np.abs(residual) <= max(_AGREEMENT_TOLERANCE * scale, 1.0)\n    return disk, float(agreeing.sum()) / _N_RAYS\n\n\ndef _standard_disk(image: np.ndarray) -> Disk:\n    height, width = image.shape\n    return Disk((width - 1) / 2.0, height / 2.0, min(height, width) * _STANDARD_RADIUS_FRACTION)\n\n\ndef _texture_disk(image: np.ndarray) -> Disk | None:\n    """Disk from local texture: chromospheric structure is busy, the halo smooth."""\n    scale = min(image.shape) / 2048.0\n    img = image.astype(np.float32)\n    detail = np.abs(img - cv2.GaussianBlur(img, (0, 0), max(3.0 * scale, 1.0)))\n    texture = cv2.GaussianBlur(detail, (0, 0), max(15.0 * scale, 2.0))\n    texture_u8 = np.clip(texture * (255.0 / max(float(texture.max()), 1e-6)), 0, 255)\n    _, binary = cv2.threshold(texture_u8.astype(np.uint8), 0, 1,\n                              cv2.THRESH_BINARY + cv2.THRESH_OTSU)\n    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)\n    if n_labels <= 1:\n        return None\n    largest = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))\n    component = (labels == largest).astype(np.uint8)\n    # Fill the component\'s interior: smooth patches on the disk (and the whole\n    # interior of a texture-free test image) leave holes.\n    flood = component.copy()\n    cv2.floodFill(flood, np.zeros((flood.shape[0] + 2, flood.shape[1] + 2), np.uint8), (0, 0), 1)\n    component = component | (1 - flood)\n    contours, _ = cv2.findContours(component, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)\n    points = max(contours, key=cv2.contourArea)[:, 0, :].astype(np.float64)\n    if len(points) < 3:\n        return None\n    disk, _ = _fit_circle(points[:, 0], points[:, 1])\n    return disk\n\n\ndef _threshold_disk(image: np.ndarray) -> Disk:\n    """Disk from an intensity threshold: the brightest large component.\n\n    Reliable for the centre of a clean frame, but the halo annulus usually\n    passes the threshold too, so the radius it gives is an upper bound.\n    """\n    blurred = cv2.GaussianBlur(image, (0, 0), sigmaX=5)\n    threshold, binary = cv2.threshold(\n        blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU\n    )\n    binary = cv2.morphologyEx(\n        binary, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (31, 31))\n    )\n\n    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)\n    if n_labels <= 1:\n        # Degenerate image: fall back to the nominal full-frame disk.\n        height, width = image.shape\n        return Disk(width / 2, height / 2, min(height, width) * 0.45)\n\n    largest = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))\n    component = (labels == largest).astype(np.uint8)\n\n    contours, _ = cv2.findContours(component, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)\n    contour = max(contours, key=cv2.contourArea)\n    (cx, cy), radius = cv2.minEnclosingCircle(contour)\n    return Disk(float(cx), float(cy), float(radius))\n\n\ndef limb_darkening_profile(\n    image: np.ndarray, disk: Disk, n_bins: int = 128, smooth: int = 9\n) -> np.ndarray:\n    """Median intensity as a function of normalised radius."""\n    radius = disk.radius_map(image.shape)\n    inside = radius <= 1.0\n    bins = np.clip((radius[inside] * n_bins).astype(np.int32), 0, n_bins - 1)\n    values = image[inside].astype(np.float32)\n\n    profile = np.ones(n_bins, dtype=np.float32)\n    order = np.argsort(bins, kind="stable")\n    bins_sorted, values_sorted = bins[order], values[order]\n    edges = np.searchsorted(bins_sorted, np.arange(n_bins + 1))\n    for b in range(n_bins):\n        lo, hi = edges[b], edges[b + 1]\n        if hi > lo:\n            profile[b] = np.median(values_sorted[lo:hi])\n\n    # Fill empty bins, then smooth so the correction has no step artefacts.\n    valid = profile > 0\n    if valid.any():\n        profile = np.interp(np.arange(n_bins), np.flatnonzero(valid), profile[valid])\n    if smooth > 1:\n        kernel = np.ones(smooth, dtype=np.float32) / smooth\n        profile = np.convolve(profile, kernel, mode="same")\n    profile[profile <= 0] = 1.0\n    return profile\n\n\ndef flatten(image: np.ndarray, disk: Disk, n_bins: int = 128) -> np.ndarray:\n    """Divide out limb darkening.\n\n    Returns a float32 image where 1.0 is "typical quiet Sun at this radius" and\n    filaments sit clearly below 1.0, independent of distance from disk centre.\n    """\n    profile = limb_darkening_profile(image, disk, n_bins=n_bins)\n    radius = disk.radius_map(image.shape)\n    bin_index = np.clip((radius * n_bins).astype(np.int32), 0, n_bins - 1)\n    expected = profile[bin_index]\n    flat = image.astype(np.float32) / np.maximum(expected, 1e-3)\n    flat[radius > 1.0] = 1.0\n    return flat\n\n', 'filament_seg/fusion.py': '"""Detector-guided instance formation.\n\nThe U-Net answers "is this pixel a filament?" well -- its matched masks reach\nan IoU of about 0.67 -- but it has no notion of which pixels belong to the\n*same* filament. Connected components and gap bridging guess that from\ngeometry alone, and that guess is where most of the Panoptic Quality goes:\nthe U-Net pipeline makes about as many false positives as true positives. An\ninstance detector answers the grouping question directly and scores every\ninstance, but its masks are coarse: predicted at a quarter of its input\nresolution, which is itself half the image\'s.\n\nFusion takes the best of both. Each detection, in descending confidence,\nclaims the full-resolution U-Net filament pixels inside a slightly grown copy\nof its coarse mask. A pixel belongs to the first detection that claims it, so\ninstances never overlap. U-Net pixels that no detection claims are dropped --\nor, with ``keep_unclaimed``, grouped the way the U-Net-only pipeline groups\nthem (gap bridging, then an area cut), for filaments the detector missed. At\n``keep_unclaimed`` equal to the U-Net pipeline\'s own ``min_area``, fusion only\nchanges the grouping where a detection is confident enough to claim pixels.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\n\nimport cv2\nimport numpy as np\n\nfrom .postprocess import (\n    PostprocessParams,\n    _ellipse,\n    bridge_and_label,\n    clean_binary,\n    fill_holes,\n    probability_to_logit,\n)\nfrom pycocotools import mask as mask_utils\n\nfrom .rle import counts_to_mask, rle_from_counts\n\n\n@dataclass(frozen=True)\nclass Detection:\n    """One detector instance: its confidence and its mask as COCO RLE counts."""\n\n    score: float\n    counts: str\n\n\n@dataclass(frozen=True)\nclass Region:\n    """A detection\'s grown mask, cropped to its bounding box."""\n\n    score: float\n    y0: int\n    y1: int\n    x0: int\n    x1: int\n    mask: np.ndarray\n\n\n@dataclass\nclass FusionParams:\n    """Knobs for detector-guided instance formation."""\n\n    #: U-Net probability above which a pixel counts as filament.\n    threshold: float = 0.5\n    #: Morphological closing applied to the U-Net mask before assignment.\n    close_radius: int = 3\n    #: Pixels the detector\'s coarse mask is grown by before claiming U-Net\n    #: pixels; it is predicted at an eighth of full resolution.\n    grow: int = 8\n    #: Detections below this confidence claim nothing.\n    min_score: float = 0.25\n    #: Instances smaller than this, after claiming, are dropped.\n    min_area: int = 400\n    #: Unclaimed U-Net pixels are grouped by bridging gaps of this many\n    #: pixels, and groups at least ``keep_unclaimed`` large are kept as\n    #: instances of their own; 0 drops every unclaimed pixel.\n    keep_unclaimed: int = 0\n    unclaimed_gap: int = 24\n\n\ndef unet_binary(\n    logits: np.ndarray, disk_mask: np.ndarray, threshold: float, close_radius: int\n) -> np.ndarray:\n    """The U-Net\'s filament pixels: thresholded, closed, holes filled, on the disk."""\n    binary = ((logits > probability_to_logit(threshold)) & disk_mask).astype(np.uint8)\n    binary = clean_binary(binary, PostprocessParams(open_radius=0, close_radius=close_radius))\n    return fill_holes(binary).astype(bool) & disk_mask\n\n\ndef grow_regions(\n    detections: list[Detection],\n    grow: int,\n    shape: tuple[int, int],\n    min_score: float = 0.0,\n) -> list[Region]:\n    """Decode, crop and grow each detection\'s mask, highest confidence first."""\n    height, width = shape\n    regions = []\n    kernel = _ellipse(grow) if grow > 0 else None\n    for detection in sorted(detections, key=lambda d: -d.score):\n        if detection.score < min_score:\n            continue\n        mask = counts_to_mask(detection.counts, height, width)\n        ys, xs = np.nonzero(mask)\n        if ys.size == 0:\n            continue\n        y0, y1 = max(int(ys.min()) - grow, 0), min(int(ys.max()) + grow + 1, height)\n        x0, x1 = max(int(xs.min()) - grow, 0), min(int(xs.max()) + grow + 1, width)\n        crop = mask[y0:y1, x0:x1].astype(np.uint8)\n        if kernel is not None:\n            crop = cv2.dilate(crop, kernel)\n        regions.append(Region(detection.score, y0, y1, x0, x1, crop.astype(bool)))\n    return regions\n\n\ndef assign(\n    binary: np.ndarray,\n    regions: list[Region],\n    min_score: float,\n    min_area: int,\n    keep_unclaimed: int = 0,\n    unclaimed_gap: int = 24,\n) -> np.ndarray:\n    """Integer instance labels: each region, best first, claims its U-Net pixels."""\n    labels = np.zeros(binary.shape, dtype=np.int32)\n    claimed = np.zeros(binary.shape, dtype=bool)\n    next_label = 1\n    for region in regions:\n        if region.score < min_score:\n            break\n        window = (slice(region.y0, region.y1), slice(region.x0, region.x1))\n        pixels = binary[window] & region.mask & ~claimed[window]\n        if int(pixels.sum()) < min_area:\n            continue\n        labels[window][pixels] = next_label\n        claimed[window] |= pixels\n        next_label += 1\n\n    if keep_unclaimed > 0:\n        components = bridge_and_label((binary & ~claimed).astype(np.uint8), unclaimed_gap)\n        counts = np.bincount(components.ravel())\n        keep = counts >= keep_unclaimed\n        keep[0] = False\n        lookup = np.zeros(counts.size, dtype=np.int32)\n        lookup[keep] = np.arange(next_label, next_label + int(keep.sum()), dtype=np.int32)\n        extra = lookup[components]\n        labels = np.where(extra > 0, extra, labels)\n    return labels\n\n\ndef fuse(\n    logits: np.ndarray,\n    disk_mask: np.ndarray,\n    detections: list[Detection],\n    params: FusionParams | None = None,\n) -> np.ndarray:\n    """U-Net logits + detector instances -> integer instance label map."""\n    params = params or FusionParams()\n    binary = unet_binary(logits, disk_mask, params.threshold, params.close_radius)\n    regions = grow_regions(detections, params.grow, logits.shape, params.min_score)\n    return assign(binary, regions, params.min_score, params.min_area, params.keep_unclaimed,\n                  params.unclaimed_gap)\n\n\ndef merge_views(views: list[list[Detection]], iou_threshold: float = 0.5) -> list[Detection]:\n    """Merge detections from flipped copies of one image into one list.\n\n    ``views`` holds each copy\'s detections, already flipped back. Detections\n    from different copies that overlap with IoU above ``iou_threshold`` are\n    the same filament; it keeps the best member\'s mask and scores the\n    average over *all* copies, counting a copy that missed it as 0. A\n    filament the detector finds from every orientation keeps its confidence;\n    one it sees in a single orientation loses most of it.\n    """\n    pool = sorted(((det, view) for view, dets in enumerate(views) for det in dets),\n                  key=lambda item: -item[0].score)\n    if not pool:\n        return []\n    rles = [rle_from_counts(det.counts) for det, _ in pool]\n    ious = np.asarray(mask_utils.iou(rles, rles, [0] * len(rles)))\n    used = np.zeros(len(pool), dtype=bool)\n    merged = []\n    for i, (detection, view) in enumerate(pool):\n        if used[i]:\n            continue\n        used[i] = True\n        seen, total = {view}, detection.score\n        for j in np.argsort(-ious[i], kind="stable"):\n            if ious[i, j] <= iou_threshold:\n                break\n            if used[j] or pool[j][1] in seen:\n                continue\n            used[j] = True\n            seen.add(pool[j][1])\n            total += pool[j][0].score\n        merged.append(Detection(total / len(views), detection.counts))\n    return merged\n\n', 'filament_seg/logit_cache.py': '"""A disk cache of full-resolution model logits, safe against staleness.\n\nTiled inference with test-time augmentation is the expensive half of every\nevaluation, and it does not depend on any post-processing setting, so its\noutput -- one 2048x2048 logit map per observation, float16 (8 MB) -- is cached\nand reused by every grid point and every later experiment. The cache is keyed\nby observation name alone, so it also records which model, TTA mode and tiling\nproduced it and refuses to mix them.\n"""\n\nfrom __future__ import annotations\n\nimport json\nimport os\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\nfrom .dataset import load_model_input\nfrom .disk import Disk\nfrom .model import ensemble_predict, load_trained\n\n\ndef cache_path(logit_dir: Path, stem: str) -> Path:\n    return logit_dir / f"{stem}.npy"\n\n\ndef check_cache_provenance(\n    logit_dir: Path,\n    checkpoint_paths: list[str],\n    tta: str,\n    tile: int,\n    overlap: int,\n    force: bool,\n) -> None:\n    """Make sure every cached logit map was produced by this model and setup.\n\n    The cache is keyed by observation name alone, so without this a sweep run\n    against a new checkpoint, TTA mode or tiling would silently reuse stale\n    logits and tune against the wrong predictions. ``force`` deletes the old\n    maps *before* recording the new provenance: otherwise a forced run that\n    died part-way would leave the previous model\'s maps under the new record,\n    and the next plain run would accept them.\n    """\n    checkpoints = []\n    for path in checkpoint_paths:\n        resolved = Path(path).resolve()\n        stat = resolved.stat()\n        checkpoints.append({"path": str(resolved), "size": stat.st_size,\n                            "mtime": int(stat.st_mtime)})\n    wanted = {"checkpoints": checkpoints, "tta": tta, "tile": tile, "overlap": overlap}\n\n    logit_dir.mkdir(parents=True, exist_ok=True)\n    meta_path = logit_dir / "cache_meta.json"\n    cached = sorted(logit_dir.glob("*.npy"))\n    if force:\n        for path in cached:\n            path.unlink()\n    elif meta_path.exists():\n        found = json.loads(meta_path.read_text(encoding="utf-8"))\n        if found != wanted:\n            raise SystemExit(\n                f"{logit_dir} holds logits from a different model or setup:\\n"\n                f"  cached: {found}\\n  wanted: {wanted}\\n"\n                "use a different --out-dir, or --force to recompute"\n            )\n    elif cached:\n        raise SystemExit(\n            f"{logit_dir} holds cached logits with no record of which model made them; "\n            "use a different --out-dir, or --force to recompute"\n        )\n    meta_path.write_text(json.dumps(wanted, indent=2), encoding="utf-8")\n\n\ndef ensure_logits(\n    stems: list[str],\n    checkpoint_paths: list[str],\n    disk_geometry: dict[str, Disk],\n    flat_dir: Path,\n    logit_dir: Path,\n    tile: int,\n    overlap: int,\n    device: torch.device,\n    force: bool,\n    tta: str = "none",\n) -> None:\n    """Cache the (ensemble-averaged, TTA-averaged) logits for every stem, once."""\n    check_cache_provenance(logit_dir, checkpoint_paths, tta, tile, overlap, force)\n    pending = [s for s in stems if not cache_path(logit_dir, s).exists()]\n    if not pending:\n        print(f"logits: {len(stems)}/{len(stems)} already cached, skipping model load")\n        return\n\n    print(f"device: {device}")\n    models = [load_trained(path, device) for path in checkpoint_paths]\n    print(f"models: {len(models)}; computing logits for {len(pending)}/{len(stems)} observations")\n    for n, stem in enumerate(pending, start=1):\n        x = load_model_input(flat_dir, stem, disk_geometry[stem])\n        if x is None:\n            raise SystemExit(f"no cached flat image for {stem} -- run scripts/preprocess.py")\n        logits = ensemble_predict(models, x, tta=tta, tile=tile, overlap=overlap, device=device)\n        # Written under a temporary name and renamed, so an interrupted run can\n        # never leave a truncated map that a later run would take as complete.\n        partial = logit_dir / f"{stem}.npy.partial"\n        with open(partial, "wb") as handle:\n            np.save(handle, logits.astype(np.float16))\n        os.replace(partial, cache_path(logit_dir, stem))\n        if n % 5 == 0 or n == len(pending):\n            print(f"  {n}/{len(pending)}", flush=True)\n\n', 'filament_seg/metrics.py': '"""Local re-implementation of the competition\'s evaluation.\n\nLeaderboard metric (competition Overview -> "Leaderboard Ranking"):\n\n    PQ(Y, Y_hat) = sum_{(y, y_hat) in TP} IoU(y, y_hat)\n                   / (|TP| + 0.5 |FP| + 0.5 |FN|)\n\nwhich is exactly standard Panoptic Quality (Kirillov et al., CVPR 2019) with the\nusual IoU > 0.5 matching rule. Above 0.5 the match is provably unique, so greedy\nassignment by descending IoU is optimal.\n\nTwo design notes:\n\n* **Aggregation is pooled.** The organisers\' self-evaluation notebook\n  (kaggle.com/code/azimahmadzadeh/self-evaluation-notebook) matches the\n  predictions against every annotator\'s view of an observation separately and\n  pools TP, FP, FN and the matched IoUs over all views of all observations;\n  ``filament_seg.scoring`` keeps those totals per observation. ``summarize``\n  reports the pooled score (``pq_pooled``) and, for diagnostics only, the mean\n  per-image PQ (``pq_per_image_mean``).\n\n* **The rubric wants more than PQ.** 70% of the final score is quantitative and\n  explicitly includes the *distributions* of Dice and IoU plus the counts of\n  one-to-many (fragmentation) and many-to-one (over-merge) relations. Those are\n  produced here too, so the report can be written from real numbers.\n\nMarginal-value rule worth remembering while tuning post-processing: a predicted\ninstance that fails to reach IoU 0.5 costs 0.5 FP + 0.5 FN = 1.0 in the\ndenominator, while simply not predicting it costs only 0.5. Emitting a doubtful\ninstance is *twice* as expensive as staying silent, so a prediction is only\nworth making when its chance of clearing the threshold is above roughly\n``PQ / (2 * E[IoU | hit])`` -- around 30% at a PQ of 0.4.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\nfrom typing import Sequence\n\nimport numpy as np\n\nfrom .config import FRAGMENT_OVERLAP_THRESHOLD, PQ_IOU_THRESHOLD\nfrom .rle import Rle, iou_matrix, rle_areas, rle_intersection_area\n\n\n@dataclass\nclass ImageResult:\n    """Per-image matching outcome."""\n\n    image_id: str\n    n_gt: int\n    n_pred: int\n    tp: int\n    fp: int\n    fn: int\n    matched_ious: list[float] = field(default_factory=list)\n    #: Ground-truth segments split across >= 2 predictions (fragmentation).\n    one_to_many: int = 0\n    #: Predictions covering >= 2 ground-truth segments (over-merge).\n    many_to_one: int = 0\n\n    @property\n    def iou_sum(self) -> float:\n        return float(sum(self.matched_ious))\n\n    @property\n    def denominator(self) -> float:\n        return self.tp + 0.5 * self.fp + 0.5 * self.fn\n\n    @property\n    def pq(self) -> float:\n        if self.denominator == 0:\n            # No ground truth and no prediction: nothing to get wrong.\n            return 1.0\n        return self.iou_sum / self.denominator\n\n    @property\n    def matched_dices(self) -> list[float]:\n        # Dice and IoU are algebraically linked: Dice = 2*IoU / (1 + IoU).\n        return [2.0 * i / (1.0 + i) for i in self.matched_ious]\n\n\ndef match_instances(\n    gt_rles: Sequence[Rle],\n    pred_rles: Sequence[Rle],\n    iou_threshold: float = PQ_IOU_THRESHOLD,\n) -> tuple[list[tuple[int, int, float]], np.ndarray]:\n    """Greedily match predictions to ground truth by descending IoU.\n\n    Returns ``(matches, iou_matrix)`` where each match is ``(gt_i, pred_j, iou)``.\n    """\n    ious = iou_matrix(gt_rles, pred_rles)\n    matches: list[tuple[int, int, float]] = []\n    if ious.size == 0:\n        return matches, ious\n\n    candidates = np.argwhere(ious > iou_threshold)\n    order = np.argsort([-ious[g, p] for g, p in candidates]) if len(candidates) else []\n    used_gt: set[int] = set()\n    used_pred: set[int] = set()\n    for idx in order:\n        g, p = candidates[idx]\n        g, p = int(g), int(p)\n        if g in used_gt or p in used_pred:\n            continue\n        used_gt.add(g)\n        used_pred.add(p)\n        matches.append((g, p, float(ious[g, p])))\n    return matches, ious\n\n\ndef count_fragmentation(\n    gt_rles: Sequence[Rle],\n    pred_rles: Sequence[Rle],\n    overlap_threshold: float = FRAGMENT_OVERLAP_THRESHOLD,\n) -> tuple[int, int]:\n    """Count one-to-many and many-to-one relations.\n\n    A ground-truth segment is *fragmented* when at least two predictions each\n    cover more than ``overlap_threshold`` of its area. A prediction is an\n    *over-merge* when it covers more than ``overlap_threshold`` of at least two\n    ground-truth segments.\n    """\n    if not gt_rles or not pred_rles:\n        return 0, 0\n\n    gt_area = rle_areas(gt_rles)\n    inter = np.zeros((len(gt_rles), len(pred_rles)), dtype=np.float64)\n    ious = iou_matrix(gt_rles, pred_rles)\n    for g in range(len(gt_rles)):\n        for p in range(len(pred_rles)):\n            if ious[g, p] > 0:\n                inter[g, p] = rle_intersection_area(gt_rles[g], pred_rles[p])\n\n    with np.errstate(divide="ignore", invalid="ignore"):\n        coverage = np.where(gt_area[:, None] > 0, inter / gt_area[:, None], 0.0)\n\n    significant = coverage > overlap_threshold\n    one_to_many = int((significant.sum(axis=1) >= 2).sum())\n    many_to_one = int((significant.sum(axis=0) >= 2).sum())\n    return one_to_many, many_to_one\n\n\ndef evaluate_image(\n    image_id: str,\n    gt_rles: Sequence[Rle],\n    pred_rles: Sequence[Rle],\n    iou_threshold: float = PQ_IOU_THRESHOLD,\n    with_fragmentation: bool = True,\n) -> ImageResult:\n    matches, _ = match_instances(gt_rles, pred_rles, iou_threshold)\n    tp = len(matches)\n    result = ImageResult(\n        image_id=image_id,\n        n_gt=len(gt_rles),\n        n_pred=len(pred_rles),\n        tp=tp,\n        fp=len(pred_rles) - tp,\n        fn=len(gt_rles) - tp,\n        matched_ious=[m[2] for m in matches],\n    )\n    if with_fragmentation:\n        result.one_to_many, result.many_to_one = count_fragmentation(gt_rles, pred_rles)\n    return result\n\n\ndef summarize(results: Sequence[ImageResult]) -> dict:\n    """Aggregate per-image results into the numbers the rubric asks for."""\n    if not results:\n        return {"n_images": 0}\n\n    tp = sum(r.tp for r in results)\n    fp = sum(r.fp for r in results)\n    fn = sum(r.fn for r in results)\n    iou_sum = sum(r.iou_sum for r in results)\n    denom = tp + 0.5 * fp + 0.5 * fn\n\n    all_ious = np.array([i for r in results for i in r.matched_ious], dtype=np.float64)\n    all_dices = np.array([d for r in results for d in r.matched_dices], dtype=np.float64)\n    per_image = np.array([r.pq for r in results], dtype=np.float64)\n\n    def distribution(values: np.ndarray) -> dict:\n        if values.size == 0:\n            return {"n": 0}\n        percentiles = np.percentile(values, [5, 25, 50, 75, 95])\n        return {\n            "n": int(values.size),\n            "mean": float(values.mean()),\n            "std": float(values.std()),\n            "min": float(values.min()),\n            "p05": float(percentiles[0]),\n            "p25": float(percentiles[1]),\n            "median": float(percentiles[2]),\n            "p75": float(percentiles[3]),\n            "p95": float(percentiles[4]),\n            "max": float(values.max()),\n        }\n\n    return {\n        "n_images": len(results),\n        # The two candidate aggregations -- see the module docstring.\n        "pq_pooled": float(iou_sum / denom) if denom else float("nan"),\n        "pq_per_image_mean": float(per_image.mean()),\n        # Standard PQ = SQ * RQ decomposition, useful for diagnosis:\n        # SQ says "how good are the masks I got right", RQ says "how often am I right".\n        "sq": float(iou_sum / tp) if tp else 0.0,\n        "rq": float(tp / denom) if denom else 0.0,\n        "tp": tp,\n        "fp": fp,\n        "fn": fn,\n        "n_gt": sum(r.n_gt for r in results),\n        "n_pred": sum(r.n_pred for r in results),\n        "one_to_many": sum(r.one_to_many for r in results),\n        "many_to_one": sum(r.many_to_one for r in results),\n        "iou_distribution": distribution(all_ious),\n        "dice_distribution": distribution(all_dices),\n        "pq_per_image_distribution": distribution(per_image),\n    }\n\n\ndef evaluate(\n    gt: dict[str, Sequence[Rle]],\n    pred: dict[str, Sequence[Rle]],\n    iou_threshold: float = PQ_IOU_THRESHOLD,\n    with_fragmentation: bool = True,\n) -> tuple[dict, list[ImageResult]]:\n    """Score a prediction dict against a ground-truth dict, keyed by image."""\n    results = [\n        evaluate_image(\n            image_id,\n            gt[image_id],\n            pred.get(image_id, []),\n            iou_threshold=iou_threshold,\n            with_fragmentation=with_fragmentation,\n        )\n        for image_id in sorted(gt)\n    ]\n    # Predictions for images absent from the ground truth are pure false positives.\n    for image_id in sorted(set(pred) - set(gt)):\n        n_pred = len(pred[image_id])\n        results.append(\n            ImageResult(image_id=image_id, n_gt=0, n_pred=n_pred, tp=0, fp=n_pred, fn=0)\n        )\n    return summarize(results), results\n\n\ndef format_report(summary: dict) -> str:\n    """Human-readable summary for the terminal and the technical report."""\n    if not summary.get("n_images"):\n        return "no images evaluated"\n\n    def dist_line(name: str, dist: dict) -> str:\n        if not dist.get("n"):\n            return f"  {name:<6} (none)"\n        return (\n            f"  {name:<6} n={dist[\'n\']:<5} mean={dist[\'mean\']:.3f}  "\n            f"p05={dist[\'p05\']:.3f} p25={dist[\'p25\']:.3f} med={dist[\'median\']:.3f} "\n            f"p75={dist[\'p75\']:.3f} p95={dist[\'p95\']:.3f}"\n        )\n\n    return "\\n".join(\n        [\n            f"images={summary[\'n_images\']}  gt={summary[\'n_gt\']}  pred={summary[\'n_pred\']}",\n            f"PQ (pooled)      {summary[\'pq_pooled\']:.4f}",\n            f"PQ (per-image)   {summary[\'pq_per_image_mean\']:.4f}",\n            f"SQ={summary[\'sq\']:.4f}  RQ={summary[\'rq\']:.4f}",\n            f"TP={summary[\'tp\']}  FP={summary[\'fp\']}  FN={summary[\'fn\']}",\n            f"fragmentation: one-to-many={summary[\'one_to_many\']}  "\n            f"many-to-one={summary[\'many_to_one\']}",\n            "distributions over matched pairs:",\n            dist_line("IoU", summary["iou_distribution"]),\n            dist_line("Dice", summary["dice_distribution"]),\n            dist_line("PQ/img", summary["pq_per_image_distribution"]),\n        ]\n    )\n\n', 'filament_seg/model.py': '"""Segmentation network, loss, device selection and tiled inference.\n\nA U-Net over a ResNet encoder is a deliberately boring choice: at native\n2048x2048 resolution there is no budget for architecture search, and a\nwell-understood encoder/decoder pair with ImageNet-pretrained weights gives\nthe fastest path to a working model. The two-channel input (flattened\nintensity + disk radius) needs its own first conv, so the pretrained weights\nonly warm-start the encoder trunk, not that very first layer.\n\n``tiled_predict`` lives here rather than duplicated in ``scripts/train.py``\n(per-epoch validation) and ``scripts/predict.py`` (submission generation) so\nthe two can never quietly drift apart on how tiles are blended.\n"""\n\nfrom __future__ import annotations\n\nimport segmentation_models_pytorch as smp\nimport numpy as np\nimport torch\nimport torch.nn as nn\n\n\ndef build_model(\n    encoder: str = "resnet34",\n    in_channels: int = 2,\n    weights: str | None = "imagenet",\n) -> nn.Module:\n    """U-Net over ``encoder``, one logit output channel (no activation).\n\n    The loss (``DiceBCELoss``) applies ``BCEWithLogitsLoss`` internally, so\n    the network stays activation-free -- this is also what keeps it\n    numerically stable under autocast.\n    """\n    return smp.Unet(\n        encoder_name=encoder,\n        encoder_weights=weights,\n        in_channels=in_channels,\n        classes=1,\n        activation=None,\n    )\n\n\nclass DiceBCELoss(nn.Module):\n    """Soft Dice + BCE-with-logits.\n\n    Filament pixels are roughly 0.2% of the disk, so plain BCE is dominated\n    by the easy background majority long before its gradient starts pushing\n    on filament boundaries. Dice is defined on the foreground overlap and is\n    immune to how much background surrounds it; BCE is kept alongside it\n    because Dice alone gives a noisy, sometimes unstable signal on crops with\n    very few (or zero) positive pixels.\n\n    ``fn_weight`` turns the Dice term into a Tversky index: above 0.5, missed\n    filament pixels cost more than extra ones (Salehi et al., 2017). At 0.5,\n    the default, it is exactly soft Dice.\n    """\n\n    def __init__(\n        self,\n        dice_weight: float = 1.0,\n        bce_weight: float = 1.0,\n        smooth: float = 1.0,\n        fn_weight: float = 0.5,\n    ) -> None:\n        super().__init__()\n        self.dice_weight = dice_weight\n        self.bce_weight = bce_weight\n        self.smooth = smooth\n        self.fn_weight = fn_weight\n        self.bce = nn.BCEWithLogitsLoss()\n\n    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:\n        bce_loss = self.bce(logits, target)\n\n        probs = torch.sigmoid(logits).flatten(1)\n        target_flat = target.flatten(1)\n        intersection = (probs * target_flat).sum(dim=1)\n        false_pos = probs.sum(dim=1) - intersection\n        false_neg = target_flat.sum(dim=1) - intersection\n        denominator = (2.0 * intersection + 2.0 * (1.0 - self.fn_weight) * false_pos\n                       + 2.0 * self.fn_weight * false_neg)\n        dice = (2.0 * intersection + self.smooth) / (denominator + self.smooth)\n        dice_loss = 1.0 - dice.mean()\n\n        return self.dice_weight * dice_loss + self.bce_weight * bce_loss\n\n\ndef select_device() -> torch.device:\n    """Prefer CUDA, then Apple MPS, then CPU."""\n    if torch.cuda.is_available():\n        return torch.device("cuda")\n    if torch.backends.mps.is_available():\n        return torch.device("mps")\n    return torch.device("cpu")\n\n\ndef _hann_window(size: int) -> np.ndarray:\n    window = np.hanning(size)\n    if window.sum() == 0:  # size == 1: np.hanning gives an all-zero window\n        window = np.ones(size)\n    return np.outer(window, window).astype(np.float32)\n\n\n@torch.no_grad()\ndef tiled_predict(\n    model: nn.Module,\n    x: np.ndarray,\n    tile: int = 512,\n    overlap: int = 128,\n    device: torch.device | None = None,\n    batch_size: int = 4,\n) -> np.ndarray:\n    """Run ``model`` over overlapping tiles of a full-resolution image.\n\n    Tiles are blended with a 2D Hann window rather than averaged uniformly: a\n    tile\'s prediction is least reliable right at its own edge (no context\n    beyond the tile boundary), so down-weighting the edges is what keeps the\n    seams between tiles from fragmenting a filament that crosses one -- and\n    fragmentation is punished roughly three times over under Panoptic\n    Quality (see ``filament_seg.postprocess``).\n\n    ``x`` is ``(C, H, W)`` float32; returns full-resolution logits ``(H, W)``.\n    """\n    model.eval()\n    device = device or next(model.parameters()).device\n    _, height, width = x.shape\n    stride = max(tile - overlap, 1)\n\n    def positions(extent: int) -> list[int]:\n        starts = list(range(0, max(extent - tile, 0) + 1, stride))\n        if not starts or starts[-1] != extent - tile:\n            starts.append(max(extent - tile, 0))\n        return starts\n\n    ys, xs = positions(height), positions(width)\n    window = _hann_window(tile)\n    logits_sum = np.zeros((height, width), dtype=np.float32)\n    weight_sum = np.zeros((height, width), dtype=np.float32)\n\n    grid = [(y, xc) for y in ys for xc in xs]\n    for start in range(0, len(grid), batch_size):\n        batch = grid[start : start + batch_size]\n        tiles = np.stack([x[:, y : y + tile, xc : xc + tile] for y, xc in batch], axis=0)\n        tensor = torch.from_numpy(tiles).to(device)\n        out = model(tensor).squeeze(1).float().cpu().numpy()\n        for (y, xc), tile_logits in zip(batch, out):\n            logits_sum[y : y + tile, xc : xc + tile] += tile_logits * window\n            weight_sum[y : y + tile, xc : xc + tile] += window\n\n    weight_sum[weight_sum == 0] = 1.0\n    return logits_sum / weight_sum\n\n\n#: Test-time augmentation: which orientations to average over, each written as\n#: (flip left-right, flip up-down, transpose). Training augmented with flips and\n#: 90-degree rotations, so the model has seen all eight orientations of a square\n#: image ("dihedral"); "flips" is the cheaper half of them.\nTTA_MODES: dict[str, list[tuple[bool, bool, bool]]] = {\n    "none": [(False, False, False)],\n    "flips": [(False, False, False), (True, False, False),\n              (False, True, False), (True, True, False)],\n    "dihedral": [(flip_lr, flip_ud, transpose)\n                 for transpose in (False, True)\n                 for flip_lr in (False, True)\n                 for flip_ud in (False, True)],\n}\n\n\ndef _orient(a: np.ndarray, flip_lr: bool, flip_ud: bool, transpose: bool) -> np.ndarray:\n    """Apply one orientation to the last two (spatial) axes."""\n    if transpose:\n        a = np.swapaxes(a, -1, -2)\n    if flip_lr:\n        a = a[..., :, ::-1]\n    if flip_ud:\n        a = a[..., ::-1, :]\n    return np.ascontiguousarray(a)\n\n\ndef _unorient(a: np.ndarray, flip_lr: bool, flip_ud: bool, transpose: bool) -> np.ndarray:\n    """Undo :func:`_orient` on a 2-D map -- the same steps, in reverse order."""\n    if flip_ud:\n        a = a[::-1, :]\n    if flip_lr:\n        a = a[:, ::-1]\n    if transpose:\n        a = a.T\n    return a\n\n\n@torch.no_grad()\ndef tiled_predict_tta(\n    model: nn.Module, x: np.ndarray, tta: str = "none", **tiled_kwargs\n) -> np.ndarray:\n    """``tiled_predict`` averaged over mirrored and rotated copies of the image.\n\n    Each copy is predicted, turned back to the original orientation, and the\n    logits are averaged. The prediction for a thin filament then no longer\n    depends on which way the tiles happened to cut across it. Both input\n    channels -- intensity and the radius map -- are transformed together, so the\n    model always sees a geometrically consistent image.\n\n    Averaging makes logits less extreme than a single pass, so a threshold tuned\n    without TTA is not automatically right with it.\n    """\n    orientations = TTA_MODES[tta]\n    total: np.ndarray | None = None\n    for orientation in orientations:\n        logits = tiled_predict(model, _orient(x, *orientation), **tiled_kwargs)\n        logits = _unorient(logits, *orientation)\n        total = logits.copy() if total is None else total + logits\n    return total / len(orientations)\n\n\ndef load_trained(path: str, device: torch.device) -> nn.Module:\n    """Rebuild a trained model from a checkpoint written by ``scripts/train.py``.\n\n    The architecture is read from the checkpoint itself, and the pretrained\n    ImageNet weights are skipped, since the trained weights replace them at once.\n    """\n    checkpoint = torch.load(path, map_location=device)\n    model = build_model(\n        encoder=checkpoint.get("encoder", "resnet34"),\n        in_channels=checkpoint.get("in_channels", 2),\n        weights=None,\n    ).to(device)\n    model.load_state_dict(checkpoint["model"])\n    model.eval()\n    return model\n\n\n@torch.no_grad()\ndef ensemble_predict(\n    models: list[nn.Module], x: np.ndarray, tta: str = "none", **tiled_kwargs\n) -> np.ndarray:\n    """Average the logits of several models, each with the same TTA applied.\n\n    An ensemble helps when its members make different mistakes. Here the model\n    trained on single-annotator labels and the one trained on the annotators\'\n    consensus each came out ahead on about half of the held-out validation\n    images, which is exactly the situation averaging exploits. With one model\n    this is simply ``tiled_predict_tta``.\n    """\n    total: np.ndarray | None = None\n    for model in models:\n        logits = tiled_predict_tta(model, x, tta=tta, **tiled_kwargs)\n        total = logits if total is None else total + logits\n    return total / len(models)\n\n', 'filament_seg/postprocess.py': '"""Turning a binary probability/threshold map into scored filament instances.\n\nThis is where Panoptic Quality is won or lost. The organisers name structural\ncontinuity as challenge #3: algorithms tend to shatter one filament into a\ncluster of dark "islands". Under PQ a shattered filament is one false negative\n*plus* several false positives, so fragmentation is punished roughly three times\nover while pixel accuracy barely moves.\n\nThe pipeline here is deliberately ordered:\n\n``threshold`` (probability map -> binary) -> ``open`` (kill speckle) ->\n``close`` (rejoin thin necks) -> ``bridge`` (join components separated by a\nsmall gap) -> ``area filter`` (drop instances too small to plausibly clear IoU\n0.5) -> ``confidence filter`` (drop instances the model is unsure of).\n\nEvery step is a tunable knob; tune them against local PQ, never against Dice.\n:func:`logits_to_instances` is the single entry point used by training\nvalidation, the post-processing sweep and the submission, so the three cannot\nquietly disagree about what a setting means.\n"""\n\nfrom __future__ import annotations\n\nimport dataclasses\nfrom dataclasses import dataclass\nfrom typing import TypeVar\n\nimport cv2\nimport numpy as np\n\n\n@dataclass\nclass PostprocessParams:\n    """Knobs for logits -> instances. Defaults are a starting point, not tuned."""\n\n    #: Probability above which a pixel counts as filament.\n    threshold: float = 0.5\n    open_radius: int = 2\n    close_radius: int = 5\n    #: Components whose gap is below this many pixels are merged into one\n    #: instance. Directly targets the fragmentation penalty.\n    bridge_gap: int = 12\n    #: Instances smaller than this (in pixels, at 2048x2048) are dropped.\n    #: A doubtful instance costs 1.0 in the PQ denominator; silence costs 0.5.\n    min_area: int = 400\n    #: Fill interior holes so a filament is one solid blob. Not swept: an\n    #: unfilled ring can only lower IoU against the solid annotated polygons.\n    fill_holes: bool = True\n    #: Instances whose mean predicted probability is below this are dropped;\n    #: 0 keeps everything. A prediction that misses costs 0.5 in the PQ\n    #: denominator for nothing, so a faint, uncertain component is often worth\n    #: more unsubmitted.\n    min_confidence: float = 0.0\n\n\n_Settings = TypeVar("_Settings")\n\n\ndef parse_settings(text: str, defaults: _Settings) -> _Settings:\n    """``"threshold=0.7,min_area=400"`` -> a copy of ``defaults`` with those fields set.\n\n    Values are converted to each field\'s type; an unknown name raises\n    ``ValueError`` rather than being silently ignored.\n    """\n    values: dict = {}\n    for item in filter(None, (part.strip() for part in text.split(","))):\n        key, _, raw = item.partition("=")\n        key = key.strip()\n        if not hasattr(defaults, key):\n            raise ValueError(f"unknown setting {key!r} for {type(defaults).__name__}")\n        kind = type(getattr(defaults, key))\n        values[key] = raw.strip().lower() in ("1", "true", "yes") if kind is bool else kind(raw)\n    return dataclasses.replace(defaults, **values)\n\n\ndef probability_to_logit(p: float) -> float:\n    """The logit threshold equivalent to probability ``p`` (clamped at 0 and 1)."""\n    if p <= 0.0:\n        return -1e9\n    if p >= 1.0:\n        return 1e9\n    return float(np.log(p / (1.0 - p)))\n\n\ndef _ellipse(radius: int) -> np.ndarray:\n    size = max(1, 2 * radius + 1)\n    return cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (size, size))\n\n\ndef clean_binary(binary: np.ndarray, params: PostprocessParams) -> np.ndarray:\n    """Morphological cleanup of a binary mask (uint8, 0/1)."""\n    out = (binary > 0).astype(np.uint8)\n    if params.open_radius > 0:\n        out = cv2.morphologyEx(out, cv2.MORPH_OPEN, _ellipse(params.open_radius))\n    if params.close_radius > 0:\n        out = cv2.morphologyEx(out, cv2.MORPH_CLOSE, _ellipse(params.close_radius))\n    return out\n\n\ndef fill_holes(binary: np.ndarray) -> np.ndarray:\n    """Fill enclosed background regions, leaving the outer background intact."""\n    mask = (binary > 0).astype(np.uint8)\n    flood = mask.copy()\n    h, w = mask.shape\n    pad = np.zeros((h + 2, w + 2), np.uint8)\n    cv2.floodFill(flood, pad, (0, 0), 1)\n    return (mask | (1 - flood)).astype(np.uint8)\n\n\ndef bridge_and_label(binary: np.ndarray, gap: int) -> np.ndarray:\n    """Label connected components, merging any that lie within ``gap`` pixels.\n\n    Dilating by ``gap/2`` and labelling *that* lets nearby fragments share a\n    label; the labels are then mapped back onto the original (undilated) pixels,\n    so instance shapes are unchanged -- only their grouping is.\n    """\n    mask = (binary > 0).astype(np.uint8)\n    if gap <= 0:\n        _, labels = cv2.connectedComponents(mask, connectivity=8)\n        return labels\n\n    dilated = cv2.dilate(mask, _ellipse(max(1, gap // 2)))\n    _, group_labels = cv2.connectedComponents(dilated, connectivity=8)\n    return np.where(mask > 0, group_labels, 0).astype(np.int32)\n\n\ndef filter_by_area(labels: np.ndarray, min_area: int) -> np.ndarray:\n    """Drop instances below ``min_area`` and relabel consecutively from 1.\n\n    One counting pass and a lookup table, rather than a full-frame comparison\n    per instance: the post-processing sweep calls this thousands of times.\n    """\n    counts = np.bincount(labels.ravel())\n    keep = counts >= min_area\n    keep[0] = False\n    lookup = np.zeros(counts.size, dtype=np.int32)\n    lookup[keep] = np.arange(1, int(keep.sum()) + 1, dtype=np.int32)\n    return lookup[labels]\n\n\ndef binary_to_instances(\n    binary: np.ndarray,\n    params: PostprocessParams | None = None,\n    restrict_to: np.ndarray | None = None,\n) -> np.ndarray:\n    """Full binary -> integer label map pipeline.\n\n    ``restrict_to`` is typically the solar disk mask: anything outside it cannot\n    be a filament and is discarded before instances are formed.\n    """\n    params = params or PostprocessParams()\n    mask = (binary > 0).astype(np.uint8)\n    if restrict_to is not None:\n        mask = (mask & restrict_to.astype(np.uint8)).astype(np.uint8)\n\n    mask = clean_binary(mask, params)\n    if params.fill_holes:\n        mask = fill_holes(mask)\n    labels = bridge_and_label(mask, params.bridge_gap)\n    return filter_by_area(labels, params.min_area)\n\n\ndef drop_low_confidence(\n    labels: np.ndarray, logits: np.ndarray, min_confidence: float\n) -> np.ndarray:\n    """Drop instances whose mean probability is below ``min_confidence``.\n\n    Survivors are relabelled consecutively from 1, as ``filter_by_area`` does.\n    Only instance pixels are passed through the sigmoid, so this stays cheap\n    on a 2048x2048 map.\n    """\n    if min_confidence <= 0.0:\n        return labels\n    inside = labels > 0\n    if not inside.any():\n        return labels\n    ids = labels[inside]\n    probs = 1.0 / (1.0 + np.exp(-logits[inside].astype(np.float64)))\n    n = int(labels.max()) + 1\n    sums = np.bincount(ids, weights=probs, minlength=n)\n    counts = np.bincount(ids, minlength=n)\n    keep = counts > 0\n    keep[keep] = sums[keep] / counts[keep] >= min_confidence\n    keep[0] = False\n    lookup = np.zeros(n, dtype=np.int32)\n    lookup[keep] = np.arange(1, int(keep.sum()) + 1, dtype=np.int32)\n    return lookup[labels]\n\n\ndef logits_to_instances(\n    logits: np.ndarray,\n    disk_mask: np.ndarray,\n    params: PostprocessParams | None = None,\n) -> np.ndarray:\n    """Model logits -> integer instance label map, restricted to the disk."""\n    params = params or PostprocessParams()\n    binary = ((logits > probability_to_logit(params.threshold)) & disk_mask).astype(np.uint8)\n    labels = binary_to_instances(binary, params, restrict_to=disk_mask)\n    return drop_low_confidence(labels, logits, params.min_confidence)\n\n', 'filament_seg/rle.py': '"""Conversion between masks, COCO polygons and the competition\'s RLE format.\n\nThe submission asks for *RLE counts only* -- one row per predicted filament,\nno ``size`` field, no quoting. The size is fixed at 2048x2048 for every image.\n\nTwo things bite people here:\n\n1. ``pycocotools`` expects Fortran-ordered (column-major) arrays. Passing a\n   C-ordered mask silently produces a transposed encoding that scores ~0.\n2. ``counts`` comes back as ``bytes``; it must be decoded to ``str`` before it\n   reaches the CSV.\n\nThe COCO counts alphabet spans ASCII 48..111 (``0``..``o``), which contains no\ncomma, quote or newline -- so a plain ``to_csv`` never adds quoting. We assert\nthis rather than trust it.\n"""\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\nfrom typing import Iterable, Sequence\n\nimport numpy as np\nimport pandas as pd\nfrom pycocotools import mask as mask_utils\n\nfrom .config import IMAGE_HEIGHT, IMAGE_SIZE, IMAGE_WIDTH\n\n#: Characters that would force CSV quoting and therefore corrupt a submission.\n_FORBIDDEN_IN_COUNTS = set(\',"\\\'\\r\\n\')\n\nRle = dict  # {"size": [h, w], "counts": bytes}\n\n\ndef mask_to_rle(mask: np.ndarray) -> Rle:\n    """Encode a boolean/0-1 mask as a COCO RLE dict."""\n    if mask.dtype != np.uint8:\n        mask = mask.astype(np.uint8)\n    return mask_utils.encode(np.asfortranarray(mask))\n\n\ndef rle_to_mask(rle: Rle) -> np.ndarray:\n    """Decode a COCO RLE dict to a boolean mask."""\n    return mask_utils.decode(rle).astype(bool)\n\n\ndef mask_to_counts(mask: np.ndarray) -> str:\n    """Encode a mask to the bare counts string expected in the submission."""\n    return counts_of(mask_to_rle(mask))\n\n\ndef counts_of(rle: Rle) -> str:\n    counts = rle["counts"]\n    if isinstance(counts, bytes):\n        counts = counts.decode("ascii")\n    return counts\n\n\ndef rle_from_counts(\n    counts: str, height: int = IMAGE_HEIGHT, width: int = IMAGE_WIDTH\n) -> Rle:\n    """Rebuild a full RLE dict from a bare counts string."""\n    return {"size": [height, width], "counts": counts.encode("ascii")}\n\n\ndef counts_to_mask(\n    counts: str, height: int = IMAGE_HEIGHT, width: int = IMAGE_WIDTH\n) -> np.ndarray:\n    return rle_to_mask(rle_from_counts(counts, height, width))\n\n\ndef polygon_to_rle(\n    polygons: Sequence[Sequence[float]],\n    height: int = IMAGE_HEIGHT,\n    width: int = IMAGE_WIDTH,\n) -> Rle:\n    """Convert a COCO ``segmentation`` field (list of flat polygons) to one RLE.\n\n    MAGFiLO stores exactly one single-piece polygon per filament, but merging\n    handles the general case without extra cost.\n    """\n    rles = mask_utils.frPyObjects([list(map(float, p)) for p in polygons], height, width)\n    return mask_utils.merge(rles)\n\n\ndef labels_to_rles(labels: np.ndarray) -> list[Rle]:\n    """Split an integer label map (0 = background) into one RLE per instance."""\n    out: list[Rle] = []\n    for value in np.unique(labels):\n        if value == 0:\n            continue\n        out.append(mask_to_rle(labels == value))\n    return out\n\n\ndef rle_areas(rles: Sequence[Rle]) -> np.ndarray:\n    if not rles:\n        return np.zeros(0, dtype=np.float64)\n    rles = list(rles)\n    # Under NumPy 2, pycocotools\' area() overflows a uint8 counter past 255\n    # masks in one call, so the masks are measured in chunks of at most 255.\n    return np.concatenate([np.asarray(mask_utils.area(rles[i : i + 255]), dtype=np.float64)\n                           for i in range(0, len(rles), 255)])\n\n\ndef rle_intersection_area(a: Rle, b: Rle) -> float:\n    """Exact intersection area of two RLEs, without decoding either to a mask."""\n    return float(mask_utils.area(mask_utils.merge([a, b], intersect=1)))\n\n\ndef iou_matrix(gt_rles: Sequence[Rle], pred_rles: Sequence[Rle]) -> np.ndarray:\n    """Pairwise IoU with shape ``(n_gt, n_pred)``.\n\n    Computed in C by pycocotools, so it stays cheap even at 2048x2048.\n    """\n    if not gt_rles or not pred_rles:\n        return np.zeros((len(gt_rles), len(pred_rles)), dtype=np.float64)\n    # mask_utils.iou(dt, gt, iscrowd) -> (n_dt, n_gt); transpose to (n_gt, n_pred).\n    ious = mask_utils.iou(list(pred_rles), list(gt_rles), [0] * len(gt_rles))\n    return np.asarray(ious, dtype=np.float64).T\n\n\n# --- Submission I/O ---------------------------------------------------------\n\n\ndef make_filament_id(image_stem: str, index: int) -> str:\n    """Build a submission id, e.g. ``20150125172714Mh_1`` (1-based index)."""\n    return f"{image_stem}_{index}"\n\n\ndef validate_counts(counts: str) -> None:\n    bad = _FORBIDDEN_IN_COUNTS.intersection(counts)\n    if bad:\n        raise ValueError(f"RLE counts contain CSV-hostile characters: {sorted(bad)}")\n\n\ndef build_submission(\n    predictions: dict[str, Sequence[Rle]],\n    validate: bool = True,\n) -> pd.DataFrame:\n    """Turn ``{image_stem: [rle, ...]}`` into the submission DataFrame.\n\n    Images with no predicted filament simply contribute no rows.\n    """\n    rows: list[dict[str, str]] = []\n    for stem in sorted(predictions):\n        for i, rle in enumerate(predictions[stem], start=1):\n            counts = counts_of(rle)\n            if validate:\n                validate_counts(counts)\n            rows.append(\n                {"filament_id": make_filament_id(stem, i), "segmentation_rle": counts}\n            )\n    frame = pd.DataFrame(rows, columns=["filament_id", "segmentation_rle"])\n    if frame["filament_id"].duplicated().any():\n        raise ValueError("duplicate filament_id values in submission")\n    return frame\n\n\ndef write_submission(frame: pd.DataFrame, path: str | Path) -> Path:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    frame.to_csv(path, index=False)\n    return path\n\n\ndef read_submission(path: str | Path) -> dict[str, list[Rle]]:\n    """Read a submission CSV back into ``{image_stem: [rle, ...]}``."""\n    frame = pd.read_csv(path, dtype=str).fillna("")\n    grouped: dict[str, list[Rle]] = {}\n    for filament_id, counts in zip(frame["filament_id"], frame["segmentation_rle"]):\n        stem = filament_id.rsplit("_", 1)[0]\n        grouped.setdefault(stem, []).append(rle_from_counts(counts, *IMAGE_SIZE))\n    return grouped\n\n\ndef iter_masks(rles: Iterable[Rle]) -> Iterable[np.ndarray]:\n    for rle in rles:\n        yield rle_to_mask(rle)\n\n', 'filament_seg/scoring.py': '"""Pooled Panoptic Quality totals and the paired bootstrap that compares them.\n\nThe competition pools TP, FP, FN and the matched IoU sum over every annotator\'s\nview of every observation. Keeping those six totals per observation, rather\nthan a per-observation PQ, is what lets comparisons resample observations and\nstill compute the pooled score exactly.\n"""\n\nfrom __future__ import annotations\n\nfrom typing import Sequence\n\nimport numpy as np\n\nfrom .metrics import evaluate_image\nfrom .rle import Rle\n\n#: Per-observation totals, in this order, summed over the observation\'s views.\nCOUNT_FIELDS = ("n_gt", "n_pred", "tp", "fp", "fn", "iou_sum")\n\n\ndef view_totals(views: Sequence[str], gt: dict[str, list[Rle]], pred_rles: list[Rle]) -> np.ndarray:\n    """One observation\'s totals: its predictions scored against each view, summed."""\n    out = np.zeros(len(COUNT_FIELDS), dtype=np.float64)\n    for image_id in views:\n        r = evaluate_image(image_id, gt[image_id], pred_rles, with_fragmentation=False)\n        out += (r.n_gt, r.n_pred, r.tp, r.fp, r.fn, r.iou_sum)\n    return out\n\n\ndef pq_of(totals: np.ndarray) -> np.ndarray:\n    """Pooled PQ from summed totals; works on any leading shape."""\n    tp, fp, fn, iou_sum = (totals[..., COUNT_FIELDS.index(k)] for k in ("tp", "fp", "fn", "iou_sum"))\n    denominator = tp + 0.5 * fp + 0.5 * fn\n    return np.divide(iou_sum, denominator, out=np.zeros_like(iou_sum), where=denominator > 0)\n\n\ndef paired_bootstrap(\n    per_stem_a: np.ndarray, per_stem_b: np.ndarray, n_boot: int = 2000, seed: int = 0\n) -> tuple[float, float, float]:\n    """PQ(a) - PQ(b) on all observations, with a 95% interval from resampling them."""\n    deltas = bootstrap_deltas(per_stem_a, per_stem_b, n_boot, seed)\n    full = float(pq_of(per_stem_a.sum(axis=0)) - pq_of(per_stem_b.sum(axis=0)))\n    low, high = np.percentile(deltas, [2.5, 97.5])\n    return full, float(low), float(high)\n\n\ndef bootstrap_deltas(\n    per_stem_a: np.ndarray, per_stem_b: np.ndarray, n_boot: int = 2000, seed: int = 0\n) -> np.ndarray:\n    """PQ(a) - PQ(b) on ``n_boot`` resamples of the observations.\n\n    Observations, not views, are resampled -- an observation\'s views share one\n    set of predictions -- and both settings see the same resample each time.\n    """\n    n = per_stem_a.shape[0]\n    weights = np.random.default_rng(seed).multinomial(n, np.full(n, 1.0 / n), size=n_boot)\n    # einsum rather than `@`: macOS\'s Accelerate BLAS raises spurious\n    # floating-point warnings in matmul, and this is far too small to need BLAS.\n    return (pq_of(np.einsum("bs,sk->bk", weights, per_stem_a))\n            - pq_of(np.einsum("bs,sk->bk", weights, per_stem_b)))\n\n', 'legacy_v14/__init__.py': '"""Read-only V1.4 inference and selection."""\n', 'legacy_v14/selection.py': '"""Build a candidate bank, learn expected matched IoU, select disjoint masks."""\n\nfrom dataclasses import dataclass\n\nimport cv2\n\nimport numpy as np\n\nfrom pycocotools import mask as cm\n\nfrom sklearn.ensemble import HistGradientBoostingRegressor\n\nfrom threadpoolctl import threadpool_limits\n\nfrom solar_v1.inference import native_coarse\n\nfrom solar_v1.postprocess import Instance, encode\n\nfrom solar_v1.pq import pairwise_iou\n\nfrom solar_v12a.features import clean, disk_mask\n\nLEVELS = (.35, .50, .65)\n\nTAUS = (.16, .20, .24, .28, .32)\n\nBEAM_WIDTH = 24\n\nFEATURES = (\n    \'base\', \'coarse\', \'refined\', \'semantic\', \'det_score\', \'coarse_refined_iou\',\n    \'threshold\', \'log_area\', \'extent\', \'elongation\', \'radial\', \'contrast\',\n    \'semantic_mean\', \'semantic_q20\', \'semantic_peak\', \'best_detector_iou\',\n    \'best_detector_support\', \'best_semantic_iou\', \'alternatives\',\n)\n\n@dataclass\nclass Candidate:\n    rle: dict\n    sources: set\n    det: float = 0.\n    pair: float = 0.\n    level: float = 0.\n\ndef add_candidate(bank, rle, source, det=0., pair=0., level=0.):\n    key = rle[\'counts\']\n    if key not in bank:\n        bank[key] = Candidate(rle, set())\n    entry = bank[key]\n    entry.sources.add(source)\n    entry.det = max(entry.det, det)\n    entry.pair = max(entry.pair, pair)\n    entry.level = max(entry.level, level)\n\ndef detector_candidates(bank, pack, data, c, disk):\n    shape = data[\'raw\'].shape\n    for i, score in enumerate(pack[\'scores\']):\n        if score < .35:\n            continue\n        coarse = clean(native_coarse(pack, i) >= c.MASK_THRESHOLD, disk, c)\n        x0, y0, x1, y1 = map(int, pack[\'ref_bounds\'][i])\n        refined = np.zeros(shape, dtype=bool)\n        refined[y0:y1, x0:x1] = cv2.resize(\n            pack[\'ref_probs\'][i], (x1-x0, y1-y0), interpolation=cv2.INTER_LINEAR\n        ) >= c.REFINER_THRESHOLD\n        refined = clean(refined, disk, c)\n        pair = float(np.count_nonzero(coarse & refined) / max(np.count_nonzero(coarse | refined), 1))\n        for source, mask in [(\'coarse\', coarse), (\'refined\', refined)]:\n            if np.count_nonzero(mask) >= c.MIN_INSTANCE_AREA:\n                add_candidate(bank, encode(mask), source, float(score), pair)\n\ndef semantic_candidates(bank, probability, disk, min_area):\n    shape = probability.shape\n    for level in LEVELS:\n        _, labels, stats, _ = cv2.connectedComponentsWithStats(\n            ((probability >= level) & disk).astype(np.uint8), connectivity=8,\n        )\n        indices = [i for i in range(1, len(stats)) if min_area <= stats[i, 4] <= 100_000]\n        if len(indices) > 64:\n            # Rank by the peak semantic support, not by size, to retain small targets.\n            peaks = {}\n            for i in indices:\n                x, y, w, h, _ = stats[i]\n                region = labels[y:y+h, x:x+w] == i\n                peaks[i] = float(probability[y:y+h, x:x+w][region].max())\n            indices = sorted(indices, key=lambda i: (-peaks[i], i))[:64]\n        canvas = np.zeros(shape, np.uint8)\n        for i in indices:\n            x, y, w, h, _ = stats[i]\n            roi = canvas[y:y+h, x:x+w]\n            roi[:] = labels[y:y+h, x:x+w] == i\n            add_candidate(bank, cm.encode(np.asfortranarray(canvas)), \'semantic\', level=level)\n            roi.fill(0)\n\ndef geometry(candidate, probability, data):\n    x, y, w, h = cm.toBbox(candidate.rle).astype(int)\n    x0, y0 = max(x-4, 0), max(y-4, 0)\n    x1 = min(x+w+4, probability.shape[1])\n    y1 = min(y+h+4, probability.shape[0])\n    mask = cm.decode(candidate.rle)[y0:y1, x0:x1]\n    ys, xs = np.nonzero(mask)\n    area = len(xs)\n    cov = np.cov(np.stack([xs, ys]), bias=True) if area > 1 else np.zeros((2, 2))\n    lo, hi = np.maximum(np.linalg.eigvalsh(cov), 0)\n    cx, cy, radius = data[\'disk\']\n    ring = cv2.dilate(mask, np.ones((7, 7), np.uint8)).astype(bool) & ~mask.astype(bool)\n    corrected = data[\'corrected\'][y0:y1, x0:x1]\n    region = mask.astype(bool)\n    contrast = (corrected[ring].mean() - corrected[region].mean()) / 255 if ring.any() else 0.\n    prob = probability[y0:y1, x0:x1][region]\n    return [np.log1p(area), area/(w*h), np.sqrt((hi+1)/(lo+1)),\n            np.hypot(x0+xs.mean()-cx, y0+ys.mean()-cy)/max(radius, 1),\n            contrast, prob.mean(), np.quantile(prob, .2), prob.max()]\n\n@dataclass\nclass PhotoBank:\n    rles: list\n    features: np.ndarray\n    conflicts: list\n    sources: list\n\ndef build_bank(pack, baseline, probability, data, config):\n    bank = {}\n    for ins in baseline:\n        add_candidate(bank, ins.rle, \'base\', ins.score)\n    disk = disk_mask(data, config)\n    if disk is None:\n        disk = np.ones(probability.shape, bool)\n    detector_candidates(bank, pack, data, config, disk)\n    semantic_candidates(bank, probability, disk, config.MIN_INSTANCE_AREA)\n    entries = list(bank.values())\n    rles = [c.rle for c in entries]\n    if not rles:\n        return PhotoBank([], np.empty((0, len(FEATURES)), np.float32), [], [])\n    ious = pairwise_iou(rles, rles)\n    det = np.array([e.det for e in entries])\n    sem = np.array([\'semantic\' in e.sources for e in entries])\n    features, conflicts = [], []\n    for i, e in enumerate(entries):\n        row = ious[i]\n        features.append([*[float(k in e.sources) for k in (\'base\', \'coarse\', \'refined\', \'semantic\')],\n                         e.det, e.pair, e.level, *geometry(e, probability, data),\n                         row[det > 0].max(initial=0), (row*det).max(initial=0),\n                         row[sem].max(initial=0), np.count_nonzero(row > .25)-1])\n        conflicts.append(sum(1 << int(j) for j in np.flatnonzero(row > 0) if j != i))\n    x = np.asarray(features, np.float32)\n    if x.shape != (len(rles), len(FEATURES)) or not np.isfinite(x).all():\n        raise ValueError(\'Candidate feature schema is invalid.\')\n    return PhotoBank(rles, x, conflicts, [sorted(e.sources) for e in entries])\n\ndef target_values(bank, views):\n    if not views:\n        raise ValueError(\'Candidate targets require an annotator view.\')\n    total = np.zeros(len(bank.rles), np.float64)\n    for gt in views.values():\n        best = pairwise_iou(gt, bank.rles).max(axis=0, initial=0)\n        total += np.where(best > .5, best, 0.)\n    return total/len(views)\n\ndef predict_quality(model, bank):\n    if not bank.rles:\n        return np.empty(0)\n    with threadpool_limits(limits=2):\n        quality = model.predict(bank.features).clip(0, 1)\n    if not np.isfinite(quality).all():\n        raise ValueError(\'Candidate quality contains a nonfinite value.\')\n    return quality\n\ndef bits(mask):\n    while mask:\n        low = mask & -mask\n        yield low.bit_length()-1\n        mask ^= low\n\ndef select(bank, quality, tau):\n    # A mask has positive utility only when its expected matched IoU clears tau.\n    gain = quality - tau\n    remaining = sum(1 << int(i) for i in np.flatnonzero(gain > 0))\n    chosen = []\n    while remaining:\n        frontier = remaining & -remaining\n        component = 0\n        while frontier:\n            component |= frontier\n            neighbours = 0\n            for i in bits(frontier):\n                neighbours |= bank.conflicts[i]\n            frontier = neighbours & remaining & ~component\n        remaining &= ~component\n        members = sorted(bits(component), key=lambda i: (-gain[i], i))\n        # Keep competing whole-mask vs split-mask solutions. Search is bounded for large groups.\n        states = [(0., 0, 0)]\n        for i in members:\n            bit = 1 << i\n            states += [(value+gain[i], picked|bit, blocked|bank.conflicts[i])\n                       for value, picked, blocked in states if not blocked & bit]\n            states = sorted(states, key=lambda s: (-s[0], s[1]))[:BEAM_WIDTH]\n        chosen.extend(bits(states[0][1]))\n    final = [Instance(float(quality[i]), bank.rles[i], source=\'+\'.join(bank.sources[i])) for i in chosen]\n    return sorted(final, key=lambda ins: tuple(cm.toBbox(ins.rle)[[1, 0]]))\n', 'legacy_v14/semantic.py': "import cv2\n\nimport numpy as np\n\nimport torch\n\nfrom pycocotools import mask as cm\n\nfrom torch import nn\n\nfrom torch.nn import functional as F\n\nfrom torchvision.models import ResNet34_Weights, resnet34\n\nfrom solar_v1.inference import smooth_window, tile_starts\n\nfrom solar_v1.preprocess import features, read_gray\n\nMODE = 'raw_corrected_radial'\n\nSIDE = 512\n\nOVERLAP = 128\n\nclass DecoderBlock(nn.Sequential):\n    def __init__(self, n_in, n_out):\n        super().__init__(\n            nn.Conv2d(n_in, n_out, 3, padding=1, bias=False),\n            nn.GroupNorm(8, n_out), nn.ReLU(inplace=True),\n        )\n\nclass SolarUNet(nn.Module):\n    def __init__(self, pretrained=False):\n        super().__init__()\n        self.encoder = resnet34(weights=ResNet34_Weights.DEFAULT if pretrained else None)\n        self.encoder.fc = nn.Identity()\n        self.decoder = nn.ModuleList([\n            DecoderBlock(512+256, 192), DecoderBlock(192+128, 96),\n            DecoderBlock(96+64, 48), DecoderBlock(48+64, 32),\n        ])\n        self.head = nn.Sequential(DecoderBlock(32+3, 16), nn.Conv2d(16, 1, 1))\n        self.register_buffer('mean', torch.tensor([.485, .456, .5])[None, :, None, None])\n        self.register_buffer('std', torch.tensor([.229, .224, .25])[None, :, None, None])\n\n    def forward(self, image):\n        x = (image - self.mean) / self.std\n        e = self.encoder\n        s0 = e.relu(e.bn1(e.conv1(x)))\n        s1 = e.layer1(e.maxpool(s0))\n        s2 = e.layer2(s1)\n        s3 = e.layer3(s2)\n        y = e.layer4(s3)\n        for block, skip in zip(self.decoder, (s3, s2, s1, s0)):\n            y = block(torch.cat([F.interpolate(y, size=skip.shape[-2:], mode='bilinear',\n                                                align_corners=False), skip], dim=1))\n        y = F.interpolate(y, size=x.shape[-2:], mode='bilinear', align_corners=False)\n        return self.head(torch.cat([y, x], dim=1))\n\n@torch.inference_mode()\ndef semantic_map(model, data, device, side=SIDE, overlap=OVERLAP):\n    h, w = data['raw'].shape\n    bounds = [(x, y, x+side, y+side) for y in tile_starts(h, side, overlap)\n              for x in tile_starts(w, side, overlap)]\n    weight = smooth_window(side)\n    result, coverage = np.zeros((h, w), np.float32), np.zeros((h, w), np.float32)\n    for start in range(0, len(bounds), 2):\n        part = bounds[start:start+2]\n        inp = torch.from_numpy(np.stack([features(data, MODE, b) for b in part])).to(device)\n        logits = model(inp).float()\n        if not torch.isfinite(logits).all():\n            raise RuntimeError('Nonfinite semantic logits during tile inference.')\n        prob = logits.sigmoid().cpu().numpy()[:, 0]\n        for p, (x0, y0, x1, y1) in zip(prob, part):\n            result[y0:y1, x0:x1] += weight*p\n            coverage[y0:y1, x0:x1] += weight\n    if np.any(coverage <= 0) or not np.isfinite(result).all():\n        raise RuntimeError('Tile fusion produced uncovered pixels or nonfinite probabilities.')\n    return result / coverage\n", 'solar_v1/__init__.py': '"""Solar Filament Segmentation V1.0."""\n__version__ = \'1.0.0\'\n', 'solar_v1/config.py': "from dataclasses import asdict, dataclass, field\nfrom pathlib import Path\nimport hashlib\nimport json\nimport os\nimport random\nimport numpy as np\nimport torch\n\n\n@dataclass\nclass Config:\n    RUN_MODE: str = 'train_calibrate_submit'\n    INPUT_ROOT: str = '/kaggle/input'\n    OUTPUT_ROOT: str = '/kaggle/working/solar_v1'\n    ANNOTATION_PATH: str = ''\n    TRAIN_IMAGE_DIR: str = ''\n    TEST_IMAGE_DIR: str = ''\n    SEED: int = 2026\n    N_FOLDS: int = 5\n    FOLD_INDEX: int = 0\n    INPUT_MODE: str = 'raw_corrected_radial'\n    ANNOTATION_MODE: str = 'all_annotators'\n    PREPROCESS: bool = True\n    DISK_MARGIN: float = 1.01\n    DETECTOR_IMAGE_SIZE: int = 1024\n    ANCHOR_SIZES: tuple = (16, 32, 64, 128, 256)\n    ANCHOR_RATIOS: tuple = (.5, 1., 2.)\n    DETECTOR_EPOCHS: int = 10\n    TRAIN_DETECTOR: bool = True\n    DETECTOR_LR: float = .005\n    DETECTOR_ACCUM_STEPS: int = 1\n    ENABLE_REFINER: bool = True\n    REFINER_SIZE: int = 384\n    REFINER_EPOCHS: int = 8\n    TRAIN_REFINER: bool = True\n    REFINER_BATCH_SIZE: int = 4\n    REFINER_ACCUM_STEPS: int = 2\n    REFINER_USE_COARSE: bool = True\n    REFINER_USE_RADIAL: bool = True\n    REFINER_CONTEXT: float = 1.8\n    REFINER_MIN_CROP: int = 96\n    REFINER_CROPS_PER_PHOTO: int = 8\n    REFINER_BOX_JITTER: float = .12\n    EMPTY_REFINER_FALLBACK: bool = True\n    SEGMENTER_LR: float = .0003\n    TVERSKY_ALPHA: float = .3\n    TVERSKY_BETA: float = .7\n    ENABLE_TILE_BRANCH: bool = False\n    TILE_SIZE: int = 512\n    TILE_OVERLAP: int = 128\n    TILE_EPOCHS: int = 8\n    TRAIN_TILE: bool = True\n    TILE_BATCH_SIZE: int = 2\n    TILE_ACCUM_STEPS: int = 2\n    TILE_SAMPLES_PER_PHOTO: int = 4\n    TILE_FUSION_WEIGHT: float = .35\n    TILE_THRESHOLD: float = .65\n    TILE_GROW: int = 8\n    TILE_KEEP_UNCLAIMED: bool = False\n    TILE_UNCLAIMED_MIN_AREA: int = 64\n    DETECTION_THRESHOLD: float = .5\n    MASK_THRESHOLD: float = .5\n    REFINER_THRESHOLD: float = .65\n    MASK_NMS_IOU: float = .5\n    MIN_INSTANCE_AREA: int = 32\n    POSTPROCESS_MODE: str = 'mask_nms'\n    SOFT_NMS_SIGMA: float = .5\n    MORPH_CLOSE_RADIUS: int = 0\n    FRAGMENT_MODE: str = 'keep'\n    FRAGMENT_MIN_AREA: int = 8\n    CACHE_SCORE_FLOOR: float = .2\n    MAX_DETECTIONS: int = 80\n    INFERENCE_BATCH_SIZE: int = 8\n    NUM_WORKERS: int = 2\n    USE_DDP: bool = True\n    AMP: bool = True\n    VALIDATE_EVERY: int = 2\n    RESUME: bool = True\n    PRETRAINED: bool = True\n    ALLOW_WEIGHT_DOWNLOAD: bool = True\n    PRETRAINED_DIR: str = ''\n    CHECKPOINT_DIR: str = ''\n    RESUME_SOURCE: str = ''\n    REFIT_ALL_DATA: bool = False\n    CROSS_FIT_CALIBRATION: bool = True\n    CALIBRATION_ROUNDS: int = 2\n    CALIBRATION_GRID: dict = field(default_factory=lambda: {\n        'DETECTION_THRESHOLD': [.3, .5, .7], 'MASK_THRESHOLD': [.4, .5, .6],\n        'REFINER_THRESHOLD': [.5, .65, .8], 'MASK_NMS_IOU': [.3, .5, .7],\n        'MIN_INSTANCE_AREA': [16, 32, 64]})\n    ENSEMBLE_CACHE_DIRS: tuple = ()\n    ENSEMBLE_WEIGHTS: tuple = ()\n    ENSEMBLE_MATCH_IOU: float = .55\n    QUALITY_WEIGHT: float = 0.\n    MAX_CACHE_GB: float = 12.\n\n    def validate(self):\n        assert self.RUN_MODE in {'train_calibrate_submit','train','prepare','validate','calibrate','predict'}\n        assert self.INPUT_MODE in {'raw_repeat','corrected_repeat','raw_corrected_radial'}\n        assert self.ANNOTATION_MODE in {'all_annotators','random_annotator','max_count'}\n        assert self.N_FOLDS >= 2 and 0 <= self.FOLD_INDEX < self.N_FOLDS\n        assert self.DETECTOR_IMAGE_SIZE in (1024,1280)\n        assert self.REFINER_SIZE in (256,384,512)\n        assert self.TILE_SIZE % 32 == 0 and 0 <= self.TILE_OVERLAP < self.TILE_SIZE\n        assert len(self.ANCHOR_SIZES)==5 and min(self.ANCHOR_SIZES)>0 and min(self.ANCHOR_RATIOS)>0\n        assert self.POSTPROCESS_MODE in {'mask_nms','soft_nms','legacy_union'}\n        assert self.FRAGMENT_MODE in {'keep','remove_small','split'}\n        assert 0 <= self.QUALITY_WEIGHT <= 1 and self.MAX_CACHE_GB>0\n        assert self.CACHE_SCORE_FLOOR <= min(self.CALIBRATION_GRID['DETECTION_THRESHOLD'])\n        assert 0<=self.CACHE_SCORE_FLOOR<=self.DETECTION_THRESHOLD<=1\n        assert all(0<getattr(self,k)<=1 for k in ['MASK_THRESHOLD','REFINER_THRESHOLD','MASK_NMS_IOU','TILE_THRESHOLD'])\n        assert self.MIN_INSTANCE_AREA>=1 and self.FRAGMENT_MIN_AREA>=1 and self.SOFT_NMS_SIGMA>0\n        assert 0<=self.TILE_FUSION_WEIGHT<=1 and self.TILE_GROW>=0 and self.MORPH_CLOSE_RADIUS>=0\n        assert all(getattr(self,k)>=1 for k in ['DETECTOR_EPOCHS','REFINER_EPOCHS','TILE_EPOCHS',\n            'REFINER_BATCH_SIZE','TILE_BATCH_SIZE','INFERENCE_BATCH_SIZE','VALIDATE_EVERY',\n            'DETECTOR_ACCUM_STEPS','REFINER_ACCUM_STEPS','TILE_ACCUM_STEPS'])\n        return self\n\n    def save(self,path):\n        atomic_json(path,asdict(self))\n\n    @classmethod\n    def load(cls,path):\n        return cls(**json.loads(Path(path).read_text())).validate()\n\n\ndef atomic_json(path,value):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    tmp=path.with_name(path.name+'.tmp')\n    tmp.write_text(json.dumps(value,indent=2,ensure_ascii=False),encoding='utf-8')\n    os.replace(tmp,path)\n\n\ndef digest(value):\n    return hashlib.sha256(json.dumps(value,sort_keys=True,default=str).encode()).hexdigest()\n\n\ndef file_hash(path):\n    h=hashlib.sha256()\n    with Path(path).open('rb') as f:\n        for b in iter(lambda:f.read(4*1024*1024),b''):h.update(b)\n    return h.hexdigest()\n\n\ndef seed_all(seed):\n    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)\n    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)\n\n\ndef architecture(c,stage):\n    common={'version':'1.0.0','stage':stage}\n    if stage=='detector':\n        common.update(size=c.DETECTOR_IMAGE_SIZE,anchors=c.ANCHOR_SIZES,ratios=c.ANCHOR_RATIOS,\n                      input_mode=c.INPUT_MODE,preprocess=c.PREPROCESS,roi_pool=28)\n    elif stage=='refiner':\n        common.update(size=c.REFINER_SIZE,coarse=c.REFINER_USE_COARSE,radial=c.REFINER_USE_RADIAL)\n    else:common.update(size=c.TILE_SIZE,input_mode=c.INPUT_MODE,preprocess=c.PREPROCESS)\n    return digest(common)\n\n\ndef autocast(c,device):\n    return torch.amp.autocast(device.type,enabled=c.AMP and device.type=='cuda',dtype=torch.float16)\n", 'solar_v1/data.py': 'from pathlib import Path\nimport json\nimport cv2\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.utils.data import Dataset\nfrom pycocotools.coco import COCO\nfrom pycocotools import mask as cm\nfrom sklearn.model_selection import StratifiedGroupKFold\nfrom .config import atomic_json,digest,file_hash\nfrom .preprocess import SolarPreprocessor,features,radial_window,fit_normalization\n\n\ndef image_files(root):\n    return sorted(p for p in Path(root).rglob(\'*\') if p.suffix.lower() in {\'.jpg\',\'.jpeg\',\'.png\',\'.tif\',\'.tiff\'})\n\n\nclass Catalog:\n    def __init__(self,c):\n        self.c=c;root=Path(c.INPUT_ROOT)\n        matches=[Path(c.ANNOTATION_PATH)] if c.ANNOTATION_PATH else list(root.rglob(\'MAGFiLO_1.0_Annotations_kaggle2026_train.json\'))\n        if len(matches)!=1:raise ValueError(\'Set ANNOTATION_PATH: expected exactly one training JSON.\')\n        self.annotation=matches[0]\n        self.train_dir=Path(c.TRAIN_IMAGE_DIR) if c.TRAIN_IMAGE_DIR else self.annotation.parent/\'train_images\'\n        self.test_dir=Path(c.TEST_IMAGE_DIR) if c.TEST_IMAGE_DIR else self.annotation.parent.parent/\'test/test_images\'\n        self.coco=COCO(str(self.annotation))\n        self.actual_ids={str(k):k for k in self.coco.imgs}\n        self.photos={};self.records={}\n        for iid,im in self.coco.imgs.items():\n            rid=str(iid);stem=Path(im[\'file_name\']).stem\n            path=self.train_dir/im[\'file_name\']\n            if not path.is_file():path=self.train_dir/Path(im[\'file_name\']).name\n            if not path.is_file():raise FileNotFoundError(path)\n            if len(stem)<16 or not stem[:14].isdigit():raise ValueError(f\'Cannot parse observation date/site: {stem}\')\n            anns=self.coco.imgToAnns[iid]\n            self.records[rid]={\'id\':rid,\'photo\':stem,\'count\':len(anns),\'area\':sum(float(a.get(\'area\',0)) for a in anns)}\n            if stem not in self.photos:self.photos[stem]={\'stem\':stem,\'date\':stem[:8],\'site\':stem[-2:],\'path\':str(path),\'records\':[]}\n            elif self.photos[stem][\'path\']!=str(path):raise ValueError(\'Duplicate observation stem with different paths\')\n            self.photos[stem][\'records\'].append(rid)\n        self.test_paths=image_files(self.test_dir)\n        if len({p.stem for p in self.test_paths})!=len(self.test_paths):raise ValueError(\'Duplicate test photo stems\')\n        if set(self.photos)&{p.stem for p in self.test_paths}:raise ValueError(\'Train/test photo ids overlap\')\n        self.hash=file_hash(self.annotation)\n\n    def rles(self,rid):\n        anns=self.coco.imgToAnns[self.actual_ids[str(rid)]]\n        return [self.coco.annToRLE(a) for a in anns if float(cm.area(self.coco.annToRLE(a)))>0]\n\n    def masks(self,rid,size=None,bounds=None):\n        result=[]\n        for rle in self.rles(rid):\n            m=cm.decode(rle)\n            if bounds is not None:\n                x0,y0,x1,y1=bounds;m=m[y0:y1,x0:x1]\n            if size is not None:m=cv2.resize(m,(size,size),interpolation=cv2.INTER_NEAREST)\n            if m.any():result.append(m)\n        if size is not None:shape=(size,size)\n        elif bounds is not None:shape=(bounds[3]-bounds[1],bounds[2]-bounds[0])\n        else:\n            im=self.coco.imgs[self.actual_ids[str(rid)]];shape=(im[\'height\'],im[\'width\'])\n        return np.stack(result).astype(np.uint8) if result else np.zeros((0,*shape),np.uint8)\n\n    def views(self,stem):return {rid:self.rles(rid) for rid in self.photos[stem][\'records\']}\n\n\ndef prepare(c):\n    out=Path(c.OUTPUT_ROOT);out.mkdir(parents=True,exist_ok=True)\n    cat=Catalog(c);pre=SolarPreprocessor(out/\'preprocessed\',c.PREPROCESS)\n    photos=[cat.photos[s].copy() for s in sorted(cat.photos)]\n    # Union dates connected by duplicate image pixels, so renamed copies also stay together.\n    parent={p[\'date\']:p[\'date\'] for p in photos};seen={}\n    def find(x):\n        while parent[x]!=x:parent[x]=parent[parent[x]];x=parent[x]\n        return x\n    for i,p in enumerate(photos,1):\n        data=pre.load(p[\'path\']);ph=str(data[\'pixel_hash\'])\n        if ph in seen:parent[find(p[\'date\'])]=find(seen[ph])\n        else:seen[ph]=p[\'date\']\n        p[\'pixel_hash\']=ph\n        p[\'disk_cx\'],p[\'disk_cy\'],p[\'disk_radius\']=map(float,data[\'disk\'])\n        if i%50==0:print(f\'Preprocess/cache: {i}/{len(photos)}\',flush=True)\n    groups=[find(p[\'date\']) for p in photos]\n    if len(set(groups))<c.N_FOLDS:raise ValueError(\'Not enough independent date groups for N_FOLDS\')\n    folds=np.full(len(photos),-1)\n    splitter=StratifiedGroupKFold(c.N_FOLDS,shuffle=True,random_state=c.SEED)\n    for fold,(_,val) in enumerate(splitter.split(np.zeros(len(photos)),[p[\'site\'] for p in photos],groups)):\n        folds[val]=fold\n    for p,f,g in zip(photos,folds,groups):p.update(fold=int(f),group=g,split=\'val\' if f==c.FOLD_INDEX else \'train\')\n    provenance={\'annotation_hash\':cat.hash,\'seed\':c.SEED,\'n_folds\':c.N_FOLDS,\'fold\':c.FOLD_INDEX,\n                \'preprocess_version\':pre.VERSION,\'preprocess_enabled\':c.PREPROCESS,\n                \'photos\':photos}\n    provenance[\'split_hash\']=digest({**provenance,\'photos\':[{k:v for k,v in p.items() if k!=\'path\'} for p in photos]})\n    path=out/\'manifest.json\'\n    if path.exists() and json.loads(path.read_text())[\'split_hash\']!=provenance[\'split_hash\']:\n        raise ValueError(\'Existing manifest differs. Choose a new OUTPUT_ROOT for a different dataset/split.\')\n    atomic_json(path,provenance)\n    rows=[{**{k:v for k,v in p.items() if k!=\'records\'},\'record_id\':rid,\n           \'n_views\':len(p[\'records\']),\'n_instances\':cat.records[rid][\'count\']} for p in photos for rid in p[\'records\']]\n    pd.DataFrame(rows).to_csv(out/\'all_annotators_manifest.csv\',index=False)\n    pd.crosstab(pd.Series([p[\'site\'] for p in photos],name=\'site\'),pd.Series([p[\'split\'] for p in photos],name=\'split\')).to_csv(out/\'site_split.csv\')\n    train=[p for p in photos if p[\'split\']==\'train\']\n    norm=fit_normalization(c,pre,train);norm[\'split_hash\']=provenance[\'split_hash\']\n    atomic_json(out/\'normalization.json\',norm)\n    atomic_json(out/\'data_profile.json\',{\'n_photos\':len(photos),\'n_annotation_views\':len(cat.records),\n        \'n_annotations\':len(cat.coco.anns),\'train_photos\':len(train),\'val_photos\':len(photos)-len(train),\n        \'n_test_photos\':len(cat.test_paths),\'station_balance\':\'StratifiedGroupKFold by site, groups join dates and duplicate pixels\',\n        \'train_val_dates_disjoint\':not({p[\'date\'] for p in train}&{p[\'date\'] for p in photos if p[\'split\']==\'val\'})})\n    return cat,provenance,norm\n\n\ndef load_manifest(c):return json.loads((Path(c.OUTPUT_ROOT)/\'manifest.json\').read_text())\n\n\ndef selected_record(cat,p,mode,rng,epoch=0,slot=0):\n    ids=p[\'records\']\n    if mode==\'max_count\':return max(ids,key=lambda r:(cat.records[r][\'count\'],cat.records[r][\'area\'],r))\n    if mode==\'all_annotators\':return ids[(epoch+slot)%len(ids)]\n    if mode==\'random_annotator\':\n        # Same record for this photo throughout an epoch, independent of workers/ranks.\n        seed=int(digest([cat.c.SEED,p[\'stem\'],epoch])[:8],16)\n        return ids[np.random.default_rng(seed).integers(len(ids))]\n    raise ValueError(mode)\n\n\ndef spatial_augment(x,m,rng):\n    k=int(rng.integers(4));x=np.rot90(x,k,axes=(-2,-1));m=np.rot90(m,k,axes=(-2,-1))\n    if rng.random()<.5:x=np.flip(x,-1);m=np.flip(m,-1)\n    return np.ascontiguousarray(x),np.ascontiguousarray(m)\n\n\ndef photometric_features(x,mode,rng):\n    """Reference gamma/contrast/gain/blur/noise, applied only to intensity channels."""\n    count=2 if mode==\'raw_corrected_radial\' else 1\n    gamma=rng.uniform(.8,1.25) if rng.random()<.7 else 1.\n    contrast,gain=(rng.uniform(.85,1.15),rng.uniform(.9,1.1)) if rng.random()<.7 else (1.,1.)\n    blur=rng.uniform(.3,.8) if rng.random()<.15 else 0.\n    noise=rng.uniform(.003,.015) if rng.random()<.25 else 0.\n    for i in range(count):\n        image=x[i].copy();valid=image>.01\n        if not valid.any():continue\n        image[valid]**=gamma;mean=image[valid].mean()\n        image[valid]=((image[valid]-mean)*contrast+mean)*gain\n        if blur:image[valid]=cv2.GaussianBlur(image,(0,0),blur)[valid]\n        if noise:image[valid]+=rng.normal(0,noise,int(valid.sum()))\n        x[i]=np.clip(image,0,1)\n    if count==1:x[1]=x[0];x[2]=x[0]\n    return x\n\n\ndef boxes_from_masks(masks):\n    result=[]\n    for m in masks:\n        y,x=np.where(m);result.append([x.min(),y.min(),x.max()+1,y.max()+1])\n    return np.asarray(result,np.float32).reshape(-1,4)\n\n\nclass EpochDataset(Dataset):\n    @property\n    def epoch(self):return int(self._epoch.item())\n\n    @epoch.setter\n    def epoch(self,value):\n        if hasattr(self,\'_epoch\'):self._epoch.fill_(value)\n        else:self._epoch=torch.tensor(value,dtype=torch.long).share_memory_()\n\n\nclass DetectorDataset(EpochDataset):\n    def __init__(self,c,cat,photos):\n        self.c,self.cat,self.photos=c,cat,photos;self.epoch=0\n        self.pre=SolarPreprocessor(Path(c.OUTPUT_ROOT)/\'preprocessed\',c.PREPROCESS)\n        if c.ANNOTATION_MODE==\'all_annotators\':self.items=[(p,r) for p in photos for r in p[\'records\']]\n        else:self.items=[(p,None) for p in photos]\n\n    def __len__(self):return len(self.items)\n\n    def __getitem__(self,index):\n        p,r=self.items[index];rng=np.random.default_rng(self.c.SEED+self.epoch*1000003+index)\n        if r is None:r=selected_record(self.cat,p,self.c.ANNOTATION_MODE,rng,self.epoch)\n        x=features(self.pre.load(p[\'path\']),self.c.INPUT_MODE,size=self.c.DETECTOR_IMAGE_SIZE)\n        m=self.cat.masks(r,size=self.c.DETECTOR_IMAGE_SIZE)\n        x,m=spatial_augment(x,m,rng)\n        x=photometric_features(x,self.c.INPUT_MODE,rng)\n        weight=len(self.items)/(len(self.photos)*len(p[\'records\'])) if self.c.ANNOTATION_MODE==\'all_annotators\' else 1.\n        target={\'boxes\':torch.from_numpy(boxes_from_masks(m)),\'labels\':torch.ones(len(m),dtype=torch.long),\n                \'masks\':torch.from_numpy(m),\'image_id\':torch.tensor([index]),\n                \'area\':torch.from_numpy(m.sum((1,2)).astype(np.float32)),\'iscrowd\':torch.zeros(len(m),dtype=torch.long)}\n        return torch.from_numpy(x),target,float(weight)\n\n\ndef square_box(box,shape,context=1.8,minimum=96,rng=None,jitter=0.):\n    x0,y0,x1,y1=map(float,box);h,w=shape\n    side=max(x1-x0,y1-y0)*context;cx=(x0+x1)/2;cy=(y0+y1)/2\n    if rng is not None and jitter:\n        cx+=rng.uniform(-jitter,jitter)*side;cy+=rng.uniform(-jitter,jitter)*side\n        side*=rng.uniform(.85,1.2)\n    side=int(min(max(np.ceil(side),minimum),h,w))\n    x=int(np.clip(round(cx-side/2),0,w-side));y=int(np.clip(round(cy-side/2),0,h-side))\n    return x,y,x+side,y+side\n\n\ndef roi_channels(data,bounds,size,coarse,c):\n    x=features(data,\'raw_repeat\',bounds,size)\n    if c.REFINER_USE_COARSE:x=np.concatenate([x,cv2.resize(coarse.astype(np.float32),(size,size),interpolation=cv2.INTER_LINEAR)[None]],0)\n    if c.REFINER_USE_RADIAL:x=np.concatenate([x,cv2.resize(radial_window(data,bounds),(size,size),interpolation=cv2.INTER_LINEAR)[None]],0)\n    return np.ascontiguousarray(x)\n\n\nclass SegDataset(EpochDataset):\n    def __init__(self,c,cat,photos,stage):\n        self.c,self.cat,self.photos,self.stage=c,cat,photos,stage;self.epoch=0\n        self.pre=SolarPreprocessor(Path(c.OUTPUT_ROOT)/\'preprocessed\',c.PREPROCESS)\n        self.samples=c.REFINER_CROPS_PER_PHOTO if stage==\'refiner\' else c.TILE_SAMPLES_PER_PHOTO\n        # Retain negative photos for tile training; refiner needs one instance.\n        if stage==\'refiner\':self.photos=[p for p in photos if any(cat.records[r][\'count\'] for r in p[\'records\'])]\n        if not self.photos:raise ValueError(f\'No training examples for {stage}\')\n\n    def __len__(self):return len(self.photos)*self.samples\n\n    def __getitem__(self,index):\n        p=self.photos[index//self.samples];slot=index%self.samples\n        rng=np.random.default_rng(self.c.SEED+self.epoch*1000003+index)\n        rid=selected_record(self.cat,p,self.c.ANNOTATION_MODE,rng,self.epoch,slot)\n        data=self.pre.load(p[\'path\']);h,w=data[\'raw\'].shape\n        if self.stage==\'refiner\':\n            if not self.cat.records[rid][\'count\']:\n                available=[r for r in p[\'records\'] if self.cat.records[r][\'count\']]\n                rid=available[int(rng.integers(len(available)))]\n            rles=self.cat.rles(rid);r=rles[int(rng.integers(len(rles)))];m=cm.decode(r)\n            bx,by,bw,bh=cm.toBbox(r)\n            bounds=square_box([bx,by,bx+bw,by+bh],(h,w),self.c.REFINER_CONTEXT,self.c.REFINER_MIN_CROP,rng,self.c.REFINER_BOX_JITTER)\n            x0,y0,x1,y1=bounds;target=m[y0:y1,x0:x1]\n            small=cv2.resize(target.astype(np.float32),(max(8,(x1-x0)//8),max(8,(y1-y0)//8)),interpolation=cv2.INTER_AREA)\n            coarse=cv2.resize(small.astype(np.float32),(x1-x0,y1-y0),interpolation=cv2.INTER_LINEAR)\n            if rng.random()<.3:coarse=cv2.erode(coarse,np.ones((3,3),np.uint8))\n            if rng.random()<.3:coarse=cv2.dilate(coarse,np.ones((3,3),np.uint8))\n            if rng.random()<.1:coarse*=0\n            size=self.c.REFINER_SIZE;x=roi_channels(data,bounds,size,coarse,self.c)\n            y=cv2.resize(target,(size,size),interpolation=cv2.INTER_NEAREST)[None].astype(np.float32)\n        else:\n            size=self.c.TILE_SIZE\n            if h<size or w<size:raise ValueError(\'TILE_SIZE exceeds native image size\')\n            rles=self.cat.rles(rid)\n            if rles and rng.random()<.7:\n                bx,by,bw,bh=cm.toBbox(rles[int(rng.integers(len(rles)))])\n                cx=rng.uniform(bx,bx+bw);cy=rng.uniform(by,by+bh)\n                x0=int(np.clip(cx-size/2,0,w-size));y0=int(np.clip(cy-size/2,0,h-size))\n            else:x0=int(rng.integers(w-size+1));y0=int(rng.integers(h-size+1))\n            bounds=(x0,y0,x0+size,y0+size)\n            x=features(data,self.c.INPUT_MODE,bounds,size)\n            masks=self.cat.masks(rid,bounds=bounds)\n            y=(masks.any(0) if len(masks) else np.zeros((size,size),bool))[None].astype(np.float32)\n        x,y=spatial_augment(x,y,rng)\n        if self.stage==\'refiner\':\n            if rng.random()<.5:x[:3]=x[:3]**rng.uniform(.85,1.2)\n            if rng.random()<.5:x[:3]=np.clip(x[:3]*rng.uniform(.9,1.1),0,1)\n        else:x=photometric_features(x,self.c.INPUT_MODE,rng)\n        return torch.from_numpy(x),torch.from_numpy(y)\n\n\ndef detection_collate(batch):return tuple(zip(*batch))\n', 'solar_v1/disk.py': '"""Solar disk geometry and limb-darkening correction.\n\nTwo preprocessing steps that matter more than the choice of model:\n\n1. **Disk masking.** Everything outside the solar limb is sky and detector\n   noise. It is uniformly dark, so any "filaments are dark" rule fires on it.\n   Masking it away removes a large class of false positives for free -- and\n   under Panoptic Quality every false positive costs 0.5 in the denominator.\n\n2. **Limb-darkening correction.** A full-disk H-alpha image is systematically\n   darker toward the limb, so a single global darkness threshold is far too\n   eager near the edge and far too shy at disk centre. Dividing by the median\n   radial intensity profile makes one threshold mean the same thing everywhere.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\n\n\n@dataclass(frozen=True)\nclass Disk:\n    cx: float\n    cy: float\n    radius: float\n\n    def mask(self, shape: tuple[int, int], shrink: float = 0.98) -> np.ndarray:\n        """Boolean mask of the disk, shrunk slightly to drop the noisy limb."""\n        height, width = shape\n        ys, xs = np.ogrid[:height, :width]\n        r2 = (xs - self.cx) ** 2 + (ys - self.cy) ** 2\n        return r2 <= (self.radius * shrink) ** 2\n\n    def radius_map(self, shape: tuple[int, int]) -> np.ndarray:\n        """Normalised radial distance (1.0 at the limb)."""\n        return self.radius_window(0, 0, *shape)\n\n    def radius_window(self, y0: int, x0: int, height: int, width: int) -> np.ndarray:\n        """``radius_map`` for rows ``y0:y0+height`` and columns ``x0:x0+width`` only."""\n        ys, xs = np.ogrid[y0 : y0 + height, x0 : x0 + width]\n        return np.sqrt((xs - self.cx) ** 2 + (ys - self.cy) ** 2) / max(self.radius, 1.0)\n\n\ndef read_image(path: str | Path) -> np.ndarray:\n    """Load a GONG H-alpha JPEG as a single-channel uint8 array.\n\n    The competition data is grayscale; reading it as BGR and averaging would\n    only add rounding noise.\n    """\n    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)\n    if image is None:\n        raise FileNotFoundError(f"could not read image: {path}")\n    return image\n\n\n#: GONG H-alpha frames are usually registered to a standard geometry: the\n#: disk centred and about 0.439 frame-widths in radius (899 px at 2048). Many\n#: frames are not, so this is only one of several starting guesses.\n_STANDARD_RADIUS_FRACTION = 0.439\n#: Rays cast from a candidate centre when locating the limb.\n_N_RAYS = 720\n#: How far inside each starting guess to look as well, in px at 2048. The halo\n#: can reach 150 px beyond the limb, so a guess locked onto it needs this.\n_INWARD_OFFSETS = (0.0, 50.0, 100.0, 150.0)\n#: A ray "agrees" with a circle if its sharpest edge lies within this many px\n#: (at 2048) of it. Fixed, not data-adaptive, so scattered edges score low.\n_AGREEMENT_TOLERANCE = 3.0\n#: A circle counts as well supported if this fraction of the best circle\'s\n#: agreeing rays also agree with it.\n_SUPPORT_RATIO = 0.8\n\n\ndef detect_disk(image: np.ndarray) -> Disk:\n    """Locate the solar disk by its limb.\n\n    GONG frames carry a smooth "halo" annulus outside the limb -- scattered\n    light, and the area the frame covered before it was registered -- that can\n    be nearly as bright as the disk itself, and its outer border is a crisp\n    circle too. Thresholding the image, which this function used to rely on\n    alone, locks onto that border: about 35 px beyond the limb on a typical\n    frame and up to 150 px on the worst.\n\n    Instead, several starting circles (an intensity threshold, a texture\n    threshold -- the disk is textured, the halo smooth -- and the standard GONG\n    geometry), each also shrunk by up to 150 px, are snapped to the sharpest\n    radial edge nearby and scored by how many rays agree with the result. The\n    halo always lies outside the limb, so of the well-supported circles the\n    smallest is the limb.\n    """\n    scale = min(image.shape) / 2048.0\n    blurred = cv2.GaussianBlur(image.astype(np.float32), (0, 0), max(1.5 * scale, 0.8))\n    seeds = [_threshold_disk(image), _texture_disk(image), _standard_disk(image)]\n\n    fits: list[tuple[Disk, float]] = []\n    for seed in seeds:\n        if seed is None:\n            continue\n        for offset in _INWARD_OFFSETS:\n            start = Disk(seed.cx, seed.cy, seed.radius - offset * scale)\n            if start.radius <= 0:\n                continue\n            disk, _ = _refine_limb(blurred, start, inside=60.0, outside=20.0)\n            # A second, narrow pass lets the centre settle once the radius is close.\n            fits.append(_refine_limb(blurred, disk, inside=10.0, outside=10.0))\n\n    best_support = max((support for _, support in fits), default=0.0)\n    # Fewer than half the rays agreeing means nothing found a clean limb; the\n    # plain threshold circle is then the least surprising answer.\n    if best_support < 0.5:\n        return seeds[0]\n    supported = [disk for disk, support in fits if support >= _SUPPORT_RATIO * best_support]\n    return min(supported, key=lambda disk: disk.radius)\n\n\ndef _fit_circle(xs: np.ndarray, ys: np.ndarray, iterations: int = 4) -> tuple[Disk, np.ndarray]:\n    """Least-squares (Kasa) circle fit, refitted without outliers.\n\n    Returns the circle and a boolean inlier mask over the input points.\n    """\n    keep = np.ones(xs.size, dtype=bool)\n    disk = Disk(0.0, 0.0, 1.0)\n    for _ in range(iterations):\n        if keep.sum() < 3:\n            break\n        design = np.column_stack([xs[keep], ys[keep], np.ones(int(keep.sum()))])\n        target = -(xs[keep] ** 2 + ys[keep] ** 2)\n        d, e, f = np.linalg.lstsq(design, target, rcond=None)[0]\n        cx, cy = -d / 2.0, -e / 2.0\n        disk = Disk(float(cx), float(cy), float(np.sqrt(max(cx * cx + cy * cy - f, 1.0))))\n        residual = np.hypot(xs - disk.cx, ys - disk.cy) - disk.radius\n        spread = 1.4826 * np.median(np.abs(residual[keep] - np.median(residual[keep])))\n        keep = np.abs(residual) <= max(3.0 * spread, 2.0)\n    return disk, keep\n\n\ndef _refine_limb(\n    blurred: np.ndarray, start: Disk, inside: float, outside: float\n) -> tuple[Disk, float]:\n    """Snap ``start`` to the sharpest radial edge within the search band.\n\n    ``blurred`` is the lightly smoothed float image. Band lengths are in px at\n    2048 and scale with the frame, so the same code works on small test\n    images. Returns the refined circle and the fraction of all rays whose\n    sharpest edge lies within ``_AGREEMENT_TOLERANCE`` of it.\n    """\n    height, width = blurred.shape\n    scale = min(height, width) / 2048.0\n    theta = np.linspace(0.0, 2.0 * np.pi, _N_RAYS, endpoint=False)[:, None]\n    radii = np.arange(start.radius - inside * scale, start.radius + outside * scale + 1.0, 1.0)\n    map_x = (start.cx + radii[None, :] * np.cos(theta)).astype(np.float32)\n    map_y = (start.cy + radii[None, :] * np.sin(theta)).astype(np.float32)\n    profiles = cv2.remap(blurred, map_x, map_y, cv2.INTER_LINEAR,\n                         borderMode=cv2.BORDER_CONSTANT, borderValue=0)\n    in_frame = (map_x >= 1) & (map_x <= width - 2) & (map_y >= 1) & (map_y <= height - 2)\n    gradient = np.abs(np.gradient(profiles, axis=1))\n    gradient[~in_frame] = -1.0\n    edge = np.argmax(gradient, axis=1)\n    valid = gradient[np.arange(_N_RAYS), edge] > 0\n    # A real edge is a peak inside the band. A maximum on the band\'s boundary\n    # is just the smooth limb-darkening slope rising toward it -- perfectly\n    # circular, so without this it would pass for a well-supported limb.\n    valid &= (edge > 1) & (edge < radii.size - 2)\n    if valid.sum() < 3:\n        return start, 0.0\n    # Sub-pixel peak position from a parabola through the peak and its two\n    # neighbours: the flattening downstream is sensitive to fractions of a\n    # pixel in the radius.\n    rows = np.flatnonzero(valid)\n    peak = edge[valid]\n    before, at, after = (gradient[rows, peak - 1], gradient[rows, peak],\n                         gradient[rows, peak + 1])\n    curvature = before - 2.0 * at + after\n    shift = np.divide(0.5 * (before - after), curvature,\n                      out=np.zeros_like(at), where=curvature < 0)\n    r = radii[peak] + np.clip(shift, -0.5, 0.5)\n    xs = start.cx + r * np.cos(theta[valid, 0])\n    ys = start.cy + r * np.sin(theta[valid, 0])\n    disk, _ = _fit_circle(xs, ys)\n    residual = np.hypot(xs - disk.cx, ys - disk.cy) - disk.radius\n    agreeing = np.abs(residual) <= max(_AGREEMENT_TOLERANCE * scale, 1.0)\n    return disk, float(agreeing.sum()) / _N_RAYS\n\n\ndef _standard_disk(image: np.ndarray) -> Disk:\n    height, width = image.shape\n    return Disk((width - 1) / 2.0, height / 2.0, min(height, width) * _STANDARD_RADIUS_FRACTION)\n\n\ndef _texture_disk(image: np.ndarray) -> Disk | None:\n    """Disk from local texture: chromospheric structure is busy, the halo smooth."""\n    scale = min(image.shape) / 2048.0\n    img = image.astype(np.float32)\n    detail = np.abs(img - cv2.GaussianBlur(img, (0, 0), max(3.0 * scale, 1.0)))\n    texture = cv2.GaussianBlur(detail, (0, 0), max(15.0 * scale, 2.0))\n    texture_u8 = np.clip(texture * (255.0 / max(float(texture.max()), 1e-6)), 0, 255)\n    _, binary = cv2.threshold(texture_u8.astype(np.uint8), 0, 1,\n                              cv2.THRESH_BINARY + cv2.THRESH_OTSU)\n    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)\n    if n_labels <= 1:\n        return None\n    largest = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))\n    component = (labels == largest).astype(np.uint8)\n    # Fill the component\'s interior: smooth patches on the disk (and the whole\n    # interior of a texture-free test image) leave holes.\n    flood = component.copy()\n    cv2.floodFill(flood, np.zeros((flood.shape[0] + 2, flood.shape[1] + 2), np.uint8), (0, 0), 1)\n    component = component | (1 - flood)\n    contours, _ = cv2.findContours(component, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)\n    points = max(contours, key=cv2.contourArea)[:, 0, :].astype(np.float64)\n    if len(points) < 3:\n        return None\n    disk, _ = _fit_circle(points[:, 0], points[:, 1])\n    return disk\n\n\ndef _threshold_disk(image: np.ndarray) -> Disk:\n    """Disk from an intensity threshold: the brightest large component.\n\n    Reliable for the centre of a clean frame, but the halo annulus usually\n    passes the threshold too, so the radius it gives is an upper bound.\n    """\n    blurred = cv2.GaussianBlur(image, (0, 0), sigmaX=5)\n    threshold, binary = cv2.threshold(\n        blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU\n    )\n    binary = cv2.morphologyEx(\n        binary, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (31, 31))\n    )\n\n    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)\n    if n_labels <= 1:\n        # Degenerate image: fall back to the nominal full-frame disk.\n        height, width = image.shape\n        return Disk(width / 2, height / 2, min(height, width) * 0.45)\n\n    largest = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))\n    component = (labels == largest).astype(np.uint8)\n\n    contours, _ = cv2.findContours(component, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)\n    contour = max(contours, key=cv2.contourArea)\n    (cx, cy), radius = cv2.minEnclosingCircle(contour)\n    return Disk(float(cx), float(cy), float(radius))\n\n\ndef limb_darkening_profile(\n    image: np.ndarray, disk: Disk, n_bins: int = 128, smooth: int = 9\n) -> np.ndarray:\n    """Median intensity as a function of normalised radius."""\n    radius = disk.radius_map(image.shape)\n    inside = radius <= 1.0\n    bins = np.clip((radius[inside] * n_bins).astype(np.int32), 0, n_bins - 1)\n    values = image[inside].astype(np.float32)\n\n    profile = np.ones(n_bins, dtype=np.float32)\n    order = np.argsort(bins, kind="stable")\n    bins_sorted, values_sorted = bins[order], values[order]\n    edges = np.searchsorted(bins_sorted, np.arange(n_bins + 1))\n    for b in range(n_bins):\n        lo, hi = edges[b], edges[b + 1]\n        if hi > lo:\n            profile[b] = np.median(values_sorted[lo:hi])\n\n    # Fill empty bins, then smooth so the correction has no step artefacts.\n    valid = profile > 0\n    if valid.any():\n        profile = np.interp(np.arange(n_bins), np.flatnonzero(valid), profile[valid])\n    if smooth > 1:\n        kernel = np.ones(smooth, dtype=np.float32) / smooth\n        profile = np.convolve(profile, kernel, mode="same")\n    profile[profile <= 0] = 1.0\n    return profile\n\n\ndef flatten(image: np.ndarray, disk: Disk, n_bins: int = 128) -> np.ndarray:\n    """Divide out limb darkening.\n\n    Returns a float32 image where 1.0 is "typical quiet Sun at this radius" and\n    filaments sit clearly below 1.0, independent of distance from disk centre.\n    """\n    profile = limb_darkening_profile(image, disk, n_bins=n_bins)\n    radius = disk.radius_map(image.shape)\n    bin_index = np.clip((radius * n_bins).astype(np.int32), 0, n_bins - 1)\n    expected = profile[bin_index]\n    flat = image.astype(np.float32) / np.maximum(expected, 1e-3)\n    flat[radius > 1.0] = 1.0\n    return flat\n\n', 'solar_v1/inference.py': '"""Threshold-independent ROI geometry + continuous coarse conditioning permit exact cached search."""\nfrom pathlib import Path\nimport importlib.metadata\nimport os\nimport time\nimport cv2\nimport numpy as np\nimport torch\nfrom .config import atomic_json,digest,file_hash,autocast\nfrom .data import square_box,roi_channels\nfrom .preprocess import features\n\n\ndef tile_starts(length,size,overlap):\n    if length<size:raise ValueError(\'Native image smaller than tile\')\n    points=list(range(0,length-size+1,size-overlap))\n    if points[-1]!=length-size:points.append(length-size)\n    return points\n\n\ndef smooth_window(size):\n    # Positive edges ensure even outermost native pixels receive weight.\n    one=np.maximum(np.hanning(size).astype(np.float32),.05)\n    return one[:,None]*one[None,:]\n\n\n@torch.inference_mode()\ndef tile_probability(model,data,c,device):\n    h,w=data[\'raw\'].shape;size=c.TILE_SIZE\n    bounds=[(x,y,x+size,y+size) for y in tile_starts(h,size,c.TILE_OVERLAP) for x in tile_starts(w,size,c.TILE_OVERLAP)]\n    result=np.zeros((h,w),np.float32);weights=np.zeros((h,w),np.float32);window=smooth_window(size)\n    for start in range(0,len(bounds),c.INFERENCE_BATCH_SIZE):\n        batch=bounds[start:start+c.INFERENCE_BATCH_SIZE]\n        x=torch.from_numpy(np.stack([features(data,c.INPUT_MODE,b,size) for b in batch])).to(device)\n        with autocast(c,device):p=model(x).sigmoid().float().cpu().numpy()[:,0]\n        for prob,(x0,y0,x1,y1) in zip(p,batch):\n            result[y0:y1,x0:x1]+=prob*window;weights[y0:y1,x0:x1]+=window\n    if np.any(weights<=0):raise RuntimeError(\'Uncovered tile pixels\')\n    return result/weights\n\n\ndef save_npz(path,pack):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    tmp=path.with_name(path.stem+f\'.{os.getpid()}.tmp.npz\')\n    np.savez_compressed(tmp,**pack);os.replace(tmp,path)\n\n\ndef read_pack(path):\n    with np.load(path,allow_pickle=False) as z:return {k:z[k] for k in z.files}\n\n\n@torch.inference_mode()\ndef detector_pack(model,data,c,device):\n    h,w=data[\'raw\'].shape;size=c.DETECTOR_IMAGE_SIZE\n    tensor=torch.from_numpy(features(data,c.INPUT_MODE,size=size)).to(device)\n    with autocast(c,device):out=model([tensor])[0]\n    scores=out[\'scores\'].float().cpu().numpy();boxes=out[\'boxes\'].float().cpu().numpy()\n    probs=out[\'masks\'][:,0].float().cpu().numpy()\n    boxes[:,[0,2]]*=w/size;boxes[:,[1,3]]*=h/size\n    pack={\'shape\':np.array([h,w]),\'detector_size\':np.array(size),\'scores\':scores,\'boxes\':boxes,\n          \'pixel_hash\':data[\'pixel_hash\']}\n    bounds=[]\n    for i,p in enumerate(probs):\n        y,x=np.nonzero(p>0)\n        b=[int(x.min()),int(y.min()),int(x.max()+1),int(y.max()+1)] if len(x) else [0,0,1,1]\n        x0,y0,x1,y1=b;bounds.append(b);pack[f\'det_{i}\']=p[y0:y1,x0:x1]\n    pack[\'det_bounds\']=np.array(bounds,np.int32).reshape(-1,4)\n    return pack\n\n\ndef native_coarse(pack,index):\n    size=int(pack[\'detector_size\']);h,w=map(int,pack[\'shape\'])\n    full=np.zeros((size,size),np.float32);x0,y0,x1,y1=map(int,pack[\'det_bounds\'][index])\n    full[y0:y1,x0:x1]=pack[f\'det_{index}\']\n    return cv2.resize(full,(w,h),interpolation=cv2.INTER_LINEAR)\n\n\n@torch.inference_mode()\ndef add_refiner(pack,data,model,c,device):\n    bounds=[square_box(b,data[\'raw\'].shape,c.REFINER_CONTEXT,c.REFINER_MIN_CROP) for b in pack[\'boxes\']]\n    result=[]\n    for start in range(0,len(bounds),c.INFERENCE_BATCH_SIZE):\n        inputs=[]\n        for i in range(start,min(start+c.INFERENCE_BATCH_SIZE,len(bounds))):\n            x0,y0,x1,y1=bounds[i];coarse=native_coarse(pack,i)[y0:y1,x0:x1]\n            inputs.append(roi_channels(data,bounds[i],c.REFINER_SIZE,coarse,c))\n        x=torch.from_numpy(np.stack(inputs)).to(device)\n        with autocast(c,device):p=model(x).sigmoid().float().cpu().numpy()[:,0]\n        result.extend(p)\n    pack=dict(pack);pack[\'ref_bounds\']=np.asarray(bounds,np.int32).reshape(-1,4)\n    pack[\'ref_probs\']=np.asarray(result,np.float32).reshape(-1,c.REFINER_SIZE,c.REFINER_SIZE)\n    return pack\n\n\nclass PredictionCache:\n    def __init__(self,c,detector_path,refiner_path=None,tile_path=None,norm=None):\n        source={p.name:file_hash(p) for p in Path(__file__).parent.glob(\'*.py\')}\n        provenance={\'detector\':file_hash(detector_path),\n                    \'refiner\':file_hash(refiner_path) if refiner_path else None,\n                    \'tile\':file_hash(tile_path) if tile_path else None,\n                    \'features\':c.INPUT_MODE,\'preprocess\':c.PREPROCESS,\'normalization\':norm,\n                    \'size\':c.DETECTOR_IMAGE_SIZE,\'score_floor\':c.CACHE_SCORE_FLOOR,\'max_det\':c.MAX_DETECTIONS,\n                    \'context\':c.REFINER_CONTEXT,\'min_crop\':c.REFINER_MIN_CROP,\'ref_size\':c.REFINER_SIZE,\n                    \'tile_size\':c.TILE_SIZE,\'tile_overlap\':c.TILE_OVERLAP,\n                    \'amp\':c.AMP,\'torch\':str(torch.__version__),\'opencv\':cv2.__version__,\n                    \'torchvision\':importlib.metadata.version(\'torchvision\'),\'numpy\':np.__version__,\n                    \'cuda\':torch.version.cuda,\'cudnn\':torch.backends.cudnn.version(),\n                    \'gpus\':[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],\n                    \'source\':source}\n        self.signature=digest(provenance)\n        self.root=Path(c.OUTPUT_ROOT)/\'prediction_cache\'/self.signature[:24]\n        self.root.mkdir(parents=True,exist_ok=True)\n        atomic_json(self.root/\'cache_manifest.json\',provenance)\n        self.c=c\n\n    def path(self,stem):return self.root/(stem+\'.npz\')\n\n    def load(self,stem,data=None):\n        pack=read_pack(self.path(stem))\n        if data is not None and str(pack[\'pixel_hash\'])!=str(data[\'pixel_hash\']):raise ValueError(\'Cache image pixels changed\')\n        return pack\n\n    def put(self,stem,pack):\n        # Fail explicitly rather than silently dropping an optional branch or candidates.\n        if sum(p.stat().st_size for p in self.root.parent.rglob(\'*.npz\'))>self.c.MAX_CACHE_GB*2**30:\n            raise RuntimeError(\'Prediction cache budget exceeded. Archive old experiment caches or increase MAX_CACHE_GB.\')\n        save_npz(self.path(stem),pack)\n\n\ndef cache_photos(c,photos,pre,cache,detector,refiner,tile,device,base_cache=None):\n    times=[]\n    for i,p in enumerate(photos,1):\n        start=time.perf_counter();stem=p[\'stem\'];data=pre.load(p[\'path\'])\n        if cache.path(stem).exists():cache.load(stem,data);continue\n        if base_cache is not None and base_cache.path(stem).exists():pack=base_cache.load(stem,data)\n        elif detector is not None:pack=detector_pack(detector,data,c,device)\n        else:raise FileNotFoundError(f\'Missing fixed detector cache: {stem}\')\n        if refiner is not None:pack=add_refiner(pack,data,refiner,c,device)\n        if tile is not None:pack[\'tile_prob\']=tile_probability(tile,data,c,device)\n        cache.put(stem,pack);times.append(time.perf_counter()-start)\n        print(f\'Inference cache {i}/{len(photos)} {stem}: {times[-1]:.2f}s, candidates={len(pack["scores"])}\',flush=True)\n    return times\n', 'solar_v1/models.py': '"""Original v2 detector and exact original ResNet18 decoder, with explicit input adapters."""\nfrom pathlib import Path\nimport math\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torchvision.models import ResNet18_Weights\nfrom torchvision.models.detection import maskrcnn_resnet50_fpn_v2,MaskRCNN_ResNet50_FPN_V2_Weights\nfrom torchvision.models.detection.faster_rcnn import FastRCNNPredictor\nfrom torchvision.models.detection.mask_rcnn import MaskRCNNPredictor\nfrom torchvision.models.detection.anchor_utils import AnchorGenerator\nfrom torchvision.ops import MultiScaleRoIAlign\nfrom .reference_models import CropUNet\nfrom .config import architecture\n\n\ndef pretrained_state(c,weights):\n    name=Path(weights.url).name\n    roots=[Path(c.PRETRAINED_DIR)] if c.PRETRAINED_DIR else []\n    roots += [Path(torch.hub.get_dir())/\'checkpoints\']\n    for root in roots:\n        path=root/name\n        if path.exists():return torch.load(path,map_location=\'cpu\',weights_only=True)\n    # Attached Kaggle inputs are searched only by exact official weight filename.\n    matches=list(Path(c.INPUT_ROOT).rglob(name)) if Path(c.INPUT_ROOT).exists() else []\n    if len(matches)==1:return torch.load(matches[0],map_location=\'cpu\',weights_only=True)\n    if not c.ALLOW_WEIGHT_DOWNLOAD:raise FileNotFoundError(f\'Attach official pretrained weights: {name}\')\n    return weights.get_state_dict(progress=True,check_hash=True)\n\n\ndef warm_pretrained(c):\n    if not c.PRETRAINED:return\n    pretrained_state(c,MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT)\n    if c.ENABLE_REFINER or c.ENABLE_TILE_BRANCH:pretrained_state(c,ResNet18_Weights.DEFAULT)\n\n\ndef mixed_input_init(conv):\n    with torch.no_grad():\n        summed=conv.weight.sum(1,keepdim=True)\n        conv.weight[:,0:1].copy_(summed*.5)\n        conv.weight[:,1:2].copy_(summed*.5)\n        conv.weight[:,2:3].zero_()\n\n\ndef build_detector(c,norm,pretrained=False):\n    model=maskrcnn_resnet50_fpn_v2(weights=None,weights_backbone=None)\n    if pretrained:model.load_state_dict(pretrained_state(c,MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT))\n    box=model.roi_heads.box_predictor.cls_score.in_features\n    old=model.roi_heads.mask_predictor\n    model.roi_heads.box_predictor=FastRCNNPredictor(box,2)\n    model.roi_heads.mask_predictor=MaskRCNNPredictor(old.conv5_mask.in_channels,old.conv5_mask.out_channels,2)\n    model.rpn.anchor_generator=AnchorGenerator(sizes=tuple((s,) for s in c.ANCHOR_SIZES),aspect_ratios=(tuple(c.ANCHOR_RATIOS),)*5)\n    # Preserve the original RPN conv stack when changing aspect-ratio count.\n    n=len(c.ANCHOR_RATIOS);head=model.rpn.head\n    if n!=3:\n        cls=nn.Conv2d(256,n,1);boxlayer=nn.Conv2d(256,n*4,1)\n        with torch.no_grad():\n            for j,r in enumerate(c.ANCHOR_RATIOS):\n                k=min(range(3),key=lambda i:abs(math.log(r/[.5,1.,2.][i])))\n                cls.weight[j].copy_(head.cls_logits.weight[k]);cls.bias[j].copy_(head.cls_logits.bias[k])\n                boxlayer.weight[4*j:4*j+4].copy_(head.bbox_pred.weight[4*k:4*k+4])\n                boxlayer.bias[4*j:4*j+4].copy_(head.bbox_pred.bias[4*k:4*k+4])\n        head.cls_logits,head.bbox_pred=cls,boxlayer\n    model.roi_heads.mask_roi_pool=MultiScaleRoIAlign([\'0\',\'1\',\'2\',\'3\'],28,2)\n    model.transform.min_size=(c.DETECTOR_IMAGE_SIZE,);model.transform.max_size=c.DETECTOR_IMAGE_SIZE\n    model.transform.image_mean=norm[\'mean\'];model.transform.image_std=norm[\'std\']\n    model.roi_heads.score_thresh=c.CACHE_SCORE_FLOOR;model.roi_heads.detections_per_img=c.MAX_DETECTIONS\n    # Match pretrained torchvision\'s three trainable ResNet stages. Mixed inputs\n    # additionally train conv1 so its initially-zero radial weights can learn.\n    for name,p in model.backbone.body.named_parameters():\n        p.requires_grad_(name.startswith((\'layer2\',\'layer3\',\'layer4\')) or\n                         (c.INPUT_MODE==\'raw_corrected_radial\' and name.startswith(\'conv1\')))\n    if c.INPUT_MODE==\'raw_corrected_radial\' and pretrained:mixed_input_init(model.backbone.body.conv1)\n    return model\n\n\nclass Segmenter(CropUNet):\n    def __init__(self,c,norm,stage,pretrained=False):\n        # Exact reference ResNet18 encoder and transposed-convolution decoder.\n        super().__init__(pretrained=False)\n        if pretrained:\n            state=pretrained_state(c,ResNet18_Weights.DEFAULT)\n            self.stem[0].weight.data.copy_(state[\'conv1.weight\'])\n            self.stem[1].load_state_dict({k.removeprefix(\'bn1.\'):v for k,v in state.items() if k.startswith(\'bn1.\')})\n            for name in [\'layer1\',\'layer2\',\'layer3\',\'layer4\']:\n                getattr(self,name).load_state_dict({k[len(name)+1:]:v for k,v in state.items() if k.startswith(name+\'.\')})\n        if stage==\'refiner\':\n            n=3+int(c.REFINER_USE_COARSE)+int(c.REFINER_USE_RADIAL)\n            old=self.stem[0]\n            conv=nn.Conv2d(n,64,7,2,3,bias=False)\n            with torch.no_grad():conv.weight.zero_();conv.weight[:,:3].copy_(old.weight)\n            self.stem[0]=conv\n            means=[.485,.456,.406]+[.5]*(n-3);stds=[.229,.224,.225]+[.25]*(n-3)\n        else:\n            means=norm[\'mean\'];stds=norm[\'std\']\n            if c.INPUT_MODE==\'raw_corrected_radial\' and pretrained:mixed_input_init(self.stem[0])\n        self.mean=torch.tensor(means).view(1,-1,1,1)\n        self.std=torch.tensor(stds).view(1,-1,1,1)\n\n\ndef bce_tversky(logits,target,alpha=.3,beta=.7):\n    # Accumulate small structures in float32 even inside AMP.\n    logits=logits.float();target=target.float();p=logits.sigmoid()\n    dims=(1,2,3);tp=(p*target).sum(dims);fp=(p*(1-target)).sum(dims);fn=((1-p)*target).sum(dims)\n    return .5*F.binary_cross_entropy_with_logits(logits,target)+.5*(1-(tp+1)/(tp+alpha*fp+beta*fn+1)).mean()\n\n\ndef checkpoint_path(c,stage,kind=\'best\',all_data=False):\n    root=Path(c.CHECKPOINT_DIR) if c.CHECKPOINT_DIR else Path(c.OUTPUT_ROOT)/\'checkpoints\'\n    return root/((\'all_\' if all_data else \'\')+stage+\'_\'+kind+\'.pt\')\n\n\ndef load_model(c,norm,stage,device,path=None):\n    path=Path(path) if path else checkpoint_path(c,stage)\n    state=torch.load(path,map_location=\'cpu\',weights_only=True)\n    if state[\'architecture\']!=architecture(c,stage):raise ValueError(f\'Checkpoint architecture/config mismatch: {path}\')\n    if any(not np.allclose(state[\'normalization\'][k],norm[k],rtol=1e-6,atol=1e-7) for k in [\'mean\',\'std\']):\n        raise ValueError(f\'Checkpoint training normalization differs from current pipeline: {path}\')\n    model=build_detector(c,norm) if stage==\'detector\' else Segmenter(c,norm,stage)\n    model.load_state_dict(state[\'model\'],strict=True)\n    return model.to(device).eval(),state\n', 'solar_v1/postprocess.py': 'from dataclasses import dataclass,replace\nfrom pathlib import Path\nimport json\nimport cv2\nimport numpy as np\nfrom pycocotools import mask as cm\nfrom .inference import native_coarse\nfrom .pq import canonical_rle,pairwise_iou\n\n\n@dataclass\nclass Instance:\n    score: float\n    rle: dict\n    quality: float = 1.\n    source: str = \'global\'\n\n\ndef encode(mask):return cm.encode(np.asfortranarray(mask.astype(np.uint8)))\n\n\ndef mask_nms(instances,threshold,soft_sigma=None,min_score=0.):\n    pool=sorted(instances,key=lambda x:-x.score);result=[]\n    while pool:\n        best=pool.pop(0)\n        if best.score<min_score:continue\n        result.append(best)\n        if not pool:break\n        ious=pairwise_iou([best.rle],[x.rle for x in pool])[0]\n        if soft_sigma is None:pool=[x for x,iou in zip(pool,ious) if iou<=threshold]\n        else:\n            pool=[replace(x,score=x.score*float(np.exp(-iou*iou/soft_sigma))) if iou>threshold else x for x,iou in zip(pool,ious)]\n            pool.sort(key=lambda x:-x.score)\n    return result\n\n\ndef components(mask,min_area,split=False):\n    n,labels,stats,_=cv2.connectedComponentsWithStats(mask.astype(np.uint8),8)\n    kept=[labels==i for i in range(1,n) if stats[i,cv2.CC_STAT_AREA]>=min_area]\n    if split:return kept\n    return [np.logical_or.reduce(kept)] if kept else []\n\n\ndef raw_instances(pack,c,disk=None):\n    h,w=map(int,pack[\'shape\']);instances=[]\n    tile=pack.get(\'tile_prob\') if c.ENABLE_TILE_BRANCH else None\n    for i,score in enumerate(pack[\'scores\']):\n        if score<c.DETECTION_THRESHOLD:continue\n        coarse=native_coarse(pack,i)\n        if c.ENABLE_REFINER:\n            if \'ref_probs\' not in pack:raise ValueError(\'Refiner enabled but probability cache has no refiner output\')\n            x0,y0,x1,y1=map(int,pack[\'ref_bounds\'][i])\n            local=cv2.resize(pack[\'ref_probs\'][i],(x1-x0,y1-y0),interpolation=cv2.INTER_LINEAR)\n            prob=np.zeros((h,w),np.float32);prob[y0:y1,x0:x1]=local\n            threshold=c.REFINER_THRESHOLD\n        else:prob=coarse.copy();threshold=c.MASK_THRESHOLD\n        if tile is not None:\n            support=coarse>=c.MASK_THRESHOLD\n            if c.TILE_GROW:support=cv2.dilate(support.astype(np.uint8),np.ones((2*c.TILE_GROW+1,)*2,np.uint8)).astype(bool)\n            prob[support]=(1-c.TILE_FUSION_WEIGHT)*prob[support]+c.TILE_FUSION_WEIGHT*tile[support]\n        mask=prob>=threshold\n        if not mask.any() and c.ENABLE_REFINER and c.EMPTY_REFINER_FALLBACK:mask=coarse>=c.MASK_THRESHOLD\n        if disk is not None:mask &= disk\n        if c.MORPH_CLOSE_RADIUS:\n            r=c.MORPH_CLOSE_RADIUS;mask=cv2.morphologyEx(mask.astype(np.uint8),cv2.MORPH_CLOSE,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(2*r+1,2*r+1))).astype(bool)\n            if disk is not None:mask &= disk\n        if not mask.any():continue\n        quality=float(prob[mask].mean())\n        s=float(score)*((1-c.QUALITY_WEIGHT)+c.QUALITY_WEIGHT*quality)\n        instances.append(Instance(s,encode(mask),quality))\n    return instances\n\n\ndef fuse_instances(views,weights=None,match_iou=.55):\n    """Seed-based instance matching, at most one vote/model; no overlap transitive union."""\n    if weights is None:weights=np.ones(len(views))\n    weights=np.asarray(weights,float)\n    if len(weights)!=len(views) or np.any(weights<=0):raise ValueError(\'One positive ensemble weight per view required\')\n    pool=sorted([(ins,model) for model,view in enumerate(views) for ins in view],key=lambda z:-z[0].score)\n    used=set();result=[]\n    for i,(seed,model) in enumerate(pool):\n        if i in used:continue\n        used.add(i);group=[i];seen={model}\n        for j,(other,m) in enumerate(pool):\n            if j in used or m in seen:continue\n            if pairwise_iou([seed.rle],[other.rle])[0,0]>match_iou:used.add(j);group.append(j);seen.add(m)\n        total=sum(weights[pool[j][1]]*pool[j][0].score for j in group)\n        canvas=np.zeros(seed.rle[\'size\'],np.float32);available=0.\n        for j in group:\n            instance,m=pool[j];a=weights[m]*instance.score\n            canvas+=a*cm.decode(instance.rle);available+=a\n        mask=canvas>=.5*available\n        if mask.any():result.append(Instance(float(total/weights.sum()),encode(mask),source=\'ensemble\'))\n    return result\n\n\ndef external_instances(root,stem,validation=False):\n    root=Path(root);meta=json.loads((root/\'ensemble_manifest.json\').read_text())\n    if validation and stem in set(meta[\'trained_photo_ids\']):raise ValueError(\'External ensemble predictions have training leakage\')\n    payload=json.loads((root/(stem+\'.json\')).read_text())\n    if payload[\'photo\']!=stem:raise ValueError(\'External prediction photo mismatch\')\n    return [Instance(float(x[\'score\']),canonical_rle(x[\'rle\']),float(x.get(\'quality\',1)),str(root)) for x in payload[\'instances\']]\n\n\ndef finalize(instances,c,shape,tile=None,disk=None):\n    if c.POSTPROCESS_MODE==\'legacy_union\':\n        # Explicit historical control only.\n        adjacency=pairwise_iou([x.rle for x in instances],[x.rle for x in instances])>0 if instances else np.zeros((0,0),bool)\n        seen=set();merged=[]\n        for i in range(len(instances)):\n            if i in seen:continue\n            stack=[i];group=[]\n            while stack:\n                j=stack.pop()\n                if j in seen:continue\n                seen.add(j);group.append(j);stack.extend(np.flatnonzero(adjacency[j]).tolist())\n            merged.append(Instance(max(instances[j].score for j in group),cm.merge([instances[j].rle for j in group])))\n        instances=merged\n    else:instances=mask_nms(instances,c.MASK_NMS_IOU,c.SOFT_NMS_SIGMA if c.POSTPROCESS_MODE==\'soft_nms\' else None,c.DETECTION_THRESHOLD)\n    occupied=np.zeros(shape,bool);result=[]\n    for ins in sorted(instances,key=lambda x:-x.score):\n        mask=cm.decode(ins.rle).astype(bool)&~occupied\n        if disk is not None:mask &= disk\n        masks=components(mask,c.FRAGMENT_MIN_AREA,c.FRAGMENT_MODE==\'split\') if c.FRAGMENT_MODE!=\'keep\' else [mask]\n        for part in masks:\n            if part.sum()<c.MIN_INSTANCE_AREA:continue\n            occupied|=part;result.append(replace(ins,rle=encode(part)))\n    if tile is not None and c.TILE_KEEP_UNCLAIMED:\n        mask=(tile>=c.TILE_THRESHOLD)&~occupied\n        if disk is not None:mask &= disk\n        for part in components(mask,c.TILE_UNCLAIMED_MIN_AREA,True):result.append(Instance(c.TILE_THRESHOLD,encode(part),source=\'tile_unclaimed\'))\n    return sorted(result,key=lambda x:(float(cm.toBbox(x.rle)[1]),float(cm.toBbox(x.rle)[0])))\n\n\ndef postprocess(pack,c,data=None,stem=None,validation=False):\n    shape=tuple(map(int,pack[\'shape\']));disk=None\n    if data is not None and c.PREPROCESS:\n        yy,xx=np.ogrid[:shape[0],:shape[1]];cx,cy,r=data[\'disk\']\n        disk=(xx-cx)**2+(yy-cy)**2<=(r*c.DISK_MARGIN)**2\n    own=raw_instances(pack,c,disk)\n    if c.ENSEMBLE_CACHE_DIRS:\n        views=[own]+[external_instances(root,stem,validation) for root in c.ENSEMBLE_CACHE_DIRS]\n        own=fuse_instances(views,c.ENSEMBLE_WEIGHTS or None,c.ENSEMBLE_MATCH_IOU)\n    return finalize(own,c,shape,pack.get(\'tile_prob\') if c.ENABLE_TILE_BRANCH else None,disk)\n', 'solar_v1/pq.py': '"""Official-notebook-compatible pooled PQ. No model or dataset downloads.\n\nReference: azimahmadzadeh/self-evaluation-notebook, version 6, retrieved\n2026-11-02. The reference counts ALL IoU > 0.5 pairs, not Hungarian matches.\nThis module deliberately preserves that behavior, including overlapping GT.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import asdict, dataclass\nfrom typing import Mapping, Sequence\n\nimport numpy as np\nfrom pycocotools import mask as coco_mask\n\n\n@dataclass\nclass Totals:\n    tp: int = 0\n    fp: int = 0\n    fn: int = 0\n    iou_sum: float = 0.0\n    ambiguous_gt: int = 0\n    ambiguous_pred: int = 0\n    fragmented_gt: int = 0\n    merged_pred: int = 0\n\n    def __add__(self, other: Totals) -> Totals:\n        return Totals(**{k: v + asdict(other)[k] for k, v in asdict(self).items()})\n\n    def summary(self) -> dict:\n        denominator = self.tp + 0.5 * self.fp + 0.5 * self.fn\n        return {\n            **asdict(self),\n            "pq": self.iou_sum / denominator if denominator else 0.0,\n            "dq": self.tp / denominator if denominator else 0.0,\n            "sq": self.iou_sum / self.tp if self.tp else 0.0,\n        }\n\n\ndef from_iou(ious: np.ndarray) -> Totals:\n    """Count exactly as the host notebook; diagnostics use any positive IoU."""\n    ious = np.asarray(ious, dtype=np.float64)\n    if ious.ndim != 2 or not np.isfinite(ious).all():\n        raise ValueError("IoU matrix must be a finite 2-D array")\n    if ((ious < 0) | (ious > 1)).any():\n        raise ValueError("IoU values must be in [0, 1]")\n    hit = ious > 0.5\n    links = ious > 0\n    return Totals(\n        tp=int(hit.sum()),\n        fp=int((hit.sum(axis=0) == 0).sum()),\n        fn=int((hit.sum(axis=1) == 0).sum()),\n        iou_sum=float(ious[hit].sum()),\n        ambiguous_gt=int((hit.sum(axis=1) > 1).sum()),\n        ambiguous_pred=int((hit.sum(axis=0) > 1).sum()),\n        fragmented_gt=int((links.sum(axis=1) > 1).sum()),\n        merged_pred=int((links.sum(axis=0) > 1).sum()),\n    )\n\n\ndef canonical_rle(rle: dict) -> dict:\n    size = list(map(int, rle["size"]))\n    if len(size) != 2 or min(size) <= 0:\n        raise ValueError("RLE size must be positive height,width")\n    counts = rle["counts"]\n    if isinstance(counts, str):\n        counts = counts.encode("ascii")\n    if isinstance(counts, list):\n        return coco_mask.frPyObjects({"size": size, "counts": counts}, *size)\n    return {"size": size, "counts": counts}\n\n\ndef pairwise_iou(gt: Sequence[dict], pred: Sequence[dict]) -> np.ndarray:\n    if not gt or not pred:\n        return np.zeros((len(gt), len(pred)), dtype=np.float64)\n    gt, pred = list(map(canonical_rle, gt)), list(map(canonical_rle, pred))\n    shapes = {tuple(r["size"]) for r in gt + pred}\n    if len(shapes) != 1:\n        raise ValueError("GT and prediction RLE sizes differ; resize before encoding")\n    return np.asarray(coco_mask.iou(pred, gt, [0] * len(gt)), dtype=np.float64).T\n\n\ndef require_disjoint(pred: Sequence[dict], shape=(2048, 2048)) -> None:\n    """Reject invalid submissions rather than silently changing their masks."""\n    occupied = np.zeros(shape, dtype=bool)\n    for r in pred:\n        r = canonical_rle(r)\n        if tuple(r["size"]) != tuple(shape):\n            raise ValueError("Prediction has unexpected canvas size")\n        current = coco_mask.decode(r).astype(bool)\n        if not current.any():\n            raise ValueError("Empty mask is not a filament instance")\n        if (occupied & current).any():\n            raise ValueError("Prediction instances overlap")\n        occupied |= current\n\n\ndef evaluate_records(\n    records: Mapping[str, tuple[str, Sequence[dict]]],\n    predictions: Mapping[str, Sequence[dict]],\n    *, validate_predictions: bool = True, shape=(2048, 2048),\n) -> dict:\n    """One prediction set per photo, scored against EVERY annotator separately.\n\n    records: {annotator_record_id: (physical_photo_stem, list_of_GT_RLEs)}.\n    Missing predictions are [], and unknown prediction photos are rejected.\n    Explicit zero-GT records are supported using the host score\'s empty-row\n    rule. Unlike its CSV-only entry discovery, the manifest retains such rows.\n    """\n    known = {stem for stem, _ in records.values()}\n    unknown = set(predictions) - known\n    if unknown:\n        raise ValueError(f"Predictions outside evaluation split: {sorted(unknown)[:3]}")\n    for stem in known:\n        if validate_predictions:\n            require_disjoint(predictions.get(stem, []), shape)\n    total = Totals()\n    by_photo: dict[str, Totals] = {}\n    by_record = {}\n    nonzero_ious, matched_ious = [], []\n    for record_id, (stem, gt) in records.items():\n        pred = predictions.get(stem, [])\n        for r in list(gt) + list(pred):\n            if tuple(r["size"]) != tuple(shape):\n                raise ValueError("RLE canvas differs from evaluation canvas")\n        ious = pairwise_iou(gt, pred)\n        result = from_iou(ious)\n        total = total + result\n        by_photo[stem] = by_photo.get(stem, Totals()) + result\n        by_record[record_id] = result.summary()\n        nonzero_ious.extend(ious[ious > 0].tolist())\n        matched_ious.extend(ious[ious > 0.5].tolist())\n    return {\n        "pooled": total.summary(),\n        "n_photos": len(known), "n_annotator_records": len(records),\n        "n_prediction_instances_physical": sum(len(predictions.get(s, [])) for s in known),\n        "per_photo": {s: t.summary() for s, t in sorted(by_photo.items())},\n        "per_record": by_record,\n        "positive_pair_ious": nonzero_ious,\n        "positive_pair_dices": [2 * x / (1 + x) for x in nonzero_ious],\n        "matched_pair_ious": matched_ious,\n        "notes": "Pooled all-pairs host semantics; no one-to-one assignment. Empty total returns 0.",\n    }\n', 'solar_v1/preprocess.py': '"""Disk fitting is adapted from ameyypawar/filament-seg (MIT; license bundled)."""\nfrom functools import lru_cache\nfrom pathlib import Path\nimport hashlib\nimport os\nimport cv2\nimport numpy as np\nfrom .config import digest\nfrom .disk import detect_disk\n\n\ndef read_gray(path):\n    image=cv2.imread(str(path),cv2.IMREAD_GRAYSCALE)\n    if image is None:raise FileNotFoundError(path)\n    return image\n\n\nclass SolarPreprocessor:\n    VERSION=\'solar-v1-radial-median-edgepad-2\'\n\n    def __init__(self,root,enabled=True):\n        self.root=Path(root);self.root.mkdir(parents=True,exist_ok=True)\n        self.enabled=enabled\n\n    def process(self,image):\n        disk=detect_disk(image)\n        radial=disk.radius_map(image.shape).astype(np.float32)\n        inside=radial<=1.\n        profile=np.full(128,np.nan,np.float32)\n        # Sample every second pixel: robust radial statistics without a large sort.\n        r=radial[::2,::2];v=image[::2,::2].astype(np.float32)\n        bins=np.minimum((r*128).astype(int),127)\n        for b in range(128):\n            values=v[(bins==b)&(r<=1.)]\n            if len(values):profile[b]=np.median(values)\n        valid=np.isfinite(profile)&(profile>0)\n        if valid.any():profile=np.interp(np.arange(128),np.flatnonzero(valid),profile[valid]).astype(np.float32)\n        else:profile.fill(max(float(image.mean()),1.))\n        profile=np.convolve(np.pad(profile,(4,4),mode=\'edge\'),np.ones(9)/9,mode=\'valid\').astype(np.float32)\n        background=np.interp(np.clip(radial*127,0,127),np.arange(128),profile).astype(np.float32)\n        corrected=image.astype(np.float32)/np.maximum(background,1.)\n        values=corrected[inside]\n        lo,hi=np.percentile(values,[1,99.5]) if len(values) else (0.,1.)\n        corrected=np.clip((corrected-lo)/max(float(hi-lo),.05),0,1)\n        corrected[~inside]=0\n        if not self.enabled:corrected=image.astype(np.float32)/255.\n        return {\'raw\':image,\'corrected\':np.rint(corrected*255).astype(np.uint8),\'disk_mask\':inside.astype(np.uint8),\n                \'disk\':np.array([disk.cx,disk.cy,disk.radius],np.float64),\'profile\':profile,\n                \'pixel_hash\':np.asarray(hashlib.sha256(image.tobytes()).hexdigest())}\n\n    def path(self,path):\n        p=Path(path);s=p.stat()\n        key=digest([self.VERSION,str(p.resolve()),s.st_size,s.st_mtime_ns,self.enabled])[:24]\n        return self.root/(p.stem+\'_\'+key+\'.npz\')\n\n    def ensure(self,path):\n        dest=self.path(path)\n        if not dest.exists():\n            data=self.process(read_gray(path))\n            tmp=dest.with_name(dest.stem+f\'.{os.getpid()}.tmp.npz\')\n            np.savez_compressed(tmp,**data);os.replace(tmp,dest)\n        return dest\n\n    @lru_cache(maxsize=2)\n    def load(self,path):\n        with np.load(self.ensure(path),allow_pickle=False) as z:\n            return {k:z[k] for k in z.files}\n\n\ndef radial_window(data,bounds=None):\n    h,w=data[\'raw\'].shape\n    x0,y0,x1,y1=(0,0,w,h) if bounds is None else map(int,bounds)\n    cx,cy,r=data[\'disk\']\n    yy,xx=np.ogrid[y0:y1,x0:x1]\n    return np.clip(np.hypot(xx-cx,yy-cy)/max(r,1.),0,1).astype(np.float32)\n\n\ndef features(data,mode,bounds=None,size=None):\n    h,w=data[\'raw\'].shape\n    x0,y0,x1,y1=(0,0,w,h) if bounds is None else map(int,bounds)\n    raw=data[\'raw\'][y0:y1,x0:x1].astype(np.float32)/255.\n    corrected=data[\'corrected\'][y0:y1,x0:x1].astype(np.float32)/255.\n    if mode==\'raw_repeat\':array=np.repeat(raw[...,None],3,axis=2)\n    elif mode==\'corrected_repeat\':array=np.repeat(corrected[...,None],3,axis=2)\n    elif mode==\'raw_corrected_radial\':array=np.stack([raw,corrected,radial_window(data,bounds)],axis=-1)\n    else:raise ValueError(mode)\n    if size is not None:\n        interp=cv2.INTER_AREA if min(array.shape[:2])>size else cv2.INTER_LINEAR\n        array=cv2.resize(array,(size,size),interpolation=interp)\n    return np.ascontiguousarray(array.transpose(2,0,1))\n\n\ndef fit_normalization(c,preprocessor,photos):\n    if c.INPUT_MODE!=\'raw_corrected_radial\':\n        return {\'mean\':[.485,.456,.406],\'std\':[.229,.224,.225],\'source\':\'ImageNet RGB convention\'}\n    total=np.zeros(3);squared=np.zeros(3);n=0\n    for p in photos:\n        f=features(preprocessor.load(p[\'path\']),c.INPUT_MODE,size=128).reshape(3,-1).astype(np.float64)\n        total+=f.sum(1);squared+=(f*f).sum(1);n+=f.shape[1]\n    mean=total/n;std=np.sqrt(np.maximum(squared/n-mean**2,1e-4))\n    return {\'mean\':mean.tolist(),\'std\':std.tolist(),\'source\':\'training physical photos only; full-frame pixels\'}\n', 'solar_v1/reference_models.py': '"""Architectures extracted unchanged from Nomannic19\'s public reference notebook.\nSource: https://www.kaggle.com/code/nomannic19/cv-solar-filament-mask-r-cnn-u-net-refiner\nReference notebook is Apache-2.0. No pretrained weights are downloaded by this module.\nOnly class/build/crop definitions are included; no training code is executed.\n"""\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torchvision.models import resnet18, ResNet18_Weights\nfrom torchvision.models.detection import maskrcnn_resnet50_fpn_v2, MaskRCNN_ResNet50_FPN_V2_Weights\nfrom torchvision.models.detection.faster_rcnn import FastRCNNPredictor\nfrom torchvision.models.detection.mask_rcnn import MaskRCNNPredictor\nfrom torchvision.models.detection.anchor_utils import AnchorGenerator\nfrom torchvision.ops import MultiScaleRoIAlign\nIMAGE_SIZE = 1024\n\ndef build_model(pretrained=True):\n    weights = MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT if pretrained else None\n    model = maskrcnn_resnet50_fpn_v2(weights=weights, weights_backbone=None)\n\n    box_features = model.roi_heads.box_predictor.cls_score.in_features\n    mask_features = model.roi_heads.mask_predictor.conv5_mask.in_channels\n    hidden = model.roi_heads.mask_predictor.conv5_mask.out_channels\n    model.roi_heads.box_predictor = FastRCNNPredictor(box_features, 2)\n    model.roi_heads.mask_predictor = MaskRCNNPredictor(mask_features, hidden, 2)\n\n    model.rpn.anchor_generator = AnchorGenerator(sizes=((16,), (32,), (64,), (128,), (256,)), aspect_ratios=((0.5, 1.0, 2.0),) * 5)\n    model.roi_heads.mask_roi_pool = MultiScaleRoIAlign(featmap_names=["0", "1", "2", "3"], output_size=28, sampling_ratio=2)\n    model.transform.min_size, model.transform.max_size = (IMAGE_SIZE,), IMAGE_SIZE\n    return model\n\ndef square_bounds(mask, context=1.8, minimum=96):\n    height, width = mask.shape\n    ys, xs = np.where(mask)\n    center_x, center_y = (xs.min() + xs.max() + 1) / 2, (ys.min() + ys.max() + 1) / 2\n    side = int(np.ceil(max(xs.max() - xs.min() + 1, ys.max() - ys.min() + 1) * context))\n    side = min(max(side, minimum), min(height, width))\n    x0, y0 = int(round(center_x - side / 2)), int(round(center_y - side / 2))\n    x1, y1 = x0 + side, y0 + side\n\n    if x0 < 0:\n        x1 -= x0; x0 = 0\n    if y0 < 0:\n        y1 -= y0; y0 = 0\n    if x1 > width:\n        x0 -= x1 - width; x1 = width\n    if y1 > height:\n        y0 -= y1 - height; y1 = height\n\n    return int(x0), int(y0), int(x1), int(y1)\n\nclass ConvBlock(nn.Sequential):\n    def __init__(self, in_channels, out_channels):\n        super().__init__(nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True), nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True))\n\nclass CropUNet(nn.Module):\n    def __init__(self, pretrained=True):\n        super().__init__()\n        encoder = resnet18(weights=ResNet18_Weights.DEFAULT if pretrained else None)\n        self.stem = nn.Sequential(encoder.conv1, encoder.bn1, encoder.relu)\n        self.pool, self.layer1, self.layer2, self.layer3, self.layer4 = encoder.maxpool, encoder.layer1, encoder.layer2, encoder.layer3, encoder.layer4\n        self.up4, self.dec4 = nn.ConvTranspose2d(512, 256, 2, 2), ConvBlock(512, 256)\n        self.up3, self.dec3 = nn.ConvTranspose2d(256, 128, 2, 2), ConvBlock(256, 128)\n        self.up2, self.dec2 = nn.ConvTranspose2d(128, 64, 2, 2), ConvBlock(128, 64)\n        self.up1, self.dec1 = nn.ConvTranspose2d(64, 32, 2, 2), ConvBlock(96, 32)\n        self.up0, self.head = nn.ConvTranspose2d(32, 16, 2, 2), nn.Conv2d(16, 1, 1)\n        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))\n        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))\n\n    def forward(self, image):\n        image = (image - self.mean) / self.std\n        x0 = self.stem(image)\n        x1 = self.layer1(self.pool(x0))\n        x2 = self.layer2(x1)\n        x3 = self.layer3(x2)\n        x4 = self.layer4(x3)\n        x = self.dec4(torch.cat([self.up4(x4), x3], 1))\n        x = self.dec3(torch.cat([self.up3(x), x2], 1))\n        x = self.dec2(torch.cat([self.up2(x), x1], 1))\n        x = self.dec1(torch.cat([self.up1(x), x0], 1))\n        return self.head(self.up0(x))\n', 'solar_v1/submission.py': "from pathlib import Path\nimport pandas as pd\nfrom pycocotools import mask as cm\nfrom .config import atomic_json\nfrom .pq import require_disjoint\n\n\ndef export_instances(root,stem,instances):\n    rows=[]\n    for x in instances:\n        r=x.rle;counts=r['counts'].decode('ascii') if isinstance(r['counts'],bytes) else r['counts']\n        rows.append({'score':float(x.score),'quality':float(x.quality),'rle':{'size':list(map(int,r['size'])),'counts':counts}})\n    atomic_json(Path(root)/(stem+'.json'),{'photo':stem,'instances':rows})\n\n\nclass SubmissionWriter:\n    def __init__(self,path,test_ids,shape=(2048,2048),coverage_path=None):\n        self.path=Path(path);self.ids=set(test_ids);self.shape=shape\n        if len(self.ids)!=len(test_ids):raise ValueError('Duplicate test observations')\n        self.coverage_path=Path(coverage_path) if coverage_path else self.path.with_name('submission_coverage.json')\n        self.rows=[];self.coverage={}\n\n    def add(self,stem,instances):\n        if stem not in self.ids or stem in self.coverage:raise ValueError('Unknown or repeated test photograph')\n        rles=[x.rle for x in instances];require_disjoint(rles,self.shape)\n        for number,r in enumerate(rles,1):\n            counts=r['counts'].decode('ascii') if isinstance(r['counts'],bytes) else r['counts']\n            # Decode check also rejects malformed/truncated COCO RLEs.\n            mask=cm.decode({'size':list(self.shape),'counts':counts.encode('ascii')})\n            if mask.shape!=self.shape or not mask.any():raise ValueError('Invalid submission mask')\n            self.rows.append({'filament_id':f'{stem}_{number}','segmentation_rle':counts})\n        # Zero-instance photos are represented in coverage, not by fabricated CSV masks.\n        self.coverage[stem]=len(rles)\n\n    def finish(self):\n        if set(self.coverage)!=self.ids:raise ValueError(f'Unprocessed test photographs: {sorted(self.ids-set(self.coverage))[:5]}')\n        frame=pd.DataFrame(self.rows,columns=['filament_id','segmentation_rle'])\n        if not frame.filament_id.is_unique:raise ValueError('Duplicate filament_id')\n        self.path.parent.mkdir(parents=True,exist_ok=True);frame.to_csv(self.path,index=False)\n        atomic_json(self.coverage_path,{'all_test_photos_processed':True,'n_photos':len(self.ids),\n            'n_instances':len(frame),'empty_photos':[s for s,n in self.coverage.items() if n==0],\n            'counts_by_photo':self.coverage,'shape':list(self.shape)})\n        return frame\n", 'solar_v12a/__init__.py': '"""V1.2a: frozen V1 models, supervised mask selection and legal submission."""\n__version__ = \'1.2a.0\'\n', 'solar_v12a/features.py': '"""Native-mask paired features; identical GT-free extraction for validation and test."""\nimport cv2\nimport numpy as np\nfrom pycocotools import mask as cm\nfrom solar_v1.inference import native_coarse\nfrom solar_v1.postprocess import encode\nfrom solar_v1.pq import canonical_rle\n\nFEATURE_VERSION = \'paired-native-features-1\'\nGEOMETRY = [\'valid\', \'log_area\', \'area_fraction\', \'bbox_aspect\', \'bbox_extent\',\n            \'n_components\', \'largest_component_fraction\', \'perimeter_normalized\',\n            \'circularity\', \'elongation\', \'cx\', \'cy\', \'radial\']\nLOCAL = [\'raw_mean\', \'raw_std\', \'corrected_mean\', \'corrected_std\', \'raw_ring_delta\',\n         \'prob_mean\', \'prob_std\', \'prob_p10\', \'prob_p50\', \'prob_p90\', \'prob_entropy\']\nCOMMON = [\'det_score\', \'box_width\', \'box_height\', \'box_aspect\', \'box_cx\', \'box_cy\',\n          \'coarse_refined_iou\', \'log_refine_area_ratio\', \'area_change\', \'refiner_fallback\']\nFEATURE_NAMES = COMMON + [\'coarse_\'+x for x in GEOMETRY] + [\'refined_\'+x for x in GEOMETRY] + LOCAL + [\'is_refined\']\n\n\ndef serial_rle(rle):\n    if rle is None:\n        return None\n    return {\'size\': list(map(int, rle[\'size\'])),\n            \'counts\': rle[\'counts\'].decode(\'ascii\') if isinstance(rle[\'counts\'], bytes) else rle[\'counts\']}\n\n\ndef decode(rle, shape=None):\n    if rle is None:\n        if shape is None:\n            raise ValueError(\'Empty RLE needs canvas shape.\')\n        return np.zeros(shape, bool)\n    return cm.decode(canonical_rle(rle)).astype(bool)\n\n\ndef disk_mask(data, c):\n    if not c.PREPROCESS:\n        return None\n    h, w = data[\'raw\'].shape\n    yy, xx = np.ogrid[:h, :w]\n    cx, cy, radius = data[\'disk\']\n    return (xx-cx)**2 + (yy-cy)**2 <= (radius*c.DISK_MARGIN)**2\n\n\ndef clean(mask, disk, c):\n    mask = mask.copy()\n    if disk is not None:\n        mask &= disk\n    if c.MORPH_CLOSE_RADIUS:\n        r = c.MORPH_CLOSE_RADIUS\n        mask = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_CLOSE,\n            cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2*r+1, 2*r+1))).astype(bool)\n        if disk is not None:\n            mask &= disk\n    return mask\n\n\ndef geometry(mask, data):\n    h, w = mask.shape\n    ys, xs = np.nonzero(mask)\n    if not len(xs):\n        return [0.] * len(GEOMETRY)\n    area = len(xs)\n    x0, y0, x1, y1 = int(xs.min()), int(ys.min()), int(xs.max())+1, int(ys.max())+1\n    crop = mask[y0:y1, x0:x1].astype(np.uint8)\n    n, _, stats, _ = cv2.connectedComponentsWithStats(crop, 8)\n    contours, _ = cv2.findContours(crop, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)\n    perimeter = sum(cv2.arcLength(v, True) for v in contours)\n    covariance = np.cov(np.stack([xs, ys]), bias=True) if area > 1 else np.zeros((2, 2))\n    eigen = np.maximum(np.linalg.eigvalsh(covariance), 0)\n    elongation = np.sqrt((eigen[-1]+1) / (eigen[0]+1))\n    cx, cy = float(xs.mean()), float(ys.mean())\n    dx, dy, radius = map(float, data[\'disk\'])\n    return [1., np.log1p(area), area/(h*w), max(x1-x0, y1-y0)/max(min(x1-x0, y1-y0), 1),\n            area/((x1-x0)*(y1-y0)), float(n-1), float(stats[1:, cv2.CC_STAT_AREA].max())/area,\n            perimeter/np.sqrt(area), 4*np.pi*area/max(perimeter*perimeter, 1), float(elongation),\n            cx/w, cy/h, np.hypot(cx-dx, cy-dy)/max(radius, 1)]\n\n\ndef local_features(mask, probability, data):\n    if not mask.any():\n        return [0.] * len(LOCAL)\n    raw = data[\'raw\'][mask].astype(np.float32)/255\n    corrected = data[\'corrected\'][mask].astype(np.float32)/255\n    p = np.clip(probability[mask].astype(np.float32), 1e-6, 1-1e-6)\n    ys, xs = np.nonzero(mask)\n    h, w = mask.shape\n    x0, x1 = max(0, int(xs.min())-4), min(w, int(xs.max())+5)\n    y0, y1 = max(0, int(ys.min())-4), min(h, int(ys.max())+5)\n    crop = mask[y0:y1, x0:x1].astype(np.uint8)\n    ring = cv2.dilate(crop, np.ones((7, 7), np.uint8)).astype(bool) & ~crop.astype(bool)\n    background = data[\'raw\'][y0:y1, x0:x1][ring].astype(np.float32)/255\n    return [float(raw.mean()), float(raw.std()), float(corrected.mean()), float(corrected.std()),\n            float(raw.mean()-background.mean()) if len(background) else 0.,\n            float(p.mean()), float(p.std()), *map(float, np.quantile(p, [.1, .5, .9])),\n            float((-(p*np.log(p)+(1-p)*np.log(1-p))).mean())]\n\n\ndef extract_candidates(pack, data, c):\n    shape = tuple(map(int, pack[\'shape\']))\n    if shape != data[\'raw\'].shape or str(pack[\'pixel_hash\']) != str(data[\'pixel_hash\']):\n        raise ValueError(\'Prediction pack belongs to different image pixels/shape.\')\n    if \'ref_probs\' not in pack or len(pack[\'ref_probs\']) != len(pack[\'scores\']):\n        raise ValueError(\'Both original detector and refiner probabilities are required.\')\n    disk = disk_mask(data, c)\n    result = []\n    for i, score in enumerate(pack[\'scores\']):\n        coarse_probability = native_coarse(pack, i)\n        coarse = clean(coarse_probability >= c.MASK_THRESHOLD, disk, c)\n        refined_probability = np.zeros(shape, np.float32)\n        x0, y0, x1, y1 = map(int, pack[\'ref_bounds\'][i])\n        if not (0 <= x0 < x1 <= shape[1] and 0 <= y0 < y1 <= shape[0]):\n            raise ValueError(\'Refiner crop is outside original image.\')\n        refined_probability[y0:y1, x0:x1] = cv2.resize(pack[\'ref_probs\'][i], (x1-x0, y1-y0), interpolation=cv2.INTER_LINEAR)\n        mask = refined_probability >= c.REFINER_THRESHOLD\n        fallback = bool(not mask.any() and c.EMPTY_REFINER_FALLBACK)\n        if fallback:\n            mask = coarse_probability >= c.MASK_THRESHOLD\n        refined = clean(mask, disk, c)\n        coarse_area, refined_area = int(coarse.sum()), int(refined.sum())\n        intersection = int((coarse & refined).sum())\n        union = coarse_area+refined_area-intersection\n        box = list(map(float, pack[\'boxes\'][i]))\n        bw, bh = max(box[2]-box[0], 0), max(box[3]-box[1], 0)\n        common = [float(score), bw/shape[1], bh/shape[0], max(bw, bh)/max(min(bw, bh), 1),\n                  (box[0]+box[2])/(2*shape[1]), (box[1]+box[3])/(2*shape[0]),\n                  intersection/max(union, 1), np.log((refined_area+1)/(coarse_area+1)),\n                  (refined_area-coarse_area)/max(coarse_area, 1), float(fallback)]\n        paired = common + geometry(coarse, data) + geometry(refined, data)\n        xc = paired + local_features(coarse, coarse_probability, data) + [0.]\n        xr = paired + local_features(refined, refined_probability, data) + [1.]\n        if len(xc) != len(FEATURE_NAMES) or not np.isfinite([xc, xr]).all():\n            raise ValueError(\'Invalid or misaligned quality features.\')\n        result.append({\'candidate_id\': i, \'det_score\': float(score), \'box\': box,\n                       \'crop\': [x0, y0, x1, y1], \'fallback\': fallback,\n                       \'coarse_mask\': serial_rle(encode(coarse)) if coarse_area else None,\n                       \'refined_mask\': serial_rle(encode(refined)) if refined_area else None,\n                       \'coarse_features\': list(map(float, xc)), \'refined_features\': list(map(float, xr)),\n                       \'coarse_area\': coarse_area, \'refined_area\': refined_area,\n                       \'refiner_probability\': {\'cache_field\': \'ref_probs\', \'index\': i,\n                           \'mean_in_mask\': float(refined_probability[refined].mean()) if refined_area else 0.},\n                       \'shape\': list(shape)})\n    return result\n', 'solar_v12a/io.py': '"""Read-only V1 inputs, leakage-safe roles, portable hash-checked artifacts."""\nfrom dataclasses import asdict, fields, replace\nimport importlib.metadata\nimport json\nimport os\nfrom pathlib import Path\nimport platform\nimport shutil\nimport time\n\nimport numpy as np\nfrom sklearn.model_selection import GroupShuffleSplit\nimport torch\n\nfrom solar_v1.config import Config as V1Config, atomic_json, digest, file_hash, architecture\nfrom solar_v1.data import image_files\n\n\ndef source_hashes():\n    root = Path(__file__).resolve().parents[1]\n    return {f\'{name}/{p.name}\': file_hash(p) for name in [\'solar_v1\', \'solar_v12a\']\n            for p in sorted((root / name).glob(\'*.py\'))}\n\n\ndef frozen_hashes():\n    root = Path(__file__).resolve().parents[1] / \'solar_v1\'\n    return {p.name: file_hash(p) for p in sorted(root.glob(\'*.py\'))}\n\n\ndef resolve_v1(explicit=\'\', input_root=\'/kaggle/input\'):\n    required = [\'config.json\', \'manifest.json\', \'normalization.json\',\n                \'checkpoints/detector_best.pt\', \'checkpoints/refiner_best.pt\']\n    def is_v1(root):\n        if not all((root / n).is_file() for n in required):\n            return False\n        saved = json.loads((root / \'config.json\').read_text())\n        return \'REFINER_INPUT_MODE\' not in saved and \'MERGE_STRATEGY\' not in saved\n    if explicit:\n        root = Path(explicit)\n        if not is_v1(root):\n            raise ValueError(\'V1_SOURCE must point to the completed original V1, not V1.1/preflight.\')\n        return root.resolve()\n    roots = sorted({p.parent.resolve() for p in Path(input_root).rglob(\'config.json\') if is_v1(p.parent)})\n    if len(roots) != 1:\n        raise ValueError(f\'Expected one complete V1 output; found {roots}. Set V1_SOURCE explicitly.\')\n    return roots[0]\n\n\ndef protect_output(out, source):\n    out, source = Path(out).resolve(), Path(source).resolve()\n    if out == source or source in out.parents or out in source.parents:\n        raise ValueError(\'V1.2a output must be separate from V1 input.\')\n    if \'/kaggle/input\' in out.as_posix() or any(p.name in {\'solar_v1\', \'solar_v11\', \'solar_v1_runtime\', \'solar_v11_runtime\'}\n                                               for p in [out, *out.parents]):\n        raise ValueError(\'Cannot write into mounted data or another experiment.\')\n    out.mkdir(parents=True, exist_ok=True)\n    return out\n\n\ndef load_base(c, source):\n    saved = json.loads((source / \'config.json\').read_text())\n    allowed = {f.name for f in fields(V1Config)}\n    if set(saved) - allowed:\n        raise ValueError(\'Source contains non-V1 configuration fields.\')\n    baseline = V1Config(**saved).validate()\n    path = source / \'calibration/thresholds.json\'\n    if not path.is_file():\n        raise FileNotFoundError(\'Attach completed V1 calibration thresholds; do not guess its baseline.\')\n    baseline = replace(baseline, **json.loads(path.read_text())[\'deployment_thresholds\'])\n    if baseline.DETECTOR_IMAGE_SIZE != 1024 or baseline.REFINER_SIZE != 384:\n        raise ValueError(\'V1.2a requires the original 1024 detector and 384 refiner.\')\n    if baseline.ENABLE_TILE_BRANCH or baseline.ENSEMBLE_CACHE_DIRS or baseline.REFIT_ALL_DATA:\n        raise ValueError(\'Expected the original two-stage Fold 0 model without Tile/ensembles/refit-all.\')\n    if baseline.POSTPROCESS_MODE != \'mask_nms\' or baseline.FRAGMENT_MODE != \'keep\':\n        raise ValueError(\'This audited baseline requires hard mask-NMS and keep-fragments mode.\')\n    if baseline.QUALITY_WEIGHT != 0 or baseline.MAX_DETECTIONS != 80:\n        raise ValueError(\'Unexpected V1 quality/candidate-cap baseline; investigate before comparing.\')\n    norm = json.loads((source / \'normalization.json\').read_text())\n    manifest = json.loads((source / \'manifest.json\').read_text())\n    if norm.get(\'split_hash\') != manifest[\'split_hash\']:\n        raise ValueError(\'V1 normalization and split mismatch.\')\n    weights = {stage: file_hash(source / f\'checkpoints/{stage}_best.pt\') for stage in [\'detector\', \'refiner\']}\n    for stage in weights:\n        state = torch.load(source / f\'checkpoints/{stage}_best.pt\', map_location=\'cpu\', weights_only=True)\n        val_ids = {p[\'stem\'] for p in manifest[\'photos\'] if p[\'split\'] == \'val\'}\n        trained = set(state.get(\'trained_photo_ids\', []))\n        if not trained or trained & val_ids or state.get(\'all_data\'):\n            raise ValueError(f\'{stage} weights are not training-excluded for V1 validation photos.\')\n        if state.get(\'architecture\') != architecture(baseline, stage):\n            raise ValueError(\'V1 checkpoint architecture differs from its saved configuration.\')\n        if any(not np.allclose(state[\'normalization\'][k], norm[k], rtol=1e-6, atol=1e-7) for k in [\'mean\', \'std\']):\n            raise ValueError(\'Source checkpoint normalization mismatch.\')\n        del state\n    base = replace(baseline, INPUT_ROOT=c.INPUT_ROOT, OUTPUT_ROOT=c.OUTPUT_ROOT,\n                   CHECKPOINT_DIR=str(source / \'checkpoints\'), RESUME_SOURCE=\'\',\n                   TRAIN_DETECTOR=False, TRAIN_REFINER=False, TRAIN_TILE=False,\n                   PRETRAINED=False, ALLOW_WEIGHT_DOWNLOAD=False, RUN_MODE=\'validate\',\n                   CACHE_SCORE_FLOOR=c.BASE_CANDIDATE_FLOOR,\n                   INFERENCE_BATCH_SIZE=c.REFINER_BATCH_SIZE)\n    return baseline, base, norm, manifest, weights\n\n\ndef train_paths(c, base):\n    ann = Path(c.ANNOTATION_PATH or base.ANNOTATION_PATH or \'__unset__\')\n    if not ann.is_file():\n        found = list(Path(c.INPUT_ROOT).rglob(\'MAGFiLO_1.0_Annotations_kaggle2026_train.json\'))\n        if len(found) != 1:\n            raise ValueError(\'Attach competition data or set ANNOTATION_PATH to the one train JSON.\')\n        ann = found[0]\n    train = Path(c.TRAIN_IMAGE_DIR) if c.TRAIN_IMAGE_DIR else ann.parent / \'train_images\'\n    if not train.is_dir():\n        raise FileNotFoundError(f\'Training image directory missing: {train}\')\n    return replace(base, ANNOTATION_PATH=str(ann), TRAIN_IMAGE_DIR=str(train))\n\n\ndef test_paths(c):\n    # Deliberately no annotation/Catalog construction in this function.\n    roots = [Path(c.TEST_IMAGE_DIR)] if c.TEST_IMAGE_DIR else sorted(\n        {p.resolve() for p in Path(c.INPUT_ROOT).rglob(\'test_images\') if p.is_dir()})\n    if len(roots) != 1 or not roots[0].is_dir():\n        raise ValueError(\'Set TEST_IMAGE_DIR to the unique official test_images directory.\')\n    paths = image_files(roots[0])\n    if not paths or len({p.stem for p in paths}) != len(paths):\n        raise ValueError(\'No test images or duplicated test photograph IDs.\')\n    return paths\n\n\ndef split_roles(photos, c):\n    if len({p[\'stem\'] for p in photos}) != len(photos):\n        raise ValueError(\'Role split requires unique physical photographs, not annotation rows.\')\n    # Join original V1 date/duplicate groups and any repeated pixel hashes.\n    parent = {p[\'stem\']: p[\'stem\'] for p in photos}\n    def find(x):\n        while parent[x] != x:\n            parent[x] = parent[parent[x]]\n            x = parent[x]\n        return x\n    seen = {}\n    for p in photos:\n        keys = [(\'date\', p[\'date\']), (\'v1_group\', p.get(\'group\', p[\'date\']))]\n        if p.get(\'pixel_hash\'):\n            keys.append((\'pixels\', str(p[\'pixel_hash\'])))\n        for key in keys:\n            if key in seen:\n                parent[find(p[\'stem\'])] = find(seen[key])\n            seen[key] = p[\'stem\']\n    groups = np.asarray([find(p[\'stem\']) for p in photos])\n    if len(set(groups)) < 8:\n        raise ValueError(\'Need at least eight independent date/pixel groups for the three roles.\')\n    idx = np.arange(len(photos))\n    test_fraction = 1 - c.QUALITY_FIT_FRACTION - c.CALIBRATION_FRACTION\n    left, hold = next(GroupShuffleSplit(n_splits=1, test_size=test_fraction,\n                                      random_state=c.SEED).split(idx, groups=groups))\n    fit_local, cal_local = next(GroupShuffleSplit(n_splits=1,\n        test_size=c.CALIBRATION_FRACTION / (1-test_fraction), random_state=c.SEED+1\n    ).split(left, groups=groups[left]))\n    assignments = {\'quality_fit\': left[fit_local], \'calibration\': left[cal_local], \'evaluation\': hold}\n    if any(len(v) < 8 for v in assignments.values()):\n        raise ValueError(\'Too few photos in a role; do not weaken isolation to train a complex quality model.\')\n    mapping = {photos[int(i)][\'stem\']: role for role, indices in assignments.items() for i in indices}\n    rows = [{k: p[k] for k in [\'stem\', \'date\', \'records\', \'pixel_hash\'] if k in p} |\n            {\'role\': mapping[p[\'stem\']], \'group\': str(groups[i])} for i, p in enumerate(photos)]\n    if any(len({r[\'role\'] for r in rows if r[\'group\'] == g}) != 1 for g in set(groups)):\n        raise AssertionError(\'Date/pixel group crossed roles.\')\n    result = {\'seed\': c.SEED, \'photos\': rows,\n              \'counts\': {k: len(v) for k, v in assignments.items()},\n              \'limitation\': \'Downstream held-out roles on V1 checkpoint-selected validation photos; not full model OOF.\'}\n    result[\'role_hash\'] = digest(result)\n    return result\n\n\ndef runtime():\n    return {\'python\': platform.python_version(), \'torch\': str(torch.__version__),\n            \'torchvision\': importlib.metadata.version(\'torchvision\'),\n            \'sklearn\': importlib.metadata.version(\'scikit-learn\'), \'numpy\': np.__version__,\n            \'gpus\': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}\n\n\ndef check_budget(c):\n    deadline = float(os.environ.setdefault(\'SOLAR_V12A_DEADLINE\', str(time.time()+c.TIME_BUDGET_HOURS*3600)))\n    if time.time() > deadline:\n        raise TimeoutError(\'V1.2a session budget reached; preserve its output for resume/diagnosis.\')\n    if shutil.disk_usage(Path(c.OUTPUT_ROOT).anchor or \'.\').free < c.MIN_FREE_DISK_GB * 2**30:\n        raise RuntimeError(\'Insufficient disk space for remaining masks/reports. No candidates are silently discarded.\')\n\n\ndef write_bundle_manifest(root, weights, role_hash):\n    root = Path(root)\n    names = [\'quality.joblib\', \'quality_meta.json\', \'deployment.json\', \'frozen_policies.json\',\n             \'roles.json\', \'validation_report.json\', \'base_config.json\', \'normalization.json\']\n    payload = {\'version\': \'1.2a.0\', \'weights\': weights, \'role_hash\': role_hash,\n               \'sources\': source_hashes(), \'sklearn\': runtime()[\'sklearn\'],\n               \'files\': {name: file_hash(root/name) for name in names}}\n    atomic_json(root/\'bundle_manifest.json\', payload)\n    return payload\n\n\ndef read_bundle(path, c, weights):\n    if path:\n        root = Path(path)\n    else:\n        found = list(Path(c.INPUT_ROOT).rglob(\'v12a_bundle/bundle_manifest.json\'))\n        if len(found) != 1:\n            raise ValueError(\'submit requires a unique mounted v12a_bundle or explicit BUNDLE_SOURCE.\')\n        root = found[0].parent\n    payload = json.loads((root/\'bundle_manifest.json\').read_text())\n    if payload[\'version\'] != c.VERSION or payload[\'weights\'] != weights or payload[\'sources\'] != source_hashes():\n        raise ValueError(\'Bundle version, frozen model weights or runtime source differs from this implementation.\')\n    if payload[\'sklearn\'] != runtime()[\'sklearn\']:\n        raise ValueError(\'Quality model sklearn version mismatch; use the recorded runtime or retrain quality in full mode.\')\n    for name, expected in payload[\'files\'].items():\n        if Path(name).name != name or file_hash(root/name) != expected:\n            raise ValueError(f\'Bundle artifact failed integrity check: {name}\')\n    return root, payload\n', 'solar_v12a/policy.py': '"""Separate candidate admission, paired mask choice, acceptance, rank and ownership."""\nfrom dataclasses import asdict, dataclass\nimport numpy as np\nfrom pycocotools import mask as cm\nfrom solar_v1.config import digest\nfrom solar_v1.postprocess import Instance, encode\nfrom solar_v1.pq import pairwise_iou, require_disjoint\nfrom .features import decode\n\n\n@dataclass(frozen=True)\nclass Policy:\n    candidate_floor: float = .7\n    mask_mode: str = \'refined\'\n    refine_margin: float = 0.\n    accept_weight: float = 0.\n    acceptance_threshold: float = .7\n    rank_weight: float = 0.\n    quality_mode: str = \'hybrid\'\n    mask_nms_iou: float = .3\n    min_area: int = 64\n\n    def validate(self):\n        if self.mask_mode not in {\'coarse\', \'refined\', \'selective\'}:\n            raise ValueError(\'Unknown coarse/refined selection mode.\')\n        if self.quality_mode not in {\'regression\', \'pass\', \'hybrid\'}:\n            raise ValueError(\'Unknown quality combination.\')\n        if any(not 0 <= getattr(self, k) <= 1 for k in [\'candidate_floor\', \'refine_margin\',\n                \'accept_weight\', \'acceptance_threshold\', \'rank_weight\', \'mask_nms_iou\']) or self.min_area < 1:\n            raise ValueError(\'Invalid candidate policy parameters.\')\n        return self\n\n    @property\n    def needs_quality(self):\n        return self.mask_mode == \'selective\' or self.accept_weight > 0 or self.rank_weight > 0\n\n    @property\n    def key(self):\n        return digest(asdict(self))\n\n\ndef baseline_policy(c):\n    return Policy(candidate_floor=c.DETECTION_THRESHOLD, acceptance_threshold=c.DETECTION_THRESHOLD,\n                  mask_nms_iou=c.MASK_NMS_IOU, min_area=c.MIN_INSTANCE_AREA).validate()\n\n\ndef _quality(row, variant, mode):\n    q = row[\'predicted_\'+variant+\'_quality\']\n    p = row[\'predicted_\'+variant+\'_pass\']\n    return q if mode == \'regression\' else p if mode == \'pass\' else .5*(q+p)\n\n\ndef process(candidates, policy, shape):\n    policy.validate()\n    rows = []\n    for candidate in candidates:\n        row = dict(candidate)\n        if policy.needs_quality and any(\'predicted_\'+v+\'_\'+k not in row\n                for v in [\'coarse\', \'refined\'] for k in [\'quality\', \'pass\']):\n            raise ValueError(\'Policy requires a trained quality model; silent fallback is forbidden.\')\n        if policy.needs_quality and any(not np.isfinite(row[\'predicted_\'+v+\'_\'+k])\n                or not 0 <= row[\'predicted_\'+v+\'_\'+k] <= 1\n                for v in [\'coarse\', \'refined\'] for k in [\'quality\', \'pass\']):\n            raise ValueError(\'Nonfinite or out-of-range predicted quality.\')\n        if policy.mask_mode == \'selective\':\n            use_refined = row[\'predicted_refined_quality\'] > row[\'predicted_coarse_quality\'] + policy.refine_margin\n            variant = \'refined\' if use_refined else \'coarse\'\n            reason = \'predicted_refined_exceeds_margin\' if use_refined else \'coarse_conservative_margin\'\n            other = \'coarse\' if variant == \'refined\' else \'refined\'\n            if row[variant+\'_mask\'] is None and row[other+\'_mask\'] is not None:\n                variant, reason = other, \'only_nonempty_variant_available\'\n        else:\n            variant = policy.mask_mode\n            reason = \'always_\'+variant\n        # Always-coarse control must stay coarse even when its mask is empty.\n        # The V1 refined empty fallback was already reproduced at feature extraction.\n        selected = row[variant+\'_mask\']\n        q = _quality(row, variant, policy.quality_mode) if policy.needs_quality else 0.\n        acceptance = (1-policy.accept_weight)*row[\'det_score\'] + policy.accept_weight*q\n        rank = (1-policy.rank_weight)*row[\'det_score\'] + policy.rank_weight*q\n        eligible = row[\'det_score\'] >= policy.candidate_floor\n        accepted = eligible and selected is not None and acceptance >= policy.acceptance_threshold\n        row.update(selected_variant=variant, selected_mask=selected, selection_reason=reason,\n                   selected_quality=q, acceptance_score=float(acceptance), rank_score=float(rank),\n                   passes_candidate_floor=bool(eligible), accepted=bool(accepted), nms_kept=False,\n                   final_mask=None, removed_by=None, winner_id=None)\n        if not eligible:\n            row[\'removed_by\'] = \'candidate_floor\'\n        elif selected is None:\n            row[\'removed_by\'] = \'empty_selected_mask\'\n        elif not accepted:\n            row[\'removed_by\'] = \'final_acceptance\'\n        rows.append(row)\n    active = [i for i, row in enumerate(rows) if row[\'accepted\']]\n    masks = [rows[i][\'selected_mask\'] for i in active]\n    overlaps = pairwise_iou(masks, masks)\n    position = {i: j for j, i in enumerate(active)}\n    pool = sorted(active, key=lambda i: -rows[i][\'rank_score\'])\n    kept = []\n    while pool:\n        winner = pool.pop(0)\n        rows[winner][\'nms_kept\'] = True\n        kept.append(winner)\n        remaining = []\n        for loser in pool:\n            iou = float(overlaps[position[winner], position[loser]])\n            if iou > policy.mask_nms_iou:\n                rows[loser].update(removed_by=\'mask_nms\', winner_id=rows[winner][\'candidate_id\'],\n                                   suppression_iou=iou)\n            else:\n                remaining.append(loser)\n        pool = remaining\n    occupied = np.zeros(shape, bool)\n    final = []\n    for i in kept:\n        mask = decode(rows[i][\'selected_mask\']) & ~occupied\n        rows[i][\'area_after_ownership\'] = int(mask.sum())\n        if int(mask.sum()) < policy.min_area:\n            rows[i][\'removed_by\'] = \'pixel_ownership_or_area\'\n            continue\n        occupied |= mask\n        rle = encode(mask)\n        rows[i][\'final_mask\'] = {\'size\': rle[\'size\'], \'counts\': rle[\'counts\'].decode(\'ascii\')}\n        final.append(Instance(rows[i][\'rank_score\'], rle, rows[i][\'selected_quality\'],\n                              source=\'v12a:\'+str(rows[i][\'candidate_id\'])))\n    final.sort(key=lambda x: (float(cm.toBbox(x.rle)[1]), float(cm.toBbox(x.rle)[0])))\n    require_disjoint([v.rle for v in final], shape)\n    return final, rows\n', 'solar_v12a/submission.py': '"""Fresh test-image inference and original V1 compressed-RLE writer plus independent checks."""\nfrom dataclasses import asdict\nimport json\nfrom pathlib import Path\nimport time\nimport pandas as pd\nfrom pycocotools import mask as cm\n\nfrom solar_v1.config import atomic_json, digest, file_hash\nfrom solar_v1.pq import canonical_rle, require_disjoint\nfrom solar_v1.submission import SubmissionWriter\nfrom .io import check_budget\nfrom .policy import process\n\n\ndef protect_existing_submission(c):\n    path = Path(c.SUBMISSION_PATH)\n    proof = path.parent/\'v12a_submission_verification.json\'\n    if path.exists() or Path(c.COVERAGE_PATH).exists():\n        if not path.is_file() or not proof.is_file():\n            raise ValueError(\'Existing submission/coverage belongs to an unverified experiment. Use a fresh Kaggle Notebook session to preserve V1.\')\n        old = json.loads(proof.read_text())\n        if old.get(\'pipeline_version\') != c.VERSION or old.get(\'csv_sha256\') != file_hash(path):\n            raise ValueError(\'Refusing to overwrite an existing non-V1.2a submission.csv.\')\n\n\ndef verify_csv(path, coverage_path, expected_photos, shapes):\n    path = Path(path)\n    frame = pd.read_csv(path, dtype=str, keep_default_na=False)\n    if list(frame.columns) != [\'filament_id\', \'segmentation_rle\']:\n        raise ValueError(\'Submission header must be filament_id,segmentation_rle without an index column.\')\n    if frame.empty or frame.eq(\'\').any().any() or not frame.filament_id.is_unique:\n        raise ValueError(\'Empty submission, missing fields, or duplicated filament_id.\')\n    coverage = json.loads(Path(coverage_path).read_text())\n    if not coverage[\'all_test_photos_processed\'] or set(coverage[\'counts_by_photo\']) != set(expected_photos):\n        raise ValueError(\'Missing or extra test photos in submission coverage.\')\n    groups = {stem: [] for stem in expected_photos}\n    expected_ids = {f\'{stem}_{i}\' for stem, n in coverage[\'counts_by_photo\'].items() for i in range(1, n+1)}\n    if set(frame.filament_id) != expected_ids:\n        raise ValueError(\'Instance ID format/sequence does not match per-photo coverage.\')\n    for row in frame.itertuples(index=False):\n        stem, number = row.filament_id.rsplit(\'_\', 1)\n        if stem not in groups or not number.isdigit() or int(number) < 1:\n            raise ValueError(\'Invalid filament_id format.\')\n        rle = {\'size\': list(shapes[stem]), \'counts\': row.segmentation_rle.encode(\'ascii\')}\n        mask = cm.decode(rle)\n        if tuple(mask.shape) != tuple(shapes[stem]) or not mask.any():\n            raise ValueError(\'Submission RLE is empty or has the wrong native image dimensions.\')\n        groups[stem].append(rle)\n    for stem, rles in groups.items():\n        require_disjoint(rles, tuple(shapes[stem]))\n    if len(frame) != coverage[\'n_instances\'] or path.stat().st_size <= len(\'filament_id,segmentation_rle\\n\'):\n        raise ValueError(\'CSV row count/size differs from final prediction coverage.\')\n    return {\'checks_passed\': True, \'n_test_photos\': len(groups), \'n_instances\': len(frame),\n            \'empty_photos\': [s for s, rs in groups.items() if not rs],\n            \'columns\': list(frame.columns), \'csv_sha256\': file_hash(path), \'csv_bytes\': path.stat().st_size,\n            \'checks\': [\'all_test_photos_once\', \'unique_exact_instance_ids\', \'nonempty_decodable_compressed_rle\',\n                       \'native_image_dimensions\', \'nonoverlapping_masks\', \'no_index_column\', \'coverage_row_counts\'],\n            \'row_count_hardcoded\': False}\n\n\ndef generate(c, paths, policy, quality_model, inference, preprocessor, provenance):\n    # This function receives no Catalog, annotation path, GT provider or target rows.\n    protect_existing_submission(c)\n    ids = [p.stem for p in paths]\n    if len(set(ids)) != len(ids) or not ids:\n        raise ValueError(\'Test image list is empty or duplicated.\')\n    writer = SubmissionWriter(c.SUBMISSION_PATH, ids, coverage_path=c.COVERAGE_PATH)\n    shapes, per_photo = {}, []\n    trace_root = Path(c.OUTPUT_ROOT)/\'test_candidate_lifecycle\'/inference.test_run_id\n    trace_root.mkdir(parents=True, exist_ok=True)\n    for index, path in enumerate(paths, 1):\n        check_budget(c)\n        started = time.perf_counter()\n        data = preprocessor.load(str(path))\n        candidates = inference.test_candidates(path, data)\n        # Always load/check/apply the trained head. Explicit gate-selected fallback\n        # can intentionally ignore its outputs; that is recorded below.\n        candidates = quality_model.apply(candidates)\n        instances, trace = process(candidates, policy, data[\'raw\'].shape)\n        shapes[path.stem] = list(data[\'raw\'].shape)\n        writer.shape = tuple(data[\'raw\'].shape)\n        writer.add(path.stem, instances)\n        atomic_json(trace_root/(path.stem+\'.json\'), {\'photo\': path.stem, \'shape\': shapes[path.stem],\n            \'policy\': asdict(policy), \'candidates\': trace, \'gt_access\': False,\n            \'new_detector_and_refiner_forward\': True})\n        per_photo.append({\'photo\': path.stem, \'n_candidates\': len(candidates), \'n_instances\': len(instances),\n                          \'seconds\': time.perf_counter()-started,\n                          \'selected_coarse\': sum(r[\'selected_variant\'] == \'coarse\' for r in trace),\n                          \'selected_refined\': sum(r[\'selected_variant\'] == \'refined\' for r in trace)})\n        print(f\'Test {index}/{len(paths)} {path.stem}: {len(candidates)} candidates -> {len(instances)} instances\', flush=True)\n    if inference.test_forward_ids != ids:\n        raise ValueError(\'Fresh test forward coverage differs from the official test image list.\')\n    writer.finish()\n    coverage = json.loads(Path(c.COVERAGE_PATH).read_text())\n    coverage.update(shapes_by_photo=shapes, test_forward_run_id=inference.test_run_id,\n                    fresh_test_image_forwards=True)\n    atomic_json(c.COVERAGE_PATH, coverage)\n    report = verify_csv(c.SUBMISSION_PATH, c.COVERAGE_PATH, ids, shapes)\n    report.update(pipeline_version=c.VERSION, submission=str(c.SUBMISSION_PATH),\n        quality_model_loaded_and_applied=True, quality_used_for_decisions=policy.needs_quality,\n        policy=asdict(policy), model_and_parameter_provenance=provenance,\n        test_gt_access=False, old_submission_copied=False, source_test_cache_read=False,\n        test_forward_run_id=inference.test_run_id, per_photo=per_photo)\n    atomic_json(Path(c.SUBMISSION_PATH).parent/\'v12a_submission_verification.json\', report)\n    pd.DataFrame(per_photo).to_csv(Path(c.OUTPUT_ROOT)/\'test_per_photo.csv\', index=False)\n    print(\'READY TO SUBMIT: \'+str(c.SUBMISSION_PATH), flush=True)\n    return report\n', 'solar_v15/__init__.py': '"""Solar Filament V1.5 Semantic Hybrid."""\n', 'solar_v15/assets.py': '"""Read-only checkpoint inventory and exact V1.4 deployment restoration."""\n\nimport warnings\nfrom importlib.metadata import version\nfrom pathlib import Path\nfrom types import SimpleNamespace\n\nimport joblib\nimport numpy as np\nimport torch\n\nfrom solar_v1.config import digest as v1_digest\nfrom solar_v1.models import load_model\nfrom solar_v1.preprocess import SolarPreprocessor\nfrom solar_v12a.io import load_base, resolve_v1, test_paths\nfrom legacy_v14 import semantic as legacy_semantic\nfrom legacy_v14 import selection as legacy_selection\nfrom .common import file_hash, freeze, one_file, read_json\nfrom .source_contract import BLENDED_ORIGINAL_SHA, BLENDED_NOTEBOOK_SOURCE_SHA, LEGACY_SEMANTIC_SHA\n\n\nGITHUB_SHA = \'1d1dfd8042b44a198d52efb4c5b928abbf861cbe0e973a1ee77174ad19bf06f9\'\n\n\nclass Assets:\n    def __init__(self, config, output):\n        self.config, self.output = config, output\n        root = Path(config.input_root)\n        missing = []\n\n        def locate(name, explicit=\'\'):\n            try:\n                return one_file(root, name, explicit)\n            except FileNotFoundError as error:\n                missing.append(str(error))\n\n        try:\n            self.v1_root = resolve_v1(config.v1_source, root)\n        except (ValueError, FileNotFoundError) as error:\n            missing.append(str(error))\n        self.github_path = locate(\'validated_unet_run3_last.pt\', config.github_checkpoint)\n        self.blended_path = locate(\'blended_shared_split_last.pt\', config.blended_checkpoint)\n        legacy_file = str(Path(config.legacy_root)/\'semantic_unet34.pt\') if config.legacy_root else \'\'\n        self.legacy_path = locate(\'semantic_unet34.pt\', legacy_file)\n        if self.legacy_path is not None:\n            for name in (\'semantic_training.json\', \'selector.joblib\', \'validation_report.json\', \'roles.json\'):\n                if not (self.legacy_path.parent/name).is_file():\n                    missing.append(f\'Missing completed V1.4 artifact: {self.legacy_path.parent/name}\')\n        if config.mode != \'submit\':\n            annotation = locate(\'MAGFiLO_1.0_Annotations_kaggle2026_train.json\', config.annotation_path)\n            if annotation is not None and not (annotation.parent/\'train_images\').is_dir():\n                missing.append(f\'Missing native training image folder: {annotation.parent/"train_images"}\')\n        bridge = SimpleNamespace(INPUT_ROOT=config.input_root, OUTPUT_ROOT=str(output),\n                                 BASE_CANDIDATE_FLOOR=.2, REFINER_BATCH_SIZE=4,\n                                 TEST_IMAGE_DIR=config.test_image_dir)\n        try:\n            self.tests = test_paths(bridge)\n            if len(self.tests) != 180:\n                missing.append(f\'Expected 180 test photos, found {len(self.tests)}\')\n        except (ValueError, FileNotFoundError) as error:\n            missing.append(str(error))\n        if missing:\n            raise FileNotFoundError(\'V1.5 input inventory failed.\\n\'+\'\\n\'.join(missing))\n        self.baseline, self.base, self.norm, self.manifest, self.v1_hashes = load_base(bridge, self.v1_root)\n        if {p.stem for p in self.tests} & {p[\'stem\'] for p in self.manifest[\'photos\']}:\n            raise ValueError(\'Official test photo IDs intersect the saved V1 training/validation collection.\')\n        if not self.base.REFINER_USE_COARSE or not self.base.REFINER_USE_RADIAL:\n            raise ValueError(\'The frozen V1 refiner must have all five original input channels.\')\n        self.v1_training = {}\n        manifest_train = {p[\'stem\'] for p in self.manifest[\'photos\'] if p[\'split\'] == \'train\'}\n        for stage in (\'detector\', \'refiner\'):\n            state = torch.load(self.v1_root/f\'checkpoints/{stage}_best.pt\', map_location=\'cpu\', weights_only=True)\n            trained = set(state[\'trained_photo_ids\'])\n            if not trained <= manifest_train:\n                raise ValueError(f\'{stage} train IDs differ from the V1 manifest.\')\n            self.v1_training[stage] = sorted(trained)\n        if file_hash(self.github_path) != GITHUB_SHA:\n            raise ValueError(\'GitHub U-Net SHA-256 differs from the validated V3 weight.\')\n        github_meta = torch.load(self.github_path, map_location=\'cpu\', weights_only=True)\n        if github_meta.get(\'in_channels\', 2) != 2 or github_meta.get(\'encoder\', \'resnet34\') != \'resnet34\':\n            raise ValueError(\'The signed GitHub semantic checkpoint has an unexpected encoder/input schema.\')\n        del github_meta\n        self.blended_state = torch.load(self.blended_path, map_location=\'cpu\', weights_only=True)\n        self.blended_signature = self.blended_state.get(\'signature\', {})\n        expected = {\'source_sha256\': BLENDED_NOTEBOOK_SOURCE_SHA,\n                    \'runtime_source_sha256\': BLENDED_ORIGINAL_SHA, \'epochs\': 24, \'train_batch\': 4}\n        if (any(self.blended_signature.get(k) != v for k, v in expected.items())\n                or self.blended_state.get(\'epochs\') != 24 or not self.blended_signature.get(\'train_ids\')):\n            raise ValueError(\'Blended must be the completed signed 24-epoch V3 model.\')\n        self.legacy_recipe = read_json(self.legacy_path.parent/\'semantic_training.json\')[\'recipe\']\n        expected_legacy = {\'semantic_code_sha256\': LEGACY_SEMANTIC_SHA, \'epochs\': 12,\n                           \'crop\': 512, \'semantic_precision\': \'float32\',\n                           \'preprocessing\': SolarPreprocessor.VERSION, \'preprocess_enabled\': self.base.PREPROCESS,\n                           \'split_hash\': self.manifest[\'split_hash\'],\n                           \'annotation_hash\': self.manifest[\'annotation_hash\']}\n        if any(self.legacy_recipe.get(k) != v for k, v in expected_legacy.items()):\n            raise ValueError(\'V1.4 semantic recipe does not match the completed V1.4.2 source and V1 split.\')\n        self.legacy_state = torch.load(self.legacy_path, map_location=\'cpu\', weights_only=True)\n        if (self.legacy_state.get(\'signature\') != v1_digest(self.legacy_recipe)\n                or [x[\'epoch\'] for x in self.legacy_state.get(\'history\', [])] != list(range(1, 13))\n                or any(not np.isfinite(x[\'loss\']) for x in self.legacy_state.get(\'history\', []))):\n            raise ValueError(\'V1.4 semantic state is incomplete or has the wrong training signature.\')\n        self.legacy_report = read_json(self.legacy_path.parent/\'validation_report.json\')\n        self.legacy_roles = read_json(self.legacy_path.parent/\'roles.json\')\n        if v1_digest({k: v for k, v in self.legacy_roles.items() if k != \'role_hash\'}) != self.legacy_roles[\'role_hash\']:\n            raise ValueError(\'The original V1.4 role manifest hash is invalid.\')\n        self.legacy_selector_path = self.legacy_path.parent/\'selector.joblib\'\n        with warnings.catch_warnings(record=True) as captured:\n            warnings.simplefilter(\'always\')\n            self.legacy_selector = joblib.load(self.legacy_selector_path)\n        self.serialization_warnings = [str(x.message) for x in captured]\n        selector = self.legacy_selector\n        if (tuple(selector[\'features\']) != legacy_selection.FEATURES\n                or selector[\'v1_weights\'] != self.v1_hashes\n                or selector[\'semantic_sha256\'] != file_hash(self.legacy_path)\n                or selector[\'role_hash\'] != self.legacy_report[\'role_hash\']\n                or selector[\'role_hash\'] != self.legacy_roles[\'role_hash\']):\n            raise ValueError(\'Legacy selector weights, features, roles or semantic checkpoint differ.\')\n        if getattr(selector[\'model\'], \'n_features_in_\', None) != len(legacy_selection.FEATURES):\n            raise ValueError(\'V1.4 selector feature dimension is incompatible.\')\n        if self.legacy_report[\'deployment\'] not in (\'v14_quality\', \'v1_fixed\'):\n            raise ValueError(\'Unknown legacy V1.4 deployment branch.\')\n        if self.legacy_report[\'deployment\'] == \'v14_quality\' and self.legacy_report.get(\'selected_tau\') is None:\n            raise ValueError(\'Legacy V1.4 deployment threshold is missing.\')\n        for name in (\'semantic_sha256\',):\n            if self.legacy_report.get(name) != file_hash(self.legacy_path):\n                raise ValueError(\'Legacy validation report model hash mismatch.\')\n        files = {\'github\': self.github_path, \'blended\': self.blended_path, \'legacy_semantic\': self.legacy_path,\n                 \'legacy_selector\': self.legacy_selector_path, \'legacy_report\': self.legacy_path.parent/\'validation_report.json\',\n                 \'v1_manifest\': self.v1_root/\'manifest.json\', \'v1_config\': self.v1_root/\'config.json\',\n                 \'v1_norm\': self.v1_root/\'normalization.json\',\n                 \'v1_thresholds\': self.v1_root/\'calibration/thresholds.json\'}\n        self.contract = {\'files\': {k: file_hash(p) for k, p in files.items()}, \'v1_weights\': self.v1_hashes,\n                         \'blended_signature\': self.blended_signature,\n                         \'packages\': {name: version(name) for name in (\'torch\', \'torchvision\', \'numpy\', \'scipy\',\n                                      \'pycocotools\', \'scikit-learn\', \'timm\', \'segmentation-models-pytorch\')},\n                         \'legacy_serialization_warnings\': self.serialization_warnings}\n        self.models = None\n\n    def load_networks(self):\n        if self.models is not None:\n            return self.models\n        if not torch.cuda.is_available():\n            raise RuntimeError(\'Select a Kaggle GPU for uncached frozen-network inference.\')\n        from filament_seg.model import load_trained\n        import blended_frozen_source as blended_source\n\n        v1_device = torch.device(\'cuda:0\')\n        semantic_device = torch.device(\'cuda:1\' if torch.cuda.device_count() > 1 else \'cuda:0\')\n        detector, _ = load_model(self.base, self.norm, \'detector\', v1_device,\n                                 self.v1_root/\'checkpoints/detector_best.pt\')\n        refiner, _ = load_model(self.base, self.norm, \'refiner\', v1_device,\n                                self.v1_root/\'checkpoints/refiner_best.pt\')\n        github = load_trained(str(self.github_path), semantic_device)\n        blended_source.DEVICE = str(semantic_device)\n        blended_source.TILE_BATCH = self.config.mosaic_batch\n        blended_source._BLEND_GPU = None\n        blended = blended_source.UNet(pretrained=False)\n        blended.load_state_dict(self.blended_state.pop(\'model\'), strict=True)\n        blended = blended.to(semantic_device, memory_format=torch.channels_last)\n        legacy = legacy_semantic.SolarUNet(pretrained=False)\n        legacy.load_state_dict(self.legacy_state.pop(\'model\'), strict=True)\n        legacy = legacy.to(semantic_device)\n        self.models = dict(detector=freeze(detector), refiner=freeze(refiner), github=freeze(github),\n                           blended=freeze(blended), legacy=freeze(legacy), source=blended_source,\n                           v1_device=v1_device, semantic_device=semantic_device)\n        print(\'Frozen networks loaded once: V1 detector/refiner + GitHub + Blended + legacy V1.4. YOLO models=0.\', flush=True)\n        return self.models\n', 'solar_v15/candidate_bank.py': '"""Native candidates, detector-family matching and a compact feature contract."""\n\nfrom dataclasses import dataclass, field\n\nimport cv2\nimport numpy as np\nfrom pycocotools import mask as cm\n\nfrom solar_v1.postprocess import encode\nfrom solar_v1.pq import pairwise_iou\nfrom .common import areas\n\n\nSOURCES = (\'base\', \'coarse\', \'refined\', \'semantic_cc\', \'cc_refined\')\nCLASSES = (\'detector\', \'matched_one\', \'orphan\', \'overmerge\')\nFEATURES = (*SOURCES, *CLASSES, \'det_score\', \'coarse_refined_iou\', \'log_area\', \'extent\',\n            \'elongation\', \'radial\', \'contrast\', \'threshold\', \'close_kernel\',\n            \'pf_mean\', \'pf_q20\', \'pf_peak\', \'pf_ring\', \'pg_mean\', \'pb_valid_mean\',\n            \'pb_valid_fraction\', \'model_disagreement\', \'family_max_iou\', \'family_containment\',\n            \'matched_family_count\', \'variant_count\', \'refine_pair_iou\', \'refine_log_area_change\')\n\n\n@dataclass\nclass HybridCandidate:\n    candidate_id: str\n    rle: dict\n    source: str\n    family_id: str | None\n    match_class: str\n    threshold: float | None = None\n    close_kernel: int | None = None\n    area: int = 0\n    bbox: tuple = ()\n    matched_detector_ids: tuple = ()\n    parent_id: str | None = None\n    info: dict = field(default_factory=dict)\n\n\n@dataclass\nclass HybridBank:\n    candidates: list\n    features: np.ndarray\n    conflicts: list\n\n    @property\n    def rles(self):\n        return [c.rle for c in self.candidates]\n\n    @property\n    def sources(self):\n        return [c.info.get(\'sources\', [c.source]) for c in self.candidates]\n\n\ndef populate_geometry(candidate):\n    candidate.area = int(cm.area(candidate.rle))\n    candidate.bbox = tuple(map(int, cm.toBbox(candidate.rle)))\n    return candidate\n\n\ndef from_families(families):\n    values = []\n    for family_id, family in families.items():\n        for i, variant in enumerate(family[\'variants\']):\n            if variant[\'source\'] != \'base\' and not family[\'candidate_admitted\']:\n                continue\n            values.append(populate_geometry(HybridCandidate(\n                f\'{family_id}:{variant["source"]}:{i}\', variant[\'rle\'], variant[\'source\'], family_id,\n                \'detector\', matched_detector_ids=(family_id,),\n                info={\'det_score\': family[\'raw_score\'], \'coarse_refined_iou\': family[\'coarse_refined_iou\'],\n                      \'admission\': \'v1_base_or_detector_floor\', \'family_aliases\': [family_id]})))\n    return values\n\n\ndef generate_cc(probability, disk, config, kernels=None, audit=None):\n    result, seen = [], {}\n    kernels = config.close_kernels if kernels is None else kernels\n    generated = 0\n    settings = []\n    for threshold in config.levels:\n        for kernel in kernels:\n            binary = ((probability > threshold) & disk).astype(np.uint8)\n            if kernel:\n                binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE,\n                    cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel, kernel)))\n                binary[~disk] = 0\n            _, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)\n            ranked = []\n            for i, (x, y, w, h, area) in enumerate(stats[1:], 1):\n                if not config.min_area <= area <= config.max_area:\n                    continue\n                region = labels[y:y+h, x:x+w] == i\n                p = probability[y:y+h, x:x+w][region]\n                ranked.append((float(p.mean()), int(area), i))\n            generated += len(ranked)\n            ranked.sort(key=lambda z: (-z[0], z[2]))\n            small = [z for z in ranked if 250 <= z[1] < 1000][:config.semantic_per_setting//2]\n            ids = {z[2] for z in small}\n            chosen = (small + [z for z in ranked if z[2] not in ids])[:config.semantic_per_setting]\n            settings.append({\'threshold\': threshold, \'close_kernel\': kernel,\n                             \'components\': len(stats)-1, \'area_eligible\': len(ranked),\n                             \'after_setting_budget\': len(chosen)})\n            for support, area, i in chosen:\n                x, y, w, h, _ = stats[i]\n                mask = np.zeros(probability.shape, np.uint8)\n                mask[y:y+h, x:x+w] = labels[y:y+h, x:x+w] == i\n                rle = encode(mask)\n                parameters = {\'threshold\': threshold, \'close_kernel\': kernel, \'min_area\': config.min_area}\n                if rle[\'counts\'] in seen:\n                    seen[rle[\'counts\']].info[\'parameter_aliases\'].append(parameters)\n                    continue\n                name = f\'cc:{threshold:.3f}:{kernel}:{i}\'\n                value = HybridCandidate(name, rle, \'semantic_cc\', name, \'orphan\', threshold, kernel,\n                                        area, (int(x), int(y), int(w), int(h)),\n                                        info={\'parameter_aliases\': [parameters], \'support\': support,\n                                              \'admission\': \'unclassified\', \'original_cc_id\': name})\n                result.append(value)\n                seen[rle[\'counts\']] = value\n    result.sort(key=lambda c: (-c.info[\'support\'], c.candidate_id))\n    small = [c for c in result if 250 <= c.area < 1000][:config.semantic_budget//2]\n    ids = {c.candidate_id for c in small}\n    result = (small + [c for c in result if c.candidate_id not in ids])[:config.semantic_budget]\n    if audit is not None:\n        audit.update(settings=settings, unique_before_global_budget=len(seen), retained=len(result),\n                     min_area=config.min_area, max_area=config.max_area, global_budget=config.semantic_budget)\n    if result:\n        a = areas([c.rle for c in result])\n        iou = pairwise_iou([c.rle for c in result], [c.rle for c in result])\n        intersection = iou*(a[:, None]+a[None, :])/(1+iou)\n        for i, c in enumerate(result):\n            parents = [j for j, p in enumerate(result) if a[j] > a[i] and intersection[i, j] >= a[i]-.01]\n            if parents:\n                parent = min(parents, key=lambda j: (a[j], result[j].candidate_id))\n                c.parent_id = result[parent].candidate_id\n            c.info[\'generated_before_budget\'] = generated\n    return result\n\n\ndef classify(candidates, families, config):\n    representatives, keys = [], []\n    for key, family in families.items():\n        for variant in family[\'variants\']:\n            representatives.append(variant[\'rle\'])\n            keys.append(key)\n    if not candidates:\n        return candidates\n    matrix = pairwise_iou([c.rle for c in candidates], representatives)\n    a, b = areas([c.rle for c in candidates]), areas(representatives)\n    inter = matrix*(a[:, None]+b[None, :])/(1+matrix)\n    for i, c in enumerate(candidates):\n        evidence = {}\n        for j, key in enumerate(keys):\n            entry = evidence.setdefault(key, dict(iou=0., coverage_cc=0., coverage_det=0., containment=0.))\n            entry[\'iou\'] = max(entry[\'iou\'], float(matrix[i, j]))\n            entry[\'coverage_cc\'] = max(entry[\'coverage_cc\'], float(inter[i, j]/max(a[i], 1)))\n            entry[\'coverage_det\'] = max(entry[\'coverage_det\'], float(inter[i, j]/max(b[j], 1)))\n            entry[\'containment\'] = max(entry[\'containment\'], float(inter[i, j]/max(min(a[i], b[j]), 1)))\n        strong = sorted(key for key, e in evidence.items() if e[\'iou\'] >= config.match_iou or\n                        (e[\'containment\'] >= config.match_containment and e[\'iou\'] >= .10))\n        c.matched_detector_ids = tuple(strong)\n        c.info[\'family_evidence\'] = evidence\n        c.info[\'family_max_iou\'] = max((e[\'iou\'] for e in evidence.values()), default=0.)\n        c.info[\'family_containment\'] = max((e[\'containment\'] for e in evidence.values()), default=0.)\n        if len(strong) == 1:\n            c.match_class = \'matched_one\'\n            c.info.update(admission=\'matched_one_alternative\', family_aliases=strong,\n                          det_score=families[strong[0]][\'raw_score\'],\n                          coarse_refined_iou=families[strong[0]][\'coarse_refined_iou\'])\n        elif len(strong) > 1:\n            c.match_class = \'overmerge\'\n            c.info.update(admission=\'overmerge_evidence_only\', overmerge_risk=True)\n        else:\n            c.match_class = \'orphan\'\n            c.info[\'admission\'] = (\'orphan_candidate\' if c.info[\'family_max_iou\'] < .10\n                                   else \'ambiguous_weak_detector_overlap\')\n    return candidates\n\n\ndef assemble(detectors, semantic, data, maps, primary_map, identity_aware):\n    # New objects keep traces independent across the A0–A6 ablations.\n    import copy\n    values = copy.deepcopy(detectors + semantic)\n    unique = {}\n    for c in values:\n        c.info.setdefault(\'sources\', [c.source])\n        if c.match_class == \'matched_one\' and identity_aware:\n            c.family_id = c.matched_detector_ids[0]\n        elif c.source in (\'semantic_cc\', \'cc_refined\') and not identity_aware:\n            c.info[\'family_aliases\'] = [c.info.get(\'original_cc_id\', c.candidate_id)]\n        else:\n            c.info.setdefault(\'family_aliases\', [c.family_id])\n        key = c.rle[\'counts\']\n        if key in unique:\n            old = unique[key]\n            old.info[\'sources\'] = sorted(set(old.info[\'sources\']+c.info[\'sources\']))\n            old.info[\'family_aliases\'] = sorted(set(old.info[\'family_aliases\']+c.info[\'family_aliases\']))\n            old.info.setdefault(\'candidate_aliases\', []).append(c.candidate_id)\n        else:\n            unique[key] = c\n    values = list(unique.values())\n    rles = [c.rle for c in values]\n    iou = pairwise_iou(rles, rles)\n    groups = [set(c.info[\'family_aliases\']) for c in values]\n    conflicts, rows = [], []\n    for i, c in enumerate(values):\n        cx, cy, radius = data[\'disk\']\n        x, y, w, h = c.bbox\n        x0, y0 = max(0, x-4), max(0, y-4)\n        x1, y1 = min(data[\'raw\'].shape[1], x+w+4), min(data[\'raw\'].shape[0], y+h+4)\n        roi = np.s_[y0:y1, x0:x1]\n        mask = cm.decode(c.rle)[roi].astype(bool)\n        yy, xx = np.nonzero(mask)\n        cov = np.cov(np.stack([xx, yy]), bias=True) if len(xx) > 1 else np.zeros((2, 2))\n        eig = np.maximum(np.linalg.eigvalsh(cov), 0)\n        ring = cv2.dilate(mask.astype(np.uint8), np.ones((7, 7), np.uint8)).astype(bool) & ~mask\n        p = primary_map[roi][mask]\n        g, b = maps[\'pg\'][roi][mask], maps[\'pb\'][roi][mask]\n        iy, ix = yy+y0, xx+x0\n        valid = (iy >= 88) & (iy < 1960) & (ix >= 88) & (ix < 1960)\n        contrast = ((data[\'corrected\'][roi][ring].mean()-data[\'corrected\'][roi][mask].mean())/255.\n                    if ring.any() else 0.)\n        c.info.update(primary_mean=float(p.mean()), primary_q20=float(np.quantile(p, .2)),\n                      contrast=float(contrast), pb_valid_fraction=float(valid.mean()))\n        aliases = c.info[\'sources\']\n        variant_count = sum(bool(groups[i] & family) for family in groups)-1\n        rows.append([*[float(s in aliases) for s in SOURCES], *[float(c.match_class == v) for v in CLASSES],\n                     c.info.get(\'det_score\', 0.), c.info.get(\'coarse_refined_iou\', 0.),\n                     np.log1p(c.area), c.area/max(w*h, 1), np.sqrt((eig[1]+1)/(eig[0]+1)),\n                     np.hypot(x+w/2-cx, y+h/2-cy)/max(radius, 1), contrast,\n                     c.threshold or 0., c.close_kernel or 0., p.mean(), np.quantile(p, .2), p.max(),\n                     primary_map[roi][ring].mean() if ring.any() else p.mean(), g.mean(),\n                     b[valid].mean() if valid.any() else 0., valid.mean(),\n                     np.abs(g[valid]-b[valid]).mean() if valid.any() else 0.,\n                     c.info.get(\'family_max_iou\', 0.), c.info.get(\'family_containment\', 0.),\n                     len(c.matched_detector_ids), variant_count, c.info.get(\'before_after_iou\', 1.),\n                     np.log(max(c.info.get(\'after_area\', c.area), 1)/max(c.info.get(\'before_area\', c.area), 1))])\n        blocked = [j for j in range(len(values)) if j != i and (iou[i, j] > 0 or groups[i] & groups[j])]\n        conflicts.append(sum(1 << j for j in blocked))\n    features = np.asarray(rows, np.float32).reshape(-1, len(FEATURES))\n    if not np.isfinite(features).all():\n        raise ValueError(\'A hybrid candidate feature is nonfinite.\')\n    return HybridBank(values, features, conflicts)\n', 'solar_v15/common.py': '"""Atomic artifacts, native RLE handling and frozen network checks."""\n\nimport hashlib\nimport json\nimport os\nimport tempfile\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom pycocotools import mask as cm\n\n\nSHAPE = (2048, 2048)\nVERSION = \'1.5.0\'\n\n\ndef convert(value):\n    if isinstance(value, bytes):\n        return value.decode(\'ascii\')\n    if isinstance(value, np.ndarray):\n        return value.tolist()\n    if isinstance(value, np.generic):\n        return value.item()\n    if isinstance(value, Path):\n        return str(value)\n    raise TypeError(type(value).__name__)\n\n\ndef digest(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, default=convert, allow_nan=False).encode()).hexdigest()\n\n\ndef file_hash(path):\n    with Path(path).open(\'rb\') as stream:\n        return hashlib.file_digest(stream, \'sha256\').hexdigest()\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with tempfile.NamedTemporaryFile(mode=\'w\', dir=path.parent, encoding=\'utf-8\', delete=False) as stream:\n        temporary = Path(stream.name)\n        json.dump(value, stream, default=convert, allow_nan=False, ensure_ascii=False)\n    os.replace(temporary, path)\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding=\'utf-8\'))\n\n\ndef areas(rles):\n    result = np.empty(len(rles), np.float64)\n    for start in range(0, len(rles), 255):\n        result[start:start+255] = cm.area(rles[start:start+255])\n    return result\n\n\ndef freeze(model):\n    model.eval()\n    model.requires_grad_(False)\n    if any(not torch.isfinite(t).all() for t in model.state_dict().values()):\n        raise ValueError(\'Frozen checkpoint has a nonfinite tensor.\')\n    return model\n\n\ndef normalize_rles(rows):\n    for row in rows:\n        value = row[\'rle\']\n        if isinstance(value[\'counts\'], str):\n            value[\'counts\'] = value[\'counts\'].encode(\'ascii\')\n    return rows\n\n\ndef one_file(root, name, explicit=\'\'):\n    matches = [Path(explicit)] if explicit else sorted({p.resolve() for p in Path(root).rglob(name)})\n    if len(matches) != 1 or not matches[0].is_file():\n        raise FileNotFoundError(f\'Expected one {name}. Found {matches}. Set its explicit path in CONFIG.\')\n    return matches[0]\n', 'solar_v15/config.py': '"""V1.5 uses frozen networks and fits only CPU candidate regressors."""\n\nfrom dataclasses import asdict, dataclass\n\n\n@dataclass\nclass Config:\n    mode: str = \'all\'\n    input_root: str = \'/kaggle/input\'\n    working_root: str = \'/kaggle/working\'\n    cache_root: str = \'/kaggle/temp/solar_v15/cache\'\n    cache_source: str = \'\'\n    bundle_source: str = \'\'\n    v1_source: str = \'\'\n    legacy_root: str = \'\'\n    github_checkpoint: str = \'\'\n    blended_checkpoint: str = \'\'\n    annotation_path: str = \'\'\n    test_image_dir: str = \'\'\n    levels: tuple = (.35, .50, .65)\n    close_kernels: tuple = (0, 5)\n    min_area: int = 64\n    max_area: int = 100_000\n    semantic_budget: int = 96\n    semantic_per_setting: int = 32\n    detector_floor: float = .35\n    match_iou: float = .35\n    match_containment: float = .85\n    orphan_floors: tuple = (.50, .60, .70)\n    taus: tuple = (.16, .20, .24, .28, .32, .36)\n    selective_refinement: bool = True\n    selective_budget: int = 6\n    selective_max_crop: int = 1024\n    min_gate_gain: float = .005\n    min_eval_photos: int = 20\n    min_eval_groups: int = 5\n    require_positive_ci: bool = True\n    cache_maps: bool = True\n    cache_test_maps: bool = False\n    max_cache_gb: float = 16.\n    max_hours: float = 10.5\n    visualization_count: int = 8\n    mosaic_batch: int = 2\n    seed: int = 15026\n    install_missing: bool = True\n\n    def validate(self):\n        if self.mode not in {\'preflight\', \'infer\', \'train_quality\', \'evaluate\', \'submit\', \'all\'}:\n            raise ValueError(f\'Unknown V1.5 mode {self.mode}\')\n        if (not 1 <= len(self.levels) <= 5 or sorted(set(self.levels)) != list(self.levels)\n                or any(not 0 < t < 1 for t in self.levels)):\n            raise ValueError(\'Use 1–5 distinct increasing probability thresholds.\')\n        if not self.close_kernels or any(k not in (0, 3, 5, 7, 9) for k in self.close_kernels):\n            raise ValueError(\'Closing kernels must be 0 or supported odd sizes.\')\n        if not 8 <= self.semantic_budget <= 192 or not 8 <= self.semantic_per_setting <= 64:\n            raise ValueError(\'Semantic candidate budget is outside the audited bounds.\')\n        if not 0 <= self.selective_budget <= 12 or not 1 <= self.mosaic_batch <= 2:\n            raise ValueError(\'Use at most 12 extra ROIs and 1–2 Blended tiles per batch.\')\n        if not 1 <= self.min_area <= self.max_area or not self.taus or not self.orphan_floors:\n            raise ValueError(\'Invalid area or calibration grid.\')\n        if (any(not 0 < t < 1 for t in (*self.taus, *self.orphan_floors))\n                or not 0 < self.match_iou < 1 or not 0 < self.match_containment <= 1):\n            raise ValueError(\'Matching and calibrated thresholds must be valid probabilities.\')\n        if self.max_cache_gb <= 0 or not 0 < self.max_hours <= 11.5 or not 0 <= self.visualization_count <= 12:\n            raise ValueError(\'Invalid storage, session or visualization budget.\')\n        return self\n\n    def to_dict(self):\n        return asdict(self)\n', 'solar_v15/feather.py': 'import numpy as np\n\ndef blend(pg, pb, center, weight, feather):\n    """Keep Pg unchanged wherever Pb has no support."""\n    start, end = center\n    size = end - start\n    if pg.shape != pb.shape or not (0 <= start < end <= min(pg.shape)):\n        raise ValueError("Invalid probability geometry")\n    distance = np.minimum(np.arange(size), np.arange(size)[::-1])\n    ramp = np.minimum(distance / feather, 1) if feather else np.ones(size)\n    alpha = (weight * np.minimum.outer(ramp, ramp)).astype(np.float32)\n    region = np.s_[start:end, start:end]\n    result = pg.copy()\n    result[region] = pg[region] + alpha * (pb[region] - pg[region])\n    return result\n', 'solar_v15/main.py': '"""Independent V1.5 orchestration. All neural networks remain frozen."""\n\nimport sys\nimport time\nfrom collections import Counter, defaultdict\nfrom dataclasses import asdict\nfrom pathlib import Path\n\nimport cv2\nimport joblib\nimport numpy as np\nimport torch\n\nfrom solar_v1.pq import Totals\nfrom .assets import Assets\nfrom .candidate_bank import FEATURES\nfrom .common import SHAPE, VERSION, digest, file_hash, read_json, write_json\nfrom .config import Config\nfrom .pipeline import ABLATIONS, PhotoPipeline\nfrom .provenance import audit\nfrom .quality import fit, predict, quality_metrics, targets\nfrom .selection import select\nfrom .validation import area_group, oracle_and_selection, paired_uncertainty, score, visualize\n\n\nBUNDLE = \'v15_bundle.json\'\nMODEL = \'quality_models.joblib\'\n\n\ndef runtime_contract(assets, config):\n    fields = (\'levels\', \'close_kernels\', \'min_area\', \'max_area\', \'semantic_budget\', \'semantic_per_setting\',\n              \'detector_floor\', \'match_iou\', \'match_containment\', \'selective_refinement\', \'selective_budget\', \'selective_max_crop\',\n              \'seed\', \'mosaic_batch\', \'taus\', \'orphan_floors\', \'min_gate_gain\', \'min_eval_photos\',\n              \'min_eval_groups\', \'require_positive_ci\')\n    code = {p.relative_to(Path(__file__).parents[1]).as_posix(): file_hash(p)\n            for directory in (Path(__file__).parent, Path(__file__).parents[1]/\'legacy_v14\',\n                              Path(__file__).parents[1]/\'filament_seg\', Path(__file__).parents[1]/\'solar_v1\',\n                              Path(__file__).parents[1]/\'solar_v12a\') for p in sorted(directory.glob(\'*.py\'))}\n    return {\'version\': VERSION, \'assets\': assets.contract, \'parameters\': {name: getattr(config, name) for name in fields},\n            \'code\': code, \'feature_names\': list(FEATURES), \'opencv\': cv2.__version__,\n            \'github_logits_precision\': \'float16_roundtrip_then_expit_float32\',\n            \'feather\': {\'center\': [88, 1960], \'weight\': .55, \'feather\': 64},\n            \'hardware\': {\'cuda\': torch.version.cuda, \'cudnn\': torch.backends.cudnn.version(),\n                         \'gpu\': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}}\n\n\ndef models_save(out, bundle, models):\n    path = out/MODEL\n    temporary = path.with_suffix(\'.tmp\')\n    joblib.dump(models, temporary)\n    temporary.replace(path)\n    bundle[\'models_sha256\'] = file_hash(path)\n    write_json(out/BUNDLE, bundle)\n\n\ndef bundle_load(out, config, contract, ready):\n    if config.bundle_source:\n        path = Path(config.bundle_source)/BUNDLE\n    elif (out/BUNDLE).is_file():\n        path = out/BUNDLE\n    else:\n        found = sorted(Path(config.input_root).rglob(BUNDLE))\n        if len(found) != 1:\n            raise FileNotFoundError(f\'Attach a unique {BUNDLE} or set bundle_source. Found {found}\')\n        path = found[0]\n    value = read_json(path)\n    if digest(value[\'contract\']) != digest(contract):\n        raise ValueError(\'V1.5 bundle weights, code, features, parameters or runtime do not match.\')\n    locked_roles = value[\'roles\']\n    if (value[\'role_hash\'] != locked_roles[\'role_hash\']\n            or digest({k: v for k, v in locked_roles.items() if k != \'role_hash\'}) != locked_roles[\'role_hash\']):\n        raise ValueError(\'The saved quality role manifest has changed.\')\n    if ready and value[\'status\'] != \'ready\':\n        raise ValueError(\'Run evaluate or all before submitting this unvalidated bundle.\')\n    model_path = path.parent/MODEL\n    if not model_path.is_file() or file_hash(model_path) != value[\'models_sha256\']:\n        raise ValueError(\'V1.5 candidate quality model file is missing or has the wrong checksum.\')\n    models = joblib.load(model_path)\n    if set(models) != set(ABLATIONS):\n        raise ValueError(\'The saved quality model ablation keys are incomplete.\')\n    for model in models.values():\n        if model is not None and model.n_features_in_ != len(FEATURES):\n            raise ValueError(\'V1.5 quality feature dimension mismatch.\')\n    return value, models\n\n\ndef train_quality(engine, roles, catalog, contract, check_time):\n    examples = {name: [] for name in ABLATIONS}\n    for i, row in enumerate(roles[\'roles\'][\'fit\'], 1):\n        check_time()\n        banks, _, _ = engine.compute(Path(catalog.photos[row[\'stem\']][\'path\']))\n        views = catalog.views(row[\'stem\'])\n        for name, bank in banks.items():\n            target, _, _ = targets(bank, views)\n            examples[name].append({\'features\': bank.features, \'target\': target, \'n_views\': len(views)})\n        if i % 10 == 0 or i == len(roles[\'roles\'][\'fit\']):\n            print(f\'Quality fit photos {i}/{len(roles["roles"]["fit"])}\', flush=True)\n    models = {name: fit(values, engine.config.seed) for name, values in examples.items()}\n    fit_info = {name: {\'photos\': len(values), \'candidates\': sum(len(e[\'target\']) for e in values),\n                       \'fitted\': models[name] is not None} for name, values in examples.items()}\n    del examples\n    if models[\'A3\'] is None:\n        raise ValueError(\'Insufficient V1.5 A3 quality supervision. Legacy V1.4 assets remain available, but V1.5 cannot be calibrated.\')\n    policies, trials = calibrate(engine, roles, catalog, models, check_time)\n    primary = \'A3\'\n    if engine.config.selective_refinement and models[\'A4\'] is not None:\n        gain = max(t[\'pq\'] for t in trials[\'A4\'])-max(t[\'pq\'] for t in trials[\'A3\'])\n        if gain >= engine.config.min_gate_gain:\n            primary = \'A4\'\n    bundle = {\'contract\': contract, \'roles\': roles, \'role_hash\': roles[\'role_hash\'],\n              \'policies\': policies, \'primary\': primary, \'calibration_trials\': trials, \'fit\': fit_info,\n              \'status\': \'calibrated\', \'deployment\': None,\n              \'selection_note\': \'Only A3 or A4 can be primary; chosen on calibration before evaluation.\'}\n    models_save(engine.out, bundle, models)\n    print(f\'Calibration locked before evaluation: primary={primary}, policy={policies[primary]}\', flush=True)\n    return bundle, models\n\n\ndef calibrate(engine, roles, catalog, models, check_time):\n    grids = {}\n    for name in ABLATIONS:\n        if models[name] is None:\n            continue\n        floors = (0.,) if name in (\'A1\', \'A2\') else engine.config.orphan_floors\n        grids[name] = {(tau, floor): Totals() for tau in engine.config.taus for floor in floors}\n    for i, row in enumerate(roles[\'roles\'][\'calibration\'], 1):\n        check_time()\n        banks, _, _ = engine.compute(Path(catalog.photos[row[\'stem\']][\'path\']))\n        views = catalog.views(row[\'stem\'])\n        for name, grid in grids.items():\n            bank = banks[name]\n            values = predict(models[name], bank)\n            for (tau, floor) in grid:\n                instances, _, _ = select(bank, values, {\'tau\': tau, \'orphan_floor\': floor}, name)\n                grid[tau, floor] += score(instances, views, SHAPE)[0]\n        if i % 10 == 0:\n            print(f\'Quality calibration photos {i}\', flush=True)\n    trials = {name: [{\'tau\': tau, \'orphan_floor\': floor, **value.summary()} for (tau, floor), value in grid.items()]\n              for name, grid in grids.items()}\n    policies = {name: {k: v for k, v in max(values, key=lambda x: (round(x[\'pq\'], 12), x[\'orphan_floor\'], x[\'tau\'])).items()\n                       if k in (\'tau\', \'orphan_floor\')} for name, values in trials.items()}\n    return policies, trials\n\n\ndef gate(roles, delta, uncertainty, config):\n    if not roles[\'gate_data_sufficient\']:\n        return \'v14_fallback\', \'insufficient_independent_photos_or_date_groups\'\n    if delta < config.min_gate_gain:\n        return \'v14_fallback\', \'held_out_pq_gain_below_minimum\'\n    if config.require_positive_ci and (uncertainty[\'lower95\'] is None or uncertainty[\'lower95\'] <= 0):\n        return \'v14_fallback\', \'group_bootstrap_gain_is_uncertain\'\n    return \'v15\', \'predeclared_gain_and_data_gate_passed\'\n\n\ndef evaluate(engine, bundle, models, catalog, check_time):\n    roles, primary = bundle[\'roles\'], bundle[\'primary\']\n    totals = {name: Totals() for name in (\'A0\', *bundle[\'policies\'])}\n    per_annotator = {name: defaultdict(Totals) for name in totals}\n    diagnostic = {name: Counter() for name in totals}\n    quality = defaultdict(list)\n    per_photo, paired = [], []\n    marginal = {source: Totals() for source in (\'base\', \'coarse\', \'refined\', \'semantic_cc\', \'cc_refined\')}\n    rows = roles[\'roles\'][\'evaluation\']\n    figures = []\n    for i, row in enumerate(rows, 1):\n        check_time()\n        callback = (visualize(engine.out/\'figures\', i, models[\'A3\'], bundle[\'policies\'][\'A3\'])\n                    if i <= engine.config.visualization_count else None)\n        banks, legacy, audit_row = engine.compute(Path(catalog.photos[row[\'stem\']][\'path\']), callback)\n        if callback is not None:\n            figures.append({\'case\': i, \'role\': \'evaluation\', \'contains_gt\': False})\n        views = catalog.views(row[\'stem\'])\n        baseline, annotators, small = score(legacy, views, SHAPE, audit_row[\'disk\'])\n        totals[\'A0\'] += baseline\n        diagnostic[\'A0\'].update(small)\n        for who, value in annotators.items():\n            per_annotator[\'A0\'][who] += value\n        per_photo.append({\'stem\': row[\'stem\'], \'group\': row[\'group\'], \'branch\': \'A0\', **baseline.summary()})\n        for name in bundle[\'policies\']:\n            bank = banks[name]\n            values = predict(models[name], bank)\n            final, chosen, reasons = select(bank, values, bundle[\'policies\'][name], name)\n            value, annotators, small = score(final, views, SHAPE, audit_row[\'disk\'])\n            totals[name] += value\n            diagnostic[name].update(small)\n            diagnostic[name].update(oracle_and_selection(bank, chosen, views))\n            diagnostic[name].update({\'candidate_count\': len(bank.candidates), \'final_count\': len(final)})\n            diagnostic[name].update(Counter(\'class_\'+c.match_class for c in bank.candidates))\n            diagnostic[name].update(Counter(\'rejection_\'+v for v in reasons.values() if v != \'selected\'))\n            for who, item in annotators.items():\n                per_annotator[name][who] += item\n            y, h, best = targets(bank, views)\n            quality[name].append((y, values, h))\n            feature_groups = defaultdict(list)\n            for j, c in enumerate(bank.candidates):\n                feature_groups[\'source:\'+c.source].append(j)\n                feature_groups[\'area:\'+area_group(c.area)].append(j)\n            for group, indices in feature_groups.items():\n                quality[name+\'/\'+group].append((y[indices], values[indices], h[indices]))\n            per_photo.append({\'stem\': row[\'stem\'], \'group\': row[\'group\'], \'branch\': name, **value.summary()})\n            if name == primary:\n                paired.append({\'stem\': row[\'stem\'], \'group\': row[\'group\'],\n                               \'baseline\': asdict(baseline), \'new\': asdict(value),\n                               \'pq_difference\': value.summary()[\'pq\']-baseline.summary()[\'pq\']})\n                for source in marginal:\n                    removed, _, _ = select(bank, values, bundle[\'policies\'][name], name, disabled_source=source)\n                    marginal[source] += score(removed, views, SHAPE)[0]\n            # Save the genuine pre/post CC pair IoU for the same permitted annotation views.\n            candidates = []\n            for j, c in enumerate(bank.candidates):\n                candidates.append({**asdict(c), \'predicted_quality\': float(values[j]),\n                                   \'target_effective_iou\': float(y[j]), \'best_iou_by_view\': best[:, j],\n                                   \'decision\': reasons[c.candidate_id]})\n            write_json(engine.out/\'validation_trace\'/row[\'stem\']/f\'{name}.json\',\n                       {\'candidates\': candidates, \'chosen\': chosen, \'final\': [asdict(x) for x in final], \'audit\': audit_row})\n        print(f\'Evaluation {i}/{len(rows)}\', flush=True)\n    for row in roles[\'roles\'][\'calibration\']:\n        if len(figures) >= engine.config.visualization_count:\n            break\n        check_time()\n        i = len(figures)+1\n        engine.compute(Path(catalog.photos[row[\'stem\']][\'path\']),\n                       visualize(engine.out/\'figures\', i, models[\'A3\'], bundle[\'policies\'][\'A3\']))\n        figures.append({\'case\': i, \'role\': \'calibration\', \'contains_gt\': False})\n    write_json(engine.out/\'figure_manifest.json\', figures)\n    experiments = {name: {**value.summary(), \'diagnostics\': dict(diagnostic[name]),\n                          \'per_annotator\': {k: v.summary() for k, v in per_annotator[name].items()}}\n                   for name, value in totals.items()}\n    delta = totals[primary].summary()[\'pq\']-totals[\'A0\'].summary()[\'pq\']\n    uncertainty = paired_uncertainty(paired, engine.config.seed)\n    deployment, reason = gate(roles, delta, uncertainty, engine.config)\n    bundle.update(status=\'ready\', deployment=deployment, deployment_reason=reason,\n                  evaluation_delta=delta, evaluation_uncertainty=uncertainty)\n    report = {\'version\': VERSION, \'experiments\': experiments, \'primary\': primary,\n              \'deployment\': deployment, \'reason\': reason, \'pq_gain_vs_same_set_v14\': delta,\n              \'uncertainty\': uncertainty, \'per_photo\': per_photo, \'paired_differences\': paired,\n              \'marginal_removal_pq\': {name: {\'pq_without_source\': value.summary()[\'pq\'],\n                                             \'pq_difference_if_removed\': value.summary()[\'pq\']-totals[primary].summary()[\'pq\']}\n                                       for name, value in marginal.items()},\n              \'exposure_audit\': roles, \'exploratory\': not roles[\'gate_data_sufficient\'],\n              \'unavailable_ablations\': [name for name in ABLATIONS if name not in bundle[\'policies\']],\n              \'caveat\': roles[\'provenance_limitations\'],\n              \'timing_note\': \'Per-photo preprocessing and bank-build times are in each trace; calibration reuses cached banks.\'}\n    write_json(engine.out/\'validation_report.json\', report)\n    write_json(engine.out/\'quality_report.json\', {\'features\': FEATURES, \'fit\': bundle[\'fit\'], \'roles\': roles,\n               \'target\': \'mean_view(max_gt_iou * (max_gt_iou > .5))\',\n               \'weights\': \'Each photograph contributes its number of annotation views, divided by its candidate count.\',\n               \'metrics\': {name: quality_metrics(*[np.concatenate(x) for x in zip(*values)]) for name, values in quality.items()},\n               \'missing_values\': \'No NaN accepted. Outside-crop Pb has zero statistics plus a separate valid fraction.\',\n               \'calibration_trials\': bundle[\'calibration_trials\']})\n    table = \'\\n\'.join(f\'| {name} | {value["pq"]:.6f} | {value["tp"]} | {value["fp"]} | {value["fn"]} |\'\n                      for name, value in experiments.items())\n    text = (f\'# V1.5 actual run report\\n\\nDeployment is `{deployment}` because `{reason}`. \'\n            f\'The calibration-selected primary branch is {primary}.\\n\\n\'\n            f\'Quality role photo counts are {roles["role_counts"]}. \'\n            f\'The jointly training-unseen pool has {roles["all_frozen_unseen_count"]} photos, \'\n            f\'with {roles["unseen_after_legacy_label_exclusion"]} remaining after legacy-label exclusion.\\n\\n\'\n            \'| Branch | PQ | TP | FP | FN |\\n| --- | --- | --- | --- | --- |\\n\'+table+\'\\n\\n\'\n            f\'Paired evaluation gain is {delta:.6f}. Group bootstrap interval is {uncertainty}.\\n\\n\'\n            f\'{roles["provenance_limitations"]}\\n\\n\'\n            \'The experiment reuses five frozen networks including the legacy semantic model. \'\n            \'No YOLO model is loaded. Figures contain image predictions only. \'\n            \'Both experimental and exact legacy CSVs are generated in the submit stage.\\n\')\n    (engine.out/\'experiment_report.md\').write_text(text, encoding=\'utf-8\')\n    models_save(engine.out, bundle, models)\n    for name, value in experiments.items():\n        print(f\'{name}: PQ={value["pq"]:.5f}, TP={value["tp"]}, FP={value["fp"]}, FN={value["fn"]}\', flush=True)\n    print(f\'Deployment={deployment} ({reason}), paired held-out gain={delta:+.5f}\', flush=True)\n    return report\n\n\ndef run(options):\n    config = Config(**options).validate()\n    started = time.monotonic()\n    out = Path(config.working_root)/\'solar_v15\'\n    out.mkdir(parents=True, exist_ok=True)\n\n    def check_time():\n        if time.monotonic()-started > config.max_hours*3600:\n            raise TimeoutError(\'V1.5 session budget reached. Reuse its signed bank caches and quality bundle.\')\n\n    cv2.setNumThreads(0)\n    torch.set_num_threads(2)\n    torch.backends.cudnn.benchmark = False\n    np.random.seed(config.seed)\n    torch.manual_seed(config.seed)\n    try:\n        assets = Assets(config, out)\n    except Exception as error:\n        write_json(out/\'preflight_report.json\', {\'passed\': False, \'reason\': str(error),\n                   \'required_inputs\': \'competition + original v1output + GitHub U-Net + signed V3 Blended + complete v1-4output\'})\n        raise\n    contract = runtime_contract(assets, config)\n    write_json(out/\'preflight_report.json\', {\'passed\': True, \'assets\': assets.contract, \'test_photos\': len(assets.tests),\n               \'yolo_models\': 0, \'v1_source\': str(assets.v1_root), \'legacy_source\': str(assets.legacy_path.parent),\n               \'notes\': \'GPU model strict-loading is performed on first uncached inference.\'})\n    write_json(out/\'runtime_contract.json\', {\'config\': config.to_dict(), \'contract\': contract})\n    if config.mode == \'preflight\':\n        return\n    engine = PhotoPipeline(assets, config, contract)\n    if config.mode == \'submit\':\n        bundle, models = bundle_load(out, config, contract, ready=True)\n        from .submission import generate\n        generate(engine, bundle, models, check_time)\n    else:\n        roles, catalog = audit(assets, config, check_time)\n        write_json(out/\'exposure_roles.json\', roles)\n        print(f\'Quality roles={roles["role_counts"]}, unseen={roles["all_frozen_unseen_count"]}, \'\n              f\'after legacy-label exclusion={roles["unseen_after_legacy_label_exclusion"]}\', flush=True)\n        if config.mode == \'infer\':\n            for values in roles[\'roles\'].values():\n                for row in values:\n                    check_time()\n                    engine.compute(Path(catalog.photos[row[\'stem\']][\'path\']))\n            return\n        if config.mode in (\'train_quality\', \'all\'):\n            bundle, models = train_quality(engine, roles, catalog, contract, check_time)\n        else:\n            bundle, models = bundle_load(out, config, contract, ready=False)\n            if bundle[\'role_hash\'] != roles[\'role_hash\']:\n                raise ValueError(\'The mounted annotations/images no longer match the locked roles.\')\n        if config.mode != \'train_quality\':\n            evaluate(engine, bundle, models, catalog, check_time)\n        del catalog\n        if config.mode == \'all\':\n            from .submission import generate\n            generate(engine, bundle, models, check_time)\n    if \'ultralytics\' in sys.modules:\n        raise RuntimeError(\'Unexpected YOLO package import in the isolated V1.5 runtime.\')\n    write_json(out/\'execution_report.json\', {\'seconds\': time.monotonic()-started, \'pipeline\': engine.stats,\n               \'cuda_peak_allocated_bytes\': [torch.cuda.max_memory_allocated(i) for i in range(torch.cuda.device_count())]})\n', 'solar_v15/pipeline.py': '"""One photo at a time: frozen inference, all ablations and GT-free caches."""\n\nimport copy\nimport gzip\nimport json\nimport shutil\nimport time\nfrom dataclasses import asdict\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\nfrom solar_v1.inference import read_pack, save_npz\nfrom solar_v1.preprocess import SolarPreprocessor, read_gray\nfrom solar_v1.postprocess import Instance\nfrom legacy_v14 import semantic as legacy_semantic\nfrom legacy_v14 import selection as legacy_selection\nfrom .candidate_bank import FEATURES, HybridBank, HybridCandidate, assemble, classify, from_families, generate_cc, populate_geometry\nfrom .common import SHAPE, convert, digest, file_hash, normalize_rles\nfrom .selective_refine import refine_semantic_candidate\nfrom .semantic_frozen import FrozenSemanticEnsemble, validate_maps\n\n\nABLATIONS = (\'A1\', \'A2\', \'A3\', \'A4\', \'A5\', \'A6\')\n\n\nclass PhotoPipeline:\n    def __init__(self, assets, config, contract):\n        self.assets, self.config = assets, config\n        self.out = assets.output\n        self.pre = SolarPreprocessor(self.out/\'preprocessed\', enabled=assets.base.PREPROCESS)\n        self.semantic = FrozenSemanticEnsemble(assets)\n        self.contract_hash = digest(contract)\n        self.cache = Path(config.cache_root)/self.contract_hash[:24]\n        self.cache.mkdir(parents=True, exist_ok=True)\n        self.external = Path(config.cache_source)/self.contract_hash[:24] if config.cache_source else None\n        if self.external is not None and not self.external.is_dir():\n            raise FileNotFoundError(f\'No matching signed cache directory in CACHE_SOURCE: {self.external}\')\n        self.stats = {\'network_loads\': 0, \'semantic_forwards\': 0, \'v1_forwards\': 0,\n                      \'cache_hits\': 0, \'bank_hits\': 0, \'yolo_models\': 0, \'yolo_forwards\': 0}\n        self.cache_bytes = sum(p.stat().st_size for p in self.cache.rglob(\'*\') if p.is_file())\n\n    def available(self, path):\n        if path.is_file():\n            return path\n        mounted = self.external/path.name if self.external is not None else None\n        return mounted if mounted is not None and mounted.is_file() else None\n\n    def check_storage(self):\n        if self.cache_bytes > self.config.max_cache_gb*2**30 or shutil.disk_usage(self.cache).free < 2**30:\n            raise RuntimeError(\'V1.5 cache storage exhausted. Preserve bank caches and increase max_cache_gb or set cache_maps=False.\')\n\n    def bank_cache(self, path):\n        return self.cache/f\'{path.stem}-{file_hash(path)}-banks.json.gz\'\n\n    def save_banks(self, dest, banks, legacy, audit):\n        payload = {\'contract_hash\': self.contract_hash, \'legacy\': [asdict(i) for i in legacy], \'audit\': audit,\n                   \'banks\': {name: {\'candidates\': [asdict(c) for c in bank.candidates],\n                                    \'features\': bank.features, \'conflicts\': bank.conflicts}\n                             for name, bank in banks.items()}}\n        self.check_storage()\n        temporary = dest.with_suffix(\'.tmp\')\n        with gzip.open(temporary, \'wt\', encoding=\'utf-8\') as stream:\n            json.dump(payload, stream, default=convert, allow_nan=False)\n        old_size = dest.stat().st_size if dest.exists() else 0\n        temporary.replace(dest)\n        self.cache_bytes += dest.stat().st_size-old_size\n\n    def load_banks(self, path):\n        with gzip.open(path, \'rt\', encoding=\'utf-8\') as stream:\n            value = json.load(stream)\n        if value[\'contract_hash\'] != self.contract_hash:\n            raise ValueError(\'Bank cache provenance mismatch.\')\n        legacy = [Instance(**row) for row in normalize_rles(value[\'legacy\'])]\n        banks = {}\n        for name, row in value[\'banks\'].items():\n            candidates = [HybridCandidate(**r) for r in normalize_rles(row[\'candidates\'])]\n            banks[name] = HybridBank(candidates, np.asarray(row[\'features\'], np.float32).reshape(len(candidates), len(FEATURES)),\n                                     row[\'conflicts\'])\n        return banks, legacy, value[\'audit\']\n\n    @torch.inference_mode()\n    def compute(self, path, visualizer=None, cache_maps=None):\n        started = time.monotonic()\n        identity = file_hash(path)\n        dest = self.bank_cache(path)\n        hit = self.available(dest)\n        if hit is not None and visualizer is None:\n            self.stats[\'bank_hits\'] += 1\n            return self.load_banks(hit)\n        data = self.pre.process(read_gray(path))\n        if data[\'raw\'].shape != SHAPE:\n            raise ValueError(\'V1.5 expects native 2048x2048 photos.\')\n        maps_path = self.cache/f\'{path.stem}-{identity}-frozen.npz\'\n        persist_maps = self.config.cache_maps if cache_maps is None else cache_maps\n        found = self.available(maps_path) if persist_maps else None\n        from .v1_frozen import instances_and_families, predict_pack\n\n        if found is not None:\n            stored = read_pack(found)\n            if str(stored[\'contract_hash\']) != self.contract_hash or str(stored[\'image_hash\']) != identity:\n                raise ValueError(\'Frozen output cache is incompatible with this image or models.\')\n            maps = {k: stored[k] for k in (\'pg\', \'pb\', \'pf\', \'disk_github\')}\n            p14 = stored[\'p14\']\n            pack = {k.removeprefix(\'pack_\'): v for k, v in stored.items() if k.startswith(\'pack_\')}\n            validate_maps(maps)\n            self.stats[\'cache_hits\'] += 1\n        else:\n            models = self.assets.load_networks()\n            self.stats[\'network_loads\'] = 1\n            pack = predict_pack(data, self.assets)\n            self.stats[\'v1_forwards\'] += 1\n            maps = self.semantic.predict_maps(path)\n            self.stats[\'semantic_forwards\'] += 1\n            p14 = legacy_semantic.semantic_map(models[\'legacy\'], data, models[\'semantic_device\'])\n            if persist_maps:\n                self.check_storage()\n                payload = {**{k: maps[k] for k in (\'pg\', \'pb\', \'pf\', \'disk_github\')}, \'p14\': p14,\n                           \'image_hash\': identity, \'contract_hash\': self.contract_hash,\n                           **{f\'pack_{k}\': v for k, v in pack.items()}}\n                save_npz(maps_path, payload)\n                self.cache_bytes += maps_path.stat().st_size\n        if p14.shape != SHAPE or not np.isfinite(p14).all():\n            raise ValueError(\'Legacy V1.4 semantic inference returned invalid probabilities.\')\n        base, families, disk_v1 = instances_and_families(pack, data, self.assets, self.config.detector_floor)\n        github_disk = maps[\'disk_github\'].astype(bool)\n        disk_agreement = np.count_nonzero(disk_v1 & github_disk)/max(np.count_nonzero(disk_v1 | github_disk), 1)\n        if disk_agreement < .80:\n            raise ValueError(f\'V1/GitHub solar disk disagreement is too large: {disk_agreement:.3f}\')\n        disk = disk_v1 & github_disk\n        detector_candidates = from_families(families)\n        if self.assets.legacy_report[\'deployment\'] == \'v14_quality\':\n            old_bank = legacy_selection.build_bank(pack, base, p14, data, self.assets.baseline)\n            old_scores = legacy_selection.predict_quality(self.assets.legacy_selector[\'model\'], old_bank)\n            legacy = legacy_selection.select(old_bank, old_scores, self.assets.legacy_report[\'selected_tau\'])\n        else:\n            legacy = base\n        # A1 uses the exact V1.4 CC routine, including its >= threshold comparison.\n        old_cc = {}\n        legacy_selection.semantic_candidates(old_cc, maps[\'pf\'], disk_v1, self.config.min_area)\n        a1cc = [populate_geometry(HybridCandidate(f\'legacycc:{i}\', c.rle, \'semantic_cc\', f\'legacycc:{i}\',\n                                                \'orphan\', c.level, 0, info={\'original_cc_id\': f\'legacycc:{i}\'}))\n                for i, c in enumerate(old_cc.values())]\n        classify(a1cc, families, self.config)\n        generator_audit = {name: {} for name in (\'pf\', \'pg\', \'pb\')}\n        pf_cc = classify(generate_cc(maps[\'pf\'], disk, self.config, audit=generator_audit[\'pf\']), families, self.config)\n        pg_cc = classify(generate_cc(maps[\'pg\'], disk, self.config, audit=generator_audit[\'pg\']), families, self.config)\n        center_disk = disk.copy()\n        center_disk[:88] = center_disk[1960:] = False\n        center_disk[:, :88] = center_disk[:, 1960:] = False\n        pb_cc = classify(generate_cc(maps[\'pb\'], center_disk, self.config, audit=generator_audit[\'pb\']), families, self.config)\n        refined_cc, refinement_log = [], []\n        if self.config.selective_refinement:\n            eligible = [c for c in pf_cc if c.match_class == \'matched_one\']\n            eligible.sort(key=lambda c: (c.info.get(\'support\', 0.), c.candidate_id))\n            for candidate in eligible[:self.config.selective_budget]:\n                if max(candidate.bbox[2:])*self.assets.base.REFINER_CONTEXT > self.config.selective_max_crop:\n                    refinement_log.append({\'candidate_id\': candidate.candidate_id, \'refine_reason\': \'roi_exceeds_size_budget\'})\n                    continue\n                models = self.assets.load_networks()\n                value, details = refine_semantic_candidate(candidate, data, models[\'refiner\'], self.assets.base,\n                                                            models[\'v1_device\'], disk)\n                refinement_log.append({\'candidate_id\': candidate.candidate_id, **details})\n                if value is not None:\n                    refined = copy.deepcopy(candidate)\n                    refined.candidate_id += \':cc_refined\'\n                    refined.source = \'cc_refined\'\n                    refined.rle = value\n                    refined.info.update(details)\n                    refined_cc.append(populate_geometry(refined))\n        banks = {\n            \'A1\': assemble(detector_candidates, a1cc, data, maps, maps[\'pf\'], False),\n            \'A2\': assemble(detector_candidates, pf_cc, data, maps, maps[\'pf\'], False),\n            \'A3\': assemble(detector_candidates, pf_cc, data, maps, maps[\'pf\'], True),\n            \'A4\': assemble(detector_candidates, pf_cc+refined_cc, data, maps, maps[\'pf\'], True),\n            \'A5\': assemble(detector_candidates, pg_cc, data, maps, maps[\'pg\'], True),\n            \'A6\': assemble(detector_candidates, pb_cc, data, maps, maps[\'pb\'], True),\n        }\n        audit = {\'stem\': path.stem, \'image_hash\': identity, \'pixel_hash\': str(data[\'pixel_hash\']),\n                 \'disk_iou\': disk_agreement, \'disk\': data[\'disk\'],\n                 \'effective_semantic_disk\': \'V1_margin_disk_intersection_GitHub_disk\',\n                 \'model_forwards_this_photo\': found is None, \'candidate_counts\': {k: len(b.candidates) for k, b in banks.items()},\n                 \'cc_classification\': {c.candidate_id: {\'class\': c.match_class, \'families\': c.matched_detector_ids,\n                                                     \'reason\': c.info[\'admission\'], \'evidence\': c.info[\'family_evidence\']}\n                                       for c in pf_cc}, \'selective_refinement\': refinement_log,\n                 \'generator\': generator_audit, \'seconds\': time.monotonic()-started, \'yolo_forwards\': 0}\n        if visualizer is not None:\n            visualizer(data, maps, base, pf_cc, banks[\'A3\'], path.stem)\n        self.save_banks(dest, banks, legacy, audit)\n        return banks, legacy, audit\n', 'solar_v15/provenance.py': '"""Photo/date/pixel exposure audit across both upstream splits and legacy selectors."""\n\nimport hashlib\n\nimport cv2\nfrom sklearn.model_selection import GroupShuffleSplit\n\nfrom filament_seg.data import load_annotations, make_split\nfrom solar_v1.data import Catalog\nfrom solar_v12a.io import train_paths\nfrom dataclasses import replace\nfrom types import SimpleNamespace\nfrom .common import SHAPE, digest, file_hash\n\n\ndef grouped_split(rows, fraction, seed):\n    if len({r[\'group\'] for r in rows}) < 2:\n        raise ValueError(\'At least two date/pixel groups are required to split roles.\')\n    a, b = next(GroupShuffleSplit(n_splits=1, test_size=fraction, random_state=seed)\n                .split(rows, groups=[r[\'group\'] for r in rows]))\n    return [rows[i] for i in a], [rows[i] for i in b]\n\n\ndef assign_roles(rows, config):\n    unseen = [r for r in rows if not r[\'any_neural_training\']]\n    # The deployed A0 selector also learned from GT. Its fit and calibration labels\n    # must not become a supposedly independent comparison set.\n    eligible = [r for r in unseen if not r[\'legacy_selector_labels\']]\n    eligible_groups = {r[\'group\'] for r in eligible}\n    if len(eligible) >= 64 and len(eligible_groups) >= 15:\n        rest, evaluation = grouped_split(eligible, .30, config.seed)\n        fit, calibration = grouped_split(rest, .30, config.seed+1)\n        scope = \'all three roles use declared all-frozen-unseen photos\'\n    else:\n        evaluation = eligible\n        remaining = [r for r in rows if r[\'group\'] not in eligible_groups]\n        # Prefer V1 holdout candidates, but record possible V3 exposure explicitly.\n        preferred = [r for r in remaining if not r[\'v1_training\']]\n        pool = preferred if len(preferred) >= 32 and len({r[\'group\'] for r in preferred}) >= 6 else remaining\n        if not evaluation:\n            pool, evaluation = grouped_split(pool, .20, config.seed)\n            scope = \'exploratory evaluation because no jointly unseen comparison group exists\'\n        else:\n            scope = \'all jointly unseen comparison photos held for evaluation; fit/calibration can be upstream-exposed\'\n        fit, calibration = grouped_split(pool, .25, config.seed+1)\n    for name, values in ((\'fit\', fit), (\'calibration\', calibration)):\n        if len(values) < 8:\n            raise ValueError(f\'Too few quality {name} photos ({len(values)}). Audit has not relaxed group isolation.\')\n    roles = {\'fit\': fit, \'calibration\': calibration, \'evaluation\': evaluation}\n    seen = set()\n    for values in roles.values():\n        groups = {r[\'group\'] for r in values}\n        if seen & groups:\n            raise AssertionError(\'A date/pixel group crosses quality roles.\')\n        seen |= groups\n    sufficient = (len(evaluation) >= config.min_eval_photos\n                  and len({r[\'group\'] for r in evaluation}) >= config.min_eval_groups\n                  and not any(r[\'any_neural_training\'] or r[\'legacy_selector_labels\'] for r in evaluation))\n    return {\'roles\': roles, \'scope\': scope, \'gate_data_sufficient\': sufficient,\n            \'all_frozen_unseen_count\': len(unseen), \'unseen_after_legacy_label_exclusion\': len(eligible),\n            \'role_counts\': {name: len(values) for name, values in roles.items()},\n            \'role_groups\': {name: len({r[\'group\'] for r in values}) for name, values in roles.items()}}\n\n\ndef audit(assets, config, check_time):\n    bridge = SimpleNamespace(INPUT_ROOT=config.input_root, ANNOTATION_PATH=config.annotation_path,\n                             TRAIN_IMAGE_DIR=\'\', TEST_IMAGE_DIR=config.test_image_dir)\n    live_config = replace(train_paths(bridge, assets.base), TEST_IMAGE_DIR=str(assets.tests[0].parent))\n    cat = Catalog(live_config)\n    if cat.hash != assets.manifest[\'annotation_hash\']:\n        raise ValueError(\'V1 training annotations differ from their signed output manifest.\')\n    annotations = load_annotations(cat.annotation)\n    split = make_split(annotations, group_by=\'date\', seed=0, val_fraction=.2)\n    if assets.blended_signature[\'annotation_sha256\'] != file_hash(cat.annotation):\n        raise ValueError(\'Blended annotation signature differs from mounted annotations.\')\n    if split[\'train\'] != assets.blended_signature[\'train_ids\']:\n        raise ValueError(\'Blended training IDs differ from the declared GitHub 563/144 date split.\')\n    github_train = {annotations.images[i].stem for i in split[\'train\']}\n    blended_train = {annotations.images[i].stem for i in assets.blended_signature[\'train_ids\']}\n    saved = {p[\'stem\']: p for p in assets.manifest[\'photos\']}\n    if set(saved) != set(cat.photos) or set(saved) != set(annotations.by_file_stem()):\n        raise ValueError(\'V1/V3 manifests do not describe the same physical photo collection.\')\n    v1_train = set(assets.v1_training[\'detector\']) | set(assets.v1_training[\'refiner\'])\n    legacy_training = {p[\'stem\'] for p in saved.values() if p[\'split\'] == \'train\'}\n    legacy_labels = {p[\'stem\'] for p in assets.legacy_roles[\'photos\'] if p[\'role\'] in (\'quality_fit\', \'calibration\')}\n    parent = {s: s for s in saved}\n\n    def find(s):\n        while parent[s] != s:\n            parent[s] = parent[parent[s]]\n            s = parent[s]\n        return s\n\n    rows, seen = [], {}\n    for i, stem in enumerate(sorted(saved), 1):\n        check_time()\n        p, current = saved[stem], cat.photos[stem]\n        gray = cv2.imread(str(current[\'path\']), cv2.IMREAD_GRAYSCALE)\n        if gray is None or gray.shape != SHAPE:\n            raise ValueError(f\'Cannot audit native competition photo {stem}\')\n        pixels = hashlib.sha256(gray.tobytes()).hexdigest()\n        if (pixels != p[\'pixel_hash\'] or p[\'date\'] != current[\'date\']\n                or set(p[\'records\']) != set(current[\'records\'])):\n            raise ValueError(f\'Date/pixel/annotation identity differs from V1 manifest for {stem}\')\n        row = {\'stem\': stem, \'date\': p[\'date\'], \'pixel_hash\': pixels,\n               \'v1_training\': stem in v1_train, \'v1_manifest_split\': p[\'split\'],\n               \'github_training\': stem in github_train, \'blended_training\': stem in blended_train,\n               \'legacy_semantic_training\': stem in legacy_training, \'legacy_selector_labels\': stem in legacy_labels}\n        rows.append(row)\n        for token in ((\'date\', p[\'date\']), (\'pixels\', pixels), (\'v1_group\', p[\'group\'])):\n            previous = seen.setdefault(token, stem)\n            parent[find(stem)] = find(previous)\n        if i % 150 == 0:\n            print(f\'Frozen training exposure audit {i}/{len(saved)}\', flush=True)\n    flags = (\'v1_training\', \'github_training\', \'blended_training\', \'legacy_semantic_training\', \'legacy_selector_labels\')\n    group_flags = {}\n    for r in rows:\n        r[\'group\'] = find(r[\'stem\'])\n        combined = group_flags.setdefault(r[\'group\'], dict.fromkeys(flags, False))\n        for name in flags:\n            combined[name] |= r[name]\n    for r in rows:\n        r.update(group_flags[r[\'group\']])\n        r[\'any_neural_training\'] = any(r[name] for name in flags[:-1])\n    result = assign_roles(rows, config)\n    v1_val = set(saved)-legacy_training\n    exposed = v1_val & (github_train | blended_train)\n    result.update(photos=rows, intersections={\n        \'V1_train\': sorted(v1_train), \'V1_val\': sorted(v1_val), \'GitHub_train_declared\': sorted(github_train),\n        \'Blended_train_signed\': sorted(blended_train), \'V1_val_intersection_V3_train\': sorted(exposed),\n        \'V1_val_intersection_V3_train_dates\': sorted({saved[s][\'date\'] for s in exposed})},\n        role_exposures={role: {flag: sum(r[flag] for r in values) for flag in flags}\n                        for role, values in result[\'roles\'].items()},\n        provenance_limitations=\'Blended and V1 have saved training IDs. GitHub membership is reconstructed from \'\n                              \'its validated-release reference split, not an independent signed checkpoint manifest. \'\n                              \'Upstream validation pools influenced model and recipe development. \'\n                              \'All-frozen-unseen means training-unseen under these recorded sources; historical \'\n                              \'checkpoint-selection exposure cannot be erased by downstream grouping.\')\n    result[\'role_hash\'] = digest(result)\n    return result, cat\n', 'solar_v15/quality.py': '"""A fresh small HGB regressor for every audited candidate distribution."""\n\nimport numpy as np\nfrom scipy.stats import spearmanr\nfrom sklearn.ensemble import HistGradientBoostingRegressor\nfrom threadpoolctl import threadpool_limits\n\nfrom solar_v1.pq import pairwise_iou\n\n\ndef targets(bank, views):\n    if not views:\n        raise ValueError(\'Quality supervision requires independent annotator views.\')\n    best = np.stack([pairwise_iou(gt, bank.rles).max(axis=0, initial=0) for gt in views.values()])\n    hit = best > .5\n    return np.where(hit, best, 0.).mean(axis=0), hit.mean(axis=0), best\n\n\ndef fit(examples, seed):\n    usable = [e for e in examples if len(e[\'target\'])]\n    if not usable:\n        raise ValueError(\'No candidates for quality training.\')\n    x = np.concatenate([e[\'features\'] for e in usable])\n    y = np.concatenate([e[\'target\'] for e in usable])\n    if len(y) < 100 or np.count_nonzero(y > 0) < 10 or np.count_nonzero(y == 0) < 10:\n        return None\n    weight = np.concatenate([np.full(len(e[\'target\']), e[\'n_views\']/len(e[\'target\'])) for e in usable])\n    model = HistGradientBoostingRegressor(max_iter=120, learning_rate=.055, max_leaf_nodes=7,\n                                          min_samples_leaf=24, l2_regularization=2.,\n                                          early_stopping=False, random_state=seed)\n    with threadpool_limits(limits=2):\n        model.fit(x, y, sample_weight=weight/weight.mean())\n    return model\n\n\ndef predict(model, bank):\n    if model is None:\n        raise ValueError(\'The V1.5 quality model is unavailable; report insufficient supervision explicitly.\')\n    if not bank.rles:\n        return np.empty(0)\n    with threadpool_limits(limits=2):\n        value = model.predict(bank.features).clip(0, 1)\n    if not np.isfinite(value).all():\n        raise ValueError(\'Nonfinite V1.5 quality prediction.\')\n    return value\n\n\ndef quality_metrics(y, predicted, hit_fraction):\n    if not len(y):\n        return {\'n\': 0}\n    correlation = float(spearmanr(y, predicted).statistic) if len(y) > 1 and np.ptp(y) and np.ptp(predicted) else None\n    return {\'n\': len(y), \'mae_effective_iou\': float(np.abs(y-predicted).mean()), \'spearman\': correlation,\n            \'bins\': [{\'lo\': float(low), \'n\': int(keep.sum()), \'predicted_effective_iou\': float(predicted[keep].mean()),\n                      \'actual_effective_iou\': float(y[keep].mean()), \'success_view_fraction\': float(hit_fraction[keep].mean())}\n                     for low in np.arange(0., 1., .1)\n                     if (keep := (predicted >= low) & (predicted < low+.1 if low < .89 else predicted <= 1.)).any()]}\n', 'solar_v15/selection.py': '"""Original V1.4 beam selection plus explicit admission and family conflicts."""\n\nfrom types import SimpleNamespace\n\nimport numpy as np\n\nfrom legacy_v14.selection import select as legacy_beam_select\n\n\ndef admission(candidate, branch, orphan_floor):\n    if any(source in (\'base\', \'coarse\', \'refined\') for source in candidate.info[\'sources\']):\n        return True, \'detector_or_base_candidate\'\n    if branch in (\'A1\', \'A2\'):\n        return True, \'uncontrolled_cc_ablation\'\n    if candidate.match_class == \'overmerge\':\n        return False, \'overmerge_evidence_only\'\n    if candidate.match_class == \'matched_one\':\n        return True, \'same_detector_family_alternative\'\n    if candidate.info.get(\'admission\') != \'orphan_candidate\':\n        return False, \'ambiguous_detector_overlap\'\n    if (candidate.info[\'primary_mean\'] < orphan_floor or candidate.info[\'primary_q20\'] < .35\n            or candidate.info[\'contrast\'] < -.01):\n        return False, \'orphan_support_or_contrast\'\n    return True, \'orphan_admitted_to_quality_selection\'\n\n\ndef select(bank, values, policy, branch, disabled_source=None):\n    permitted = np.ones(len(bank.candidates), bool)\n    reasons = {}\n    for i, c in enumerate(bank.candidates):\n        accepted, reason = admission(c, branch, policy[\'orphan_floor\'])\n        if disabled_source and disabled_source in c.info[\'sources\']:\n            accepted, reason = False, \'source_ablation\'\n        permitted[i] = accepted\n        reasons[c.candidate_id] = reason if not accepted else \'below_quality_tau\'\n    score = np.where(permitted, values, -1.)\n    legacy_bank = SimpleNamespace(rles=bank.rles, conflicts=bank.conflicts, sources=bank.sources)\n    instances = legacy_beam_select(legacy_bank, score, policy[\'tau\'])\n    by_mask = {c.rle[\'counts\']: i for i, c in enumerate(bank.candidates)}\n    chosen = [by_mask[ins.rle[\'counts\']] for ins in instances]\n    chosen_set = set(chosen)\n    for i, c in enumerate(bank.candidates):\n        if i in chosen_set:\n            reasons[c.candidate_id] = \'selected\'\n        elif permitted[i] and score[i] > policy[\'tau\']:\n            reasons[c.candidate_id] = (\'pixel_or_family_conflict\' if any(bank.conflicts[i] & (1 << j) for j in chosen)\n                                       else \'beam_budget_or_objective\')\n    return instances, chosen, reasons\n', 'solar_v15/selective_refine.py': '"""True CC-prompt refinement using the frozen original five-channel V1 model."""\n\nimport cv2\nimport numpy as np\nimport torch\nfrom pycocotools import mask as cm\n\nfrom solar_v1.config import autocast\nfrom solar_v1.data import roi_channels, square_box\nfrom solar_v1.postprocess import encode\nfrom solar_v12a.features import clean\n\n\n@torch.inference_mode()\ndef refine_semantic_candidate(candidate, data, model, config, device, disk):\n    if candidate.match_class != \'matched_one\':\n        raise ValueError(\'Only matched-one CC prompts are enabled in this version.\')\n    if model.training or any(p.requires_grad for p in model.parameters()):\n        raise ValueError(\'Selective refinement requires an explicitly frozen eval model.\')\n    x, y, w, h = candidate.bbox\n    bounds = square_box((x, y, x+w, y+h), data[\'raw\'].shape,\n                        config.REFINER_CONTEXT, config.REFINER_MIN_CROP)\n    x0, y0, x1, y1 = bounds\n    original = cm.decode(candidate.rle).astype(bool)\n    prompt = original[y0:y1, x0:x1].astype(np.float32)\n    channels = roi_channels(data, bounds, 384, prompt, config)\n    if channels.shape != (5, 384, 384) or not np.isfinite(channels).all():\n        raise ValueError(\'Semantic prompt does not match the V1 five-channel ROI contract.\')\n    with autocast(config, device):\n        probability = model(torch.from_numpy(channels[None]).to(device)).sigmoid().float().cpu().numpy()[0, 0]\n    if not np.isfinite(probability).all():\n        raise ValueError(\'Nonfinite frozen refiner output for a semantic CC.\')\n    mask = np.zeros(data[\'raw\'].shape, bool)\n    mask[y0:y1, x0:x1] = cv2.resize(probability, (x1-x0, y1-y0), interpolation=cv2.INTER_LINEAR) >= config.REFINER_THRESHOLD\n    mask = clean(mask, disk, config)\n    details = {\'roi\': list(bounds), \'prompt_mode\': \'raw_x3+cc_prompt+radial\', \'input_shape\': list(channels.shape),\n               \'components_before\': int(cv2.connectedComponents(original.astype(np.uint8), connectivity=8)[0]-1),\n               \'components_after\': int(cv2.connectedComponents(mask.astype(np.uint8), connectivity=8)[0]-1),\n               \'before_area\': int(original.sum()), \'after_area\': int(mask.sum()),\n               \'changed_area\': int(np.count_nonzero(original ^ mask)),\n               \'before_after_iou\': float(np.count_nonzero(original & mask)/max(np.count_nonzero(original | mask), 1))}\n    if mask.sum() < config.MIN_INSTANCE_AREA:\n        return None, {**details, \'refine_reason\': \'empty_or_too_small\'}\n    return encode(mask), {**details, \'refine_reason\': \'added_competing_variant\'}\n', 'solar_v15/semantic_frozen.py': '"""The exact two semantic forward paths of V3, with no detector dependency."""\n\nimport cv2\nimport numpy as np\nimport torch\nfrom scipy.special import expit\n\nfrom filament_seg.disk import detect_disk, flatten, read_image\nfrom filament_seg.model import tiled_predict_tta\nfrom .common import SHAPE, file_hash\nfrom .feather import blend\n\n\nclass FrozenSemanticEnsemble:\n    def __init__(self, assets):\n        self.assets = assets\n        self.calls = 0\n\n    @torch.inference_mode()\n    def predict_maps(self, path):\n        models = self.assets.load_networks()\n        image = read_image(path)\n        gray = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)\n        if image.shape != SHAPE or gray is None or gray.shape != SHAPE:\n            raise ValueError(f\'Unexpected native image size for {path}\')\n        disk = detect_disk(image)\n        flat = (np.clip(flatten(image, disk), 0, 2)*127.5).astype(np.uint8)\n        inputs = np.stack([flat.astype(np.float32)/255., disk.radius_map(SHAPE).astype(np.float32)])\n        logits = tiled_predict_tta(models[\'github\'], inputs, tta=\'dihedral\', tile=512, overlap=128,\n                                   device=models[\'semantic_device\']).astype(np.float16).astype(np.float32)\n        pg = expit(logits).astype(np.float32)\n        pb = models[\'source\'].mosaic_predict(models[\'blended\'], gray).cpu().numpy().astype(np.float32)\n        maps = dict(pg=pg, pb=pb, pf=blend(pg, pb, (88, 1960), .55, 64),\n                    disk_github=disk.mask(SHAPE), image_hash=file_hash(path))\n        validate_maps(maps)\n        self.calls += 1\n        return maps\n\n\ndef validate_maps(maps):\n    for key in (\'pg\', \'pb\', \'pf\'):\n        x = maps[key]\n        if x.shape != SHAPE or x.dtype != np.float32 or not np.isfinite(x).all() or np.any((x < 0) | (x > 1)):\n            raise ValueError(f\'Invalid frozen semantic output {key}\')\n    outer = np.ones(SHAPE, bool)\n    outer[88:1960, 88:1960] = False\n    if np.any(maps[\'pb\'][outer] != 0) or not np.array_equal(maps[\'pg\'][outer], maps[\'pf\'][outer]):\n        raise ValueError(\'Blended crop or Feather outer-region geometry mismatch.\')\n    expected = blend(maps[\'pg\'], maps[\'pb\'], (88, 1960), .55, 64)\n    if not np.array_equal(maps[\'pf\'], expected):\n        raise ValueError(\'Feather differs from the frozen V3 arithmetic.\')\n', 'solar_v15/source_contract.py': "BLENDED_ORIGINAL_SHA = '5a7645769c230045c46edf54d03296e40c2946175cc9f0fea10bba255beaeb6b'\nBLENDED_NOTEBOOK_SOURCE_SHA = '3f3815d5f2b3be9ce7604bc982ee6c87c6ecc18f84cf23dfd79973769a3cd830'\nLEGACY_SEMANTIC_SHA = 'c5149387d9099e6b134aac42bce9886ee6b0f9460a3f6cf04e9747fdcf65a3b4'\nAUDITED_NOTEBOOKS = {'fork-of-fork-of-cv-v1-2a (5).ipynb': 'd7a2aa2c3e1f8c060999e67f35babf90fe9b650f15185ff1b4c6e48af16a0b22', 'fork-of-fork-of-fork-of-cv-v1-2a (2).ipynb': '4cc7968e13fdba402d7e26a7e0d5a2aac52cb8237093881a09affdfccd3af5b8'}\n", 'solar_v15/submission.py': '"""Produce and audit both V1.4 fallback and V1.5 experimental submissions."""\n\nimport os\nimport shutil\nimport tempfile\nimport time\nfrom dataclasses import asdict\nfrom pathlib import Path\n\nfrom solar_v1.submission import SubmissionWriter\nfrom solar_v12a.submission import verify_csv\nfrom .common import SHAPE, VERSION, file_hash, read_json, write_json\nfrom .quality import predict\nfrom .selection import select\n\n\ndef generate(engine, bundle, models, check_time):\n    work, out = Path(engine.config.working_root), engine.out\n    target, proof = work/\'submission.csv\', out/\'submission_audit.json\'\n    if target.exists():\n        old = read_json(proof) if proof.is_file() else {}\n        if old.get(\'version\') != VERSION or old.get(\'csv_sha256\') != file_hash(target):\n            raise FileExistsError(\'Existing submission.csv belongs to another experiment. Use a fresh Kaggle session.\')\n    primary = bundle[\'primary\']\n    if bundle[\'deployment\'] not in (\'v14_fallback\', \'v15\') or bundle[\'status\'] != \'ready\':\n        raise ValueError(\'Submission requires an explicit ready deployment decision.\')\n    if models.get(primary) is None:\n        raise FileNotFoundError(\'Cannot produce the V1.5 experimental CSV without its trained quality model.\')\n    paths = engine.assets.tests\n    ids = [p.stem for p in paths]\n    if len(ids) != 180 or len(set(ids)) != 180:\n        raise ValueError(\'Submission requires all 180 distinct official test photos.\')\n    started = time.monotonic()\n    with tempfile.TemporaryDirectory(prefix=\'submission-\', dir=out) as temporary:\n        directory = Path(temporary)\n        fallback = SubmissionWriter(directory/\'submission_v14_fallback.csv\', ids, SHAPE,\n                                    directory/\'fallback_coverage.json\')\n        experimental = SubmissionWriter(directory/\'submission_v15_experimental.csv\', ids, SHAPE,\n                                        directory/\'experimental_coverage.json\')\n        photo_stats = []\n        for i, path in enumerate(paths, 1):\n            check_time()\n            banks, legacy, audit = engine.compute(path, cache_maps=engine.config.cache_test_maps)\n            bank = banks[primary]\n            values = predict(models[primary], bank)\n            final, chosen, reasons = select(bank, values, bundle[\'policies\'][primary], primary)\n            fallback.add(path.stem, legacy)\n            experimental.add(path.stem, final)\n            write_json(out/\'test_trace\'/f\'{path.stem}.json\', {\n                \'image_hash\': audit[\'image_hash\'], \'primary\': primary, \'audit\': audit, \'gt_access\': False,\n                \'quality_model_applied\': True, \'candidates\': [dict(asdict(c), predicted_quality=float(values[j]),\n                    decision=reasons[c.candidate_id]) for j, c in enumerate(bank.candidates)],\n                \'selected_indices\': chosen, \'v15_final\': [asdict(x) for x in final],\n                \'v14_final\': [asdict(x) for x in legacy]})\n            photo_stats.append({\'stem\': path.stem, \'candidates\': len(bank.candidates),\n                                \'v14_instances\': len(legacy), \'v15_instances\': len(final)})\n            if i == 1 or i % 20 == 0 or i == len(paths):\n                print(f\'Test {i}/180: V1.4={len(legacy)}, V1.5={len(final)}\', flush=True)\n        fallback.finish()\n        experimental.finish()\n        shapes = dict.fromkeys(ids, SHAPE)\n        reports = {\'v14_fallback\': verify_csv(fallback.path, fallback.coverage_path, ids, shapes),\n                   \'v15\': verify_csv(experimental.path, experimental.coverage_path, ids, shapes)}\n        chosen_writer = fallback if bundle[\'deployment\'] == \'v14_fallback\' else experimental\n        canonical = directory/\'submission.csv\'\n        shutil.copyfile(chosen_writer.path, canonical)\n        coverage = directory/\'submission_coverage.json\'\n        shutil.copyfile(chosen_writer.coverage_path, coverage)\n        selected_report = verify_csv(canonical, coverage, ids, shapes)\n        for writer in (fallback, experimental):\n            os.replace(writer.path, work/writer.path.name)\n            os.replace(writer.coverage_path, out/writer.coverage_path.name)\n        os.replace(coverage, work/\'submission_coverage.json\')\n        os.replace(canonical, target)\n    report = {\'version\': VERSION, **selected_report, \'deployment\': bundle[\'deployment\'],\n              \'deployment_reason\': bundle[\'deployment_reason\'], \'primary\': primary, \'both_submissions\': reports,\n              \'quality_model_applied_to_v15\': True, \'test_gt_access\': False, \'old_submission_read\': False,\n              \'yolo_models\': 0, \'yolo_forwards\': 0, \'policy\': bundle[\'policies\'][primary],\n              \'runtime_seconds\': time.monotonic()-started, \'contract_hash\': engine.contract_hash,\n              \'role_hash\': bundle[\'role_hash\'], \'per_photo\': photo_stats,\n              \'recommended_file\': str(target), \'pipeline_statistics\': engine.stats}\n    write_json(proof, report)\n    print(f\'READY TO SUBMIT: {target} | {selected_report["n_instances"]} instances | {bundle["deployment"]}\', flush=True)\n', 'solar_v15/v1_frozen.py': '"""Original V1 predictions with exact detector-index lineage."""\n\nfrom dataclasses import replace\n\nimport cv2\nimport numpy as np\nimport torch\n\nfrom solar_v1.inference import add_refiner, detector_pack, native_coarse\nfrom solar_v1.postprocess import finalize, postprocess, raw_instances, encode\nfrom solar_v12a.features import clean, disk_mask\n\n\n@torch.inference_mode()\ndef predict_pack(data, assets):\n    models = assets.load_networks()\n    pack = detector_pack(models[\'detector\'], data, assets.base, models[\'v1_device\'])\n    pack = add_refiner(pack, data, models[\'refiner\'], assets.base, models[\'v1_device\'])\n    for name in (\'scores\', \'boxes\', \'ref_probs\'):\n        if not np.isfinite(pack[name]).all():\n            raise ValueError(f\'Nonfinite frozen V1 prediction {name}\')\n    return pack\n\n\ndef instances_and_families(pack, data, assets, floor):\n    config = assets.baseline\n    disk = disk_mask(data, config)\n    if disk is None:\n        disk = np.ones(data[\'raw\'].shape, bool)\n    tagged, families = [], {}\n    for i, score in enumerate(pack[\'scores\']):\n        family = f\'detector:{i}\'\n        one = {\'shape\': pack[\'shape\'], \'detector_size\': pack[\'detector_size\'],\n               \'scores\': pack[\'scores\'][i:i+1], \'det_bounds\': pack[\'det_bounds\'][i:i+1],\n               \'det_0\': pack[f\'det_{i}\'], \'ref_bounds\': pack[\'ref_bounds\'][i:i+1],\n               \'ref_probs\': pack[\'ref_probs\'][i:i+1]}\n        tagged.extend(replace(ins, source=family) for ins in raw_instances(one, config, disk))\n        # Families exist even below the later candidate admission floor.\n        variants = []\n        coarse = clean(native_coarse(pack, i) >= config.MASK_THRESHOLD, disk, config)\n        x0, y0, x1, y1 = map(int, pack[\'ref_bounds\'][i])\n        refined = np.zeros(data[\'raw\'].shape, bool)\n        refined[y0:y1, x0:x1] = cv2.resize(pack[\'ref_probs\'][i], (x1-x0, y1-y0),\n                                          interpolation=cv2.INTER_LINEAR) >= config.REFINER_THRESHOLD\n        refined = clean(refined, disk, config)\n        for source, mask in ((\'coarse\', coarse), (\'refined\', refined)):\n            if mask.sum() >= config.MIN_INSTANCE_AREA:\n                variants.append(dict(source=source, rle=encode(mask)))\n        pair = float(np.count_nonzero(coarse & refined)/max(np.count_nonzero(coarse | refined), 1))\n        families[family] = dict(detector_index=i, raw_score=float(score), variants=variants,\n                                coarse_refined_iou=pair, candidate_admitted=bool(score >= floor))\n    base = finalize(tagged, config, data[\'raw\'].shape, disk=disk)\n    original = postprocess(pack, config, data)\n    if [x.rle for x in base] != [x.rle for x in original]:\n        raise AssertionError(\'Detector lineage reconstruction changed original V1 postprocessing.\')\n    for ins in base:\n        families[ins.source][\'variants\'].append(dict(source=\'base\', rle=ins.rle))\n    return base, families, disk\n', 'solar_v15/validation.py': '"""Legal official PQ and separate per-view candidate diagnostics."""\n\nfrom collections import Counter, defaultdict\n\nimport cv2\nimport numpy as np\nfrom pycocotools import mask as cm\n\nfrom solar_v1.pq import Totals, from_iou, pairwise_iou, require_disjoint\nfrom .common import areas\n\n\ndef area_group(area):\n    return (\'under250\', \'250_999\', \'1000_3999\', \'4000plus\')[int(np.searchsorted([250, 1000, 4000], area, side=\'right\'))]\n\n\ndef score(instances, views, shape, disk=None):\n    rles = [i.rle for i in instances]\n    require_disjoint(rles, shape)\n    total = Totals()\n    by_annotator, small = {}, Counter()\n    cx, cy, radius = disk if disk is not None else (shape[1]/2, shape[0]/2, min(shape)*.439)\n    for record, gt in views.items():\n        matrix = pairwise_iou(gt, rles)\n        value = from_iou(matrix)\n        total += value\n        group = record.split(\'-\', 1)[0]\n        by_annotator[group] = by_annotator.get(group, Totals())+value\n        a = areas(gt)\n        mask = (a >= 250) & (a < 1000)\n        small.update(small_gt=int(mask.sum()), small_matched=int(((matrix > .5).any(axis=1) & mask).sum()))\n        boxes = cm.toBbox(gt) if gt else []\n        for j, (area, box) in enumerate(zip(a, boxes)):\n            radial = np.hypot(box[0]+box[2]/2-cx, box[1]+box[3]/2-cy)/max(radius, 1.)\n            for group in (area_group(area), \'center\' if radial < .65 else \'outer\'):\n                hit = matrix[j] > .5\n                small.update({group+\'_gt\': 1, group+\'_tp\': int(hit.sum()), group+\'_fn\': int(not hit.any()),\n                              group+\'_iou_sum\': float(matrix[j, hit].sum())})\n        pred_areas = areas(rles)\n        pred_boxes = cm.toBbox(rles) if rles else []\n        for j, (area, box) in enumerate(zip(pred_areas, pred_boxes)):\n            if (matrix[:, j] > .5).any():\n                continue\n            radial = np.hypot(box[0]+box[2]/2-cx, box[1]+box[3]/2-cy)/max(radius, 1.)\n            for group in (area_group(area), \'center\' if radial < .65 else \'outer\'):\n                small[group+\'_fp\'] += 1\n    return total, by_annotator, small\n\n\ndef oracle_and_selection(bank, chosen, views):\n    counts = Counter()\n    source = np.asarray([any(s in c.info[\'sources\'] for s in (\'base\', \'coarse\', \'refined\'))\n                         for c in bank.candidates], bool)\n    semantic = np.asarray([any(s in c.info[\'sources\'] for s in (\'semantic_cc\', \'cc_refined\'))\n                           for c in bank.candidates], bool)\n    selected_orphans = [i for i in chosen if bank.candidates[i].match_class == \'orphan\'\n                        and not source[i]]\n    refined = [(i, c) for i, c in enumerate(bank.candidates) if c.source == \'cc_refined\']\n    originals = {c.info.get(\'original_cc_id\'): i for i, c in enumerate(bank.candidates) if c.source == \'semantic_cc\'}\n    for gt in views.values():\n        best = pairwise_iou(gt, bank.rles)\n        good = best > .5\n        v1_hit, sem_hit = good[:, source].any(axis=1), good[:, semantic].any(axis=1)\n        small = (areas(gt) >= 250) & (areas(gt) < 1000)\n        counts.update(gt=len(gt), v1_candidate_covered=int(v1_hit.sum()), semantic_candidate_covered=int(sem_hit.sum()),\n                      union_candidate_covered=int(good.any(axis=1).sum()), cc_unique_gt=int((sem_hit & ~v1_hit).sum()),\n                      candidate_small_gt=int(small.sum()), small_union_candidate_covered=int((good.any(axis=1) & small).sum()))\n        counts[\'selected_orphan_tp_pairs\'] += int(good[:, selected_orphans].sum())\n        counts[\'selected_orphan_fp\'] += int((~good[:, selected_orphans].any(axis=0)).sum())\n        quality = best.max(axis=0, initial=0)\n        for j, candidate in refined:\n            i = originals.get(candidate.info.get(\'original_cc_id\'))\n            if i is not None:\n                counts[\'cc_refinement_rescued_per_view\'] += int(quality[i] <= .5 < quality[j])\n                counts[\'cc_refinement_degraded_per_view\'] += int(quality[j] <= .5 < quality[i])\n    counts[\'selected_orphan_candidates\'] = len(selected_orphans)\n    return dict(counts)\n\n\ndef paired_uncertainty(rows, seed, repetitions=300):\n    groups = defaultdict(lambda: [Totals(), Totals()])\n    for row in rows:\n        for i, name in enumerate((\'baseline\', \'new\')):\n            groups[row[\'group\']][i] += Totals(**row[name])\n    if len(groups) < 2:\n        return {\'date_groups\': len(groups), \'lower95\': None, \'upper95\': None, \'reason\': \'fewer_than_two_groups\'}\n    values = list(groups.values())\n    rng = np.random.default_rng(seed)\n    deltas = []\n    for _ in range(repetitions):\n        a, b = Totals(), Totals()\n        for index in rng.integers(len(values), size=len(values)):\n            a += values[index][0]\n            b += values[index][1]\n        deltas.append(b.summary()[\'pq\']-a.summary()[\'pq\'])\n    return {\'date_groups\': len(groups), \'repetitions\': repetitions,\n            \'lower95\': float(np.quantile(deltas, .025)), \'upper95\': float(np.quantile(deltas, .975))}\n\n\ndef visualize(directory, case_number, model, policy):\n    """Eight anonymous research panels. This callback receives no GT."""\n    def callback(data, maps, base, cc, bank, stem):\n        from .quality import predict\n        from .selection import select\n\n        def overlay(rles):\n            image = cv2.cvtColor(data[\'raw\'], cv2.COLOR_GRAY2BGR)\n            for i, rle in enumerate(rles):\n                mask = cm.decode(rle).astype(bool)\n                color = np.asarray(((53+71*i)%255, (140+43*i)%255, (220+23*i)%255), np.float32)\n                image[mask] = (.5*image[mask]+.5*color).astype(np.uint8)\n            return image\n\n        final, chosen, _ = select(bank, predict(model, bank), policy, \'A3\')\n        panels = [overlay([i.rle for i in base])]\n        panels.extend(cv2.applyColorMap((maps[k]*255).astype(np.uint8), cv2.COLORMAP_VIRIDIS)\n                      for k in (\'pg\', \'pb\', \'pf\'))\n        panels.append(overlay([c.rle for c in cc if c.close_kernel == 0]))\n        panels.append(overlay([c.rle for c in cc if c.close_kernel != 0]))\n        matching = cv2.cvtColor(data[\'raw\'], cv2.COLOR_GRAY2BGR)\n        colors = {\'matched_one\': (20, 220, 20), \'orphan\': (0, 210, 250), \'overmerge\': (40, 40, 240)}\n        for c in cc:\n            contours, _ = cv2.findContours(cm.decode(c.rle), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)\n            cv2.drawContours(matching, contours, -1, colors[c.match_class], 3)\n        panels.extend([matching, overlay([i.rle for i in final])])\n        names = (\'V1 instances\', \'GitHub Pg\', \'Blended Pb\', \'Feather Pf\', \'Raw CC\', \'Closed CC\', \'A/B/C matching\', \'A3 selected\')\n        resized = []\n        for panel, title in zip(panels, names):\n            view = cv2.resize(panel, (384, 384), interpolation=cv2.INTER_AREA)\n            cv2.putText(view, title, (10, 24), cv2.FONT_HERSHEY_SIMPLEX, .6, (255, 255, 255), 1)\n            resized.append(view)\n        directory.mkdir(parents=True, exist_ok=True)\n        if not cv2.imwrite(str(directory/f\'case_{case_number:03}.png\'),\n                           np.vstack([np.hstack(resized[:4]), np.hstack(resized[4:])])):\n            raise IOError(\'Failed to save research comparison image.\')\n    return callback\n', 'solar_gq/__init__.py': '"""Independent G/Q experiments over the frozen standalone V1.5 A3 recipe."""\n\nVERSION = \'gq-20261106.1\'\n', 'solar_gq/config.py': "from dataclasses import dataclass, asdict\n\n\n@dataclass\nclass Config:\n    line: str = 'Q'\n    input_root: str = '/kaggle/input'\n    working_root: str = '/kaggle/working'\n    cache_source: str = ''\n    original_bundle_source: str = ''\n    v1_source: str = ''\n    legacy_root: str = ''\n    github_checkpoint: str = ''\n    blended_checkpoint: str = ''\n    annotation_path: str = ''\n    test_image_dir: str = ''\n    install_missing: bool = True\n    max_hours: float = 10.5\n    max_cache_gib: float = 24.\n    min_free_gib: float = 1.\n    seed: int = 15026\n    resume: bool = True\n    reuse_v1_cache: bool = True\n    # User cancelled historical A3 reproduction / CSV equality acceptance.\n    historical_csv_comparison: bool = False\n    tree_levels: int = 4096\n    tree_budget: int = 48\n    tree_min_persistence: float = .015\n    tree_min_level: float = .25\n    graph_roi_pixels: int = 1200000\n    graph_max_nodes: int = 12000\n    repair_budget: int = 8\n    repair_pairs: int = 4\n    repair_max_gap: float = 64.\n    repair_min_support: float = .25\n    repair_max_detour: float = 2.5\n    repair_max_mean_turn: float = .40\n    repair_max_area_ratio: float = 1.8\n    repair_split_families: int = 4\n    graph_beam_width: int = 32\n    graph_weights: tuple = (0., .02, .05)\n    stability_deltas: tuple = (.025, .05, .10)\n    stability_roi_margin: int = 16\n    taus: tuple = (.16, .20, .24, .28, .32, .36)\n    orphan_floors: tuple = (.50, .60, .70)\n    visualization_count: int = 8\n\n    @property\n    def stages(self):\n        return ('G0', 'G1', 'G2', 'G3', 'G4') if self.line == 'G' else ('Q0', 'Q1', 'Q2')\n\n    def validate(self):\n        if self.line not in ('G', 'Q') or self.historical_csv_comparison:\n            raise ValueError('Choose G/Q. Historical CSV reproduction was cancelled by the user.')\n        if not 1 <= self.tree_budget <= 128 or not 256 <= self.tree_levels <= 65536:\n            raise ValueError('Invalid bounded max-tree settings.')\n        if not 0 < self.max_hours <= 24 or self.max_cache_gib <= 0:\n            raise ValueError('Invalid runtime/storage budget.')\n        if not 1 <= self.repair_budget <= 16 or not 1 <= self.repair_pairs <= 8:\n            raise ValueError('Graph repair budget must remain bounded.')\n        if self.graph_roi_pixels < 100 or self.graph_max_nodes < 10:\n            raise ValueError('Graph resource limits are too small.')\n        if not 1<=self.graph_beam_width<=128 or not 2<=self.repair_split_families<=8:\n            raise ValueError('Invalid bounded graph search/family settings.')\n        if not 0<=self.repair_max_mean_turn<=2 or not 0<self.repair_max_gap<=256:\n            raise ValueError('Invalid path curvature/gap bound.')\n        if any(not 0 < x < .25 for x in self.stability_deltas):\n            raise ValueError('Stability offsets are probabilities, not SAM logit offsets.')\n        if any(not 0 < x < 1 for x in (*self.taus, *self.orphan_floors)):\n            raise ValueError('Invalid calibration grid.')\n        return self\n\n    def to_dict(self):\n        return asdict(self)\n", 'solar_gq/diagnostics.py': "from collections import Counter, defaultdict\n\nimport numpy as np\nfrom pycocotools import mask as cm\n\nfrom solar_v1.pq import pairwise_iou\nfrom solar_v15.quality import targets\nfrom solar_v15.selection import admission\nfrom .io import rle_hash\n\n\ndef diagnose(bank,values,heads,chosen,reasons,views,policy,stem,group,role,stage):\n    admitted=np.array([admission(c,'A3',policy['orphan_floor'])[0] for c in bank.candidates],bool)\n    quality=admitted&(values>policy['tau'])\n    selected=np.zeros(len(bank.candidates),bool);selected[chosen]=True\n    masks={'before_admission':np.ones(len(bank.candidates),bool),'after_admission':admitted,\n           'after_quality':quality,'after_selection':selected}\n    flow={name:{'candidates':int(keep.sum()),'gt_view_instances':0,'covered_gt_view_instances':0}\n          for name,keep in masks.items()}\n    fate=[]\n    for view,gt in views.items():\n        matrix=pairwise_iou(gt,bank.rles);good=matrix>.5\n        for name,keep in masks.items():\n            flow[name]['gt_view_instances']+=len(gt)\n            flow[name]['covered_gt_view_instances']+=int(good[:,keep].any(axis=1).sum())\n        areas=np.asarray(cm.area(gt),float) if len(gt)<=255 else np.concatenate([\n            np.asarray(cm.area(gt[i:i+255]),float) for i in range(0,len(gt),255)])\n        for j in range(len(gt)):\n            row={'stem':stem,'group':group,'role':role,'stage':stage,'annotation_view':view,'gt_index':j,'gt_area':float(areas[j])}\n            for name,keep in masks.items():\n                row['best_iou_'+name]=float(matrix[j,keep].max(initial=0))\n                row['covered_'+name]=bool(good[j,keep].any())\n            opportunities=np.flatnonzero(good[j])\n            row['matching_candidates']=[{'id':bank.candidates[k].candidate_id,'quality':float(values[k]),\n                                         'iou':float(matrix[j,k]),'reason':reasons[bank.candidates[k].candidate_id]}\n                                         for k in opportunities]\n            row['first_loss_stage']=next((name for name in ('after_admission','after_quality','after_selection')\n                if row['covered_before_admission'] and not row['covered_'+name]),\n                'selected_coverage' if row['covered_after_selection'] else 'no_matching_candidate')\n            fate.append(row)\n    y,hit,best=targets(bank,views)\n    decisions=[]\n    for i,c in enumerate(bank.candidates):\n        conflicts=[bank.candidates[j].candidate_id for j in range(len(bank.candidates)) if bank.conflicts[i]&(1<<j)]\n        row={'stem':stem,'group':group,'role':role,'stage':stage,'candidate_id':c.candidate_id,\n             'source':c.source,'family_id':c.family_id,'matched_detector_ids':c.matched_detector_ids,\n             'parent_id':c.parent_id,'rle_sha256':rle_hash(c.rle),'area':c.area,'threshold':c.threshold,\n             'match_class':c.match_class,'q_eff':float(values[i]),'target_effective_iou':float(y[i]),\n             'target_hit_view_fraction':float(hit[i]),'best_iou_by_view':best[:,i].tolist(),\n             'admitted':bool(admitted[i]),'after_quality':bool(quality[i]),'selected':bool(selected[i]),\n             'reason':reasons[c.candidate_id],'conflicts':conflicts,\n             'rle_storage':'prepared/<experiment-signature>/<stem>.json.gz',\n             'is_604_pixel_semantic_candidate':bool(c.area==604 and 'semantic_cc' in c.info.get('sources',[c.source]))}\n        if 'p_hit' in heads:row.update(p_hit=float(heads['p_hit'][i]),q_pos=float(heads['q_pos'][i]))\n        decisions.append(row)\n    quality_row={'target':y,'quality':values,'hit':hit}\n    if 'p_hit' in heads:quality_row.update(p_hit=heads['p_hit'],q_pos=heads['q_pos'])\n    ranking=Counter()\n    families=defaultdict(list)\n    for i,c in enumerate(bank.candidates):\n        for family in c.info.get('family_aliases',[c.family_id]):families[family].append(i)\n    for members in families.values():\n        for a,first in enumerate(members):\n            for second in members[a+1:]:\n                truth=y[first]-y[second]\n                if abs(truth)>1e-8:\n                    ranking['comparable_pairs']+=1\n                    ranking['correct_pairs']+=int(truth*(values[first]-values[second])>0)\n    return flow,fate,decisions,quality_row,dict(ranking)\n\n\ndef add_flow(total,value):\n    for name,row in value.items():\n        for key,count in row.items():total.setdefault(name,{}).setdefault(key,0);total[name][key]+=count\n", 'solar_gq/frozen.py': '"""Only this module may invoke frozen networks; all stages consume its output."""\nimport copy\nimport time\nfrom dataclasses import asdict\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\nfrom solar_v1.inference import read_pack, save_npz\nfrom solar_v1.preprocess import SolarPreprocessor, read_gray\nfrom solar_v15.assets import Assets\nfrom solar_v15.candidate_bank import assemble, classify, from_families, generate_cc\nfrom solar_v15.common import SHAPE, digest, file_hash, freeze, read_json, write_json\nfrom solar_v15.feather import blend\nfrom solar_v15.semantic_frozen import FrozenSemanticEnsemble, validate_maps\nfrom solar_v15.v1_frozen import instances_and_families, predict_pack\nfrom .io import bank_dict, load_bank, read_gz, write_gz\n\n\nclass FrozenAssets(Assets):\n    def load_networks(self):\n        if self.models is not None:\n            return self.models\n        if not torch.cuda.is_available():\n            raise RuntimeError(\'A missing frozen prediction needs a Kaggle GPU. Mount a complete compatible cache for CPU replay.\')\n        from solar_v1.models import load_model\n        from filament_seg.model import load_trained\n        import blended_frozen_source as source\n        v1_device = torch.device(\'cuda:0\')\n        semantic_device = torch.device(\'cuda:1\' if torch.cuda.device_count() > 1 else \'cuda:0\')\n        detector, _ = load_model(self.base, self.norm, \'detector\', v1_device,\n                                 self.v1_root/\'checkpoints/detector_best.pt\')\n        refiner, _ = load_model(self.base, self.norm, \'refiner\', v1_device,\n                                self.v1_root/\'checkpoints/refiner_best.pt\')\n        github = load_trained(str(self.github_path), semantic_device)\n        source.DEVICE, source.TILE_BATCH, source._BLEND_GPU = str(semantic_device), 2, None\n        blended = source.UNet(pretrained=False)\n        blended.load_state_dict(self.blended_state.pop(\'model\'), strict=True)\n        blended = blended.to(semantic_device, memory_format=torch.channels_last)\n        self.models = dict(detector=freeze(detector), refiner=freeze(refiner), github=freeze(github),\n                           blended=freeze(blended), source=source, v1_device=v1_device,\n                           semantic_device=semantic_device)\n        print(\'Frozen A3 networks loaded once; A0 semantic/A4 extra refinement are outside these experiments.\', flush=True)\n        return self.models\n\n\nclass FrozenStore:\n    def __init__(self, assets, original_config, config, budget, source_hash):\n        self.assets, self.original, self.config, self.budget = assets, original_config, config, budget\n        self.pre = SolarPreprocessor(assets.output/\'preprocessed\', assets.base.PREPROCESS)\n        self.semantic = FrozenSemanticEnsemble(assets)\n        # G/Q postprocessing settings deliberately do not invalidate frozen predictions.\n        self.contract = {\'schema\': 1, \'original_notebook_sha256\': source_hash,\n                         \'assets\': assets.contract, \'opencv\': __import__(\'cv2\').__version__,\n                         \'precision\': \'original_GitHub_logits_f16_roundtrip;maps_f32;lossless_storage\',\n                         \'neural_recipe\': {\'detector\': 1024, \'refiner\': 384, \'github_tta\': 8,\n                                           \'center\': [88, 1960], \'feather\': [.55, 64]}}\n        self.signature = digest(self.contract)\n        self.root = Path(config.working_root)/\'solar_gq_frozen_cache\'/self.signature[:24]\n        self.root.mkdir(parents=True, exist_ok=True)\n        write_json(self.root/\'frozen_cache_manifest.json\', {\'signature\': self.signature, \'contract\': self.contract})\n        candidates = [Path(config.cache_source)] if config.cache_source else list(Path(config.input_root).rglob(\'frozen_cache_manifest.json\'))\n        self.external = []\n        for p in candidates:\n            if p.is_dir():\n                candidates2 = list(p.rglob(\'frozen_cache_manifest.json\'))\n            else:\n                candidates2 = [p]\n            for manifest in candidates2:\n                value = read_json(manifest)\n                if value.get(\'signature\') == self.signature and digest(value[\'contract\']) == self.signature:\n                    self.external.append(manifest.parent)\n        if config.cache_source and not self.external:\n            raise ValueError(\'Explicit shared cache is missing or incompatible; fix CACHE_SOURCE rather than silently recomputing.\')\n        self.stats = dict(frozen_cache_hits=0, neural_photos=0, v1_cache_hits=0, v1_forwards=0,\n                          semantic_forwards=0, bank_hits=0, historical_csv_read=False)\n        self.forward_ids = set()\n        self.v1_caches = self.find_v1_caches() if config.reuse_v1_cache else []\n\n    def find_v1_caches(self):\n        import cv2\n        c = self.assets.base\n        expected = {\'detector\': self.assets.v1_hashes[\'detector\'], \'refiner\': self.assets.v1_hashes[\'refiner\'],\n                    \'tile\': None, \'features\': c.INPUT_MODE, \'preprocess\': c.PREPROCESS,\n                    \'normalization\': self.assets.norm, \'size\': c.DETECTOR_IMAGE_SIZE,\n                    \'score_floor\': c.CACHE_SCORE_FLOOR, \'max_det\': c.MAX_DETECTIONS,\n                    \'context\': c.REFINER_CONTEXT, \'min_crop\': c.REFINER_MIN_CROP, \'ref_size\': c.REFINER_SIZE,\n                    \'amp\': c.AMP, \'torch\': str(torch.__version__), \'opencv\': cv2.__version__}\n        source_root = Path(__file__).parents[1]/\'solar_v1\'\n        critical = (\'inference.py\', \'preprocess.py\', \'data.py\', \'models.py\', \'config.py\', \'reference_models.py\', \'disk.py\')\n        wanted_source = {name: file_hash(source_root/name) for name in critical}\n        accepted, rejected = [], []\n        for p in Path(self.config.input_root).rglob(\'cache_manifest.json\'):\n            if \'prediction_cache\' not in p.parts:\n                continue\n            value = read_json(p)\n            differences = [k for k, v in expected.items() if value.get(k) != v]\n            differences += [\'source:\'+k for k, v in wanted_source.items() if value.get(\'source\', {}).get(k) != v]\n            if differences:\n                rejected.append({\'path\': str(p), \'differences\': differences})\n            else:\n                accepted.append(p.parent)\n        self.v1_cache_inventory={\'accepted\': list(map(str, accepted)), \'rejected\': rejected}\n        return accepted\n\n    def path(self, image):\n        return self.root/(image.stem+\'-\'+file_hash(image)+\'.npz\')\n\n    def get(self, image):\n        self.budget.check()\n        image = Path(image)\n        dest = self.path(image)\n        found = next((p for root in [self.root, *self.external] if (p := root/dest.name).is_file()), None)\n        data = self.pre.load(str(image))\n        if data[\'raw\'].shape != SHAPE:\n            raise ValueError(\'Expected a native 2048 x 2048 competition photo.\')\n        if found:\n            sidecar = found.with_suffix(\'.json\')\n            if not sidecar.is_file() or read_json(sidecar).get(\'sha256\') != file_hash(found):\n                raise ValueError(\'Frozen cache file is unsigned or corrupted.\')\n            z = read_pack(found)\n            if str(z[\'signature\']) != self.signature or str(z[\'image_hash\']) != file_hash(image):\n                raise ValueError(\'Frozen prediction belongs to another image/recipe.\')\n            pg = z[\'pg\']; pb = np.zeros(SHAPE, np.float32); pb[88:1960, 88:1960] = z[\'pb_center\']\n            maps = dict(pg=pg, pb=pb, pf=blend(pg, pb, (88, 1960), .55, 64), disk_github=z[\'disk_github\'])\n            pack = {k[5:]: v for k, v in z.items() if k.startswith(\'pack_\')}\n            self.stats[\'frozen_cache_hits\'] += 1\n        else:\n            started = time.monotonic()\n            if image.stem in self.forward_ids:\n                raise RuntimeError(\'A photo attempted a second frozen neural pass in this run.\')\n            pack = None\n            for root in self.v1_caches:\n                p = root/(image.stem+\'.npz\')\n                if p.is_file():\n                    value = read_pack(p)\n                    if str(value.get(\'pixel_hash\', \'\')) != str(data[\'pixel_hash\']):\n                        raise ValueError(\'V1 cache has changed photo pixels.\')\n                    if \'ref_probs\' in value and \'det_bounds\' in value and len(value[\'ref_probs\']) == len(value[\'scores\']):\n                        pack = value; self.stats[\'v1_cache_hits\'] += 1; break\n            if pack is None:\n                pack = predict_pack(data, self.assets)\n                self.stats[\'v1_forwards\'] += 1\n            maps = self.semantic.predict_maps(image)\n            self.stats[\'semantic_forwards\'] += 1\n            self.stats[\'neural_photos\'] += 1\n            self.forward_ids.add(image.stem)\n            used = sum(p.stat().st_size for p in self.root.glob(\'*.npz\'))\n            if used > self.config.max_cache_gib*2**30:\n                raise RuntimeError(\'Persistent frozen-cache budget exhausted; all remaining stages are incomplete.\')\n            save_npz(dest, {\'pg\': maps[\'pg\'], \'pb_center\': maps[\'pb\'][88:1960, 88:1960],\n                            \'disk_github\': maps[\'disk_github\'], \'signature\': self.signature,\n                            \'image_hash\': file_hash(image), **{\'pack_\'+k: v for k, v in pack.items()}})\n            write_json(dest.with_suffix(\'.json\'), {\'sha256\': file_hash(dest), \'image_hash\': file_hash(image),\n                       \'signature\': self.signature, \'seconds\': time.monotonic()-started})\n            found = dest\n        validate_maps(maps)\n        base, families, disk_v1 = instances_and_families(pack, data, self.assets, .35)\n        github_disk = maps[\'disk_github\'].astype(bool)\n        agreement = np.count_nonzero(disk_v1 & github_disk)/max(np.count_nonzero(disk_v1 | github_disk), 1)\n        if agreement < .8:\n            raise ValueError(\'V1/GitHub solar disks disagree.\')\n        disk = disk_v1 & github_disk\n        bank_path = next((p for p in (dest.with_suffix(\'.a3.json.gz\'),found.with_suffix(\'.a3.json.gz\')) if p.is_file()),\n                         dest.with_suffix(\'.a3.json.gz\'))\n        if bank_path.is_file():\n            value = read_gz(bank_path)\n            if value[\'signature\'] != self.signature or value[\'image_hash\'] != file_hash(image):\n                raise ValueError(\'A3 bank provenance mismatch.\')\n            bank = load_bank(value[\'bank\']); self.stats[\'bank_hits\'] += 1\n        else:\n            cc = classify(generate_cc(maps[\'pf\'], disk, self.original), families, self.original)\n            bank = assemble(from_families(families), cc, data, maps, maps[\'pf\'], True)\n            # External inputs are read-only; derived banks go to this run\'s cache.\n            bank_path = dest.with_suffix(\'.a3.json.gz\')\n            write_gz(bank_path, {\'signature\': self.signature, \'image_hash\': file_hash(image), \'bank\': bank_dict(bank)})\n        return dict(image=image, data=data, maps=maps, pack=pack, families=families, disk=disk,\n                    bank=bank, image_hash=file_hash(image), frozen_signature=self.signature)\n', 'solar_gq/io.py': "import csv\nimport gzip\nimport hashlib\nimport json\nimport os\nimport shutil\nimport time\nfrom dataclasses import asdict\nfrom pathlib import Path\n\nimport numpy as np\nimport psutil\n\nfrom solar_v15.common import convert, digest, file_hash, normalize_rles, write_json\nfrom solar_v15.candidate_bank import HybridBank, HybridCandidate\n\n\ndef write_gz(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + '.tmp')\n    with gzip.open(tmp, 'wt', encoding='utf-8') as stream:\n        json.dump(value, stream, default=convert, allow_nan=False)\n    os.replace(tmp, path)\n\n\ndef read_gz(path):\n    with gzip.open(path, 'rt', encoding='utf-8') as stream:\n        return json.load(stream)\n\n\ndef bank_dict(bank):\n    return {'candidates': [asdict(c) for c in bank.candidates], 'features': bank.features,\n            'conflicts': bank.conflicts, 'mask_hash': mask_hash(bank)}\n\n\ndef load_bank(value):\n    candidates = [HybridCandidate(**v) for v in normalize_rles(value['candidates'])]\n    matrix = np.asarray(value['features'], np.float32)\n    if matrix.ndim != 2 or matrix.shape[0] != len(candidates) or not np.isfinite(matrix).all():\n        raise ValueError('Invalid persisted feature bank.')\n    bank = HybridBank(candidates, matrix, value['conflicts'])\n    if mask_hash(bank) != value['mask_hash']:\n        raise ValueError('Candidate cache RLE/identity signature mismatch.')\n    return bank\n\n\ndef mask_hash(bank):\n    return digest([{'id': c.candidate_id, 'rle': c.rle, 'family': c.family_id,\n                    'matched': c.matched_detector_ids} for c in bank.candidates])\n\n\ndef rle_hash(rle):\n    return digest(rle)\n\n\ndef write_csv(path, rows, fields=None):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    if fields is None:\n        fields = list(dict.fromkeys(k for row in rows for k in row))\n    if not fields:\n        fields = ['status']\n    tmp = path.with_name(path.name + '.tmp')\n    with tmp.open('w', encoding='utf-8', newline='') as stream:\n        writer = csv.DictWriter(stream, fieldnames=fields, extrasaction='raise')\n        writer.writeheader()\n        for row in rows:\n            writer.writerow({k: json.dumps(v, default=convert, allow_nan=False) if isinstance(v, (list, dict, tuple))\n                             else v for k, v in row.items()})\n    os.replace(tmp, path)\n\n\nclass Budget:\n    def __init__(self, config, out):\n        self.config, self.out = config, Path(out)\n        self.started = time.monotonic()\n        self.peak_rss = 0\n\n    def check(self):\n        rss = psutil.Process().memory_info().rss\n        self.peak_rss = max(self.peak_rss, rss)\n        if time.monotonic() - self.started > self.config.max_hours * 3600:\n            raise TimeoutError('G/Q runtime budget reached. Stages remain explicitly incomplete; resume signed output.')\n        if shutil.disk_usage(self.out).free < self.config.min_free_gib * 2**30:\n            raise RuntimeError('Insufficient disk capacity. No experiment stage has been silently removed.')\n\n    def report(self):\n        self.check()\n        return {'seconds': time.monotonic() - self.started, 'rss_peak_sampled_bytes': self.peak_rss,\n                'rss_measurement': 'Sampled at photo/stage checkpoints; not a continuous OS peak.'}\n", 'solar_gq/main.py': '"""Run All: locked roles -> once-only frozen predictions -> all ablations -> durable reports."""\nimport copy\nimport gc\nimport json\nimport os\nimport time\nimport traceback\nimport importlib.metadata\nfrom collections import Counter, defaultdict\nfrom dataclasses import asdict\nfrom pathlib import Path\n\nimport cv2\nimport joblib\nimport numpy as np\nimport torch\n\nfrom solar_v1.pq import Totals, require_disjoint\nfrom solar_v1.submission import SubmissionWriter\nfrom solar_v12a.submission import verify_csv\nfrom solar_v15.config import Config as OriginalConfig\nfrom solar_v15.common import SHAPE, digest, file_hash, read_json, write_json\nfrom solar_v15.provenance import audit\nfrom solar_v15.validation import score\nfrom . import VERSION\nfrom .config import Config\nfrom .io import Budget, write_csv, write_gz, mask_hash\nfrom .frozen import FrozenAssets, FrozenStore\nfrom .prepare import Prepared, visualize\nfrom .quality import fit, predict, original_model, metrics\nfrom .selection import select\nfrom .diagnostics import diagnose, add_flow\n\n\ndef save_model(path,model,signature):\n    tmp=path.with_suffix(\'.tmp\');joblib.dump(model,tmp);os.replace(tmp,path)\n    write_json(path.with_suffix(\'.json\'),{\'experiment_signature\':signature,\'sha256\':file_hash(path),\'info\':model[\'info\']})\n\n\ndef load_model(path,signature):\n    manifest=read_json(path.with_suffix(\'.json\'))\n    if manifest[\'experiment_signature\']!=signature or manifest[\'sha256\']!=file_hash(path):\n        raise ValueError(\'Fitted model resume checksum/experiment mismatch.\')\n    return joblib.load(path)\n\n\ndef fit_all(prepared,roles,catalog,config,assets,signature,budget,manifest):\n    models={};directory=assets.output/\'models\';directory.mkdir(exist_ok=True)\n    baseline=config.stages[0]\n    for stage in config.stages:\n        if stage==\'G4\':\n            models[stage]=models[\'G3\'];manifest[\'stages\'][stage][\'fit\']=\'complete_shared_G3_model\';continue\n        budget.check();path=directory/(stage+\'.joblib\')\n        if config.resume and path.is_file():\n            models[stage]=load_model(path,signature)\n        else:\n            model=None\n            if stage==baseline:\n                model,rejected=original_model(config,assets,roles)\n                write_json(assets.output/\'original_quality_model_inventory.json\',{\'rejected\':rejected,\'loaded\':model is not None,\n                           \'historical_reproduction_cancelled\':True})\n            if model is None:\n                examples=[]\n                for row in roles[\'roles\'][\'fit\']:\n                    budget.check();banks,_=prepared.load(row[\'stem\'])\n                    examples.append({\'bank\':banks[stage],\'views\':catalog.views(row[\'stem\'])})\n                model=fit(examples,stage,config.seed)\n                if stage==baseline:model[\'info\'].update(source=\'refit_original_A3_recipe\',historical_reproduction_cancelled=True,\n                                                       fixed_policy={\'tau\':.24,\'orphan_floor\':.70})\n                del examples\n            save_model(path,model,signature);models[stage]=model\n        manifest[\'stages\'][stage][\'fit\']=\'complete\'\n        write_json(assets.output/\'run_manifest.json\',manifest)\n        print(stage,\'quality fitted/loaded:\',models[stage][\'info\'],flush=True)\n    return models\n\n\ndef calibrate(prepared,roles,catalog,models,config,budget,out,signature):\n    path=out/\'locked_policies.json\'\n    if config.resume and path.is_file():\n        value=read_json(path)\n        if value[\'experiment_signature\']!=signature:raise ValueError(\'Calibration resume signature mismatch.\')\n        return value[\'policies\'],value[\'trials\']\n    baseline=config.stages[0];policies={baseline:{\'tau\':.24,\'orphan_floor\':.70}};trials={}\n    for stage in config.stages[1:]:\n        if stage==\'G4\':\n            grid=[{**policies[\'G3\'],\'graph_weight\':weight} for weight in config.graph_weights]\n        else:grid=[{\'tau\':tau,\'orphan_floor\':floor} for tau in config.taus for floor in config.orphan_floors]\n        totals=[Totals() for _ in grid]\n        for row in roles[\'roles\'][\'calibration\']:\n            budget.check();banks,_=prepared.load(row[\'stem\']);bank=banks[stage]\n            values,_=predict(models[stage],bank);views=catalog.views(row[\'stem\'])\n            for i,policy in enumerate(grid):\n                instances,_,_=select(bank,values,policy,stage,config)\n                totals[i]+=score(instances,views,SHAPE)[0]\n        trial=[{**policy,**total.summary()} for policy,total in zip(grid,totals)]\n        best=max(range(len(grid)),key=lambda i:(round(trial[i][\'pq\'],12),grid[i][\'orphan_floor\'],grid[i][\'tau\'],\n                                               -grid[i].get(\'graph_weight\',0.)))\n        policies[stage]=grid[best];trials[stage]=trial\n        print(stage,\'calibration locked\',policies[stage],\'PQ\',trial[best][\'pq\'],flush=True)\n    value={\'experiment_signature\':signature,\'policies\':policies,\'trials\':trials,\n           \'evaluation_gt_read_during_calibration\':False,\'test_gt_read\':False}\n    write_json(path,value)\n    return policies,trials\n\n\ndef run_reports(prepared,roles,catalog,models,policies,config,budget,out,manifest):\n    totals=defaultdict(Totals);dates=defaultdict(Totals);flow={};quality=defaultdict(list)\n    per_photo=[];all_decisions=[];all_fate=[];rank=defaultdict(Counter);stability=[]\n    source_quality=defaultdict(list);subgroup_quality=defaultdict(list);special=[];failures=[];generation=[]\n    for role,rows in roles[\'roles\'].items():\n        for number,row in enumerate(rows,1):\n            budget.check();banks,audit_row=prepared.load(row[\'stem\']);views=catalog.views(row[\'stem\'])\n            stage_results={};prediction_cache={}\n            generation.append({\'stem\':row[\'stem\'],\'role\':role,\'audit\':{k:v for k,v in audit_row.items() if k!=\'stability\'}})\n            stability.extend({\'stem\':row[\'stem\'],\'role\':role,**v} for v in audit_row.get(\'stability\',[]))\n            for stage in config.stages:\n                bank=banks[stage]\n                cache_key=\'G3\' if stage==\'G4\' else stage\n                if cache_key not in prediction_cache:prediction_cache[cache_key]=predict(models[stage],bank)\n                values,heads=prediction_cache[cache_key]\n                instances,chosen,reasons=select(bank,values,policies[stage],stage,config)\n                summary,_,_=score(instances,views,SHAPE)\n                totals[role,stage]+=summary;dates[role,stage,row[\'group\']]+=summary\n                stage_results[stage]=summary.summary()\n                per_photo.append({\'role\':role,\'stage\':stage,\'stem\':row[\'stem\'],\'date\':row[\'date\'],\'group\':row[\'group\'],\n                                  \'candidates\':len(bank.candidates),\'selected_physical_instances\':len(chosen),**summary.summary()})\n                part,fate,decisions,qrow,ranking=diagnose(bank,values,heads,chosen,reasons,views,policies[stage],\n                                               row[\'stem\'],row[\'group\'],role,stage)\n                add_flow(flow.setdefault(role,{}).setdefault(stage,{}),part)\n                all_fate.extend(fate);all_decisions.extend(decisions);quality[role,stage].append(qrow)\n                rank[role,stage].update(ranking)\n                for source in {c.source for c in bank.candidates}:\n                    ids=np.array([i for i,c in enumerate(bank.candidates) if c.source==source],int)\n                    source_quality[role,stage,source].append({k:v[ids] for k,v in qrow.items()})\n                groups=defaultdict(list)\n                for i,c in enumerate(bank.candidates):\n                    area=\'under250\' if c.area<250 else \'250_999\' if c.area<1000 else \'1000_3999\' if c.area<4000 else \'4000plus\'\n                    groups[\'area:\'+area].append(i);groups[\'identity:\'+c.match_class].append(i)\n                    elongation=float(bank.features[i,13])\n                    groups[\'elongation:\'+(\'under3\' if elongation<3 else \'3_10\' if elongation<10 else \'10plus\')].append(i)\n                for name,indices in groups.items():\n                    ids=np.asarray(indices,int)\n                    subgroup_quality[role,stage,name].append({k:v[ids] for k,v in qrow.items()})\n                if role==\'evaluation\' and stage==config.stages[0]:\n                    special.extend(v for v in decisions if v[\'is_604_pixel_semantic_candidate\'])\n                # Preserve complete stage output masks and quality heads for every labelled photo.\n                write_gz(out/\'predictions\'/role/row[\'stem\']/(stage+\'.json.gz\'),\n                         {\'stage\':stage,\'policy\':policies[stage],\'mask_hash\':mask_hash(bank),\n                          \'selected_indices\':chosen,\'instances\':[asdict(v) for v in instances],\n                          \'quality_heads\':heads,\'reasons\':reasons,\'metrics\':summary.summary()})\n            if role==\'evaluation\':\n                delta=stage_results[config.stages[-1]][\'pq\']-stage_results[config.stages[0]][\'pq\']\n                failures.append({\'stem\':row[\'stem\'],\'pq_delta_last_vs_baseline\':delta})\n            if number%10==0 or number==len(rows):print(role,number,\'/\',len(rows),\'all stages evaluated\',flush=True)\n            # Durable photo-level progress even if a later photo fails.\n            write_json(out/\'report_progress.json\',{\'role\':role,\'last_photo\':row[\'stem\'],\'completed\':number,\'total\':len(rows)})\n    role_rows=[{\'role\':role,\'stage\':stage,\'rq\':value.summary()[\'dq\'],**value.summary()}\n               for (role,stage),value in totals.items()]\n    write_csv(out/\'role_metrics.csv\',role_rows)\n    write_csv(out/\'ablation.csv\',[r for r in role_rows if r[\'role\']==\'evaluation\'])\n    write_csv(out/\'per_photo_metrics.csv\',per_photo)\n    write_csv(out/\'per_date_metrics.csv\',[{\'role\':role,\'stage\':stage,\'group\':group,\'rq\':v.summary()[\'dq\'],**v.summary()}\n                                         for (role,stage,group),v in dates.items()])\n    write_csv(out/\'candidate_decisions.csv\',all_decisions)\n    write_csv(out/\'gt_fates.csv\',all_fate)\n    for stages in flow.values():\n        for portions in stages.values():\n            for portion in portions.values():\n                portion[\'recall\']=portion[\'covered_gt_view_instances\']/portion[\'gt_view_instances\'] if portion[\'gt_view_instances\'] else None\n    fate_lookup={(v[\'role\'],v[\'stem\'],v[\'annotation_view\'],v[\'gt_index\'],v[\'stage\']):v for v in all_fate}\n    unique_gains={}\n    for stage in config.stages[1:]:\n        rows=[]\n        for key,value in fate_lookup.items():\n            if key[-1]!=stage:continue\n            original=fate_lookup[(*key[:-1],config.stages[0])]\n            if value[\'covered_before_admission\'] and not original[\'covered_before_admission\']:\n                rows.append({k:value[k] for k in (\'role\',\'stem\',\'annotation_view\',\'gt_index\',\'first_loss_stage\')})\n        unique_gains[stage]={\'new_covered_gt_views_vs_baseline\':len(rows),\'gt_views\':rows}\n    write_json(out/\'candidate_flow.json\',flow)\n    write_json(out/\'oracle_report.json\',{\'flow\':flow,\'unique_candidate_gains\':unique_gains,\'unit\':\'GT annotation-view instances\',\n               \'oracle_is_legal_joint_upper_bound\':False,\'explanation\':\'Independent per-GT candidate coverage; competing masks may not coexist.\'})\n    write_json(out/\'quality_report.json\',{\'roles\':roles[\'role_counts\'],\'policies\':policies,\n               \'metrics\':{role+\'/\'+stage:metrics(values) for (role,stage),values in quality.items()},\n               \'by_source\':{role+\'/\'+stage+\'/\'+source:metrics(values) for (role,stage,source),values in source_quality.items()},\n               \'by_area_elongation_identity\':{role+\'/\'+stage+\'/\'+name:metrics(values)\n                         for (role,stage,name),values in subgroup_quality.items()},\n               \'family_pair_ranking\':{role+\'/\'+stage:dict(values) for (role,stage),values in rank.items()},\n               \'models\':{k:v[\'info\'] for k,v in models.items()},\'fit_metrics_are_in_sample\':True,\n               \'baseline_historical_reproduction_required\':False,\'provenance_limitations\':roles[\'provenance_limitations\']})\n    write_json(out/\'generation_diagnostics.json\',generation)\n    write_json(out/\'legacy_604_trace.json\',{\'matched_original_candidates\':special,\n               \'unique_candidate_located\':len(special)==1,\n               \'note\':\'Area 604 is an identification aid only. Zero/multiple matches are reported as ambiguous; never used for calibration.\'})\n    if config.line==\'Q\':write_csv(out/\'stability_diagnostics.csv\',stability)\n    for stage in config.stages:manifest[\'stages\'][stage][\'evaluation\']=\'complete\'\n    return role_rows,sorted(failures,key=lambda r:r[\'pq_delta_last_vs_baseline\'])\n\n\ndef generate_test(prepared,assets,models,policies,config,budget,out):\n    ids=[p.stem for p in assets.tests]\n    writers={stage:SubmissionWriter(out/(\'submission_\'+stage+(\'_v15_A3\' if stage.endswith(\'0\') else \'\')+\'.csv\'),ids,SHAPE,\n                                    out/(stage+\'_coverage.json\')) for stage in config.stages}\n    decisions=[];counts=[]\n    for number,path in enumerate(assets.tests,1):\n        budget.check();banks,audit_row=prepared.load(path.stem);prediction_cache={}\n        for stage in config.stages:\n            bank=banks[stage];key=\'G3\' if stage==\'G4\' else stage\n            if key not in prediction_cache:prediction_cache[key]=predict(models[stage],bank)\n            values,heads=prediction_cache[key]\n            instances,chosen,reasons=select(bank,values,policies[stage],stage,config)\n            require_disjoint([v.rle for v in instances],SHAPE);writers[stage].add(path.stem,instances)\n            trace={\'stem\':path.stem,\'stage\':stage,\'policy\':policies[stage],\'gt_access\':False,\n                   \'selected_indices\':chosen,\'instances\':[asdict(v) for v in instances],\'quality_heads\':heads,\'reasons\':reasons,\n                   \'candidate_bank_mask_hash\':mask_hash(bank)}\n            write_gz(out/\'test_decisions\'/path.stem/(stage+\'.json.gz\'),trace)\n            counts.append({\'stem\':path.stem,\'stage\':stage,\'candidate_count\':len(bank.candidates),\'selected\':len(chosen)})\n            decisions.extend({\'stem\':path.stem,\'role\':\'test\',\'stage\':stage,\'candidate_id\':c.candidate_id,\'source\':c.source,\n                              \'family_id\':c.family_id,\'parent_id\':c.parent_id,\'rle_sha256\':__import__(\'solar_gq.io\',fromlist=[\'rle_hash\']).rle_hash(c.rle),\n                              \'q_eff\':float(values[i]),\'p_hit\':float(heads[\'p_hit\'][i]) if \'p_hit\' in heads else None,\n                              \'q_pos\':float(heads[\'q_pos\'][i]) if \'q_pos\' in heads else None,\n                              \'reason\':reasons[c.candidate_id],\'selected\':i in chosen,\'gt_access\':False,\n                              \'conflicting_selected_ids\':[bank.candidates[j].candidate_id for j in chosen if bank.conflicts[i]&(1<<j)]}\n                              for i,c in enumerate(bank.candidates))\n        if number%20==0 or number==1:print(\'Test\',number,\'/180, all stage predictions persisted\',flush=True)\n    reports={}\n    for stage,writer in writers.items():\n        writer.finish();reports[stage]=verify_csv(writer.path,writer.coverage_path,ids,dict.fromkeys(ids,SHAPE))\n        reports[stage][\'file\']=str(writer.path)\n    write_csv(out/\'test_candidate_decisions.csv\',decisions)\n    write_csv(out/\'test_prediction_counts.csv\',counts)\n    write_json(out/\'submission_audit.json\',{\'stages\':reports,\'gt_access\':False,\'automatic_submission\':False,\n               \'default_submission_csv_written\':False,\'historical_instance_count_enforced\':False})\n    return reports\n\n\ndef failure_figures(failures,store,prepared,config,out,models,policies):\n    for number,row in enumerate(failures[:config.visualization_count]):\n        image=Path(row[\'path\'])\n        photo=store.get(image);banks,_=prepared.load(image.stem);panels=[]\n        for stage in (config.stages[0],config.stages[-1]):\n            bank=banks[stage];values,_=predict(models[stage],bank)\n            instances,_,_=select(bank,values,policies[stage],stage,config)\n            canvas=cv2.cvtColor(photo[\'data\'][\'raw\'],cv2.COLOR_GRAY2BGR)\n            from pycocotools import mask as cm\n            for i,value in enumerate(instances):\n                mask=cm.decode(value.rle).astype(bool);color=np.array([(31+i*71)%255,(147+i*43)%255,(220+i*29)%255])\n                canvas[mask]=(.55*canvas[mask]+.45*color).astype(np.uint8)\n            canvas=cv2.resize(canvas,(640,640));cv2.putText(canvas,stage+\' predictions; no GT overlay\',(8,24),\n                           cv2.FONT_HERSHEY_SIMPLEX,.5,(255,255,255),1);panels.append(canvas)\n        directory=out/\'failure_figures\';directory.mkdir(exist_ok=True)\n        if not cv2.imwrite(str(directory/f\'{number:02}_{image.stem}.png\'),np.hstack(panels)):raise IOError(\'Failure image write failed.\')\n    write_json(out/\'failure_figures_manifest.json\',{\'selection\':\'evaluation last-stage PQ delta ascending\',\n               \'contains_gt_overlay\':False,\'photos\':failures[:config.visualization_count]})\n\n\ndef run(raw_config,source_hash):\n    config=Config(**raw_config).validate();out=Path(config.working_root)/(\'solar_gq_\'+config.line.lower());out.mkdir(parents=True,exist_ok=True)\n    budget=Budget(config,out)\n    code={p.name:file_hash(p) for p in Path(__file__).parent.glob(\'*.py\')}\n    manifest={\'version\':VERSION,\'line\':config.line,\'status\':\'running\',\'historical_A3_reproduction_cancelled\':True,\n              \'stages\':{stage:{\'prepare\':\'pending\',\'fit\':\'pending\',\'calibration\':\'pending\',\'evaluation\':\'pending\',\'test\':\'pending\'}\n                        for stage in config.stages},\'config\':config.to_dict(),\'code_sha256\':code,\n              \'metrics_provenance\':\'Computed from mounted labelled roles; test has no PQ\',\n              \'test_gt_access\':False,\'automatic_submission\':False}\n    preserve_existing_manifest=False\n    original=OriginalConfig(input_root=config.input_root,working_root=config.working_root,\n        v1_source=config.v1_source,legacy_root=config.legacy_root,github_checkpoint=config.github_checkpoint,\n        blended_checkpoint=config.blended_checkpoint,annotation_path=config.annotation_path,test_image_dir=config.test_image_dir).validate()\n    torch.set_num_threads(2);cv2.setNumThreads(0);torch.manual_seed(config.seed);np.random.seed(config.seed)\n    torch.backends.cudnn.benchmark=False\n    try:\n        assets=FrozenAssets(original,out)\n        roles,catalog=audit(assets,original,budget.check)\n        store=FrozenStore(assets,original,config,budget,source_hash)\n        stable_config={k:v for k,v in config.to_dict().items() if k not in (\n            \'working_root\',\'input_root\',\'cache_source\',\'original_bundle_source\',\'max_hours\',\'max_cache_gib\',\'min_free_gib\',\'resume\')}\n        feature_runtime={name:importlib.metadata.version(name) for name in (\'scikit-image\',\'numpy\',\'scipy\',\'scikit-learn\')}\n        signature=digest({\'config\':stable_config,\'code\':code,\'frozen_signature\':store.signature,\n                          \'role_hash\':roles[\'role_hash\'],\'feature_runtime\':feature_runtime})\n        manifest.update(experiment_signature=signature,frozen_signature=store.signature,role_counts=roles[\'role_counts\'],\n                        feature_runtime=feature_runtime,\n                        evaluation_limitations=roles[\'provenance_limitations\'],roles_locked_before_fitting=True)\n        old=read_json(out/\'run_manifest.json\') if (out/\'run_manifest.json\').is_file() else None\n        if old and old.get(\'experiment_signature\') and old[\'experiment_signature\']!=signature:\n            preserve_existing_manifest=True\n            raise FileExistsError(\'Output belongs to another G/Q experiment signature. Preserve it and choose a new working_root.\')\n        if config.resume and old and old.get(\'status\')==\'complete\' and old.get(\'artifact_checksums\'):\n            for name,expected in old[\'artifact_checksums\'].items():\n                path=out/name\n                if not path.is_file() or file_hash(path)!=expected:\n                    raise ValueError(\'Completed-run artifact missing/corrupted: \'+name)\n            print(\'COMPLETE signed output already exists; all persisted artifact checksums verified.\',out,flush=True)\n            return\n        write_json(out/\'exposure_roles.json\',roles)\n        write_json(out/\'v1_cache_compatibility.json\',getattr(store,\'v1_cache_inventory\',{\'enabled\':config.reuse_v1_cache}))\n        write_json(out/\'run_manifest.json\',manifest)\n        prepared=Prepared(config,out,signature)\n        development=[Path(catalog.photos[r[\'stem\']][\'path\']) for rows in roles[\'roles\'].values() for r in rows]\n        paths=development+assets.tests\n        if len({p.stem for p in paths})!=len(paths):raise ValueError(\'Photo roles/test overlap.\')\n        for number,path in enumerate(paths,1):\n            budget.check();dest=prepared.path(path.stem)\n            if config.resume and dest.is_file():\n                _,cache_audit=prepared.load(path.stem)\n                if cache_audit[\'image_hash\']!=file_hash(path):raise ValueError(\'Prepared cache image changed.\')\n            else:\n                photo=store.get(path);photo[\'original_config\']=original\n                row=prepared.build(photo)\n                if number<=config.visualization_count:visualize(photo,row,out/\'candidate_figures\'/(path.stem+\'.png\'))\n                del photo\n            if number%10==0 or number==len(paths):\n                print(\'Shared predictions/candidates\',number,\'/\',len(paths),flush=True)\n                write_json(out/\'prepare_progress.json\',{\'completed\':number,\'total\':len(paths),\'stats\':store.stats,\'resources\':budget.report()})\n                gc.collect()\n        for stage in config.stages:manifest[\'stages\'][stage][\'prepare\']=\'complete\'\n        write_json(out/\'run_manifest.json\',manifest)\n        models=fit_all(prepared,roles,catalog,config,assets,signature,budget,manifest)\n        policies,trials=calibrate(prepared,roles,catalog,models,config,budget,out,signature)\n        for stage in config.stages:manifest[\'stages\'][stage][\'calibration\']=\'complete\'\n        write_json(out/\'run_manifest.json\',manifest)\n        results,failures=run_reports(prepared,roles,catalog,models,policies,config,budget,out,manifest)\n        reports=generate_test(prepared,assets,models,policies,config,budget,out)\n        for row in failures:row[\'path\']=catalog.photos[row[\'stem\']][\'path\']\n        failure_figures(failures,store,prepared,config,out,models,policies)\n        for stage in config.stages:manifest[\'stages\'][stage][\'test\']=\'complete\'\n        # Best branch is chosen using calibration only, never evaluation or LB.\n        calibration={r[\'stage\']:r[\'pq\'] for r in results if r[\'role\']==\'calibration\'}\n        best=max(config.stages,key=lambda s:(round(calibration[s],12),-config.stages.index(s)))\n        manifest.update(status=\'complete\',frozen_stats=store.stats,resources=budget.report(),\n                        best_stage_by_calibration=best,submissions=reports,\n                        historical_A3_reproduction_status=\'cancelled_by_user; no CSV equality or 1132-count acceptance\',\n                        cuda_peak_allocated_bytes=[torch.cuda.max_memory_allocated(i) for i in range(torch.cuda.device_count())])\n        write_json(out/\'frozen_cache_manifest.json\',{\'signature\':store.signature,\'contract\':store.contract,\n                   \'cache_root\':str(store.root),\'note\':\'Attach solar_gq_frozen_cache output directory to share predictions with the other notebook.\'})\n        write_json(out/\'candidate_schema_version.json\',{\'version\':1,\'original_feature_count\':32,\'experiment_signature\':signature,\n                   \'full_candidates_rle_features_family_provenance\':\'prepared/\',\'stages\':config.stages})\n        text=[\'# Actual \'+config.line+\' experiment results\',\'\',\n              \'Historical A3 reproduction was cancelled. G0/Q0 uses the A3 recipe; original CSV equality is not asserted.\',\n              \'Evaluation: \'+str(roles[\'role_counts\'][\'evaluation\'])+\' photos. \'+roles[\'provenance_limitations\'],\'\',\n              \'| Stage | PQ | SQ | RQ | TP | FP | FN |\',\'|---|---:|---:|---:|---:|---:|---:|\']\n        for r in results:\n            if r[\'role\']==\'evaluation\':\n                text.append(f"| {r[\'stage\']} | {r[\'pq\']:.6f} | {r[\'sq\']:.6f} | {r[\'dq\']:.6f} | {r[\'tp\']} | {r[\'fp\']} | {r[\'fn\']} |")\n        text+=[\'\',\'Best by calibration: \'+best+\'. No Kaggle submission was made.\',\n               \'Resources: \'+json.dumps(manifest[\'resources\']),\n               \'Fit metrics are in sample; test has no GT and no reported PQ.\']\n        (out/\'experiment_report.md\').write_text(\'\\n\'.join(text),encoding=\'utf-8\')\n        manifest[\'artifact_checksums\']={p.relative_to(out).as_posix():file_hash(p) for p in out.rglob(\'*\')\n            if p.is_file() and p.name!=\'run_manifest.json\' and \'preprocessed\' not in p.relative_to(out).parts\n            and not p.name.endswith(\'.tmp\')}\n        write_json(out/\'run_manifest.json\',manifest)\n        print(\'\\n\'.join(text),flush=True)\n        print(\'COMPLETE. Stage CSVs and diagnostics:\',out,flush=True)\n    except Exception as error:\n        manifest.update(status=\'failed_incomplete\',error_type=type(error).__name__,error=str(error),\n                        elapsed_seconds=time.monotonic()-budget.started,rss_peak_sampled_bytes=budget.peak_rss)\n        failure_manifest=out/(\'rejected_new_run_\'+str(time.time_ns())+\'.json\') if preserve_existing_manifest else out/\'run_manifest.json\'\n        write_json(failure_manifest,manifest)\n        traceback_path=failure_manifest.with_suffix(\'.traceback.txt\') if preserve_existing_manifest else out/\'failure_traceback.txt\'\n        traceback_path.write_text(traceback.format_exc(),encoding=\'utf-8\')\n        raise\n', 'solar_gq/prepare.py': 'import copy\nimport time\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nfrom pycocotools import mask as cm\n\nfrom solar_v15.candidate_bank import FEATURES, HybridBank, assemble, classify\nfrom solar_v15.common import digest, file_hash, write_json\nfrom solar_v1.pq import pairwise_iou\nfrom .io import bank_dict, load_bank, mask_hash, read_gz, write_gz\nfrom .structure import GraphFeatures, GRAPH_NAMES, max_tree_candidates\nfrom .stability import StabilityFeatures, feature_names\nfrom .repair import repaired_candidates\n\n\ndef assemble_cached(original_candidates,new_candidates,photo,cache):\n    """Original bank semantics, but expensive per-mask features are evaluated only once.\n\n    Family variant_count depends on the current pool and is recomputed separately.\n    """\n    unique={}\n    for candidate in copy.deepcopy(original_candidates+new_candidates):\n        candidate.info.setdefault(\'sources\',[candidate.source])\n        if candidate.match_class==\'matched_one\':\n            candidate.family_id=candidate.matched_detector_ids[0]\n        candidate.info.setdefault(\'family_aliases\',[candidate.family_id])\n        key=candidate.rle[\'counts\']\n        if key in unique:\n            previous=unique[key]\n            previous.info[\'sources\']=sorted(set(previous.info[\'sources\']+candidate.info[\'sources\']))\n            previous.info[\'family_aliases\']=sorted(set(previous.info[\'family_aliases\']+candidate.info[\'family_aliases\']))\n            previous.info.setdefault(\'candidate_aliases\',[]).append(candidate.candidate_id)\n        else:unique[key]=candidate\n    candidates=list(unique.values());groups=[set(c.info[\'family_aliases\']) for c in candidates]\n    overlaps=pairwise_iou([c.rle for c in candidates],[c.rle for c in candidates])\n    rows=[];conflicts=[]\n    for i,candidate in enumerate(candidates):\n        key=digest(candidate.rle)\n        if key not in cache:\n            one=assemble([candidate],[],photo[\'data\'],photo[\'maps\'],photo[\'maps\'][\'pf\'],True)\n            candidate=one.candidates[0];candidates[i]=candidate\n            cache[key]=(one.features[0].copy(),{k:candidate.info[k] for k in\n                        (\'primary_mean\',\'primary_q20\',\'contrast\',\'pb_valid_fraction\')})\n        vector,info=cache[key];vector=vector.copy();candidate.info.update(info)\n        from solar_v15.candidate_bank import SOURCES,CLASSES\n        vector[:len(SOURCES)]=[float(s in candidate.info[\'sources\']) for s in SOURCES]\n        vector[len(SOURCES):len(SOURCES)+len(CLASSES)]=[float(candidate.match_class==s) for s in CLASSES]\n        vector[29]=sum(bool(groups[i]&group) for group in groups)-1\n        rows.append(vector)\n        conflicts.append(sum(1<<j for j in range(len(candidates))\n                             if j!=i and (overlaps[i,j]>0 or groups[i]&groups[j])))\n    return HybridBank(candidates,np.asarray(rows,np.float32).reshape(len(candidates),len(FEATURES)),conflicts)\n\n\nclass Prepared:\n    def __init__(self,config,out,signature):\n        self.config,self.out,self.signature=config,Path(out),signature\n        self.root=self.out/\'prepared\'/signature[:24];self.root.mkdir(parents=True,exist_ok=True)\n\n    def path(self,stem):return self.root/(stem+\'.json.gz\')\n\n    def load(self,stem):\n        value=read_gz(self.path(stem))\n        if value[\'signature\']!=self.signature:raise ValueError(\'Experiment candidate cache signature mismatch.\')\n        banks={k:load_bank(v) for k,v in value[\'banks\'].items()}\n        if self.config.line==\'G\':banks[\'G4\']=banks[\'G3\']\n        else:banks[\'Q2\']=banks[\'Q1\']\n        return banks,value[\'audit\']\n\n    def build(self,photo):\n        started=time.monotonic();b0=copy.deepcopy(photo[\'bank\']);audit={\n            \'image_hash\':photo[\'image_hash\'],\'frozen_signature\':photo[\'frozen_signature\'],\n            \'baseline_features\':len(FEATURES),\'gt_access\':False,\'historical_csv_comparison\':False}\n        if b0.features.shape[1]!=32:raise ValueError(\'The source A3 recipe must contain the actual 32 original features.\')\n        if self.config.line==\'Q\':\n            stability=StabilityFeatures(photo,self.config);vectors=[];details=[]\n            for c in b0.candidates:\n                vector,rows=stability.get(c);vectors.append(vector)\n                details.extend({\'candidate_id\':c.candidate_id,**row} for row in rows)\n            extra=np.asarray(vectors,np.float32).reshape(len(b0.candidates),len(feature_names(self.config)))\n            b1=HybridBank(copy.deepcopy(b0.candidates),np.concatenate([b0.features,extra],axis=1),b0.conflicts.copy())\n            banks={\'Q0\':b0,\'Q1\':b1};names={\'Q0\':list(FEATURES),\'Q1\':[*FEATURES,*feature_names(self.config)]}\n            if mask_hash(b0)!=mask_hash(b1):raise AssertionError(\'Q1 changed the A3 candidate masks/identity.\')\n            audit.update(stability=details,candidate_pool_frozen=True)\n        else:\n            feature_cache={digest(c.rle):(b0.features[i].copy(),{k:c.info[k] for k in\n                          (\'primary_mean\',\'primary_q20\',\'contrast\',\'pb_valid_fraction\')})\n                           for i,c in enumerate(b0.candidates)}\n            additional,tree_audit=max_tree_candidates(photo,self.config)\n            classify(additional,photo[\'families\'],photo[\'original_config\'])\n            overmerged=[c for c in b0.candidates if c.match_class==\'overmerge\']\n            overlaps=pairwise_iou([c.rle for c in overmerged],[c.rle for c in additional])\n            split_evidence=[]\n            for i,parent in enumerate(overmerged):\n                children=[]\n                for j,child in enumerate(additional):\n                    intersection=overlaps[i,j]*(parent.area+child.area)/(1+overlaps[i,j])\n                    if intersection>=.95*child.area and child.match_class==\'matched_one\':\n                        children.append(child)\n                independent_pairs=[]\n                if len(children)>=2:\n                    matrix=pairwise_iou([c.rle for c in children],[c.rle for c in children])\n                    independent_pairs=[(a.candidate_id,b.candidate_id) for j,a in enumerate(children)\n                        for k,b in enumerate(children[j+1:],j+1)\n                        if a.matched_detector_ids!=b.matched_detector_ids and matrix[j,k]==0]\n                split_evidence.append({\'parent_id\':parent.candidate_id,\'child_ids\':[c.candidate_id for c in children],\n                    \'distinct_family_disjoint_pairs\':independent_pairs,\'candidate_decomposition_found\':bool(independent_pairs)})\n            tree_audit.update(overmerge_decompositions=split_evidence,\n                              overmerge_decomposable_count=sum(v[\'candidate_decomposition_found\'] for v in split_evidence),\n                              decomposition_is_not_a_legal_TP_claim=True)\n            b1=assemble_cached(b0.candidates,additional,photo,feature_cache)\n            graphs=GraphFeatures(photo,self.config)\n            extras=np.asarray([graphs.get(c) for c in b1.candidates],np.float32).reshape(len(b1.candidates),len(GRAPH_NAMES))\n            b2=HybridBank(copy.deepcopy(b1.candidates),np.concatenate([b1.features,extras],axis=1),b1.conflicts.copy())\n            repaired,repair_audit=repaired_candidates(b2,photo,graphs,self.config)\n            classify(repaired,photo[\'families\'],photo[\'original_config\'])\n            b3=assemble_cached(b2.candidates,repaired,photo,feature_cache)\n            extras3=np.asarray([graphs.get(c) for c in b3.candidates],np.float32).reshape(len(b3.candidates),len(GRAPH_NAMES))\n            b3.features=np.concatenate([b3.features,extras3],axis=1)\n            banks={\'G0\':b0,\'G1\':b1,\'G2\':b2,\'G3\':b3}\n            names={k:list(FEATURES)+(list(GRAPH_NAMES) if k in (\'G2\',\'G3\') else []) for k in banks}\n            if mask_hash(b1)!=mask_hash(b2):raise AssertionError(\'G2 changed the G1 candidate masks/identity.\')\n            original_masks={digest(c.rle) for c in b0.candidates}\n            if not original_masks<={digest(c.rle) for c in b1.candidates}:raise AssertionError(\'G1 removed an original A3 mask.\')\n            if not {digest(c.rle) for c in b2.candidates}<={digest(c.rle) for c in b3.candidates}:\n                raise AssertionError(\'G3 removed a preceding candidate.\')\n            audit.update(max_tree=tree_audit,repair=repair_audit,graph_masks_computed=len(graphs.cache),\n                         original_feature_masks_computed_once=len(feature_cache),\n                         g1_new_masks=len(b1.candidates)-len(b0.candidates),g3_new_masks=len(b3.candidates)-len(b2.candidates))\n        audit.update(seconds=time.monotonic()-started,feature_names=names,\n                     candidate_counts={k:len(b.candidates) for k,b in banks.items()},\n                     mask_hashes={k:mask_hash(b) for k,b in banks.items()})\n        write_gz(self.path(photo[\'image\'].stem),{\'signature\':self.signature,\'image_hash\':photo[\'image_hash\'],\n                  \'banks\':{k:bank_dict(v) for k,v in banks.items()},\'audit\':audit})\n        return audit\n\n\ndef visualize(photo,audit,path):\n    def overlay(bank):\n        canvas=cv2.cvtColor(photo[\'data\'][\'raw\'],cv2.COLOR_GRAY2BGR)\n        for i,c in enumerate(bank.candidates):\n            mask=cm.decode(c.rle).astype(bool)\n            color=np.array([(71*i+30)%255,(43*i+130)%255,(29*i+210)%255])\n            canvas[mask]=(.6*canvas[mask]+.4*color).astype(np.uint8)\n        return canvas\n    original=overlay(photo[\'bank\'])\n    probability=cv2.applyColorMap(np.uint8(photo[\'maps\'][\'pf\']*255),cv2.COLORMAP_VIRIDIS)\n    panels=[]\n    for name,array in [(\'Original A3 candidate pool (no GT)\',original),(\'Frozen Pf (no GT)\',probability)]:\n        array=cv2.resize(array,(640,640));cv2.putText(array,name,(8,24),cv2.FONT_HERSHEY_SIMPLEX,.5,(255,255,255),1)\n        panels.append(array)\n    Path(path).parent.mkdir(parents=True,exist_ok=True)\n    if not cv2.imwrite(str(path),np.hstack(panels)):raise IOError(\'Failed to save GT-free diagnostic image.\')\n', 'solar_gq/quality.py': "import warnings\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nfrom scipy.stats import spearmanr\nfrom sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor\nfrom threadpoolctl import threadpool_limits\n\nfrom solar_v15.candidate_bank import FEATURES\nfrom solar_v15.common import file_hash, read_json\nfrom solar_v15.quality import targets\n\n\ndef regressor(seed):\n    return HistGradientBoostingRegressor(max_iter=120,learning_rate=.055,max_leaf_nodes=7,\n            min_samples_leaf=24,l2_regularization=2.,early_stopping=False,random_state=seed)\n\n\ndef fit(examples,stage,seed):\n    usable=[e for e in examples if len(e['bank'].candidates)]\n    if not usable:raise ValueError(f'{stage}: no candidates in the locked fitting role.')\n    x=np.concatenate([e['bank'].features for e in usable])\n    labels=[targets(e['bank'],e['views']) for e in usable]\n    y=np.concatenate([v[0] for v in labels]);hit=np.concatenate([v[1] for v in labels])\n    weights=np.concatenate([np.full(len(e['bank'].candidates),len(e['views'])/len(e['bank'].candidates)) for e in usable])\n    weights=weights/weights.mean()\n    if len(y)<100 or np.count_nonzero(y>0)<10 or np.count_nonzero(y==0)<10:\n        raise ValueError(f'{stage}: insufficient positive/negative candidate supervision; no substitute stage is permitted.')\n    info={'stage':stage,'photos':len(usable),'candidates':len(y),'positive_candidates':int((hit>0).sum()),\n          'features':x.shape[1],'target':'mean_view(best_iou * (best_iou > .5))',\n          'weighting':'each candidate has n_views / n_candidates_in_photo; roles are photo/date grouped'}\n    with threadpool_limits(limits=2):\n        if stage!='Q2':\n            model=regressor(seed).fit(x,y,sample_weight=weights)\n            return {'kind':'effective_regression','regressor':model,'info':info}\n        # Weighted positive/negative pseudo-observations exactly represent each candidate's view counts.\n        positive=hit>0;negative=hit<1\n        xx=np.concatenate([x[positive],x[negative]])\n        yy=np.concatenate([np.ones(positive.sum(),int),np.zeros(negative.sum(),int)])\n        ww=np.concatenate([weights[positive]*hit[positive],weights[negative]*(1-hit[negative])])\n        classifier=HistGradientBoostingClassifier(max_iter=120,learning_rate=.055,max_leaf_nodes=7,\n            min_samples_leaf=24,l2_regularization=2.,early_stopping=False,random_state=seed)\n        classifier.fit(xx,yy,sample_weight=ww)\n        conditional=np.divide(y,hit,out=np.zeros_like(y),where=hit>0)\n        pos_model=regressor(seed).fit(x[positive],conditional[positive],sample_weight=weights[positive]*hit[positive])\n        info.update(target_hit='fraction of annotation views with best IoU > .5',\n                    target_positive='mean best IoU conditional on a hit',\n                    positive_weight='base_photo_candidate_weight * hit_view_fraction',\n                    hit_pseudo_observations=len(yy))\n        return {'kind':'hurdle','hit':classifier,'positive':pos_model,'info':info}\n\n\ndef predict(model,bank):\n    if not bank.candidates:\n        z=np.empty(0);return z,{'q_eff':z,'p_hit':z,'q_pos':z}\n    with threadpool_limits(limits=2):\n        if model['kind']=='hurdle':\n            hit=model['hit'].predict_proba(bank.features)[:,list(model['hit'].classes_).index(1)]\n            pos=model['positive'].predict(bank.features).clip(.5,1.)\n            values=hit*pos\n            heads={'q_eff':values,'p_hit':hit,'q_pos':pos}\n        else:\n            values=model['regressor'].predict(bank.features).clip(0,1)\n            heads={'q_eff':values}\n    if not np.isfinite(values).all():raise ValueError('Nonfinite candidate quality prediction.')\n    return values,heads\n\n\ndef original_model(config,assets,roles):\n    paths=[Path(config.original_bundle_source)/'v15_bundle.json'] if config.original_bundle_source else list(\n        Path(config.input_root).rglob('v15_bundle.json'))\n    candidates=[];rejected=[]\n    for path in paths:\n        try:\n            bundle=read_json(path);model_path=path.parent/'quality_models.joblib'\n            if bundle['contract']['assets']['v1_weights']!=assets.v1_hashes:raise ValueError('V1 weights differ')\n            if bundle['contract']['assets']['files']!=assets.contract['files']:raise ValueError('Frozen/source assets differ')\n            if bundle['role_hash']!=roles['role_hash']:raise ValueError('Quality photo/date roles differ')\n            if bundle['contract']['feature_names']!=list(FEATURES):raise ValueError('Original feature schema differs')\n            if file_hash(model_path)!=bundle['models_sha256']:raise ValueError('Model checksum differs')\n            with warnings.catch_warnings(record=True) as captured:\n                warnings.simplefilter('always');models=joblib.load(model_path)\n            if captured:raise ValueError('Original model serialization warning: '+str(captured[0].message))\n            model=models['A3']\n            if model.n_features_in_!=len(FEATURES):raise ValueError('A3 feature dimension differs')\n            candidates.append((path,model))\n        except Exception as error:\n            rejected.append({'path':str(path),'reason':str(error)})\n    if len(candidates)>1:\n        hashes={file_hash(p.parent/'quality_models.joblib') for p,_ in candidates}\n        if len(hashes)>1:raise ValueError('Several different compatible A3 models exist; specify original_bundle_source.')\n    if candidates:\n        path,model=candidates[0]\n        return {'kind':'effective_regression','regressor':model,\n                'info':{'source':'original_saved_A3_model','path':str(path),'features':len(FEATURES),\n                        'historical_csv_comparison':False,'historical_row_count_enforced':False}},rejected\n    return None,rejected\n\n\ndef metrics(rows):\n    if not rows:return {'n':0}\n    y=np.concatenate([r['target'] for r in rows]);q=np.concatenate([r['quality'] for r in rows])\n    hit=np.concatenate([r['hit'] for r in rows])\n    if not len(y):return {'n':0}\n    result={'n':len(y),'mae_effective_iou':float(np.abs(y-q).mean()),\n            'spearman':float(spearmanr(y,q).statistic) if np.ptp(y)>0 and np.ptp(q)>0 else None,\n            'bins':[]}\n    for low in np.arange(0,1,.1):\n        keep=(q>=low)&(q<(low+.1) if low<.89 else q<=1)\n        if keep.any():result['bins'].append({'lo':float(low),'n':int(keep.sum()),'predicted':float(q[keep].mean()),\n                            'actual_effective_iou':float(y[keep].mean()),'actual_hit_fraction':float(hit[keep].mean())})\n    if all('p_hit' in r for r in rows):\n        p=np.clip(np.concatenate([r['p_hit'] for r in rows]),1e-7,1-1e-7)\n        pos=np.concatenate([r['q_pos'] for r in rows]);positive=hit>0\n        result.update(brier_soft_view_target=float(np.mean((p-hit)**2)),\n                      view_weighted_log_loss=float(np.mean(-hit*np.log(p)-(1-hit)*np.log(1-p))),\n                      conditional_iou_mae=float(np.average(np.abs(pos[positive]-y[positive]/hit[positive]),\n                                              weights=hit[positive])) if positive.any() else None)\n    return result\n", 'solar_gq/repair.py': '"""GT-free graph repair proposals; original masks are never overwritten."""\nimport copy\nimport time\n\nimport cv2\nimport numpy as np\nfrom pycocotools import mask as cm\nfrom scipy import ndimage as ndi\nfrom scipy.sparse.csgraph import dijkstra\nfrom skimage.graph import route_through_array\nfrom skimage.morphology import skeletonize\n\nfrom solar_v1.postprocess import encode\nfrom solar_v15.candidate_bank import HybridCandidate, populate_geometry\nfrom .io import rle_hash\nfrom .structure import pixel_graph\n\n\ndef family_masks(photo):\n    return {key: np.logical_or.reduce([cm.decode(v[\'rle\']).astype(bool) for v in family[\'variants\']])\n            for key,family in photo[\'families\'].items() if family[\'variants\']}\n\n\ndef make_candidate(parent,mask,suffix,info):\n    name=parent.candidate_id+\':graph:\'+suffix\n    candidate=HybridCandidate(name,encode(mask),\'graph_repaired\',name,\'orphan\',parent.threshold,0,\n                             parent_id=parent.candidate_id,\n                             info={\'sources\':[\'semantic_cc\',\'graph_repaired\'],\'original_cc_id\':name,\n                                   \'admission\':\'unclassified\',\'graph_repair\':info,\n                                   \'provenance\':{\'uses_gt\':False,\'method\':\'bounded_existing_skeleton_probability_graph\'},\n                                   \'support\':info.get(\'path_support\',0.)})\n    return populate_geometry(candidate)\n\n\ndef split_by_skeleton(parent,photo,masks,config):\n    x,y,w,h=parent.bbox\n    original=cm.decode(parent.rle).astype(bool)\n    mask=original[y:y+h,x:x+w]\n    if w*h>config.graph_roi_pixels:\n        return [],[{\'reason\':\'split_roi_budget\'}]\n    skeleton=skeletonize(mask)\n    if not skeleton.any() or skeleton.sum()>config.graph_max_nodes:\n        return [],[{\'reason\':\'split_skeleton_budget\'}]\n    probability=photo[\'maps\'][\'pf\'][y:y+h,x:x+w]\n    graph,yy,xx=pixel_graph(skeleton,probability)\n    keys=[k for k in parent.matched_detector_ids if k in masks]\n    if len(keys)<2 or len(keys)>config.repair_split_families:\n        return [],[{\'reason\':\'split_family_budget_or_missing\'}]\n    distances=[];seed_nodes=[]\n    for key in keys:\n        support=masks[key][y:y+h,x:x+w][yy,xx]\n        indices=np.flatnonzero(support)\n        if not len(indices):\n            return [],[{\'reason\':\'split_family_has_no_skeleton_anchor\',\'family\':key}]\n        # Distance to any supported skeleton node, not only the family centroid.\n        distance=dijkstra(graph,directed=False,indices=indices,min_only=True)\n        distances.append(distance);seed_nodes.append(indices.tolist())\n    stack=np.stack(distances)\n    reachable=np.isfinite(stack).any(0)\n    node_labels=np.argmin(stack,axis=0)+1;node_labels[~reachable]=0\n    skeleton_labels=np.zeros(mask.shape,np.int32);skeleton_labels[yy,xx]=node_labels\n    _,nearest=ndi.distance_transform_edt(~skeleton,return_indices=True)\n    pixel_labels=skeleton_labels[nearest[0],nearest[1]]\n    values=[];logs=[]\n    for label,key in enumerate(keys,1):\n        child=mask&(pixel_labels==label)\n        blocked=np.zeros(mask.shape,bool)\n        for other,other_mask in masks.items():\n            if other!=key and photo[\'families\'][other][\'raw_score\']>=.7:\n                blocked|=other_mask[y:y+h,x:x+w]\n        removed=int(np.count_nonzero(child&blocked));child&=~blocked\n        area=int(child.sum())\n        if not 64<=area<=100000:\n            logs.append({\'reason\':\'split_child_area\',\'family\':key,\'area\':area});continue\n        canvas=np.zeros(original.shape,bool);canvas[y:y+h,x:x+w]=child\n        support=float(probability[child].mean())\n        value=make_candidate(parent,canvas,\'split:\'+key,{\'kind\':\'skeleton_family_split\',\'family_anchor\':key,\n                            \'path_support\':support,\'seed_nodes\':seed_nodes[label-1],\n                            \'other_family_pixels_excluded\':removed,\n                            \'skeleton_nodes\':len(yy),\'disconnected_unreachable_nodes\':int((~reachable).sum())})\n        values.append(value);logs.append({\'reason\':\'split_generated\',\'candidate_id\':value.candidate_id,\'family\':key})\n    return values,logs\n\n\ndef bridge_fragments(parent,photo,masks,config):\n    meta=parent.info.get(\'graph\',{})\n    points=np.asarray(meta.get(\'endpoints\',[]),float).reshape(-1,2)\n    tangents=np.asarray(meta.get(\'outward_tangents\',[]),float).reshape(-1,2)\n    if len(points)<2:\n        return [],[{\'reason\':\'no_endpoint_pair\'}]\n    original=cm.decode(parent.rle).astype(bool)\n    _,component_labels=cv2.connectedComponents(original.astype(np.uint8),connectivity=8)\n    pairs=[]\n    for i in range(len(points)):\n        for j in range(i+1,len(points)):\n            a,b=points[i],points[j];delta=b-a;length=float(np.linalg.norm(delta))\n            if not 1<length<=config.repair_max_gap:\n                continue\n            if component_labels[int(a[1]),int(a[0])]==component_labels[int(b[1]),int(b[0])]:\n                continue\n            direction=delta/length\n            alignment=min(float(tangents[i]@direction),float(tangents[j]@-direction))\n            if alignment<.25:\n                continue\n            pairs.append((length/(.5+alignment),i,j,alignment))\n    pairs.sort();values=[];logs=[]\n    blocked=np.zeros(original.shape,bool)\n    for key,mask in masks.items():\n        if key not in parent.matched_detector_ids and photo[\'families\'][key][\'raw_score\']>=.7:\n            blocked|=mask\n    for number,(_,i,j,alignment) in enumerate(pairs[:config.repair_pairs]):\n        a,b=points[i].astype(int),points[j].astype(int)\n        margin=max(12,int(np.linalg.norm(a-b)/2))\n        x0,y0=np.maximum(np.minimum(a,b)-margin,0)\n        x1,y1=np.minimum(np.maximum(a,b)+margin+1,np.asarray(original.shape[::-1]))\n        roi=np.s_[y0:y1,x0:x1];p=photo[\'maps\'][\'pf\'][roi]\n        local_parent=original[roi];valid=photo[\'disk\'][roi]&~blocked[roi]\n        aa=(int(a[1]-y0),int(a[0]-x0));bb=(int(b[1]-y0),int(b[0]-x0))\n        width=ndi.distance_transform_edt(local_parent)\n        scale=float(np.clip((width[aa]+width[bb])/2,1,8))\n        distance=ndi.distance_transform_edt(~local_parent)\n        cost=(1+4*(1-p))*(1+distance/(4*scale))\n        # Penalize paths away from the endpoint tangent corridor.\n        yy,xx=np.indices(p.shape);direction=(b-a)/max(np.linalg.norm(b-a),1)\n        perpendicular=np.abs((xx-aa[1])*direction[1]-(yy-aa[0])*direction[0])\n        cost+=perpendicular/max(scale*4,1)\n        cost[~valid]=np.inf\n        if not valid[aa] or not valid[bb]:\n            logs.append({\'reason\':\'endpoint_other_family_conflict\',\'pair\':[i,j]});continue\n        try:\n            path,cost_sum=route_through_array(cost,aa,bb,fully_connected=True,geometric=True)\n        except (ValueError,RuntimeError):\n            logs.append({\'reason\':\'no_admissible_path\',\'pair\':[i,j]});continue\n        points_path=np.asarray(path,int)\n        support=float(p[points_path[:,0],points_path[:,1]].mean())\n        length=float(np.linalg.norm(np.diff(points_path,axis=0),axis=1).sum())\n        ratio=length/max(float(np.linalg.norm(a-b)),1)\n        steps=np.diff(points_path,axis=0).astype(float)\n        directions=steps/np.maximum(np.linalg.norm(steps,axis=1,keepdims=True),1.)\n        turns=1-np.sum(directions[1:]*directions[:-1],axis=1)\n        curvature=float(turns.mean()) if len(turns) else 0.\n        energy=float(cost_sum+4*turns.sum())\n        if (not np.isfinite(cost_sum) or support<config.repair_min_support or ratio>config.repair_max_detour\n                or curvature>config.repair_max_mean_turn):\n            logs.append({\'reason\':\'path_support_detour_or_curvature\',\'pair\':[i,j],\'support\':support,\n                         \'detour\':ratio,\'mean_turn_cost\':curvature});continue\n        added=np.zeros(p.shape,np.uint8)\n        for py,px in points_path:\n            cv2.circle(added,(int(px),int(py)),max(1,int(round(scale))),1,-1)\n        repaired=original.copy();repaired[roi]|=added.astype(bool)&valid\n        if (not 64<=repaired.sum()<=100000 or repaired.sum()>original.sum()*config.repair_max_area_ratio\n                or np.array_equal(repaired,original)):\n            logs.append({\'reason\':\'area_growth_or_unchanged\',\'pair\':[i,j]});continue\n        value=make_candidate(parent,repaired,f\'bridge:{number}\',{\'kind\':\'probability_tangent_path\',\n                            \'path_support\':support,\'path_cost\':energy,\'geometric_cost\':float(cost_sum),\n                            \'mean_turn_cost\':curvature,\'curvature_handling\':\'post-route bounded curvature acceptance and energy penalty\',\n                            \'path_length\':length,\n                            \'detour\':ratio,\'alignment\':alignment,\'radius_pixels\':scale,\n                            \'path_xy\':(points_path[:,::-1]+np.array([x0,y0])).tolist()})\n        values.append(value);logs.append({\'reason\':\'bridge_generated\',\'candidate_id\':value.candidate_id})\n    if not pairs:\n        logs.append({\'reason\':\'no_geometrically_admissible_cross_component_pair\'})\n    return values,logs\n\n\ndef repaired_candidates(bank,photo,graph_features,config):\n    started=time.monotonic();eligible=[]\n    for c in bank.candidates:\n        graph_features.get(c)\n        graph=c.info.get(\'graph\',{})\n        if c.match_class==\'overmerge\' or (graph.get(\'status\')==\'ok\' and graph[\'components\']>1):\n            eligible.append(c)\n    eligible.sort(key=lambda c:(c.match_class!=\'overmerge\',-c.info.get(\'primary_mean\',0.),c.candidate_id))\n    masks=family_masks(photo);seen={rle_hash(c.rle) for c in bank.candidates};new=[];logs=[]\n    for parent in eligible[:config.repair_budget]:\n        if parent.match_class==\'overmerge\':\n            proposed,attempts=split_by_skeleton(parent,photo,masks,config)\n        else:\n            proposed,attempts=bridge_fragments(parent,photo,masks,config)\n        for value in proposed:\n            signature=rle_hash(value.rle)\n            if signature in seen:\n                attempts.append({\'reason\':\'exact_mask_duplicate\',\'candidate_id\':value.candidate_id});continue\n            seen.add(signature)\n            old_area=parent.area;intersection=int((cm.decode(parent.rle)&cm.decode(value.rle)).sum())\n            value.info[\'graph_repair\'].update(parent_area=old_area,new_area=value.area,\n                changed_area=old_area+value.area-2*intersection,parent_iou=intersection/max(old_area+value.area-intersection,1))\n            new.append(value)\n        logs.append({\'parent_id\':parent.candidate_id,\'attempts\':attempts})\n    logs.extend({\'parent_id\':c.candidate_id,\'attempts\':[{\'reason\':\'photo_repair_budget\'}]}\n                for c in eligible[config.repair_budget:])\n    return new,{\'seconds\':time.monotonic()-started,\'eligible_parents\':len(eligible),\n                \'processed_parents\':min(len(eligible),config.repair_budget),\'new_unique_candidates\':len(new),\'log\':logs}\n', 'solar_gq/selection.py': "import numpy as np\n\nfrom solar_v15.selection import select as original_select, admission\nfrom solar_v1.postprocess import Instance\n\n\ndef pair_compatibility(a,b):\n    ga,gb=a.info.get('graph',{}),b.info.get('graph',{})\n    ea=np.asarray(ga.get('endpoints',[]),float).reshape(-1,2)\n    eb=np.asarray(gb.get('endpoints',[]),float).reshape(-1,2)\n    value=0.\n    if len(ea) and len(eb):\n        distances=np.linalg.norm(ea[:,None,:]-eb[None,:,:],axis=2)\n        i,j=np.unravel_index(np.argmin(distances),distances.shape)\n        if distances[i,j]<64:\n            va=np.asarray(ga['outward_tangents'][i]);vb=np.asarray(gb['outward_tangents'][j])\n            # Different identities that appear to be a continuation are treated as fragmentation risk.\n            value-=.2*max(0.,float(-va@vb))*(1-float(distances[i,j])/64)\n    ra,rb=a.info.get('graph_repair',{}),b.info.get('graph_repair',{})\n    if (a.parent_id and a.parent_id==b.parent_id and ra.get('kind')=='skeleton_family_split'\n            and rb.get('kind')=='skeleton_family_split' and a.family_id!=b.family_id):\n        value+=.2*min(ra.get('path_support',0.),rb.get('path_support',0.))\n    return value\n\n\ndef select(bank,values,policy,stage,config):\n    if stage!='G4':\n        return original_select(bank,values,policy,'A3')\n    strength=float(policy.get('graph_weight',0.))\n    if strength==0:\n        return original_select(bank,values,policy,'A3')\n    active=[];reasons={}\n    for i,c in enumerate(bank.candidates):\n        permitted,reason=admission(c,'A3',policy['orphan_floor'])\n        reasons[c.candidate_id]=reason if not permitted else 'below_quality_tau'\n        if permitted and values[i]>policy['tau']:\n            active.append(i);reasons[c.candidate_id]='graph_objective_or_conflict'\n    order=sorted(active,key=lambda i:(-values[i],bank.candidates[i].candidate_id))\n    # A fixed-width candidate-level beam: bounded O(N * width * selected_count).\n    states=[(0.,0,0)]\n    for i in order:\n        candidate=bank.candidates[i];graph=candidate.info.get('graph',{}).get('values',{})\n        repair=candidate.info.get('graph_repair',{})\n        unary=float(values[i]-policy['tau'])+strength*(\n            .2*(graph.get('skeleton_support',.5)-.5)-.1*graph.get('low_support_fraction',0.)\n            +.2*(repair.get('path_support',.5)-.5))\n        successors=[]\n        for score,picked,blocked in states:\n            if blocked&(1<<i):\n                continue\n            chosen=[j for j in order if picked&(1<<j)]\n            compatible=sum(pair_compatibility(candidate,bank.candidates[j]) for j in chosen)\n            successors.append((score+unary+strength*compatible,picked|(1<<i),blocked|bank.conflicts[i]))\n        states=sorted(states+successors,key=lambda s:(-s[0],s[1]))[:config.graph_beam_width]\n    picked=states[0][1]\n    chosen=[i for i in order if picked&(1<<i)]\n    for i in active:\n        if i in chosen:reasons[bank.candidates[i].candidate_id]='selected'\n        elif any(bank.conflicts[i]&(1<<j) for j in chosen):\n            reasons[bank.candidates[i].candidate_id]='pixel_or_family_conflict'\n        else:reasons[bank.candidates[i].candidate_id]='graph_objective_or_beam_budget'\n    instances=[Instance(float(values[i]),bank.candidates[i].rle,source=bank.candidates[i].source) for i in chosen]\n    from pycocotools import mask as cm\n    instances.sort(key=lambda v:tuple(cm.toBbox(v.rle)[[1,0]]))\n    return instances,chosen,reasons\n", 'solar_gq/stability.py': "import cv2\nimport numpy as np\nfrom pycocotools import mask as cm\nfrom .io import rle_hash\n\n\ndef feature_names(config):\n    per_delta=('stability','area_change','components_low','components_high','merge_count','seed_missing','roi_clipped')\n    return tuple(f'{name}_d{delta:.3f}_{stat}' for name in ('pf','pg','pb')\n                 for delta in config.stability_deltas for stat in per_delta)+tuple(\n                 f'{name}_{stat}' for name in ('pf','pg','pb') for stat in ('stability_min','stability_mean','valid_fraction'))\n\n\ndef supported_region(probability,mask,valid,threshold):\n    binary=((probability>threshold)&valid).astype(np.uint8)\n    n,labels=cv2.connectedComponents(binary,connectivity=8)\n    counts=np.bincount(labels[mask & valid],minlength=n)\n    counts[0]=0\n    ids=np.flatnonzero(counts)\n    # All seed components overlapping the candidate are relevant; never select unrelated neighbours.\n    result=np.isin(labels,ids) if len(ids) else np.zeros(mask.shape,bool)\n    return result,len(ids),labels\n\n\ndef extract(candidate,photo,config):\n    x,y,w,h=candidate.bbox;margin=config.stability_roi_margin\n    height,width=photo['disk'].shape\n    x0,y0=max(0,x-margin),max(0,y-margin);x1,y1=min(width,x+w+margin),min(height,y+h+margin)\n    mask=cm.decode(candidate.rle)[y0:y1,x0:x1].astype(bool)\n    reference=float(candidate.threshold) if candidate.source=='semantic_cc' and candidate.threshold is not None else .50\n    values=[];summary=[];details=[]\n    for name in ('pf','pg','pb'):\n        probability=photo['maps'][name][y0:y1,x0:x1]\n        valid=photo['disk'][y0:y1,x0:x1].copy()\n        if name=='pb':\n            yy,xx=np.ogrid[y0:y1,x0:x1]\n            valid &= (yy>=88)&(yy<1960)&(xx>=88)&(xx<1960)\n        fraction=float(np.count_nonzero(mask&valid)/max(mask.sum(),1));curve=[]\n        for delta in config.stability_deltas:\n            lo,nlo,ll=supported_region(probability,mask,valid,max(0.,reference-delta))\n            hi,nhi,hl=supported_region(probability,mask,valid,min(1.,reference+delta))\n            union=np.count_nonzero(lo|hi);intersection=np.count_nonzero(lo&hi)\n            stable=intersection/max(union,1);curve.append(stable)\n            clipped=bool(lo[0].any() or lo[-1].any() or lo[:,0].any() or lo[:,-1].any())\n            # Count multiple high-threshold children that collapse into a low component.\n            merge=0\n            for label in np.unique(ll[hi]):\n                if label:\n                    merge+=max(0,len(np.unique(hl[(ll==label)&hi]))-1)\n            area_change=float((lo.sum()-hi.sum())/max(lo.sum(),1))\n            row=[stable,area_change,nlo,nhi,merge,int(not hi.any()),int(clipped)]\n            values.extend(row)\n            details.append({'map':name,'delta':delta,'reference_threshold':reference,'stability':stable,\n                            'low_area':int(lo.sum()),'high_area':int(hi.sum()),'low_components':nlo,\n                            'high_components':nhi,'merge_count':merge,'valid_fraction':fraction,'roi_clipped':clipped})\n        summary.extend([min(curve),float(np.mean(curve)),fraction])\n    result=np.asarray(values+summary,np.float32)\n    if len(result)!=len(feature_names(config)) or not np.isfinite(result).all():\n        raise ValueError('Invalid stability feature vector.')\n    return result,details\n\n\nclass StabilityFeatures:\n    def __init__(self,photo,config):\n        self.photo,self.config,self.cache=photo,config,{}\n\n    def get(self,candidate):\n        # Same mask can originate at different thresholds. Both are part of this feature's identity.\n        key=(rle_hash(candidate.rle),candidate.source,candidate.threshold)\n        if key not in self.cache:\n            self.cache[key]=extract(candidate,self.photo,self.config)\n        return self.cache[key]\n", 'solar_gq/structure.py': '"""Bounded max-tree and FilFinder-style skeleton measurements, without neural training."""\nimport math\nimport time\n\nimport cv2\nimport numpy as np\nfrom pycocotools import mask as cm\nfrom scipy import ndimage as ndi\nfrom scipy.sparse import coo_matrix\nfrom scipy.sparse.csgraph import connected_components, dijkstra\nfrom skimage.morphology import max_tree, skeletonize\n\nfrom solar_v1.postprocess import encode\nfrom solar_v15.candidate_bank import HybridCandidate, populate_geometry\nfrom .io import rle_hash\n\ntry:\n    from numba import njit\nexcept ImportError:\n    def njit(*args, **kwargs):\n        return lambda f: f\n\n\n@njit(cache=False)\ndef tree_attributes(values, parent, order, width):\n    n = len(values)\n    area = np.ones(n, np.int64)\n    peak = values.copy()\n    xmin = np.arange(n) % width; xmax = xmin.copy()\n    ymin = np.arange(n) // width; ymax = ymin.copy()\n    representative = np.empty(n, np.int64)\n    for x in order:\n        p = parent[x]\n        representative[x] = representative[p] if p != x and values[p] == values[x] else x\n    for i in range(n-1, -1, -1):\n        x = order[i]; p = parent[x]\n        if p != x:\n            area[p] += area[x]\n            peak[p] = max(peak[p], peak[x])\n            xmin[p] = min(xmin[p], xmin[x]); xmax[p] = max(xmax[p], xmax[x])\n            ymin[p] = min(ymin[p], ymin[x]); ymax[p] = max(ymax[p], ymax[x])\n    return area, peak, xmin, xmax, ymin, ymax, representative\n\n\ndef max_tree_candidates(photo, config):\n    started = time.monotonic()\n    pf, disk = photo[\'maps\'][\'pf\'], photo[\'disk\']\n    levels = config.tree_levels-1\n    quantized = np.rint(np.where(disk, pf, 0)*levels).astype(np.uint16)\n    parents, order = max_tree(quantized, connectivity=2)\n    flat, par = quantized.ravel(), parents.ravel()\n    area, peak, xmin, xmax, ymin, ymax, rep = tree_attributes(flat, par, order, pf.shape[1])\n    indices = np.arange(flat.size)\n    canonical = (flat[par] != flat) | (par == indices)\n    nodes = indices[canonical]\n    canonical_parent = rep[par[nodes]]\n    child_count = np.bincount(canonical_parent[nodes != canonical_parent], minlength=flat.size)\n    level = flat[nodes].astype(np.float64)/levels\n    persistence = (peak[nodes].astype(np.float64)-flat[par[nodes]])/levels\n    crossing = np.floor(level*20) != np.floor(flat[par[nodes]].astype(float)/levels*20)\n    meaningful = (child_count[nodes] != 1) | crossing\n    eligible = ((area[nodes] >= 64) & (area[nodes] <= 100000) & (level >= config.tree_min_level)\n                & (persistence >= config.tree_min_persistence) & meaningful)\n    ranked = list(zip(nodes[eligible].tolist(), persistence[eligible].tolist()))\n    # Fixed GT-free ranking. Branch events and small regions both retain representation.\n    ranked.sort(key=lambda z: (-int(child_count[z[0]] >= 2), -z[1], -int(flat[z[0]]), z[0]))\n    small = [v for v in ranked if 250 <= area[v[0]] < 1000][:config.tree_budget//3]\n    used = {v[0] for v in small}\n    selected = (small+[v for v in ranked if v[0] not in used])[:config.tree_budget]\n    selected_ids = {node: f\'max_tree:{node}:{int(flat[node])}\' for node, _ in selected}\n    ancestry={}\n    for node,_ in selected:\n        ancestor=int(rep[par[node]])\n        while ancestor not in selected_ids and int(par[ancestor])!=ancestor:\n            ancestor=int(rep[par[ancestor]])\n        ancestry[node]=selected_ids.get(ancestor) if ancestor!=node else None\n    node_records=[]\n    values, seen, deduplicated = [], {rle_hash(c.rle) for c in photo[\'bank\'].candidates}, 0\n    for node, life in selected:\n        x0, x1 = int(xmin[node]), int(xmax[node])+1\n        y0, y1 = int(ymin[node]), int(ymax[node])+1\n        binary = (quantized[y0:y1, x0:x1] >= flat[node]) & disk[y0:y1, x0:x1]\n        _, labels = cv2.connectedComponents(binary.astype(np.uint8), connectivity=8)\n        y, x = divmod(node, pf.shape[1]); label = labels[y-y0, x-x0]\n        if not label:\n            raise AssertionError(\'Max-tree canonical node lost its threshold component.\')\n        mask = np.zeros(pf.shape, np.uint8); mask[y0:y1, x0:x1] = labels == label\n        rle = encode(mask); signature = rle_hash(rle)\n        children=[selected_ids[n] for n,parent_id in ancestry.items() if parent_id==selected_ids[node]]\n        node_records.append({\'id\':selected_ids[node],\'node\':node,\'parent_id\':ancestry[node],\'children\':children,\n                             \'level\':float(flat[node]/levels),\'persistence\':float(life),\'rle_sha256\':signature,\n                             \'duplicate_mask\':signature in seen})\n        if signature in seen:\n            deduplicated += 1; continue\n        seen.add(signature)\n        parent_id = ancestry[node]\n        name = selected_ids[node]\n        c = HybridCandidate(name, rle, \'max_tree\', name, \'orphan\', float(flat[node]/levels), 0,\n                            parent_id=parent_id, info={\'sources\': [\'semantic_cc\', \'max_tree\'],\n                            \'original_cc_id\': name, \'admission\': \'unclassified\', \'support\': float(pf[mask.astype(bool)].mean()),\n                            \'tree\': {\'node\': node, \'parent_node\': int(rep[par[node]]),\n                                     \'level\': float(flat[node]/levels), \'persistence\': float(life),\n                                     \'children_count\': int(child_count[node]), \'children\':children,\'parent_id\': parent_id,\n                                     \'quantization_levels\': config.tree_levels},\n                            \'provenance\': {\'method\': \'Pf max-tree\', \'uses_gt\': False}})\n        values.append(populate_geometry(c))\n    return values, {\'seconds\': time.monotonic()-started, \'canonical_nodes\': len(nodes),\n                    \'eligible_nodes\': len(ranked), \'budgeted_nodes\': len(selected),\n                    \'new_unique_candidates\': len(values), \'exact_duplicates\': deduplicated,\n                    \'quantization_max_absolute_error_bound\': .5/levels,\n                    \'quantization_actual_max_error\': float(np.max(np.abs(quantized.astype(float)/levels-np.where(disk,pf,0)))),\n                    \'node_budget\': config.tree_budget, \'foreground_probability_tree\': True,\'selected_tree_nodes\':node_records}\n\n\nGRAPH_NAMES = (\'skeleton_length\', \'main_path_proxy\', \'endpoint_count\', \'junction_clusters\',\n               \'branch_count\', \'length_area_ratio\', \'tortuosity\', \'width_mean\', \'width_cv\',\n               \'skeleton_support\', \'low_support_fraction\', \'skeleton_components\', \'cycle_rank\',\n               \'graph_missing\', \'tree_persistence\', \'tree_level\')\n\n\ndef pixel_graph(skeleton, probability=None):\n    yy, xx = np.nonzero(skeleton)\n    index = np.full(skeleton.shape, -1, np.int32); index[yy, xx] = np.arange(len(yy))\n    rows, cols, weights = [], [], []\n    for dy, dx in ((0, 1), (1, -1), (1, 0), (1, 1)):\n        ny, nx = yy+dy, xx+dx\n        valid = (ny >= 0)&(ny < skeleton.shape[0])&(nx >= 0)&(nx < skeleton.shape[1])\n        source = np.flatnonzero(valid)\n        target = index[ny[valid], nx[valid]]\n        keep = target >= 0; source, target = source[keep], target[keep]\n        w = np.full(len(source), math.hypot(dy, dx))\n        if probability is not None:\n            support = (probability[yy[source],xx[source]]+probability[yy[target],xx[target]])/2\n            w *= 1+3*(1-support)\n        rows.extend([source,target]); cols.extend([target,source]); weights.extend([w,w])\n    if rows:\n        graph = coo_matrix((np.concatenate(weights),(np.concatenate(rows),np.concatenate(cols))),\n                           shape=(len(yy),len(yy))).tocsr()\n    else:\n        from scipy.sparse import csr_matrix\n        graph = csr_matrix((len(yy),len(yy)), dtype=float)\n    return graph, yy, xx\n\n\ndef branch_graph(graph,yy,xx,degree,junction_labels,probability,width,x0,y0):\n    """Compress degree-2 chains and junction clusters; retain branch evidence, including cycles."""\n    visited=set();branches=[]\n    groups=junction_labels[yy,xx].astype(np.int64)\n    next_group=int(groups.max(initial=0))+1\n    for i in np.flatnonzero(degree<2):\n        groups[i]=next_group;next_group+=1\n\n    def trace(start,following):\n        route=[int(start)];previous=int(start);current=int(following);length=0.\n        while True:\n            edge=(min(previous,current),max(previous,current))\n            if edge in visited:break\n            visited.add(edge);length+=float(np.hypot(xx[current]-xx[previous],yy[current]-yy[previous]))\n            route.append(current)\n            if current==start or degree[current]!=2:break\n            neighbours=graph.indices[graph.indptr[current]:graph.indptr[current+1]]\n            following=int(neighbours[0] if neighbours[0]!=previous else neighbours[1])\n            previous,current=current,following\n        if len(route)<2:return\n        ids=np.asarray(route,int);support=probability[yy[ids],xx[ids]];widths=width[yy[ids],xx[ids]]*2\n        step=max(1,int(np.ceil(len(route)/64)));sample=ids[::step]\n        if sample[-1]!=ids[-1]:sample=np.append(sample,ids[-1])\n        branches.append({\'from_node\':int(groups[route[0]]),\'to_node\':int(groups[route[-1]]),\n                         \'cycle\':route[0]==route[-1],\'length\':length,\'support_mean\':float(support.mean()),\n                         \'support_min\':float(support.min()),\'low_support_fraction\':float(np.mean(support<.35)),\n                         \'width_mean\':float(widths.mean()),\'path_nodes\':len(route),\n                         \'path_xy_sample\':np.stack([xx[sample]+x0,yy[sample]+y0],1).tolist()})\n    for start in np.flatnonzero(degree!=2):\n        for other in graph.indices[graph.indptr[start]:graph.indptr[start+1]]:\n            if degree[start]>=3 and degree[other]>=3 and groups[start]==groups[other]:\n                visited.add((min(int(start),int(other)),max(int(start),int(other))));continue\n            if (min(int(start),int(other)),max(int(start),int(other))) not in visited:trace(start,other)\n    for start in np.flatnonzero(degree==2):\n        for other in graph.indices[graph.indptr[start]:graph.indptr[start+1]]:\n            if (min(int(start),int(other)),max(int(start),int(other))) not in visited:trace(start,other)\n    return branches\n\n\ndef analyze(candidate, photo, config):\n    x,y,w,h = candidate.bbox\n    margin = 3\n    x0,y0=max(0,x-margin),max(0,y-margin)\n    x1,y1=min(photo[\'disk\'].shape[1],x+w+margin),min(photo[\'disk\'].shape[0],y+h+margin)\n    tree = candidate.info.get(\'tree\', {})\n    missing = np.zeros(len(GRAPH_NAMES),np.float32)\n    missing[13:]=[1.,tree.get(\'persistence\',0.),tree.get(\'level\',0.)]\n    if (x1-x0)*(y1-y0)>config.graph_roi_pixels:\n        return missing, {\'status\':\'roi_budget\',\'bbox\':[x0,y0,x1,y1]}\n    mask=cm.decode(candidate.rle)[y0:y1,x0:x1].astype(bool)\n    skeleton=skeletonize(mask)\n    if not skeleton.any() or skeleton.sum()>config.graph_max_nodes:\n        return missing, {\'status\':\'empty_or_node_budget\',\'bbox\':[x0,y0,x1,y1]}\n    graph,yy,xx=pixel_graph(skeleton)\n    degree=np.diff(graph.indptr)\n    endpoints=np.flatnonzero(degree==1)\n    junction=np.zeros(skeleton.shape,np.uint8);junction[yy[degree>=3],xx[degree>=3]]=1\n    junction_labels,n_junction=ndi.label(junction,structure=np.ones((3,3)))\n    n_junction=int(n_junction)\n    n_components,labels=connected_components(graph,directed=False)\n    if n_components>64:\n        return missing, {\'status\':\'component_budget\',\'bbox\':[x0,y0,x1,y1], \'components\':n_components}\n    length=float(graph.sum()/2)\n    longest,straight=0.,0.\n    # A two-sweep geodesic main-path proxy; exact graph diameter is not claimed.\n    for component in range(n_components):\n        members=np.flatnonzero(labels==component)\n        first=int(members[0]);distance=dijkstra(graph,directed=False,indices=first)\n        far=int(members[np.argmax(distance[members])])\n        distance=dijkstra(graph,directed=False,indices=far)\n        other=int(members[np.argmax(distance[members])])\n        if distance[other]>longest:\n            longest=float(distance[other]);straight=float(np.hypot(xx[other]-xx[far],yy[other]-yy[far]))\n    width_map=ndi.distance_transform_edt(mask)\n    width=2*width_map[yy,xx]\n    probability=photo[\'maps\'][\'pf\'][y0:y1,x0:x1]\n    p=probability[yy,xx]\n    edges=graph.nnz//2;cycle=max(0,edges-len(yy)+n_components)\n    branches=branch_graph(graph,yy,xx,degree,junction_labels,probability,width_map,x0,y0)\n    values=np.asarray([length,longest,len(endpoints),n_junction,len(branches),length/max(candidate.area,1),\n                       longest/max(straight,1.),width.mean(),width.std()/max(width.mean(),1.),\n                       p.mean(),np.mean(p<.35),n_components,cycle,0.,\n                       tree.get(\'persistence\',0.),tree.get(\'level\',0.)],np.float32)\n    vectors=[]\n    measured_endpoints=endpoints[:64]\n    for i in measured_endpoints:\n        distances=dijkstra(graph,directed=False,indices=int(i),limit=8.)\n        nearby=np.flatnonzero(np.isfinite(distances)&(distances>0))\n        j=int(nearby[np.argmax(distances[nearby])]) if len(nearby) else int(i)\n        vec=np.array([xx[i]-xx[j],yy[i]-yy[j]],float)\n        vectors.append((vec/max(np.linalg.norm(vec),1.)).tolist())\n    return values, {\'status\':\'ok\',\'bbox\':[x0,y0,x1,y1], \'endpoints\':np.stack([xx[measured_endpoints]+x0,yy[measured_endpoints]+y0],1).tolist(),\n                    \'outward_tangents\':vectors,\'nodes\':len(yy),\'components\':n_components,\n                    \'branches\':branches,\'branch_graph_method\':\'degree-2-chain compression; junction clusters; cycle tracing\',\n                    \'endpoint_tangent_budget\':64,\'endpoint_tangents_truncated\':len(endpoints)>64,\n                    \'main_path_method\':\'two_sweep_geodesic_proxy\',\'values\':dict(zip(GRAPH_NAMES,map(float,values)))}\n\n\nclass GraphFeatures:\n    def __init__(self,photo,config):\n        self.photo,self.config,self.cache=photo,config,{}\n\n    def get(self,candidate):\n        key=rle_hash(candidate.rle)\n        if key not in self.cache:\n            self.cache[key]=analyze(candidate,self.photo,self.config)\n        values,meta=self.cache[key]\n        # Tree provenance may differ for exactly identical masks; geometry remains shared.\n        values=values.copy();tree=candidate.info.get(\'tree\',{})\n        values[-2:]=[tree.get(\'persistence\',0.),tree.get(\'level\',0.)]\n        candidate.info[\'graph\']=meta\n        return values\n'}

if not Path(CONFIG['input_root']).is_dir():
    raise RuntimeError('Run on Kaggle with the required mounted inputs. No competition data is downloaded by this notebook.')
dependencies = {'numpy':'numpy', 'scipy':'scipy', 'cv2':'opencv-python-headless',
    'pycocotools':'pycocotools==2.0.11', 'timm':'timm==1.0.29',
    'segmentation_models_pytorch':'segmentation-models-pytorch==0.5.0',
    'sklearn':'scikit-learn', 'joblib':'joblib', 'threadpoolctl':'threadpoolctl',
    'skimage':'scikit-image>=0.23,<0.27', 'psutil':'psutil', 'numba':'numba'}
missing = [package for module,package in dependencies.items() if importlib.util.find_spec(module) is None]
if missing:
    if not CONFIG['install_missing']:
        raise RuntimeError('Prepare offline dependencies first: '+str(missing))
    subprocess.run([sys.executable,'-m','pip','install',*missing],check=True)
runtime=Path('/kaggle/temp/solar_gq_d653a9936b1e1e5e/source')
for name,source in MODULES.items():
    path=runtime/name
    path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(source,encoding='utf-8',newline='\n')
env=dict(os.environ,PYTHONUNBUFFERED='1',OMP_NUM_THREADS='2',MKL_NUM_THREADS='2')
subprocess.run([sys.executable,'-c',
    'import json,sys;from solar_gq.main import run;run(json.loads(sys.argv[1]),sys.argv[2])',
    json.dumps(CONFIG),ORIGINAL_NOTEBOOK_SHA256],cwd=runtime,env=env,check=True)
